In [1]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load all macro snapshots
all_files = s.list_files('processed/macro/')

all_df = []
for f in all_files:
    date = f.split('/')[-1].replace('.parquet', '')
    try:
        df = db.read_parquet(f'processed/macro/{date}.parquet')
        df['date'] = date
        all_df.append(df)
        print(f'macro/{date}: {len(df)} snapshots')
    except Exception as e:
        print(f'ERROR macro/{date}: {e}')

db.close()
df_macro = pd.concat(all_df, ignore_index=True).reset_index(drop=True)

# Parse timestamp
df_macro['ts_ist'] = pd.to_datetime(df_macro['ts_ist'])

print(f'\nTotal: {len(df_macro):,} snapshots across {df_macro["date"].nunique()} days')
print(f'Dates: {sorted(df_macro["date"].unique())}')
print(f'Columns: {list(df_macro.columns)}')
print(f'\nSample row:')
print(df_macro.iloc[0])

macro/2026-10-05: 1 snapshots
macro/2026-10-06: 32 snapshots
macro/2026-10-07: 32 snapshots
macro/2026-10-08: 24 snapshots
macro/2026-10-09: 41 snapshots
macro/2026-10-10: 41 snapshots
macro/2026-10-11: 41 snapshots
macro/2026-10-12: 41 snapshots
macro/2026-10-13: 41 snapshots
macro/2026-10-14: 41 snapshots
macro/2026-10-15: 41 snapshots
macro/2026-10-16: 41 snapshots
macro/2026-10-17: 41 snapshots
macro/2026-10-18: 41 snapshots
macro/2026-10-19: 41 snapshots
macro/2026-10-20: 41 snapshots

Total: 581 snapshots across 16 days
Dates: ['2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-10', '2026-10-11', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18', '2026-10-19', '2026-10-20']
Columns: ['ts_ist', 'ist_sec', 'vix', 'vix_chg', 'vix_chg_pct', 'vix_prev', 'vix_1w_ago', 'vix_1m_ago', 'vix_regime', 'spot_usdinr', 'spot_eurusd', 'spot_gbpusd', 'spot_date', 'dxy', 'dxy_prev', 'dxy_chg', 'dxy_chg_pct', 'dxy_regime', '

In [2]:
df_macro.head()

,ts_ist,ist_sec,vix,vix_chg,vix_chg_pct,vix_prev,vix_1w_ago,vix_1m_ago,vix_regime,spot_usdinr,...,month,quarter_end,inr_pressure_index,inr_pressure_regime,rate_differential_direction,dii_net_equity,fii_net_equity,fii_selling,fii_buying,institutional_net
0,2026-10-05 23:43:13.057952+05:30,85393,14.67,0.21,1.47,14.78,13.64,10.68,NORMAL,96.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-10-07 00:14:27.114576+05:30,867,13.59,-1.19,-8.03,13.61,13.41,10.68,NORMAL,96.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-10-07 00:29:28.213020+05:30,1768,13.59,-1.19,-8.03,13.61,13.41,10.68,NORMAL,96.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-10-07 00:44:29.229979+05:30,2669,13.59,-1.19,-8.03,13.61,13.41,10.68,NORMAL,96.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-10-07 00:59:30.245327+05:30,3570,13.59,-1.19,-8.03,13.61,13.41,10.68,NORMAL,96.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
x = df_macro.copy()
print("\n========== DATASET OVERVIEW ==========")
print("Shape:", x.shape)
print("Columns:", x.columns.tolist())
print("Dtypes:\n", x.dtypes.to_string())

print("\n========== TIMESTAMP CHECK ==========")
if "ts_ist" in x.columns:
    x["ts_ist"] = pd.to_datetime(x["ts_ist"], errors="coerce")
    x["_date"] = x["ts_ist"].dt.date
    x["_clock"] = x["ts_ist"].dt.strftime("%H:%M:%S")
    print("Timestamp missing:", x["ts_ist"].isna().sum())
    print("First:", x["ts_ist"].min())
    print("Last:", x["ts_ist"].max())
else:
    raise ValueError("ts_ist column not found")

print("\n========== SESSION COVERAGE ==========")
sessions = x.groupby("_date").agg(
    rows=("ts_ist", "size"),
    first_time=("_clock", "min"),
    last_time=("_clock", "max")
)
print(sessions.to_string())
print("\nNumber of days:", len(sessions))

print("\n========== CONTRACT CHECK ==========")
if "symbol" in x.columns:
    contracts = x.groupby(["symbol", "_date"]).size()
    print(contracts.to_string())
else:
    print("No symbol column found")

print("\n========== DUPLICATE SECONDS ==========")
keys = ["ts_ist"]
if "symbol" in x.columns:
    keys = ["symbol", "ts_ist"]
print("Duplicate symbol × timestamps:",
      x.duplicated(keys).sum())

print("\n========== SESSION GRID CHECK ==========")
print("Expected seconds for 09:00–17:00:", 28800)
print("Session row statistics:")
print(sessions["rows"].describe())

print("\n========== EVENT FEATURES ==========")
event_cols = [
    "days_to_rbi", "days_to_fomc", "days_to_expiry",
    "near_rbi", "near_fomc", "near_expiry",
    "day_of_week", "week_of_month"
]
print(x[[c for c in event_cols if c in x.columns]]
      .isna().mean().sort_values().to_string())

print("\n========== MARKET VARIABLES ==========")
market_cols = [
    c for c in x.columns
    if any(s in c.lower() for s in
           ["spread", "depth", "tick_count",
            "update_rate", "rv_", "ofi_activity",
            "had_update", "is_observed"])
]
print(market_cols)

print("\n========== EXISTING CLOCK FEATURES ==========")
for c in ["ist_sec", "day_of_week", "week_of_month"]:
    if c in x.columns:
        print(c, "unique:", x[c].nunique(),
              "missing:", round(x[c].isna().mean(), 5))
        print("Sample:", x[c].dropna().unique()[:10])

print("\n========== DIAGNOSTIC COMPLETE ==========")


========== DATASET OVERVIEW ==========
Shape: (581, 84)
Columns: ['ts_ist', 'ist_sec', 'vix', 'vix_chg', 'vix_chg_pct', 'vix_prev', 'vix_1w_ago', 'vix_1m_ago', 'vix_regime', 'spot_usdinr', 'spot_eurusd', 'spot_gbpusd', 'spot_date', 'dxy', 'dxy_prev', 'dxy_chg', 'dxy_chg_pct', 'dxy_regime', 'days_to_rbi', 'days_to_fomc', 'days_to_expiry', 'near_rbi', 'near_fomc', 'near_expiry', 'day_of_week', 'week_of_month', 'usdinr_dxy_corr_proxy', 'date', 'spot_jpyusd', 'crudeoil_price', 'crudeoil_chg', 'crudeoil_chg_pct', 'crudeoil_oi', 'gold_price', 'gold_chg', 'gold_chg_pct', 'gold_oi', 'silver_price', 'silver_chg', 'silver_chg_pct', 'silver_oi', 'copper_price', 'copper_chg', 'copper_chg_pct', 'copper_oi', 'natgas_price', 'natgas_chg', 'natgas_chg_pct', 'natgas_oi', 'oil_regime', 'crudeoil_usd_approx', 'gold_oil_ratio', 'copper_momentum', 'india_10y_level', 'india_10y_chg', 'india_mid_level', 'india_mid_chg', 'india_lng_level', 'india_lng_chg', 'yield_curve_slope', 'yield_curve_mid', 'india_yield

In [4]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load all clean USDINR data (both contracts)
all_files  = s.list_files('processed/1s/USDINR26SEPFUT/')
all_files += s.list_files('processed/1s/USDINR26OCTFUT/')

all_df = []
for f in all_files:
    parts = f.split('/')
    sym   = parts[2]
    date  = parts[3].replace('.parquet','')
    try:
        df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
        df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
        df = df[(df['ist_sec'] >= 9*3600) &
                (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
        df['date']   = date
        df['symbol'] = sym
        all_df.append(df)
        print(f'{sym}/{date}: {len(df)} bars')
    except Exception as e:
        print(f'ERROR {sym}/{date}: {e}')

db.close()
df_raw = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
print(f'\nTotal: {len(df_raw):,} bars across {df_raw["date"].nunique()} days')
print(f'Dates: {sorted(df_raw["date"].unique())}')
print(f'Columns: {len(df_raw.columns)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-22: 12568 bars
USDINR26SEPFUT/2026-09-23: 11031 bars
USDINR26SEPFUT/2026-09-24: 10385 bars
USDINR26SEPFUT/2026-09-25: 15469 bars
USDINR26SEPFUT/2026-09-28: 2921 bars
USDINR26OCTFUT/2026-09-30: 16701 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-01: 16554 bars
USDINR26OCTFUT/2026-10-05: 14944 bars
USDINR26OCTFUT/2026-10-06: 18290 bars
USDINR26OCTFUT/2026-10-07: 19336 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-08: 20968 bars


... (2 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-13: 18435 bars
USDINR26OCTFUT/2026-10-14: 17290 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-15: 17281 bars
USDINR26OCTFUT/2026-10-16: 16527 bars
USDINR26OCTFUT/2026-10-19: 17632 bars

Total: 284,944 bars across 18 days
Dates: ['2026-09-22', '2026-09-23', '2026-09-24', '2026-09-25', '2026-09-28', '2026-09-30', '2026-10-01', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-19']
Columns: 55


In [5]:
df = (df_raw.sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True))
print(df.shape)
print(df["date"].nunique())
print(df.head())

(284944, 55)
18
           symbol      ts_sec     open  high      low  close  volume_delta  \
0  USDINR26OCTFUT  1790739000  96.2725  96.3  96.2725   96.3             1   
1  USDINR26OCTFUT  1790739001  96.3000  96.3  96.3000   96.3             0   
2  USDINR26OCTFUT  1790739002  96.3000  96.3  96.3000   96.3             0   
3  USDINR26OCTFUT  1790739003  96.3000  96.3  96.3000   96.3             0   
4  USDINR26OCTFUT  1790739004  96.3000  96.3  96.3000   96.3             0   

   tick_count       vwap      oi  ...  imbalance_ma_30s  imbalance_ma_60s  \
0           2  96.272647  961002  ...          0.696133          0.696133   
1           1  96.272647  961002  ...          0.671351          0.671351   
2           1  96.272647  961002  ...          0.655217          0.655217   
3           1  96.272647  961002  ...          0.654130          0.654130   
4           1  96.272647  961002  ...          0.653632          0.653632   

   spread_zscore  volume_ratio  price_mom_10s  price

In [6]:
duplicates = df.duplicated(subset=[
        "symbol",
        "date",
        "ts_sec"],
    keep=False)
print("Duplicate symbol-date-second rows:", duplicates.sum())

Duplicate symbol-date-second rows: 0


In [7]:
df["is_observed"] = 1
df["had_update"] = 1

In [8]:
def regularize_day(day):
    day = day.sort_values("ts_sec").copy()
    start = int(day["ts_sec"].min())
    end = int(day["ts_sec"].max())
    full_seconds = pd.DataFrame({
        "ts_sec": np.arange(
            start,
            end + 1)
    })
    symbol = day["symbol"].iloc[0]
    date = day["date"].iloc[0]
    full_seconds["symbol"] = symbol
    full_seconds["date"] = date

    out = full_seconds.merge(day,
            on=[
            "symbol",
            "date",
            "ts_sec"], how="left")

    return out

In [9]:
df_1s = (
    df
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(regularize_day)
    .reset_index(drop=True)
)

In [10]:
df_1s["is_observed"] = (
    df_1s["is_observed"]
    .fillna(0)
    .astype(int)
)

df_1s["had_update"] = (
    df_1s["had_update"]
    .fillna(0)
    .astype(int)
)

In [11]:
state_cols = [
    "oi",

    "bid_p1", "bid_q1",
    "bid_p2", "bid_q2",
    "bid_p3", "bid_q3",
    "bid_p4", "bid_q4",
    "bid_p5", "bid_q5",

    "ask_p1", "ask_q1",
    "ask_p2", "ask_q2",
    "ask_p3", "ask_q3",
    "ask_p4", "ask_q4",
    "ask_p5", "ask_q5",
]

In [12]:
state_cols = [
    c for c in state_cols
    if c in df_1s.columns
]

In [13]:
df_1s[state_cols] = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )[state_cols]
    .ffill()
)

In [14]:
flow_cols = [
    "volume_delta",
    "tick_count"
]

for col in flow_cols:

    if col in df_1s.columns:

        df_1s[col] = (
            df_1s[col]
            .fillna(0)
        )

In [15]:
df_1s["mid"] = (
    df_1s["bid_p1"]
    +
    df_1s["ask_p1"]
) / 2
df_1s["spread"] = (
    df_1s["ask_p1"]
    -
    df_1s["bid_p1"]
)
eps = 1e-12

df_1s["spread_bps_calc"] = (
    1e4
    *
    df_1s["spread"]
    /
    (df_1s["mid"] + eps)
)
df_1s["log_mid"] = np.log(
    df_1s["mid"]
)
df_1s["ret_1s"] = (
    df_1s["log_mid"]
    -
    df_1s
    .groupby(
        ["symbol", "date"]
    )["log_mid"]
    .shift(1)
)
bid_cols = [
    f"bid_q{i}"
    for i in range(1, 6)
]

ask_cols = [
    f"ask_q{i}"
    for i in range(1, 6)
]

df_1s["bid_depth_l5"] = (
    df_1s[bid_cols]
    .sum(axis=1)
)

df_1s["ask_depth_l5"] = (
    df_1s[ask_cols]
    .sum(axis=1)
)

df_1s["total_depth_l5"] = (
    df_1s["bid_depth_l5"]
    +
    df_1s["ask_depth_l5"]
)
session_summary = (
    df_1s
    .groupby(["symbol", "date"])
    .agg(
        first_sec=("ts_sec", "min"),
        last_sec=("ts_sec", "max"),
        rows=("ts_sec", "size"),
        observed_rows=("is_observed", "sum")
    )
)

session_summary["observation_rate"] = (
    session_summary["observed_rows"]
    /
    session_summary["rows"]
)

print(session_summary)

                            first_sec    last_sec   rows  observed_rows  \
symbol         date                                                       
USDINR26OCTFUT 2026-09-30  1790739000  1790767748  28749          16701   
               2026-10-01  1790825780  1790854181  28402          16554   
               2026-10-05  1791171000  1791199766  28767          14944   
               2026-10-06  1791257400  1791286154  28755          18290   
               2026-10-07  1791343800  1791372551  28752          19336   
               2026-10-08  1791430200  1791458948  28749          20968   
               2026-10-09  1791516600  1791545347  28748          20206   
               2026-10-12  1791775800  1791804553  28754          18406   
               2026-10-13  1791862200  1791890992  28793          18435   
               2026-10-14  1791948600  1791977391  28792          17290   
               2026-10-15  1792035000  1792063796  28797          17281   
               2026-10-16

In [16]:
x = df_1s.copy()

required = ["symbol", "ts_sec", "close", "tick_count"]

missing = [c for c in required if c not in x.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Use epoch timestamps as the authoritative clock.
x["timestamp"] = (
    pd.to_datetime(x["ts_sec"], unit="s", utc=True)
      .dt.tz_convert("Asia/Kolkata")
)

x["date"] = x["timestamp"].dt.date
x["clock_sec"] = (
    x["timestamp"].dt.hour * 3600
    + x["timestamp"].dt.minute * 60
    + x["timestamp"].dt.second
)

# Restrict to the regular 09:00–17:00 clock window.
# This is an exploratory filter, not an expiry-day rule.
start_sec = 9 * 3600
end_sec = 17 * 3600

x = x.loc[
    (x["clock_sec"] >= start_sec)
    & (x["clock_sec"] < end_sec)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol × timestamp found.")

# =====================================================
# 1. SESSION COVERAGE
# =====================================================

sessions = (
    x.groupby(["symbol", "date"])
     .agg(
         rows=("ts_sec", "size"),
         first_ts=("timestamp", "min"),
         last_ts=("timestamp", "max")
     )
     .reset_index()
)

sessions["coverage_pct"] = (
    100 * sessions["rows"] / 28800
)

print("\n===== SESSION COVERAGE =====")
print(sessions.to_string(index=False))

print("\nTotal unique sessions:", len(sessions))
print("Date range:", x["date"].min(), x["date"].max())

# =====================================================
# 2. PRIMITIVE MICROSTRUCTURE MEASURES
# =====================================================

# Price movements: never calculate across sessions.
g = x.groupby(["symbol", "date"], sort=False)

x["ret_1s"] = g["close"].transform(
    lambda s: np.log(s.where(s > 0)).diff()
)

# Activity measures.
x["tick_activity"] = x["tick_count"].fillna(0)

if "is_observed" in x.columns:
    x["observation_activity"] = (
        x["is_observed"].astype(float)
    )
elif "had_update" in x.columns:
    x["observation_activity"] = (
        x["had_update"].astype(float)
    )
else:
    x["observation_activity"] = np.nan
    print(
        "\nWARNING: No observed-update indicator. "
        "Tick count may not equal actual update intensity."
    )

# Detect L1 bid/ask prices using common naming variants.
bid_candidates = ["bid_px_00", "bid_price_1", "bid_px_0"]
ask_candidates = ["ask_px_00", "ask_price_1", "ask_px_0"]

bid_col = next(
    (c for c in bid_candidates if c in x.columns), None
)
ask_col = next(
    (c for c in ask_candidates if c in x.columns), None
)

if bid_col and ask_col:
    x["spread_l1"] = x[ask_col] - x[bid_col]
    x.loc[x["spread_l1"] < 0, "spread_l1"] = np.nan
else:
    x["spread_l1"] = np.nan
    print("\nL1 price columns not identified: spread skipped.")

# =====================================================
# 3. 5-MINUTE TIME BINS
# =====================================================

x["time_bin"] = (
    (x["clock_sec"] - start_sec) // 300
).astype(int)

x["bin_clock"] = (
    pd.Timestamp("2026-01-01 09:00:00")
    + pd.to_timedelta(x["time_bin"] * 5, unit="m")
).dt.strftime("%H:%M")

# Session-bin aggregation first.
# This prevents long sessions from dominating profiles.
daily_bins = (
    x.groupby(["symbol", "date", "time_bin", "bin_clock"])
     .agg(
         seconds_present=("ts_sec", "size"),
         mean_ticks=("tick_activity", "mean"),
         update_rate=("observation_activity", "mean"),
         mean_spread=("spread_l1", "mean"),
         mean_ret_sq=("ret_1s", lambda s: np.nanmean(s**2)
                      if s.notna().any() else np.nan),
         nonzero_return_rate=(
             "ret_1s",
             lambda s: (s.dropna() != 0).mean()
             if s.notna().any() else np.nan
         )
     )
     .reset_index()
)

daily_bins["bin_coverage"] = (
    daily_bins["seconds_present"] / 300
)

# Keep well-covered bins for descriptive profiles.
valid_bins = daily_bins.loc[
    daily_bins["bin_coverage"] >= 0.95
].copy()

profiles = (
    valid_bins.groupby(["time_bin", "bin_clock"])
    .agg(
        n_sessions=("date", "size"),
        mean_ticks=("mean_ticks", "mean"),
        update_rate=("update_rate", "mean"),
        mean_spread=("mean_spread", "mean"),
        mean_ret_sq=("mean_ret_sq", "mean"),
        nonzero_return_rate=(
            "nonzero_return_rate", "mean"
        )
    )
    .reset_index()
)

print("\n===== INTRADAY PROFILE =====")
print(profiles.to_string(index=False))

print("\nNumber of well-covered session-bins:",
      len(valid_bins))

# =====================================================
# 4. PLOTS
# =====================================================

plot_cols = [
    ("mean_ticks", "Mean tick count per second"),
    ("update_rate", "Observed update rate"),
    ("mean_spread", "Mean L1 spread"),
    ("mean_ret_sq", "Mean squared 1-second return"),
    ("nonzero_return_rate", "Nonzero return frequency"),
]

# for col, title in plot_cols:
#     if col not in profiles or profiles[col].notna().sum() == 0:
#         continue

#     fig, ax = plt.subplots(figsize=(11, 4))

#     ax.plot(profiles["time_bin"], profiles[col])

#     xticks = np.arange(0, 96, 12)
#     labels = [
#         (pd.Timestamp("2026-01-01 09:00")
#          + pd.Timedelta(minutes=int(i * 5)))
#         .strftime("%H:%M")
#         for i in xticks
#     ]

#     ax.set_xticks(xticks, labels)
#     ax.set_xlabel("Time of day (IST)")
#     ax.set_ylabel(col)
#     ax.set_title(title)
#     ax.grid(alpha=0.3)

#     plt.tight_layout()
#     plt.show()

# =====================================================
# 5. COVERAGE SUMMARY
# =====================================================

print("\n===== COVERAGE DIAGNOSTICS =====")

print(
    daily_bins["bin_coverage"]
    .describe()
    .to_string()
)

print("\nWell-covered bins:", len(valid_bins))
print("Possible bins per full session:", 96)

print("\n===== EXPERIMENT 1 COMPLETE =====")



===== SESSION COVERAGE =====
        symbol       date  rows                  first_ts                   last_ts  coverage_pct
USDINR26OCTFUT 2026-09-30 28749 2026-09-30 09:00:00+05:30 2026-09-30 16:59:08+05:30     99.822917
USDINR26OCTFUT 2026-10-01 28402 2026-10-01 09:06:20+05:30 2026-10-01 16:59:41+05:30     98.618056
USDINR26OCTFUT 2026-10-05 28767 2026-10-05 09:00:00+05:30 2026-10-05 16:59:26+05:30     99.885417
USDINR26OCTFUT 2026-10-06 28755 2026-10-06 09:00:00+05:30 2026-10-06 16:59:14+05:30     99.843750
USDINR26OCTFUT 2026-10-07 28752 2026-10-07 09:00:00+05:30 2026-10-07 16:59:11+05:30     99.833333
USDINR26OCTFUT 2026-10-08 28749 2026-10-08 09:00:00+05:30 2026-10-08 16:59:08+05:30     99.822917
USDINR26OCTFUT 2026-10-09 28748 2026-10-09 09:00:00+05:30 2026-10-09 16:59:07+05:30     99.819444
USDINR26OCTFUT 2026-10-12 28754 2026-10-12 09:00:00+05:30 2026-10-12 16:59:13+05:30     99.840278
USDINR26OCTFUT 2026-10-13 28793 2026-10-13 09:00:00+05:30 2026-10-13 16:59:52+05:30     

In [17]:
print(df_1s.columns.tolist())

['ts_sec', 'symbol', 'date', 'open', 'high', 'low', 'close', 'volume_delta', 'tick_count', 'vwap', 'oi', 'spread_mean', 'spread_max', 'spread_bps', 'imbalance_mean', 'imbalance_std', 'imbalance_last', 'total_bid_qty', 'total_ask_qty', 'weighted_mid', 'price_impact', 'bid_p1', 'bid_q1', 'bid_p2', 'bid_q2', 'bid_p3', 'bid_q3', 'bid_p4', 'bid_q4', 'bid_p5', 'bid_q5', 'ask_p1', 'ask_q1', 'ask_p2', 'ask_q2', 'ask_p3', 'ask_q3', 'ask_p4', 'ask_q4', 'ask_p5', 'ask_q5', 'realized_vol_10s', 'realized_vol_30s', 'realized_vol_60s', 'realized_vol_300s', 'imbalance_ma_10s', 'imbalance_ma_30s', 'imbalance_ma_60s', 'spread_zscore', 'volume_ratio', 'price_mom_10s', 'price_mom_30s', 'price_mom_60s', 'ts_ist', 'ist_sec', 'is_observed', 'had_update', 'mid', 'spread', 'spread_bps_calc', 'log_mid', 'ret_1s', 'bid_depth_l5', 'ask_depth_l5', 'total_depth_l5']


In [18]:
# ==========================================================
# USDINR RESEARCH — EXPERIMENT 2
# Intraday seasonality, normalization, stability and coverage
# No matplotlib, scipy, or machine learning required.
#
# Expected input: df containing regularized USDINR futures.
# ==========================================================

BIN_SECONDS = 300
START = 9 * 3600
END = 17 * 3600
MIN_BIN_COVERAGE = 0.95
MIN_FULL_DAY_COVERAGE = 0.97
EPS = 1e-12
required = [
    "symbol", "ts_sec", "mid",
    "bid_p1", "ask_p1", "is_observed"
]

missing = [c for c in required if c not in x.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

print("INPUT DATA SHAPE:", x.shape)

# ----------------------------------------------------------
# 1. CLOCK AND CONTRACT VALIDATION
# ----------------------------------------------------------

x["timestamp"] = (
    pd.to_datetime(x["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

x["date"] = x["timestamp"].dt.date
x["clock_sec"] = (
    x["timestamp"].dt.hour * 3600
    + x["timestamp"].dt.minute * 60
    + x["timestamp"].dt.second
)

x = x[
    (x["clock_sec"] >= START)
    & (x["clock_sec"] < END)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp rows detected.")

x["session_id"] = (
    x["symbol"].astype(str)
    + "|"
    + x["date"].astype(str)
)

x["time_bin"] = (
    (x["clock_sec"] - START) // BIN_SECONDS
).astype(int)

# ----------------------------------------------------------
# 2. SESSION COVERAGE AND EXPIRY TAGGING
# ----------------------------------------------------------

session_info = (
    x.groupby(["session_id", "symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_sec=("clock_sec", "min"),
        last_sec=("clock_sec", "max")
    )
    .reset_index()
)

session_info["full_day_coverage"] = (
    session_info["rows"] / (END - START)
)

# September 28 is treated as a known shortened session
# from the prior research; not assumed to be a normal day.
session_info["short_session"] = (
    session_info["date"].astype(str).eq("2026-09-28")
)

session_info["full_day_eligible"] = (
    (session_info["full_day_coverage"]
     >= MIN_FULL_DAY_COVERAGE)
    & (~session_info["short_session"])
)

print("\n===== SESSION INVENTORY =====")
print(session_info.to_string(index=False))

print("\nTotal rows after filtering:", len(x))
print("Total symbol-date sessions:", len(session_info))
print(
    "Full-day eligible sessions:",
    session_info["full_day_eligible"].sum()
)

# ----------------------------------------------------------
# 3. PRIMITIVE MARKET MEASURES
# ----------------------------------------------------------

x["mid"] = pd.to_numeric(x["mid"], errors="coerce")
x["bid_p1"] = pd.to_numeric(
    x["bid_p1"], errors="coerce"
)
x["ask_p1"] = pd.to_numeric(
    x["ask_p1"], errors="coerce"
)

x["spread_l1"] = x["ask_p1"] - x["bid_p1"]

valid_quote = (
    (x["bid_p1"] > 0)
    & (x["ask_p1"] >= x["bid_p1"])
)

x.loc[~valid_quote, "spread_l1"] = np.nan

x["spread_bps_l1"] = (
    10000 * x["spread_l1"]
    / ((x["ask_p1"] + x["bid_p1"]) / 2)
)

if "total_depth_l5" in x.columns:
    x["depth_l5"] = pd.to_numeric(
        x["total_depth_l5"], errors="coerce"
    )
else:
    depth_cols = [
        f"{side}_q{i}"
        for side in ["bid", "ask"]
        for i in range(1, 6)
    ]
    if all(c in x.columns for c in depth_cols):
        x["depth_l5"] = x[depth_cols].sum(
            axis=1, min_count=len(depth_cols)
        )
    else:
        x["depth_l5"] = np.nan

x.loc[x["depth_l5"] < 0, "depth_l5"] = np.nan

x["update_flag"] = pd.to_numeric(
    x["is_observed"], errors="coerce"
)

bad_update = ~x["update_flag"].isin([0, 1])
if bad_update.any():
    raise ValueError("is_observed must be binary and nonmissing.")

# Compute log returns only for consecutive seconds
# inside the same contract and session.
g = x.groupby("session_id", sort=False)

x["log_mid_clean"] = np.log(x["mid"].where(x["mid"] > 0))
x["ret_1s_clean"] = g["log_mid_clean"].diff()

delta_sec = g["ts_sec"].diff()
x.loc[delta_sec.ne(1), "ret_1s_clean"] = np.nan

x["ret_sq"] = x["ret_1s_clean"] ** 2
x["price_changed"] = (
    x["ret_1s_clean"].ne(0)
    .where(x["ret_1s_clean"].notna(), np.nan)
    .astype(float)
)

# ----------------------------------------------------------
# 4. SESSION × FIVE-MINUTE BIN STATISTICS
# ----------------------------------------------------------

bins = (
    x.groupby(
        ["session_id", "symbol", "date", "time_bin"]
    )
    .agg(
        n_seconds=("ts_sec", "size"),
        n_valid_returns=("ret_sq", "count"),
        update_rate=("update_flag", "mean"),
        mean_spread=("spread_bps_l1", "mean"),
        mean_depth=("depth_l5", "mean"),
        mean_ret_sq=("ret_sq", "mean"),
        move_rate=("price_changed", "mean")
    )
    .reset_index()
)

bins["coverage"] = bins["n_seconds"] / BIN_SECONDS

# A volatility estimate additionally requires a good
# number of genuine consecutive one-second returns.
bins.loc[
    bins["n_valid_returns"] < BIN_SECONDS * 0.90,
    "mean_ret_sq"
] = np.nan

bins = bins.merge(
    session_info[
        ["session_id", "full_day_eligible"]
    ],
    on="session_id",
    how="left",
    validate="many_to_one"
)

# Common eligible session set, good bin coverage
valid = bins[
    bins["full_day_eligible"]
    & (bins["coverage"] >= MIN_BIN_COVERAGE)
].copy()

print("\n===== VALID BIN COVERAGE =====")
print("Usable session-bins:", len(valid))
print(
    "Usable sessions:",
    valid["session_id"].nunique()
)

# ----------------------------------------------------------
# 5. WITHIN-DAY NORMALIZATION
# ----------------------------------------------------------

metrics = [
    "update_rate",
    "mean_spread",
    "mean_depth",
    "mean_ret_sq",
    "move_rate"
]

# Only available, sufficiently covered bins are used.
# Weighted mean avoids differences in bin coverage.
for metric in metrics:
    numerator = (
        valid[metric] * valid["n_seconds"]
    ).where(valid[metric].notna())

    denominator = valid["n_seconds"].where(
        valid[metric].notna()
    )

    temp = valid[[
        "session_id"
    ]].copy()

    temp["num"] = numerator
    temp["den"] = denominator

    totals = temp.groupby("session_id")[
        ["num", "den"]
    ].transform("sum")

    day_mean = (
        totals["num"] / totals["den"].replace(0, np.nan)
    )

    valid[f"{metric}_rel"] = (
        valid[metric] / day_mean.where(day_mean > EPS)
    )

# ----------------------------------------------------------
# 6. INTRADAY PROFILE: MEDIAN AND IQR
# ----------------------------------------------------------

relative_metrics = [
    f"{m}_rel" for m in metrics
]

profile = valid.groupby("time_bin")[
    relative_metrics
].agg(["count", "median",
       lambda s: s.quantile(0.25),
       lambda s: s.quantile(0.75)])

profile.columns = [
    "_".join(map(str, col))
    for col in profile.columns
]

profile = profile.reset_index()

def clock_label(b):
    seconds = START + int(b) * BIN_SECONDS
    return f"{seconds//3600:02d}:{(seconds%3600)//60:02d}"

profile["clock"] = profile["time_bin"].map(clock_label)

# Select representative clock times.
selected_times = [
    0, 6, 12, 18, 24, 30, 36,
    42, 48, 54, 60, 66, 72, 78, 84, 90
]

print("\n===== NORMALIZED INTRADAY PROFILE =====")

cols = [
    "clock",
    "update_rate_rel_median",
    "mean_spread_rel_median",
    "mean_depth_rel_median",
    "mean_ret_sq_rel_median",
    "move_rate_rel_median"
]

print(
    profile.loc[
        profile["time_bin"].isin(selected_times),
        cols
    ].round(4).to_string(index=False)
)

# ----------------------------------------------------------
# 7. PAIRED EARLY VS LATE SESSION COMPARISON
# ----------------------------------------------------------

# Early: 09:30–10:30
# Late: 15:30–16:30
# Deliberately exclude opening and closing edge bins.

valid["period"] = np.select(
    [
        valid["time_bin"].between(6, 17),
        valid["time_bin"].between(78, 89)
    ],
    ["early", "late"],
    default="other"
)

comparison = valid[
    valid["period"].isin(["early", "late"])
].copy()

period_counts = (
    comparison.groupby(["session_id", "period"])
    ["time_bin"]
    .nunique()
    .unstack()
)

eligible_ids = period_counts.index[
    (period_counts.get("early", 0) >= 10)
    & (period_counts.get("late", 0) >= 10)
]

comparison = comparison[
    comparison["session_id"].isin(eligible_ids)
]

print("\n===== PAIRED EARLY-LATE COMPARISON =====")

for metric in metrics:
    p = comparison.pivot_table(
        index="session_id",
        columns="period",
        values=metric,
        aggfunc="mean"
    ).dropna(subset=["early", "late"])

    if p.empty:
        print(metric, ": insufficient data")
        continue

    diff = p["early"] - p["late"]

    print(f"\n{metric}")
    print("Paired sessions:", len(p))
    print("Early mean:", round(p["early"].mean(), 8))
    print("Late mean:", round(p["late"].mean(), 8))
    print("Median difference:", round(diff.median(), 8))
    print("Fraction early > late:",
          round((diff > 0).mean(), 3))

# ----------------------------------------------------------
# 8. LEAVE-ONE-SESSION-OUT PATTERN STABILITY
# ----------------------------------------------------------

print("\n===== LEAVE-ONE-SESSION-OUT STABILITY =====")

stability_results = []

for metric in relative_metrics:
    matrix = valid.pivot_table(
        index="session_id",
        columns="time_bin",
        values=metric
    )

    for session_id in matrix.index:
        heldout = matrix.loc[session_id]
        train = matrix.drop(index=session_id)

        if len(train) < 3:
            continue

        reference = train.median(axis=0)

        mask = heldout.notna() & reference.notna()

        if mask.sum() < 48:
            continue

        corr = heldout[mask].corr(
            reference[mask], method="spearman"
        )

        stability_results.append({
            "metric": metric,
            "session_id": session_id,
            "spearman": corr,
            "n_bins": int(mask.sum())
        })

stability = pd.DataFrame(stability_results)

if not stability.empty:
    summary = (
        stability.groupby("metric")["spearman"]
        .agg(["count", "mean", "median", "min"])
    )

    summary["positive_fraction"] = (
        stability.groupby("metric")["spearman"]
        .apply(lambda s: (s > 0).mean())
    )

    print(summary.round(4).to_string())
else:
    print("Insufficient data for stability assessment.")

# ----------------------------------------------------------
# 9. CONTRACT-LEVEL PROFILE COMPARISON
# ----------------------------------------------------------

print("\n===== CONTRACT DESCRIPTIVE SUMMARY =====")

contract_summary = valid.groupby("symbol")[
    metrics
].mean()

print(contract_summary.round(8).to_string())

# ----------------------------------------------------------
# 10. RESEARCH OUTPUTS
# ----------------------------------------------------------

print("\n===== FINAL DATA QUALITY CHECK =====")
print("Full-day candidate sessions:",
      int(session_info["full_day_eligible"].sum()))

print("Observed valid sessions:",
      valid["session_id"].nunique())

print("Number of valid five-minute bins:", len(valid))

print("Spread observations:",
      int(valid["mean_spread"].notna().sum()))

print("Depth observations:",
      int(valid["mean_depth"].notna().sum()))

print("Return variance observations:",
      int(valid["mean_ret_sq"].notna().sum()))

# Preserve diagnostics for next experiments.
intraday_session_info = session_info
intraday_bins = valid
intraday_profile = profile
intraday_stability = stability

print("\nEXPERIMENT 2 COMPLETE")


INPUT DATA SHAPE: (495578, 72)

===== SESSION INVENTORY =====
               session_id         symbol       date  rows  first_sec  last_sec  full_day_coverage  short_session  full_day_eligible
USDINR26OCTFUT|2026-09-30 USDINR26OCTFUT 2026-09-30 28749      32400     61148           0.998229          False               True
USDINR26OCTFUT|2026-10-01 USDINR26OCTFUT 2026-10-01 28402      32780     61181           0.986181          False               True
USDINR26OCTFUT|2026-10-05 USDINR26OCTFUT 2026-10-05 28767      32400     61166           0.998854          False               True
USDINR26OCTFUT|2026-10-06 USDINR26OCTFUT 2026-10-06 28755      32400     61154           0.998437          False               True
USDINR26OCTFUT|2026-10-07 USDINR26OCTFUT 2026-10-07 28752      32400     61151           0.998333          False               True
USDINR26OCTFUT|2026-10-08 USDINR26OCTFUT 2026-10-08 28749      32400     61148           0.998229          False               True
USDINR26OCTFUT

In [19]:

import numpy as np
import pandas as pd

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 3
# Chronological testing of expected intraday book depth
#
# No matplotlib or scipy.
# No future information enters historical forecasts.
# ============================================================

BIN_SECONDS = 300
START_SEC = 9 * 3600
END_SEC = 17 * 3600

MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95
MIN_HISTORY_DAYS = 5

# ------------------------------------------------------------
# 1. PREPARE DATA
# ------------------------------------------------------------

required = ["symbol", "ts_sec"]

for col in required:
    if col not in x.columns:
        raise ValueError(f"Missing column: {col}")

x["timestamp"] = (
    pd.to_datetime(x["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

x["date"] = x["timestamp"].dt.date

x["clock_sec"] = (
    x["timestamp"].dt.hour * 3600
    + x["timestamp"].dt.minute * 60
    + x["timestamp"].dt.second
)

x = x[
    (x["clock_sec"] >= START_SEC)
    & (x["clock_sec"] < END_SEC)
].copy()

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp rows")

x["time_bin"] = (
    (x["clock_sec"] - START_SEC) // BIN_SECONDS
).astype(int)

# Build L5 depth from primitive quantity columns.
depth_columns = [
    f"{side}_q{level}"
    for side in ["bid", "ask"]
    for level in range(1, 6)
]

if all(c in x.columns for c in depth_columns):
    quantities = x[depth_columns].apply(
        pd.to_numeric, errors="coerce"
    )

    quantities = quantities.where(quantities >= 0)

    x["depth_l5_exp3"] = quantities.sum(
        axis=1, min_count=len(depth_columns)
    )

elif "total_depth_l5" in x.columns:
    x["depth_l5_exp3"] = pd.to_numeric(
        x["total_depth_l5"], errors="coerce"
    )

    x.loc[
        x["depth_l5_exp3"] < 0,
        "depth_l5_exp3"
    ] = np.nan
else:
    raise ValueError("L5 depth fields are missing")

# ------------------------------------------------------------
# 2. SESSION ELIGIBILITY
# ------------------------------------------------------------

sessions = (
    x.groupby(["date", "symbol"])
    .agg(
        rows=("ts_sec", "size")
    )
    .reset_index()
)

sessions["coverage"] = (
    sessions["rows"] / (END_SEC - START_SEC)
)

sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

print("\n===== SESSION INVENTORY =====")
print(sessions.to_string(index=False))

x = x.merge(
    sessions[["date", "symbol", "eligible"]],
    on=["date", "symbol"],
    how="left",
    validate="many_to_one"
)

x = x[x["eligible"]].copy()

# ------------------------------------------------------------
# 3. FIVE-MINUTE DEPTH OBSERVATIONS
# ------------------------------------------------------------

daily_bins = (
    x.groupby(["date", "symbol", "time_bin"])
    .agg(
        n_seconds=("ts_sec", "size"),
        n_valid_depth=("depth_l5_exp3", "count"),
        depth=("depth_l5_exp3", "mean")
    )
    .reset_index()
)

daily_bins["coverage"] = (
    daily_bins["n_seconds"] / BIN_SECONDS
)

daily_bins = daily_bins[
    (daily_bins["coverage"] >= MIN_BIN_COVERAGE)
    & (daily_bins["n_valid_depth"]
       >= MIN_BIN_COVERAGE * BIN_SECONDS)
    & daily_bins["depth"].notna()
].copy()

daily_bins = daily_bins.sort_values(
    ["date", "symbol", "time_bin"]
).reset_index(drop=True)

print("\n===== DEPTH DATA =====")
print("Usable sessions:",
      daily_bins[["date", "symbol"]]
      .drop_duplicates().shape[0])
print("Usable bins:", len(daily_bins))

# ------------------------------------------------------------
# 4. CHRONOLOGICAL FORECASTING
# ------------------------------------------------------------

unique_dates = sorted(daily_bins["date"].unique())

records = []

for evaluation_date in unique_dates:

    # Strict chronological separation.
    historical = daily_bins[
        daily_bins["date"] < evaluation_date
    ].copy()

    test = daily_bins[
        daily_bins["date"] == evaluation_date
    ].copy()

    historical_dates = historical["date"].nunique()

    if historical_dates < MIN_HISTORY_DAYS:
        continue

    # Model A: time-independent historical median.
    global_baseline = historical["depth"].median()

    # Model B: historical median at the same clock bin.
    clock_baseline = (
        historical.groupby("time_bin")["depth"]
        .median()
    )

    historical_bin_counts = (
        historical.groupby("time_bin")["date"]
        .nunique()
    )

    test["prediction_constant"] = global_baseline

    test["prediction_clock"] = (
        test["time_bin"].map(clock_baseline)
    )

    test["history_bin_days"] = (
        test["time_bin"].map(historical_bin_counts)
    )

    # Require five historical sessions for both models
    # and valid positive baseline depth.
    test = test[
        (test["history_bin_days"] >= MIN_HISTORY_DAYS)
        & (test["prediction_constant"] > 0)
        & (test["prediction_clock"] > 0)
    ].copy()

    if test.empty:
        continue

    test["error_constant"] = (
        test["depth"] - test["prediction_constant"]
    ).abs()

    test["error_clock"] = (
        test["depth"] - test["prediction_clock"]
    ).abs()

    test["improvement"] = (
        test["error_constant"] - test["error_clock"]
    )

    # Candidate depth-surprise feature.
    # This is retrospective evaluation only.
    test["depth_surprise_log"] = np.log(
        (test["depth"] + 1e-8)
        / (test["prediction_clock"] + 1e-8)
    )

    test["history_days"] = historical_dates

    records.append(test)

if not records:
    raise ValueError(
        "Insufficient historical sessions for evaluation."
    )

results = pd.concat(records, ignore_index=True)

# ------------------------------------------------------------
# 5. SESSION-LEVEL FORECAST RESULTS
# ------------------------------------------------------------

session_results = (
    results.groupby(["date", "symbol"])
    .agg(
        bins=("time_bin", "size"),
        history_days=("history_days", "first"),
        mae_constant=("error_constant", "mean"),
        mae_clock=("error_clock", "mean"),
        fraction_clock_better=(
            "improvement",
            lambda s: (s > 0).mean()
        )
    )
    .reset_index()
)

session_results["improvement_pct"] = (
    100 * (
        1 - session_results["mae_clock"]
        / session_results["mae_constant"]
    )
)

print("\n===== CHRONOLOGICAL SESSION RESULTS =====")
print(
    session_results.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 6. OVERALL FORECAST PERFORMANCE
# ------------------------------------------------------------

mae_constant = results["error_constant"].mean()
mae_clock = results["error_clock"].mean()

overall_improvement = (
    100 * (1 - mae_clock / mae_constant)
)

print("\n===== OVERALL PERFORMANCE =====")

print("Evaluation sessions:", len(session_results))
print("Evaluation bins:", len(results))
print("Constant baseline MAE:", round(mae_constant, 6))
print("Clock baseline MAE:", round(mae_clock, 6))
print("Clock improvement (%):",
      round(overall_improvement, 4))

print(
    "Fraction of sessions with positive improvement:",
    round(
        (session_results["improvement_pct"] > 0).mean(),
        4
    )
)

print(
    "Median session improvement (%):",
    round(
        session_results["improvement_pct"].median(),
        4
    )
)

# ------------------------------------------------------------
# 7. RESULTS BY CONTRACT
# ------------------------------------------------------------

print("\n===== CONTRACT-LEVEL PERFORMANCE =====")

for symbol, group in results.groupby("symbol"):

    constant_mae = group["error_constant"].mean()
    clock_mae = group["error_clock"].mean()

    improvement = (
        100 * (1 - clock_mae / constant_mae)
    )

    print("\nContract:", symbol)
    print("Evaluation bins:", len(group))
    print("Constant MAE:", round(constant_mae, 6))
    print("Clock MAE:", round(clock_mae, 6))
    print("Improvement (%):", round(improvement, 4))

# ------------------------------------------------------------
# 8. DEPTH SURPRISE DIAGNOSTICS
# ------------------------------------------------------------

print("\n===== DEPTH SURPRISE DISTRIBUTION =====")

print(
    results["depth_surprise_log"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .round(5)
    .to_string()
)

print("\n===== WIN / LOSS SUMMARY =====")

session_wins = (
    session_results["improvement_pct"] > 0
).sum()

print("Sessions improved:", int(session_wins))
print("Sessions evaluated:", len(session_results))

# Preserve objects for subsequent research.
calendar_exp3_results = results
calendar_exp3_sessions = session_results

print("\n===== EXPERIMENT 3 COMPLETE =====")



===== SESSION INVENTORY =====
      date         symbol  rows  coverage  eligible
2026-09-22 USDINR26SEPFUT 27175  0.943576     False
2026-09-23 USDINR26SEPFUT 28437  0.987396      True
2026-09-24 USDINR26SEPFUT 28607  0.993299      True
2026-09-25 USDINR26SEPFUT 28440  0.987500      True
2026-09-28 USDINR26SEPFUT  9281  0.322257     False
2026-09-30 USDINR26OCTFUT 28749  0.998229      True
2026-10-01 USDINR26OCTFUT 28402  0.986181      True
2026-10-05 USDINR26OCTFUT 28767  0.998854      True
2026-10-06 USDINR26OCTFUT 28755  0.998437      True
2026-10-07 USDINR26OCTFUT 28752  0.998333      True
2026-10-08 USDINR26OCTFUT 28749  0.998229      True
2026-10-09 USDINR26OCTFUT 28748  0.998194      True
2026-10-12 USDINR26OCTFUT 28754  0.998403      True
2026-10-13 USDINR26OCTFUT 28793  0.999757      True
2026-10-14 USDINR26OCTFUT 28792  0.999722      True
2026-10-15 USDINR26OCTFUT 28797  0.999896      True
2026-10-16 USDINR26OCTFUT 28796  0.999861      True
2026-10-19 USDINR26OCTFUT 28784  

In [20]:

import numpy as np
import pandas as pd

# =========================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 4
# Contract-specific rolling liquidity-level vs shape
# Historical depth forecast comparison
# pandas + numpy only
# =========================================================

HISTORY_SESSIONS = 3
BIN_SIZE = 300
MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95
MIN_HIST_BIN_DAYS = 2

x = df_1s.copy()

needed = ["symbol", "ts_sec"] + [
    f"{side}_q{i}"
    for side in ("bid", "ask")
    for i in range(1, 6)
]

missing = [c for c in needed if c not in x.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

x["ts"] = (
    pd.to_datetime(x["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)
x["date"] = x["ts"].dt.date
x["sec"] = (
    x["ts"].dt.hour * 3600
    + x["ts"].dt.minute * 60
    + x["ts"].dt.second
)

x = x[(x.sec >= 32400) & (x.sec < 61200)].copy()
x = x.sort_values(["symbol", "date", "ts_sec"])

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp")

x["bin"] = ((x.sec - 32400) // BIN_SIZE).astype(int)

q = x[needed[2:]].apply(pd.to_numeric, errors="coerce")
q = q.where(q >= 0)

x["depth"] = q.sum(axis=1, min_count=10)

# ---------------------------------------------------------
# 1. SESSION ELIGIBILITY
# ---------------------------------------------------------

session = (
    x.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        valid_depth=("depth", "count")
    )
    .reset_index()
)

session["eligible"] = (
    (session.rows / 28800 >= MIN_DAY_COVERAGE)
    & (session.valid_depth / session.rows >= 0.95)
    & (session.date.astype(str) != "2026-09-28")
)

x = x.merge(
    session[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

x = x[x.eligible].copy()

# ---------------------------------------------------------
# 2. DAILY BIN DEPTHS
# ---------------------------------------------------------

bins = (
    x.groupby(["symbol", "date", "bin"])
    .agg(
        n=("ts_sec", "size"),
        n_depth=("depth", "count"),
        depth=("depth", "mean")
    )
    .reset_index()
)

bins = bins[
    (bins.n >= BIN_SIZE * MIN_BIN_COVERAGE)
    & (bins.n_depth >= BIN_SIZE * MIN_BIN_COVERAGE)
    & bins.depth.notna()
    & (bins.depth > 0)
].copy()

# ---------------------------------------------------------
# 3. CONTRACT-SPECIFIC WALK-FORWARD FORECASTS
# ---------------------------------------------------------

predictions = []

for symbol, contract in bins.groupby("symbol"):

    dates = sorted(contract.date.unique())

    for i in range(HISTORY_SESSIONS, len(dates)):

        test_date = dates[i]
        train_dates = dates[
            i - HISTORY_SESSIONS:i
        ]

        train = contract[
            contract.date.isin(train_dates)
        ].copy()

        test = contract[
            contract.date == test_date
        ].copy()

        # Model A: rolling constant depth.
        constant = train.depth.median()

        # Model B: rolling raw clock-bin depth.
        clock = train.groupby("bin").depth.median()

        counts = train.groupby("bin").date.nunique()

        # Model C:
        # Historical daily level * normalized clock shape.
        day_level = train.groupby("date").depth.median()

        train["day_level"] = train.date.map(day_level)

        train["relative_depth"] = (
            train.depth / train.day_level
        )

        shape = (
            train.groupby("bin")
            .relative_depth.median()
        )

        expected_level = day_level.median()

        test["pred_constant"] = constant
        test["pred_clock"] = test.bin.map(clock)
        test["pred_shape"] = (
            expected_level * test.bin.map(shape)
        )

        test["hist_bin_days"] = test.bin.map(counts)

        test = test[
            (test.hist_bin_days >= MIN_HIST_BIN_DAYS)
            & test[
                ["pred_constant", "pred_clock", "pred_shape"]
            ].notna().all(axis=1)
        ].copy()

        if test.empty:
            continue

        test["history_sessions"] = HISTORY_SESSIONS

        for model in ["constant", "clock", "shape"]:
            test[f"ae_{model}"] = (
                test.depth - test[f"pred_{model}"]
            ).abs()

        predictions.append(test)

if not predictions:
    raise ValueError(
        "Not enough contract-specific history to evaluate."
    )

r = pd.concat(predictions, ignore_index=True)

# ---------------------------------------------------------
# 4. SESSION PERFORMANCE
# ---------------------------------------------------------

daily = (
    r.groupby(["symbol", "date"])
    .agg(
        bins=("bin", "size"),
        constant_mae=("ae_constant", "mean"),
        clock_mae=("ae_clock", "mean"),
        shape_mae=("ae_shape", "mean")
    )
    .reset_index()
)

daily["clock_improvement_pct"] = (
    100 * (1 - daily.clock_mae / daily.constant_mae)
)

daily["shape_improvement_pct"] = (
    100 * (1 - daily.shape_mae / daily.constant_mae)
)

print("\n===== EXPERIMENT 4: DAILY RESULTS =====")
print(daily.round(4).to_string(index=False))

# ---------------------------------------------------------
# 5. AGGREGATE COMPARISON
# ---------------------------------------------------------

print("\n===== AGGREGATE FORECAST PERFORMANCE =====")

base = r.ae_constant.mean()

for model in ["constant", "clock", "shape"]:
    mae = r[f"ae_{model}"].mean()
    improvement = 100 * (1 - mae / base)

    print(
        f"{model:10s} | MAE={mae:.4f} "
        f"| improvement={improvement:+.3f}%"
    )

print("\n===== DAY-LEVEL ROBUSTNESS =====")

for model in ["clock", "shape"]:
    col = f"{model}_improvement_pct"

    print(f"\n{model.upper()}")
    print("Evaluated sessions:", len(daily))
    print("Winning sessions:", int((daily[col] > 0).sum()))
    print("Median improvement:", round(daily[col].median(), 4))
    print("Worst day:", round(daily[col].min(), 4))
    print("Best day:", round(daily[col].max(), 4))

# ---------------------------------------------------------
# 6. RELATIVE DEPTH DISTRIBUTIONS
# ---------------------------------------------------------

r["surprise_clock"] = np.log(
    r.depth / r.pred_clock
)

r["surprise_shape"] = np.log(
    r.depth / r.pred_shape
)

print("\n===== CANDIDATE SURPRISE FEATURES =====")

print(
    r[["surprise_clock", "surprise_shape"]]
    .describe(percentiles=[.05, .25, .5, .75, .95])
    .round(5)
    .to_string()
)

# Preserve exploratory outputs.
calendar_exp4_predictions = r
calendar_exp4_daily = daily

print("\nEXPERIMENT 4 COMPLETE")



===== EXPERIMENT 4: DAILY RESULTS =====
        symbol       date  bins  constant_mae  clock_mae  shape_mae  clock_improvement_pct  shape_improvement_pct
USDINR26OCTFUT 2026-10-06    95     1166.0206  1395.2912  1312.3116               -19.6627               -12.5462
USDINR26OCTFUT 2026-10-07    95     1617.5678  1607.9981  1536.5253                 0.5916                 5.0101
USDINR26OCTFUT 2026-10-08    95     3378.5717  2896.4414  3030.4576                14.2702                10.3036
USDINR26OCTFUT 2026-10-09    95     2683.9679  2831.7509  2800.4528                -5.5061                -4.3400
USDINR26OCTFUT 2026-10-12    95     1914.4971  2268.7550  2031.0105               -18.5040                -6.0858
USDINR26OCTFUT 2026-10-13    95     1785.3628  2224.7524  2308.6069               -24.6107               -29.3074
USDINR26OCTFUT 2026-10-14    95     1575.3382  1741.5361  1701.8392               -10.5500                -8.0301
USDINR26OCTFUT 2026-10-15    96      661.0430  

In [21]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 5
#
# Does historical intraday liquidity seasonality condition
# the predictive relationship between L1 queue imbalance
# and future mid-price returns?
#
# MASTER DATASET: df_1s
# Do not modify df_1s or use the old x DataFrame.
#
# OUTPUTS:
#   calendar_exp5_data
#   calendar_exp5_ic
#   calendar_exp5_paired
#   calendar_exp5_baselines
#   calendar_exp5_regime_counts
#
# Dependencies: pandas, numpy
# ============================================================


# ============================================================
# 0. CONFIGURATION
# ============================================================

HORIZONS = [30, 60, 300]

MIN_HISTORY_SESSIONS = 3
MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95
MIN_REGIME_OBS = 300

BIN_SECONDS = 300
START_SEC = 9 * 3600
END_SEC = 17 * 3600


# ============================================================
# 1. VALIDATE MASTER DATASET
# ============================================================

depth_cols = [
    f"{side}_q{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
]

required = [
    "symbol",
    "ts_sec",
    "mid"
] + depth_cols

# Check duplicate names in the master dataset.
duplicate_cols = (
    df_1s.columns[df_1s.columns.duplicated()]
    .unique()
    .tolist()
)

if duplicate_cols:
    print("Duplicate columns detected:", duplicate_cols)

    # Never silently discard conflicting market data.
    for col in duplicate_cols:
        copies = df_1s.loc[:, df_1s.columns == col]

        identical = all(
            copies.iloc[:, 0].equals(copies.iloc[:, j])
            for j in range(1, copies.shape[1])
        )

        if not identical:
            raise ValueError(
                f"Conflicting duplicate column: {col}"
            )

    # Use a deduplicated view without modifying df_1s.
    source = df_1s.loc[
        :, ~df_1s.columns.duplicated(keep="first")
    ]
else:
    source = df_1s

missing = [
    c for c in required
    if c not in source.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

if len(required) != len(set(required)):
    raise ValueError("Required column list has duplicates.")

work = source.loc[:, required].copy()

assert work.columns.is_unique

print("\n===== DATASET VALIDATION =====")
print("Master shape:", df_1s.shape)
print("Working shape:", work.shape)
print("Unique working columns:", work.columns.is_unique)


# ============================================================
# 2. TIMESTAMPS AND SESSION BOUNDARIES
# ============================================================

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Missing or invalid ts_sec values.")

work["ts"] = (
    pd.to_datetime(
        work["ts_sec"],
        unit="s",
        utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START_SEC)
    & (work["sec"] < END_SEC)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError(
        "Duplicate symbol x timestamp observations."
    )

work["bin"] = (
    (work["sec"] - START_SEC) // BIN_SECONDS
).astype(int)


# ============================================================
# 3. REBUILD ORDER BOOK FEATURES
# ============================================================

q = work[depth_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

q = q.where(q >= 0)

# Total resting depth across five bid and five ask levels.
# Require all ten quantities.
work["depth_l5"] = q.sum(
    axis=1,
    min_count=len(depth_cols)
)

bid1 = q["bid_q1"]
ask1 = q["ask_q1"]

denom = bid1 + ask1

work["queue_imbalance"] = (
    (bid1 - ask1) / denom.where(denom > 0)
)

work["mid"] = pd.to_numeric(
    work["mid"], errors="coerce"
)

work.loc[work["mid"] <= 0, "mid"] = np.nan

work["log_mid"] = np.log(work["mid"])

print("\n===== BOOK DATA QUALITY =====")

for col in ["depth_l5", "queue_imbalance", "mid"]:
    print(
        f"{col}: missing={work[col].isna().mean():.6f}"
    )


# ============================================================
# 4. SESSION COVERAGE
# ============================================================

sessions = (
    work.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_sec=("sec", "min"),
        last_sec=("sec", "max")
    )
    .reset_index()
)

sessions["coverage"] = (
    sessions["rows"] / (END_SEC - START_SEC)
)

# Exclude the known shortened September 28 session.
# Also exclude sessions with <97% regular-clock coverage.
sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (
        sessions["date"].astype(str)
        != "2026-09-28"
    )
)

print("\n===== SESSION INVENTORY =====")
print(sessions.to_string(index=False))

print(
    "\nEligible sessions:",
    int(sessions["eligible"].sum())
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    how="left",
    validate="many_to_one"
)

work = work[work["eligible"]].copy()


# ============================================================
# 5. EXACT FORWARD RETURNS
#
# Calculate before historical-baseline filtering.
# Match actual t+h timestamps, not row positions.
#
# Return(t,h) = log(mid(t+h) / mid(t))
#
# Joining by symbol and date prevents cross-session returns.
# ============================================================

price_grid = work[
    ["symbol", "date", "ts_sec", "mid"]
].copy()

for h in HORIZONS:

    future = price_grid.copy()

    future["ts_sec"] = future["ts_sec"] - h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    work[f"return_{h}"] = np.log(
        work[f"future_mid_{h}"] / work["mid"]
    )

print("\n===== FORWARD RETURN COVERAGE =====")

for h in HORIZONS:
    print(
        f"{h}s: "
        f"{work[f'return_{h}'].notna().mean():.6f}"
    )


# ============================================================
# 6. HISTORICAL FIVE-MINUTE DEPTH BASELINE
#
# Use only PREVIOUS eligible sessions of SAME contract.
# Use previous 3 sessions, not future/current-day bins.
#
# ExpectedDepth(d,b) =
#     Median(MeanDepth(j,b))
#
# for historical sessions j < d.
# ============================================================

bin_depth = (
    work.groupby(["symbol", "date", "bin"])
    .agg(
        n_seconds=("ts_sec", "size"),
        n_valid_depth=("depth_l5", "count"),
        mean_depth=("depth_l5", "mean")
    )
    .reset_index()
)

bin_depth["coverage"] = (
    bin_depth["n_seconds"] / BIN_SECONDS
)

bin_depth.loc[
    (bin_depth["coverage"] < MIN_BIN_COVERAGE)
    | (
        bin_depth["n_valid_depth"]
        < MIN_BIN_COVERAGE * BIN_SECONDS
    ),
    "mean_depth"
] = np.nan

baseline_records = []

for symbol, contract in bin_depth.groupby("symbol"):

    dates = sorted(contract["date"].unique())

    for i in range(
        MIN_HISTORY_SESSIONS,
        len(dates)
    ):

        evaluation_date = dates[i]

        history_dates = dates[
            i - MIN_HISTORY_SESSIONS:i
        ]

        historical = contract[
            contract["date"].isin(history_dates)
        ].copy()

        historical = historical[
            historical["mean_depth"].notna()
        ]

        baseline = (
            historical.groupby("bin")
            .agg(
                expected_depth=(
                    "mean_depth", "median"
                ),
                history_days=(
                    "date", "nunique"
                )
            )
            .reset_index()
        )

        baseline = baseline[
            baseline["history_days"]
            >= MIN_HISTORY_SESSIONS
        ].copy()

        baseline["symbol"] = symbol
        baseline["date"] = evaluation_date

        baseline_records.append(baseline)

if not baseline_records:
    raise ValueError(
        "Insufficient same-contract historical sessions."
    )

baselines = pd.concat(
    baseline_records,
    ignore_index=True
)

calendar_exp5_baselines = baselines.copy()

print("\n===== BASELINE COVERAGE =====")
print("Baseline bins:", len(baselines))
print(
    "Evaluation sessions:",
    baselines[["symbol", "date"]]
    .drop_duplicates().shape[0]
)


# ============================================================
# 7. HISTORICAL DEPTH SURPRISE
#
# DepthSurprise(t) =
#    log(CurrentDepth(t) / ExpectedDepth(clock_bin(t)))
#
# Negative -> thinner than expected
# Positive -> deeper than expected
# ============================================================

work = work.merge(
    baselines[
        ["symbol", "date", "bin", "expected_depth"]
    ],
    on=["symbol", "date", "bin"],
    how="inner",
    validate="many_to_one"
)

work = work[
    (work["depth_l5"] > 0)
    & (work["expected_depth"] > 0)
    & work["queue_imbalance"].notna()
    & work["log_mid"].notna()
].copy()

work["depth_surprise"] = np.log(
    work["depth_l5"] / work["expected_depth"]
)

work["liquidity_regime"] = np.where(
    work["depth_surprise"] < 0,
    "thin",
    "deep"
)

print("\n===== LIQUIDITY REGIME DISTRIBUTION =====")

print(
    work["liquidity_regime"]
    .value_counts(normalize=True)
    .round(4)
    .to_string()
)


# ============================================================
# 8. DAILY CONDITIONAL SPEARMAN IC
#
# IC = Spearman(queue imbalance, future mid return)
#
# Evaluate daily to avoid interpreting many correlated
# one-second observations as independent samples.
# ============================================================

def safe_spearman(a, b, min_obs=MIN_REGIME_OBS):

    z = pd.DataFrame({
        "signal": np.asarray(a),
        "target": np.asarray(b)
    })

    z = (
        z.replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(z) < min_obs:
        return np.nan

    if (
        z["signal"].nunique() < 3
        or z["target"].nunique() < 3
    ):
        return np.nan

    return z["signal"].corr(
        z["target"],
        method="spearman"
    )


records = []

for (symbol, date), day in work.groupby(
    ["symbol", "date"],
    sort=True
):

    for h in HORIZONS:

        target = f"return_{h}"

        full_sample = day[
            ["queue_imbalance", target]
        ].dropna()

        full_ic = safe_spearman(
            full_sample["queue_imbalance"],
            full_sample[target]
        )

        for regime in ["thin", "deep"]:

            sample = day[
                day["liquidity_regime"] == regime
            ][["queue_imbalance", target]].dropna()

            conditional_ic = safe_spearman(
                sample["queue_imbalance"],
                sample[target]
            )

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "regime": regime,
                "n": len(sample),
                "ic": conditional_ic,
                "full_day_ic": full_ic
            })

ic_table = pd.DataFrame(records)

print("\n===== DAILY CONDITIONED IC =====")

print(
    ic_table.round(6).to_string(index=False)
)


# ============================================================
# 9. PAIRED THIN VS DEEP IC COMPARISON
# ============================================================

paired_results = []

print("\n===== PAIRED LIQUIDITY REGIME COMPARISON =====")

for h in HORIZONS:

    subset = ic_table[
        ic_table["horizon"] == h
    ].copy()

    pivot = subset.pivot_table(
        index=["symbol", "date"],
        columns="regime",
        values="ic"
    )

    if not {"thin", "deep"}.issubset(pivot.columns):
        print(f"\n{h}s: insufficient regime coverage")
        continue

    pivot = pivot.dropna(subset=["thin", "deep"])

    if pivot.empty:
        print(f"\n{h}s: no valid paired sessions")
        continue

    diff = pivot["thin"] - pivot["deep"]

    result = {
        "horizon": h,
        "paired_sessions": len(pivot),
        "mean_thin_ic": pivot["thin"].mean(),
        "mean_deep_ic": pivot["deep"].mean(),
        "mean_difference": diff.mean(),
        "median_difference": diff.median(),
        "fraction_thin_higher": (diff > 0).mean(),
        "fraction_abs_thin_stronger": (
            pivot["thin"].abs() > pivot["deep"].abs()
        ).mean()
    }

    paired_results.append(result)

    print(f"\nHorizon: {h}s")
    print("Paired sessions:", len(pivot))
    print(
        "Mean thin IC:",
        f"{result['mean_thin_ic']:+.6f}"
    )
    print(
        "Mean deep IC:",
        f"{result['mean_deep_ic']:+.6f}"
    )
    print(
        "Mean difference:",
        f"{result['mean_difference']:+.6f}"
    )
    print(
        "Median difference:",
        f"{result['median_difference']:+.6f}"
    )
    print(
        "Fraction thin IC higher:",
        f"{result['fraction_thin_higher']:.4f}"
    )
    print(
        "Fraction thin |IC| stronger:",
        f"{result['fraction_abs_thin_stronger']:.4f}"
    )

paired_table = pd.DataFrame(paired_results)


# ============================================================
# 10. UNCONDITIONED DAILY QUEUE IC
# ============================================================

print("\n===== UNCONDITIONED DAILY QUEUE IC =====")

for h in HORIZONS:

    subset = ic_table[
        ic_table["horizon"] == h
    ]

    daily_ic = (
        subset.groupby(["symbol", "date"])["full_day_ic"]
        .first()
        .dropna()
    )

    if daily_ic.empty:
        continue

    print(
        f"{h}s | "
        f"Mean IC={daily_ic.mean():+.6f} | "
        f"Median={daily_ic.median():+.6f} | "
        f"Positive days={(daily_ic > 0).mean():.3f} | "
        f"Days={len(daily_ic)}"
    )


# ============================================================
# 11. REGIME SAMPLE BALANCE
# ============================================================

print("\n===== DAILY REGIME COVERAGE =====")

regime_counts = (
    work.groupby(
        ["symbol", "date", "liquidity_regime"]
    )
    .size()
    .unstack(fill_value=0)
)

print(regime_counts.to_string())

print("\n===== DEPTH SURPRISE DISTRIBUTION =====")

print(
    work["depth_surprise"]
    .describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
    .round(6)
    .to_string()
)


# ============================================================
# 12. PRESERVE EXPERIMENT OUTPUTS
# ============================================================

calendar_exp5_data = work
calendar_exp5_ic = ic_table
calendar_exp5_paired = paired_table
calendar_exp5_regime_counts = regime_counts

print("\n===== EXPERIMENT 5 COMPLETE =====")
print("Master df_1s remains unchanged.")
print("No new feature has been added to the validated core.")



===== DATASET VALIDATION =====
Master shape: (495578, 65)
Working shape: (495578, 13)
Unique working columns: True

===== BOOK DATA QUALITY =====
depth_l5: missing=0.000000
queue_imbalance: missing=0.000000
mid: missing=0.000000

===== SESSION INVENTORY =====
        symbol       date  rows  first_sec  last_sec  coverage  eligible
USDINR26OCTFUT 2026-09-30 28749      32400     61148  0.998229      True
USDINR26OCTFUT 2026-10-01 28402      32780     61181  0.986181      True
USDINR26OCTFUT 2026-10-05 28767      32400     61166  0.998854      True
USDINR26OCTFUT 2026-10-06 28755      32400     61154  0.998437      True
USDINR26OCTFUT 2026-10-07 28752      32400     61151  0.998333      True
USDINR26OCTFUT 2026-10-08 28749      32400     61148  0.998229      True
USDINR26OCTFUT 2026-10-09 28748      32400     61147  0.998194      True
USDINR26OCTFUT 2026-10-12 28754      32400     61153  0.998403      True
USDINR26OCTFUT 2026-10-13 28793      32400     61192  0.999757      True
USDINR26O

In [22]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 6
# CLOCK-TIME CONDITIONING OF QUEUE-IMBALANCE ALPHA
#
# Master dataset: df_1s
# Does not modify the master dataset.
# Dependencies: pandas and numpy only
#
# Output:
# calendar_exp6_data
# calendar_exp6_daily
# calendar_exp6_summary
# calendar_exp6_paired
# ============================================================

HORIZONS = [30, 60, 300]
MIN_DAY_COVERAGE = 0.97
MIN_WINDOW_OBS = 300

# Predefined clock-time windows, IST seconds from midnight
WINDOWS = [
    ("opening",   32400, 34200),
    ("morning",   34200, 39600),
    ("midday",    39600, 46800),
    ("afternoon", 46800, 54000),
    ("late",      54000, 59400),
]

# ------------------------------------------------------------
# 1. LOAD FROM MASTER DATASET
# ------------------------------------------------------------

required = [
    "symbol", "ts_sec", "mid", "bid_q1", "ask_q1"
]

if not df_1s.columns.is_unique:
    raise ValueError("Master df_1s has duplicate column names")

missing = [c for c in required if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= 32400)
    & (work["sec"] < 61200)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp observations")

# ------------------------------------------------------------
# 2. SESSION ELIGIBILITY
# ------------------------------------------------------------

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["coverage"] = sessions["rows"] / 28800

sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

print("\n===== ELIGIBLE SESSIONS =====")
print(sessions.to_string(index=False))

# ------------------------------------------------------------
# 3. RECONSTRUCT L1 QUEUE IMBALANCE
# ------------------------------------------------------------

bid = pd.to_numeric(work["bid_q1"], errors="coerce")
ask = pd.to_numeric(work["ask_q1"], errors="coerce")

bid = bid.where(bid >= 0)
ask = ask.where(ask >= 0)

denom = bid + ask

work["queue_imbalance"] = (
    (bid - ask) / denom.where(denom > 0)
)

work["mid"] = pd.to_numeric(
    work["mid"], errors="coerce"
)

work.loc[work["mid"] <= 0, "mid"] = np.nan

# ------------------------------------------------------------
# 4. EXACT FUTURE RETURNS
# ------------------------------------------------------------

price_grid = work[
    ["symbol", "date", "ts_sec", "mid"]
].copy()

for h in HORIZONS:

    future = price_grid.copy()
    future["ts_sec"] = future["ts_sec"] - h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    work[f"return_{h}"] = np.log(
        work[f"future_mid_{h}"] / work["mid"]
    )

# ------------------------------------------------------------
# 5. ASSIGN PREDEFINED TIME WINDOWS
# ------------------------------------------------------------

work["clock_regime"] = "excluded"

for name, start, end in WINDOWS:
    mask = (
        (work["sec"] >= start)
        & (work["sec"] < end)
    )
    work.loc[mask, "clock_regime"] = name

work = work[
    work["clock_regime"] != "excluded"
].copy()

# ------------------------------------------------------------
# 6. DAILY SPEARMAN IC
# ------------------------------------------------------------

def safe_spearman(a, b):
    z = pd.DataFrame({
        "a": np.asarray(a),
        "b": np.asarray(b)
    }).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if len(z) < MIN_WINDOW_OBS:
        return np.nan

    if z["a"].nunique() < 3 or z["b"].nunique() < 3:
        return np.nan

    return z["a"].corr(z["b"], method="spearman")


records = []

for (symbol, date), day in work.groupby(
    ["symbol", "date"]
):

    for h in HORIZONS:

        target = f"return_{h}"

        full_ic = safe_spearman(
            day["queue_imbalance"],
            day[target]
        )

        for name, _, _ in WINDOWS:

            subset = day[
                day["clock_regime"] == name
            ]

            valid = subset[
                ["queue_imbalance", target]
            ].dropna()

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "window": name,
                "observations": len(valid),
                "ic": safe_spearman(
                    valid["queue_imbalance"],
                    valid[target]
                ),
                "full_ic": full_ic
            })

daily_ic = pd.DataFrame(records)

# ------------------------------------------------------------
# 7. SUMMARY BY CLOCK WINDOW
# ------------------------------------------------------------

summary = (
    daily_ic.groupby(["horizon", "window"])
    .agg(
        sessions=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_fraction=(
            "ic", lambda s: (s.dropna() > 0).mean()
            if s.notna().any() else np.nan
        ),
        median_observations=("observations", "median")
    )
    .reset_index()
)

summary["window"] = pd.Categorical(
    summary["window"],
    categories=[w[0] for w in WINDOWS],
    ordered=True
)

summary = summary.sort_values(
    ["horizon", "window"]
)

print("\n===== CLOCK-WINDOW IC SUMMARY =====")
print(summary.round(5).to_string(index=False))

# ------------------------------------------------------------
# 8. PAIRED MORNING VS LATE COMPARISON
# ------------------------------------------------------------

# Compare same-date IC across windows.
# Treat days, not individual seconds, as replication units.

paired_records = []

print("\n===== PAIRED MORNING VS LATE IC =====")

for h in HORIZONS:

    subset = daily_ic[
        daily_ic["horizon"] == h
    ]

    p = subset.pivot_table(
        index=["symbol", "date"],
        columns="window",
        values="ic"
    )

    p = p.dropna(subset=["morning", "late"])

    if p.empty:
        continue

    difference = p["morning"] - p["late"]

    result = {
        "horizon": h,
        "sessions": len(p),
        "morning_mean_ic": p["morning"].mean(),
        "late_mean_ic": p["late"].mean(),
        "mean_difference": difference.mean(),
        "median_difference": difference.median(),
        "fraction_morning_higher": (
            difference > 0
        ).mean()
    }

    paired_records.append(result)

    print(f"\nHorizon: {h}s")
    print("Sessions:", len(p))
    print("Morning mean IC:", f"{result['morning_mean_ic']:+.6f}")
    print("Late mean IC:", f"{result['late_mean_ic']:+.6f}")
    print("Mean difference:", f"{result['mean_difference']:+.6f}")
    print("Median difference:", f"{result['median_difference']:+.6f}")
    print(
        "Morning IC higher fraction:",
        f"{result['fraction_morning_higher']:.4f}"
    )

paired = pd.DataFrame(paired_records)

# ------------------------------------------------------------
# 9. SESSION COVERAGE BY CLOCK REGIME
# ------------------------------------------------------------

print("\n===== OBSERVATION COVERAGE =====")

coverage = (
    work.groupby(["symbol", "date", "clock_regime"])
    .size()
    .unstack(fill_value=0)
)

coverage = coverage.reindex(
    columns=[w[0] for w in WINDOWS],
    fill_value=0
)

print(coverage.to_string())

# ------------------------------------------------------------
# 10. PRESERVE RESEARCH OUTPUTS
# ------------------------------------------------------------

calendar_exp6_data = work
calendar_exp6_daily = daily_ic
calendar_exp6_summary = summary
calendar_exp6_paired = paired

print("\n===== EXPERIMENT 6 COMPLETE =====")
print("df_1s remains unchanged.")
print("No feature added to validated core.")



===== ELIGIBLE SESSIONS =====
        symbol       date  rows  coverage  eligible
USDINR26OCTFUT 2026-09-30 28749  0.998229      True
USDINR26OCTFUT 2026-10-01 28402  0.986181      True
USDINR26OCTFUT 2026-10-05 28767  0.998854      True
USDINR26OCTFUT 2026-10-06 28755  0.998437      True
USDINR26OCTFUT 2026-10-07 28752  0.998333      True
USDINR26OCTFUT 2026-10-08 28749  0.998229      True
USDINR26OCTFUT 2026-10-09 28748  0.998194      True
USDINR26OCTFUT 2026-10-12 28754  0.998403      True
USDINR26OCTFUT 2026-10-13 28793  0.999757      True
USDINR26OCTFUT 2026-10-14 28792  0.999722      True
USDINR26OCTFUT 2026-10-15 28797  0.999896      True
USDINR26OCTFUT 2026-10-16 28796  0.999861      True
USDINR26OCTFUT 2026-10-19 28784  0.999444      True
USDINR26SEPFUT 2026-09-22 27175  0.943576     False
USDINR26SEPFUT 2026-09-23 28437  0.987396      True
USDINR26SEPFUT 2026-09-24 28607  0.993299      True
USDINR26SEPFUT 2026-09-25 28440  0.987500      True
USDINR26SEPFUT 2026-09-28  9281  

In [23]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 7
# CHRONOLOGICAL UPDATE-INTENSITY SEASONALITY
#
# Input: df_1s
# Dependencies: pandas, numpy
# Master dataset remains unchanged.
# ============================================================

HISTORY_DAYS = 3
BIN_SIZE = 300
MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95
START = 9 * 3600
END = 17 * 3600

required = ["symbol", "ts_sec", "is_observed"]

missing = [c for c in required if c not in df_1s.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate columns in df_1s")

work = df_1s[required].copy()

# ------------------------------------------------------------
# 1. TIMESTAMPS
# ------------------------------------------------------------

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol timestamps")

work["bin"] = (
    (work["sec"] - START) // BIN_SIZE
).astype(int)

work["update"] = pd.to_numeric(
    work["is_observed"], errors="coerce"
)

if not work["update"].isin([0, 1]).all():
    raise ValueError("is_observed must contain only 0/1")

# ------------------------------------------------------------
# 2. SESSION COVERAGE
# ------------------------------------------------------------

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["coverage"] = sessions["rows"] / (END - START)

sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ------------------------------------------------------------
# 3. FIVE-MINUTE ACTIVITY
# ------------------------------------------------------------

bins = (
    work.groupby(["symbol", "date", "bin"])
    .agg(
        n=("update", "size"),
        activity=("update", "mean")
    )
    .reset_index()
)

bins = bins[
    bins["n"] >= BIN_SIZE * MIN_BIN_COVERAGE
].copy()

# ------------------------------------------------------------
# 4. CONTRACT-SPECIFIC CHRONOLOGICAL FORECASTS
# ------------------------------------------------------------

records = []

for symbol, contract in bins.groupby("symbol"):

    dates = sorted(contract["date"].unique())

    for i in range(HISTORY_DAYS, len(dates)):

        test_date = dates[i]
        history_dates = dates[i-HISTORY_DAYS:i]

        train = contract[
            contract["date"].isin(history_dates)
        ].copy()

        test = contract[
            contract["date"] == test_date
        ].copy()

        # Constant historical update-intensity baseline
        constant = train["activity"].median()

        # Clock-time historical baseline
        clock = train.groupby("bin")["activity"].median()

        counts = train.groupby("bin")["date"].nunique()

        test["pred_constant"] = constant
        test["pred_clock"] = test["bin"].map(clock)
        test["history_bin_days"] = test["bin"].map(counts)

        test = test[
            test["history_bin_days"] >= HISTORY_DAYS
        ].copy()

        if test.empty:
            continue

        test["ae_constant"] = (
            test["activity"] - test["pred_constant"]
        ).abs()

        test["ae_clock"] = (
            test["activity"] - test["pred_clock"]
        ).abs()

        records.append(test)

if not records:
    raise ValueError("Insufficient historical sessions")

results = pd.concat(records, ignore_index=True)

# ------------------------------------------------------------
# 5. DAILY FORECAST PERFORMANCE
# ------------------------------------------------------------

daily = (
    results.groupby(["symbol", "date"])
    .agg(
        bins=("bin", "size"),
        mae_constant=("ae_constant", "mean"),
        mae_clock=("ae_clock", "mean")
    )
    .reset_index()
)

daily["improvement_pct"] = (
    100 * (1 - daily["mae_clock"] / daily["mae_constant"])
)

print("\n===== DAILY UPDATE-INTENSITY FORECASTS =====")
print(daily.round(5).to_string(index=False))

# ------------------------------------------------------------
# 6. AGGREGATE PERFORMANCE
# ------------------------------------------------------------

mae_a = results["ae_constant"].mean()
mae_b = results["ae_clock"].mean()

print("\n===== AGGREGATE PERFORMANCE =====")

print("Evaluation sessions:", len(daily))
print("Evaluation bins:", len(results))
print("Constant MAE:", f"{mae_a:.6f}")
print("Clock MAE:", f"{mae_b:.6f}")

print(
    "Overall improvement (%):",
    f"{100 * (1 - mae_b / mae_a):+.4f}"
)

print(
    "Winning sessions:",
    int((daily["improvement_pct"] > 0).sum()),
    "/",
    len(daily)
)

print(
    "Median daily improvement (%):",
    f"{daily['improvement_pct'].median():+.4f}"
)

# ------------------------------------------------------------
# 7. BIAS AND CORRELATION DIAGNOSTICS
# ------------------------------------------------------------

results["residual_clock"] = (
    results["activity"] - results["pred_clock"]
)

results["residual_constant"] = (
    results["activity"] - results["pred_constant"]
)

print("\n===== FORECAST BIAS =====")

print(
    "Clock mean residual:",
    f"{results['residual_clock'].mean():+.6f}"
)

print(
    "Constant mean residual:",
    f"{results['residual_constant'].mean():+.6f}"
)

print("\n===== DAILY BIAS =====")

bias = (
    results.groupby(["symbol", "date"])
    .agg(
        clock_bias=("residual_clock", "mean"),
        constant_bias=("residual_constant", "mean")
    )
    .reset_index()
)

print(bias.round(5).to_string(index=False))

# ------------------------------------------------------------
# 8. STORE RESEARCH OUTPUTS
# ------------------------------------------------------------

calendar_exp7_results = results
calendar_exp7_daily = daily
calendar_exp7_bias = bias

print("\n===== EXPERIMENT 7 COMPLETE =====")
print("Master df_1s unchanged.")
print("No features added to validated core.")



===== DAILY UPDATE-INTENSITY FORECASTS =====
        symbol       date  bins  mae_constant  mae_clock  improvement_pct
USDINR26OCTFUT 2026-10-06    93       0.22577    0.23925         -5.96920
USDINR26OCTFUT 2026-10-07    93       0.23695    0.24710         -4.28074
USDINR26OCTFUT 2026-10-08    95       0.20502    0.28505        -39.03817
USDINR26OCTFUT 2026-10-09    95       0.16239    0.17270         -6.35264
USDINR26OCTFUT 2026-10-12    95       0.26116    0.22021         15.67916
USDINR26OCTFUT 2026-10-13    95       0.20147    0.22537        -11.85998
USDINR26OCTFUT 2026-10-14    95       0.18032    0.23747        -31.69877
USDINR26OCTFUT 2026-10-15    95       0.24800    0.20330         18.02490
USDINR26OCTFUT 2026-10-16    96       0.19455    0.19957         -2.58290
USDINR26OCTFUT 2026-10-19    95       0.20398    0.25351        -24.27969

===== AGGREGATE PERFORMANCE =====
Evaluation sessions: 10
Evaluation bins: 947
Constant MAE: 0.211860
Clock MAE: 0.228260
Overall improveme

In [24]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 8
# ACTIVITY ACCELERATION AND CONDITIONAL QUEUE ALPHA
#
# Input: df_1s (master remains unchanged)
# Dependencies: pandas, numpy
#
# Outputs:
#   calendar_exp8_data
#   calendar_exp8_daily_ic
#   calendar_exp8_conditioned_ic
#   calendar_exp8_paired
# ============================================================

HORIZONS = [30, 60, 300]
WINDOWS = [30, 60, 300]

START = 9 * 3600
END = 17 * 3600

MIN_DAY_COVERAGE = 0.97
MIN_REGIME_OBS = 300

required = [
    "symbol", "ts_sec", "mid",
    "bid_q1", "ask_q1", "is_observed"
]

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate columns in df_1s")

missing = [c for c in required if c not in df_1s.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

# ------------------------------------------------------------
# 1. TIMESTAMP AND SESSION SETUP
# ------------------------------------------------------------

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp observations")

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["eligible"] = (
    (sessions["rows"] / (END - START) >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ------------------------------------------------------------
# 2. ACTIVITY AND QUEUE IMBALANCE
# ------------------------------------------------------------

work["update"] = pd.to_numeric(
    work["is_observed"], errors="coerce"
)

if not work["update"].isin([0, 1]).all():
    raise ValueError("is_observed must be binary")

bid = pd.to_numeric(work["bid_q1"], errors="coerce")
ask = pd.to_numeric(work["ask_q1"], errors="coerce")

bid = bid.where(bid >= 0)
ask = ask.where(ask >= 0)

denom = bid + ask

work["queue_imbalance"] = (
    (bid - ask) / denom.where(denom > 0)
)

work["mid"] = pd.to_numeric(
    work["mid"], errors="coerce"
)
work.loc[work["mid"] <= 0, "mid"] = np.nan

# ------------------------------------------------------------
# 3. CAUSAL TRAILING ACTIVITY RATES
#
# Require all w consecutive seconds.
# This prevents rolling over gaps or crossing days.
# ------------------------------------------------------------

group_keys = ["symbol", "date"]
g = work.groupby(group_keys, sort=False)

work["run_id"] = (
    g["ts_sec"].diff().ne(1)
    .groupby([work["symbol"], work["date"]])
    .cumsum()
)

run_keys = ["symbol", "date", "run_id"]

for w in WINDOWS:
    work[f"activity_{w}s"] = (
        work.groupby(run_keys, sort=False)["update"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=w
            ).mean()
        )
    )

work["activity_accel_30_300"] = (
    work["activity_30s"] - work["activity_300s"]
)

work["activity_accel_60_300"] = (
    work["activity_60s"] - work["activity_300s"]
)

# A predetermined threshold, not optimized from labels.
work["activity_regime"] = np.select(
    [
        work["activity_accel_30_300"] > 0,
        work["activity_accel_30_300"] < 0
    ],
    ["accelerating", "decelerating"],
    default="neutral"
)

# ------------------------------------------------------------
# 4. EXACT FORWARD MID-PRICE RETURNS
# ------------------------------------------------------------

price_grid = work[
    ["symbol", "date", "ts_sec", "mid"]
].copy()

for h in HORIZONS:
    future = price_grid.copy()
    future["ts_sec"] -= h
    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    work[f"return_{h}"] = np.log(
        work[f"future_mid_{h}"] / work["mid"]
    )

# ------------------------------------------------------------
# 5. SAFE DAILY SPEARMAN IC
# ------------------------------------------------------------

def safe_ic(a, b, min_obs=MIN_REGIME_OBS):
    z = pd.DataFrame({
        "x": np.asarray(a),
        "y": np.asarray(b)
    }).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if len(z) < min_obs:
        return np.nan

    if z["x"].nunique() < 3:
        return np.nan

    if z["y"].nunique() < 3:
        return np.nan

    return z["x"].corr(z["y"], method="spearman")


signals = [
    "queue_imbalance",
    "activity_30s",
    "activity_60s",
    "activity_300s",
    "activity_accel_30_300",
    "activity_accel_60_300"
]

ic_records = []

for (symbol, date), day in work.groupby(group_keys):
    for h in HORIZONS:
        target = f"return_{h}"

        for signal in signals:
            ic_records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "signal": signal,
                "ic": safe_ic(day[signal], day[target])
            })

daily_ic = pd.DataFrame(ic_records)

summary = (
    daily_ic.groupby(["signal", "horizon"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: (s.dropna() > 0).mean()
            if s.notna().any() else np.nan
        ),
        negative_days=(
            "ic",
            lambda s: (s.dropna() < 0).mean()
            if s.notna().any() else np.nan
        )
    )
    .reset_index()
)

print("\n===== STANDALONE ACTIVITY IC =====")
print(summary.round(5).to_string(index=False))

# ------------------------------------------------------------
# 6. QUEUE IC CONDITIONAL ON ACTIVITY ACCELERATION
# ------------------------------------------------------------

conditioned_records = []

for (symbol, date), day in work.groupby(group_keys):
    for h in HORIZONS:
        target = f"return_{h}"

        for regime in ["accelerating", "decelerating"]:
            sub = day[
                day["activity_regime"] == regime
            ][["queue_imbalance", target]].dropna()

            conditioned_records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "regime": regime,
                "n": len(sub),
                "queue_ic": safe_ic(
                    sub["queue_imbalance"],
                    sub[target]
                )
            })

conditioned = pd.DataFrame(conditioned_records)

conditioned_summary = (
    conditioned.groupby(["horizon", "regime"])
    .agg(
        sessions=("queue_ic", "count"),
        mean_ic=("queue_ic", "mean"),
        median_ic=("queue_ic", "median"),
        positive_days=(
            "queue_ic",
            lambda s: (s.dropna() > 0).mean()
            if s.notna().any() else np.nan
        )
    )
    .reset_index()
)

print("\n===== CONDITIONAL QUEUE IC =====")
print(
    conditioned_summary.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 7. PAIRED SESSION COMPARISONS
# ------------------------------------------------------------

paired_rows = []

for h in HORIZONS:
    z = conditioned[conditioned["horizon"] == h]

    p = z.pivot_table(
        index=["symbol", "date"],
        columns="regime",
        values="queue_ic"
    )

    if not {
        "accelerating", "decelerating"
    }.issubset(p.columns):
        continue

    p = p.dropna(
        subset=["accelerating", "decelerating"]
    )

    if p.empty:
        continue

    diff = p["accelerating"] - p["decelerating"]

    paired_rows.append({
        "horizon": h,
        "sessions": len(p),
        "mean_accel_ic": p["accelerating"].mean(),
        "mean_decel_ic": p["decelerating"].mean(),
        "mean_difference": diff.mean(),
        "median_difference": diff.median(),
        "fraction_accel_higher": (diff > 0).mean(),
        "fraction_abs_accel_higher": (
            p["accelerating"].abs()
            > p["decelerating"].abs()
        ).mean()
    })

paired = pd.DataFrame(paired_rows)

print("\n===== PAIRED ACTIVITY-REGIME IC =====")
print(paired.round(6).to_string(index=False))

# ------------------------------------------------------------
# 8. COVERAGE / DISTRIBUTION
# ------------------------------------------------------------

print("\n===== ACTIVITY REGIME COVERAGE =====")

coverage = (
    work.groupby(
        ["symbol", "date", "activity_regime"]
    )
    .size()
    .unstack(fill_value=0)
)

print(coverage.to_string())

print("\n===== ACTIVITY ACCELERATION DISTRIBUTION =====")
print(
    work["activity_accel_30_300"]
    .describe(percentiles=[.05, .25, .5, .75, .95])
    .round(6)
    .to_string()
)

print("\n===== CORRELATION AMONG ACTIVITY FEATURES =====")

print(
    work[
        [
            "activity_30s",
            "activity_60s",
            "activity_300s",
            "activity_accel_30_300",
            "activity_accel_60_300"
        ]
    ].corr(method="spearman").round(3).to_string()
)

# ------------------------------------------------------------
# 9. PRESERVE OUTPUTS
# ------------------------------------------------------------

calendar_exp8_data = work
calendar_exp8_daily_ic = daily_ic
calendar_exp8_conditioned_ic = conditioned
calendar_exp8_paired = paired

print("\n===== EXPERIMENT 8 COMPLETE =====")
print("df_1s unchanged. No features validated.")



===== STANDALONE ACTIVITY IC =====
               signal  horizon  days  mean_ic  median_ic  positive_days  negative_days
        activity_300s       30    16 -0.01268   -0.01218         0.3125         0.6875
        activity_300s       60    16 -0.02064   -0.02170         0.2500         0.7500
        activity_300s      300    16 -0.04181   -0.03498         0.3125         0.6875
         activity_30s       30    16 -0.02302   -0.01594         0.0625         0.9375
         activity_30s       60    16 -0.03401   -0.02522         0.1250         0.8750
         activity_30s      300    16 -0.04438   -0.04216         0.2500         0.7500
         activity_60s       30    16 -0.02107   -0.01547         0.1875         0.8125
         activity_60s       60    16 -0.03060   -0.03000         0.0625         0.9375
         activity_60s      300    16 -0.04236   -0.04195         0.2500         0.7500
activity_accel_30_300       30    16 -0.01917   -0.01975         0.2500         0.7500
activit

In [25]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 9
# CLOCK-CONTROLLED ACTIVITY ACCELERATION INTERACTION
#
# Input: calendar_exp8_data
# Output: calendar_exp9_*
# pandas + numpy only
# ============================================================

HORIZONS = [30, 60, 300]
MIN_OBS = 150

required = [
    "symbol", "date", "sec",
    "queue_imbalance", "activity_300s",
    "activity_accel_30_300",
    "return_30", "return_60", "return_300"
]

missing = [
    c for c in required
    if c not in calendar_exp8_data.columns
]
if missing:
    raise ValueError(f"Missing columns: {missing}")

z = calendar_exp8_data[required].copy()

# ------------------------------------------------------------
# 1. FIXED CLOCK WINDOWS
# ------------------------------------------------------------

windows = [
    ("opening", 32400, 34200),
    ("morning", 34200, 39600),
    ("midday", 39600, 46800),
    ("afternoon", 46800, 54000),
    ("late", 54000, 59400)
]

z["window"] = pd.NA

for name, start, end in windows:
    z.loc[
        (z["sec"] >= start) & (z["sec"] < end),
        "window"
    ] = name

z = z[z["window"].notna()].copy()

# Exclude unavailable acceleration rather than
# silently assigning it to a neutral regime.
z = z[z["activity_accel_30_300"].notna()].copy()

z["regime"] = np.select(
    [
        z["activity_accel_30_300"] > 0,
        z["activity_accel_30_300"] < 0
    ],
    ["accelerating", "decelerating"],
    default="neutral"
)

# ------------------------------------------------------------
# 2. SAFE RANK CORRELATION
# ------------------------------------------------------------

def rank_ic(a, b):
    frame = pd.DataFrame({
        "signal": np.asarray(a),
        "target": np.asarray(b)
    }).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if len(frame) < MIN_OBS:
        return np.nan

    if frame["signal"].nunique() < 3:
        return np.nan

    if frame["target"].nunique() < 3:
        return np.nan

    return frame["signal"].corr(
        frame["target"], method="spearman"
    )

# ------------------------------------------------------------
# 3. CONTROL FOR LONG-WINDOW ACTIVITY LEVEL
#
# Within each session, split activity_300s at its
# contemporaneously measured median.
#
# IMPORTANT:
# This is retrospective stratification only.
# Do not use today's completed-day median as a
# live feature or trading threshold.
# ------------------------------------------------------------

day_median = z.groupby(
    ["symbol", "date"]
)["activity_300s"].transform("median")

z["activity_level"] = np.where(
    z["activity_300s"] >= day_median,
    "high",
    "low"
)

# ------------------------------------------------------------
# 4. CONDITIONED IC WITH CLOCK + LEVEL CONTROLS
# ------------------------------------------------------------

records = []

for keys, sample in z.groupby(
    ["symbol", "date", "window", "activity_level"],
    sort=True
):

    symbol, date, window, level = keys

    for h in HORIZONS:
        target = f"return_{h}"

        for regime in ["accelerating", "decelerating"]:

            sub = sample[
                sample["regime"] == regime
            ]

            records.append({
                "symbol": symbol,
                "date": date,
                "window": window,
                "activity_level": level,
                "horizon": h,
                "regime": regime,
                "n": sub[
                    ["queue_imbalance", target]
                ].dropna().shape[0],
                "ic": rank_ic(
                    sub["queue_imbalance"],
                    sub[target]
                )
            })

detail = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. MATCH ACCELERATING / DECELERATING
# ------------------------------------------------------------

keys = [
    "symbol", "date", "window",
    "activity_level", "horizon"
]

p = detail.pivot_table(
    index=keys,
    columns="regime",
    values="ic"
)

if not {
    "accelerating", "decelerating"
}.issubset(p.columns):
    raise ValueError("Insufficient matched regimes")

p = p.dropna(
    subset=["accelerating", "decelerating"]
).reset_index()

p["ic_diff"] = (
    p["accelerating"] - p["decelerating"]
)

# Equal-weight matched clock/level cells within each day.
# Do not treat the cells as independent trading days.
daily = (
    p.groupby(["symbol", "date", "horizon"])
    .agg(
        matched_cells=("ic_diff", "size"),
        mean_diff=("ic_diff", "mean"),
        median_diff=("ic_diff", "median"),
        positive_cells=(
            "ic_diff", lambda s: (s > 0).mean()
        )
    )
    .reset_index()
)

summary = (
    daily.groupby("horizon")
    .agg(
        sessions=("mean_diff", "count"),
        mean_day_difference=("mean_diff", "mean"),
        median_day_difference=("mean_diff", "median"),
        positive_day_fraction=(
            "mean_diff", lambda s: (s > 0).mean()
        ),
        min_day_difference=("mean_diff", "min"),
        max_day_difference=("mean_diff", "max"),
        median_matched_cells=("matched_cells", "median")
    )
    .reset_index()
)

print("\n===== CLOCK + ACTIVITY CONTROLLED IC =====")
print(summary.round(6).to_string(index=False))

print("\n===== DAILY CONTROLLED DIFFERENCES =====")
print(daily.round(6).to_string(index=False))

# ------------------------------------------------------------
# 6. WINDOW-LEVEL DIAGNOSTICS
# ------------------------------------------------------------

window_summary = (
    p.groupby(["horizon", "window"])
    .agg(
        matched_cells=("ic_diff", "size"),
        mean_difference=("ic_diff", "mean"),
        median_difference=("ic_diff", "median"),
        positive_fraction=(
            "ic_diff", lambda s: (s > 0).mean()
        )
    )
    .reset_index()
)

print("\n===== CLOCK-WINDOW ROBUSTNESS =====")
print(
    window_summary.round(6).to_string(index=False)
)

print("\n===== MATCHED SAMPLE COVERAGE =====")
print(
    p.groupby("horizon")
    .size()
    .to_string()
)

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

calendar_exp9_data = z
calendar_exp9_detail = detail
calendar_exp9_matched = p
calendar_exp9_daily = daily
calendar_exp9_summary = summary
calendar_exp9_windows = window_summary

print("\nEXPERIMENT 9 COMPLETE")
print("Original datasets unchanged.")
print("No new features validated.")



===== CLOCK + ACTIVITY CONTROLLED IC =====
 horizon  sessions  mean_day_difference  median_day_difference  positive_day_fraction  min_day_difference  max_day_difference  median_matched_cells
      30        16             0.006679              -0.026024                 0.4375           -0.126764            0.198643                   9.0
      60        16             0.006172               0.002324                 0.5000           -0.074664            0.209110                   9.0
     300        16            -0.017007              -0.016018                 0.4375           -0.183720            0.090191                   9.0

===== DAILY CONTROLLED DIFFERENCES =====
        symbol       date  horizon  matched_cells  mean_diff  median_diff  positive_cells
USDINR26OCTFUT 2026-09-30       30              8  -0.126764    -0.056821        0.375000
USDINR26OCTFUT 2026-09-30       60              8  -0.064332    -0.055680        0.250000
USDINR26OCTFUT 2026-09-30      300              8  -

In [26]:

import pandas as pd
import numpy as np

# ==========================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 10
# INTRADAY VOLATILITY DECOMPOSITION
#
# Input: df_1s
# Dependencies: pandas, numpy
# No changes to the master dataset.
# ==========================================================

START = 9 * 3600
END = 17 * 3600
BIN_SECONDS = 300
MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95

required = ["symbol", "ts_sec", "mid"]

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate master columns detected")

missing = [c for c in required if c not in df_1s.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

# ----------------------------------------------------------
# 1. TIME AND SESSION SETUP
# ----------------------------------------------------------

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol timestamps")

session = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

session["coverage"] = session["rows"] / (END - START)

session["eligible"] = (
    (session["coverage"] >= MIN_DAY_COVERAGE)
    & (session["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    session[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

work["bin"] = (
    (work["sec"] - START) // BIN_SECONDS
).astype(int)

# ----------------------------------------------------------
# 2. CONSECUTIVE SECOND MID-PRICE RETURNS
# ----------------------------------------------------------

work["mid"] = pd.to_numeric(
    work["mid"], errors="coerce"
)

work.loc[work["mid"] <= 0, "mid"] = np.nan

work["log_mid"] = np.log(work["mid"])

g = work.groupby(["symbol", "date"], sort=False)

work["ret"] = g["log_mid"].diff()

# Only exact one-second transitions are valid.
elapsed = g["ts_sec"].diff()

work.loc[elapsed.ne(1), "ret"] = np.nan

work["ret_sq"] = work["ret"] ** 2

work["nonzero"] = (
    work["ret"].ne(0)
    .where(work["ret"].notna(), np.nan)
    .astype(float)
)

# ----------------------------------------------------------
# 3. FIVE-MINUTE RV DECOMPOSITION
# ----------------------------------------------------------

bins = (
    work.groupby(["symbol", "date", "bin"])
    .agg(
        seconds=("ts_sec", "size"),
        valid_returns=("ret", "count"),
        n_moves=("nonzero", "sum"),
        rv=("ret_sq", "sum"),
        max_abs_return=(
            "ret", lambda s: s.abs().max()
        )
    )
    .reset_index()
)

bins["coverage"] = bins["seconds"] / BIN_SECONDS

bins = bins[
    (bins["coverage"] >= MIN_BIN_COVERAGE)
    & (bins["valid_returns"] >= 0.90 * BIN_SECONDS)
].copy()

bins["move_rate"] = (
    bins["n_moves"] / bins["valid_returns"]
)

bins["mean_sq_move"] = (
    bins["rv"]
    / bins["n_moves"].replace(0, np.nan)
)

bins["mean_abs_move_proxy"] = np.sqrt(
    bins["mean_sq_move"]
)

# Fraction of RV contributed by the largest return.
# It indicates variance concentration, not necessarily
# a statistically identified jump.
bins["largest_move_rv_share"] = (
    bins["max_abs_return"] ** 2
    / bins["rv"].replace(0, np.nan)
)

# Verify the mathematical decomposition.
check = (
    bins["n_moves"] * bins["mean_sq_move"]
    - bins["rv"]
)

print("\n===== DECOMPOSITION CHECK =====")
print(
    "Maximum absolute reconstruction error:",
    np.nanmax(np.abs(check))
)

# ----------------------------------------------------------
# 4. NORMALIZED CLOCK-TIME PROFILES
# ----------------------------------------------------------

metrics = [
    "rv",
    "move_rate",
    "mean_sq_move",
    "largest_move_rv_share"
]

for metric in ["rv", "move_rate", "mean_sq_move"]:

    daily_mean = bins.groupby(
        ["symbol", "date"]
    )[metric].transform("mean")

    bins[f"{metric}_relative"] = (
        bins[metric]
        / daily_mean.where(daily_mean > 0)
    )

profile = (
    bins.groupby("bin")
    .agg(
        sessions=("date", "size"),
        median_relative_rv=(
            "rv_relative", "median"
        ),
        median_relative_move_rate=(
            "move_rate_relative", "median"
        ),
        median_relative_move_size=(
            "mean_sq_move_relative", "median"
        ),
        median_largest_move_share=(
            "largest_move_rv_share", "median"
        )
    )
    .reset_index()
)

def clock_label(b):
    s = START + int(b) * BIN_SECONDS
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}"

profile["clock"] = profile["bin"].map(clock_label)

selected = list(range(0, 96, 6))

print("\n===== INTRADAY VOLATILITY DECOMPOSITION =====")
print(
    profile[
        profile["bin"].isin(selected)
    ][[
        "clock",
        "sessions",
        "median_relative_rv",
        "median_relative_move_rate",
        "median_relative_move_size",
        "median_largest_move_share"
    ]].round(5).to_string(index=False)
)

# ----------------------------------------------------------
# 5. OPENING VS REST OF SESSION
# ----------------------------------------------------------

bins["period"] = np.where(
    bins["bin"] < 6,
    "opening_30m",
    "rest"
)

periods = (
    bins.groupby(["symbol", "date", "period"])
    .agg(
        rv=("rv", "sum"),
        moves=("n_moves", "sum"),
        valid_returns=("valid_returns", "sum")
    )
    .reset_index()
)

periods["move_rate"] = (
    periods["moves"] / periods["valid_returns"]
)

periods["mean_sq_move"] = (
    periods["rv"]
    / periods["moves"].replace(0, np.nan)
)

paired = {}

for metric in ["move_rate", "mean_sq_move"]:

    p = periods.pivot_table(
        index=["symbol", "date"],
        columns="period",
        values=metric
    ).dropna()

    if not {"opening_30m", "rest"}.issubset(p.columns):
        continue

    diff = p["opening_30m"] - p["rest"]

    paired[metric] = p

    print(f"\n===== {metric.upper()} =====")
    print("Sessions:", len(p))
    print(
        "Opening mean:",
        f"{p['opening_30m'].mean():.10g}"
    )
    print(
        "Rest mean:",
        f"{p['rest'].mean():.10g}"
    )
    print(
        "Opening greater fraction:",
        f"{(diff > 0).mean():.3f}"
    )

# ----------------------------------------------------------
# 6. OPENING SPIKE SENSITIVITY
# ----------------------------------------------------------

opening = work[
    (work["sec"] >= START)
    & (work["sec"] < START + 300)
].copy()

opening["first_30s"] = (
    opening["sec"] < START + 30
)

opening_rv = (
    opening.groupby(["symbol", "date"])
    .agg(
        first_bin_rv=("ret_sq", "sum"),
        first_30s_rv=(
            "ret_sq",
            lambda s: s.iloc[:30].sum()
        )
    )
    .reset_index()
)

opening_rv["first_30s_share"] = (
    opening_rv["first_30s_rv"]
    / opening_rv["first_bin_rv"].replace(0, np.nan)
)

print("\n===== OPENING RV CONCENTRATION =====")

print(
    opening_rv.round(6).to_string(index=False)
)

print(
    "\nMedian first-30s share:",
    opening_rv["first_30s_share"].median()
)

# ----------------------------------------------------------
# 7. PRESERVE OUTPUTS
# ----------------------------------------------------------

calendar_exp10_bins = bins
calendar_exp10_profile = profile
calendar_exp10_periods = periods
calendar_exp10_opening = opening_rv

print("\n===== EXPERIMENT 10 COMPLETE =====")
print("Master df_1s unchanged.")
print("No volatility feature validated yet.")



===== DECOMPOSITION CHECK =====
Maximum absolute reconstruction error: 1.6940658945086007e-21

===== INTRADAY VOLATILITY DECOMPOSITION =====
clock  sessions  median_relative_rv  median_relative_move_rate  median_relative_move_size  median_largest_move_share
09:00        12            12.37043                    2.53400                    5.70321                    0.24933
09:30        16             1.11855                    1.13982                    0.98173                    0.22175
10:00        16             0.65623                    0.87371                    0.51274                    0.28116
10:30        16             0.87282                    0.85925                    0.89577                    0.28351
11:00        16             0.47566                    0.87962                    0.38918                    0.21926
11:30        16             0.98146                    1.44659                    0.83868                    0.23735
12:00        16             0.41996    

In [27]:

import numpy as np
import pandas as pd

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 11
# OPENING VOLATILITY: DURATION, CONCENTRATION, ROBUSTNESS
#
# Master: df_1s (unchanged)
# Dependencies: pandas, numpy
#
# Outputs:
# calendar_exp11_interval_stats
# calendar_exp11_daily
# calendar_exp11_summary
# calendar_exp11_concentration
# ============================================================

START = 9 * 3600
END = 17 * 3600
MIN_DAY_COVERAGE = 0.97
MIN_INTERVAL_RETURN_COVERAGE = 0.90

required = ["symbol", "ts_sec", "mid"]
missing = [c for c in required if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate columns in df_1s")

work = df_1s[required].copy()

# ------------------------------------------------------------
# 1. TIMESTAMPS AND SESSIONS
# ------------------------------------------------------------

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)
work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START) & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows")

sessions = (
    work.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_sec=("sec", "min")
    )
    .reset_index()
)

sessions["eligible"] = (
    (sessions["rows"] / (END - START) >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ------------------------------------------------------------
# 2. STRICTLY CONSECUTIVE MID RETURNS
# ------------------------------------------------------------

work["mid"] = pd.to_numeric(
    work["mid"], errors="coerce"
).where(lambda s: s > 0)

work["log_mid"] = np.log(work["mid"])

group = work.groupby(["symbol", "date"], sort=False)

work["ret"] = group["log_mid"].diff()
elapsed = group["ts_sec"].diff()

work.loc[elapsed.ne(1), "ret"] = np.nan

work["ret_sq"] = work["ret"] ** 2

work["move"] = (
    work["ret"].ne(0)
    .where(work["ret"].notna(), np.nan)
    .astype(float)
)

# ------------------------------------------------------------
# 3. PREDEFINED NONOVERLAPPING CLOCK INTERVALS
# ------------------------------------------------------------

intervals = [
    ("open_00_30s", 0, 30),
    ("open_30_60s", 30, 60),
    ("open_01_05m", 60, 300),
    ("open_05_30m", 300, 1800),
    ("rest_30m_end", 1800, END - START)
]

interval_frames = []

for name, begin, finish in intervals:

    sub = work[
        (work["sec"] >= START + begin)
        & (work["sec"] < START + finish)
    ].copy()

    stats = (
        sub.groupby(["symbol", "date"])
        .agg(
            n_returns=("ret", "count"),
            rv=("ret_sq", "sum"),
            n_moves=("move", "sum"),
            max_abs_return=(
                "ret", lambda s: s.abs().max()
            )
        )
        .reset_index()
    )

    stats["interval"] = name
    stats["duration_seconds"] = finish - begin

    stats["return_coverage"] = (
        stats["n_returns"] / stats["duration_seconds"]
    )

    stats["valid"] = (
        stats["return_coverage"]
        >= MIN_INTERVAL_RETURN_COVERAGE
    )

    stats["rv_per_second"] = (
        stats["rv"] / stats["n_returns"].replace(0, np.nan)
    )

    stats["move_rate"] = (
        stats["n_moves"]
        / stats["n_returns"].replace(0, np.nan)
    )

    stats["sq_per_move"] = (
        stats["rv"] / stats["n_moves"].replace(0, np.nan)
    )

    interval_frames.append(stats)

interval_stats = pd.concat(
    interval_frames, ignore_index=True
)

# Mark invalid intervals as missing for comparison.
metrics = [
    "rv_per_second", "move_rate", "sq_per_move"
]

interval_stats.loc[
    ~interval_stats["valid"], metrics
] = np.nan

print("\n===== INTERVAL COVERAGE =====")
print(
    interval_stats.groupby("interval")
    .agg(
        available_sessions=("valid", "size"),
        valid_sessions=("valid", "sum"),
        median_return_coverage=(
            "return_coverage", "median"
        )
    )
    .to_string()
)

# ------------------------------------------------------------
# 4. WITHIN-DAY COMPARISON AGAINST REST OF SESSION
# ------------------------------------------------------------

daily_rows = []

for metric in metrics:

    pivot = interval_stats.pivot_table(
        index=["symbol", "date"],
        columns="interval",
        values=metric
    )

    if "rest_30m_end" not in pivot.columns:
        continue

    rest = pivot["rest_30m_end"]

    for interval in [
        "open_00_30s",
        "open_30_60s",
        "open_01_05m",
        "open_05_30m"
    ]:

        if interval not in pivot.columns:
            continue

        paired = pd.DataFrame({
            "opening": pivot[interval],
            "rest": rest
        }).dropna()

        paired = paired[
            paired["rest"] > 0
        ].copy()

        if paired.empty:
            continue

        paired["ratio"] = (
            paired["opening"] / paired["rest"]
        )

        paired["difference"] = (
            paired["opening"] - paired["rest"]
        )

        for (symbol, date), row in paired.iterrows():
            daily_rows.append({
                "symbol": symbol,
                "date": date,
                "metric": metric,
                "interval": interval,
                "ratio": row["ratio"],
                "difference": row["difference"]
            })

daily = pd.DataFrame(daily_rows)

summary = (
    daily.groupby(["metric", "interval"])
    .agg(
        sessions=("ratio", "count"),
        median_ratio=("ratio", "median"),
        mean_ratio=("ratio", "mean"),
        positive_day_fraction=(
            "difference", lambda s: (s > 0).mean()
        ),
        min_ratio=("ratio", "min"),
        max_ratio=("ratio", "max")
    )
    .reset_index()
)

print("\n===== OPENING VS REST: PAIRED RATIOS =====")
print(summary.round(5).to_string(index=False))

# ------------------------------------------------------------
# 5. EXACT FIRST-30-SECOND RV CONCENTRATION
# ------------------------------------------------------------

# Require a sufficiently complete 09:00–09:05 interval.
# The first 30 seconds contribute at most 29 valid
# consecutive returns because the first day's return
# at 09:00:00 has no previous-session reference.

concentration_rows = []

for (symbol, date), day in work.groupby(["symbol", "date"]):

    first_5m = day[
        (day["sec"] >= START)
        & (day["sec"] < START + 300)
    ]

    first_30s = first_5m[
        first_5m["sec"] < START + 30
    ]

    first_60s = first_5m[
        first_5m["sec"] < START + 60
    ]

    n_5m = first_5m["ret"].count()
    n_30s = first_30s["ret"].count()
    n_60s = first_60s["ret"].count()

    # Reject incomplete opening intervals rather than
    # comparing the first available 30 observations.
    if n_5m < 0.95 * 299:
        continue

    if n_30s < 0.90 * 29:
        continue

    if n_60s < 0.90 * 59:
        continue

    rv_5m = first_5m["ret_sq"].sum()
    rv_30s = first_30s["ret_sq"].sum()
    rv_60s = first_60s["ret_sq"].sum()

    if rv_5m <= 0:
        continue

    concentration_rows.append({
        "symbol": symbol,
        "date": date,
        "n_5m": n_5m,
        "n_30s": n_30s,
        "n_60s": n_60s,
        "rv_first_5m": rv_5m,
        "share_30s": rv_30s / rv_5m,
        "share_60s": rv_60s / rv_5m
    })

concentration = pd.DataFrame(concentration_rows)

print("\n===== EXACT OPENING RV CONCENTRATION =====")

if not concentration.empty:

    display_cols = [
        "symbol", "date",
        "n_30s", "n_60s",
        "share_30s", "share_60s"
    ]

    print(
        concentration[display_cols]
        .round(5)
        .to_string(index=False)
    )

    print("\nMedian first-30s RV share:",
          f"{concentration['share_30s'].median():.4f}")

    print("Median first-60s RV share:",
          f"{concentration['share_60s'].median():.4f}")

    print("Sessions:", len(concentration))

else:
    print("Insufficient fully covered opening intervals.")

# ------------------------------------------------------------
# 6. OUTLIER AND STABILITY DIAGNOSTICS
# ------------------------------------------------------------

print("\n===== DAY-LEVEL OPENING RV =====")

first_bins = interval_stats[
    interval_stats["interval"] == "open_00_30s"
][["symbol", "date", "rv", "n_returns", "valid"]]

print(
    first_bins.to_string(
        index=False,
        float_format=lambda v: f"{v:.8e}"
    )
)

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

calendar_exp11_interval_stats = interval_stats
calendar_exp11_daily = daily
calendar_exp11_summary = summary
calendar_exp11_concentration = concentration

print("\n===== EXPERIMENT 11 COMPLETE =====")
print("Master df_1s remains unchanged.")
print("No volatility feature validated yet.")



===== INTERVAL COVERAGE =====
              available_sessions  valid_sessions  median_return_coverage
interval                                                                
open_00_30s                   12              12                0.966667
open_01_05m                   14              12                1.000000
open_05_30m                   16              16                1.000000
open_30_60s                   12              12                1.000000
rest_30m_end                  16              16                0.998704

===== OPENING VS REST: PAIRED RATIOS =====
       metric    interval  sessions  median_ratio  mean_ratio  positive_day_fraction  min_ratio  max_ratio
    move_rate open_00_30s        12       6.61952     6.77077                1.00000    1.53844   16.13083
    move_rate open_01_05m        12       1.98843     2.40849                1.00000    1.27035    4.91523
    move_rate open_05_30m        16       1.54592     1.85196                0.93750    0.933

In [28]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 12
# OPENING VOLATILITY QUALITY AND OUTLIER ROBUSTNESS
#
# INPUT:
#   df_1s
#
# OUTPUTS:
#   calendar_exp12_data
#   calendar_exp12_quality
#   calendar_exp12_jumps
#   calendar_exp12_robustness
#   calendar_exp12_summary
#   calendar_exp12_summary_no_outlier
#
# No plotting packages required.
# Master df_1s remains unchanged.
# ============================================================

START_SEC = 9 * 3600
END_SEC = 17 * 3600

MIN_DAY_COVERAGE = 0.97
MIN_INTERVAL_COVERAGE = 0.90

INTERVALS = [
    ("first_30s", 0, 30),
    ("next_30s", 30, 60),
    ("minute_1_5", 60, 300),
    ("minute_5_30", 300, 1800),
    ("rest", 1800, 28800),
]

# ============================================================
# 1. VALIDATE DATASET
# ============================================================

required = [
    "symbol",
    "ts_sec",
    "mid",
    "bid_p1",
    "ask_p1"
]

if not df_1s.columns.is_unique:
    raise ValueError(
        "Duplicate column names found in df_1s."
    )

missing = [
    c for c in required
    if c not in df_1s.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s.loc[:, required].copy()

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid ts_sec values detected.")

print("\n===== DATASET VALIDATION =====")
print("Master shape:", df_1s.shape)
print("Working shape:", work.shape)
print("Unique columns:", work.columns.is_unique)


# ============================================================
# 2. BUILD IST TIMESTAMPS
# ============================================================

work["ts"] = (
    pd.to_datetime(
        work["ts_sec"],
        unit="s",
        utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work.loc[
    (work["sec"] >= START_SEC)
    & (work["sec"] < END_SEC)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError(
        "Duplicate symbol x timestamp observations detected."
    )


# ============================================================
# 3. FILTER ELIGIBLE FULL SESSIONS
# ============================================================

sessions = (
    work.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_sec=("sec", "min"),
        last_sec=("sec", "max")
    )
    .reset_index()
)

sessions["coverage"] = (
    sessions["rows"] / (END_SEC - START_SEC)
)

sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (
        sessions["date"].astype(str)
        != "2026-09-28"
    )
)

print("\n===== SESSION COVERAGE =====")
print(sessions.to_string(index=False))

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    how="left",
    validate="many_to_one"
)

work = work.loc[work["eligible"]].copy()


# ============================================================
# 4. VALIDATE L1 QUOTES
#
# A valid quote must have:
#   bid > 0
#   ask > 0
#   ask >= bid
#
# We additionally track locked quotes (ask == bid).
# No assumption that locked quotes are invalid is made here.
# ============================================================

for col in ["mid", "bid_p1", "ask_p1"]:
    work[col] = pd.to_numeric(
        work[col], errors="coerce"
    )

work["quote_valid"] = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] >= work["bid_p1"])
)

work["locked_quote"] = (
    work["quote_valid"]
    & (work["ask_p1"] == work["bid_p1"])
)

work["crossed_quote"] = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] < work["bid_p1"])
)

work["quote_mid"] = (
    work["bid_p1"] + work["ask_p1"]
) / 2

work.loc[
    ~work["quote_valid"],
    "quote_mid"
] = np.nan

work["mid_difference"] = (
    work["mid"] - work["quote_mid"]
).abs()

work["spread_bps"] = (
    10000
    * (work["ask_p1"] - work["bid_p1"])
    / work["quote_mid"]
)

work.loc[
    ~work["quote_valid"],
    "spread_bps"
] = np.nan

work["log_quote_mid"] = np.log(
    work["quote_mid"]
)


# ============================================================
# 5. CONSECUTIVE ONE-SECOND RETURNS
#
# Reconstruct returns from valid L1 bid/ask quotes.
# Never cross session boundaries or timestamp gaps.
# ============================================================

g = work.groupby(
    ["symbol", "date"],
    sort=False
)

work["ret"] = g["log_quote_mid"].diff()

elapsed = g["ts_sec"].diff()

previous_valid = (
    g["quote_valid"].shift(1)
    .fillna(False)
    .astype(bool)
)

valid_transition = (
    elapsed.eq(1)
    & work["quote_valid"]
    & previous_valid
)

work.loc[
    ~valid_transition,
    "ret"
] = np.nan

work["ret_sq"] = work["ret"] ** 2

work["abs_ret"] = work["ret"].abs()

work["nonzero_move"] = (
    work["ret"].ne(0)
    .where(work["ret"].notna(), np.nan)
    .astype(float)
)


# ============================================================
# 6. ASSIGN EXACT NONOVERLAPPING IST INTERVALS
# ============================================================

work["interval"] = "excluded"

for name, begin, finish in INTERVALS:

    mask = (
        (work["sec"] >= START_SEC + begin)
        & (work["sec"] < START_SEC + finish)
    )

    work.loc[mask, "interval"] = name

work = work.loc[
    work["interval"] != "excluded"
].copy()


# ============================================================
# 7. QUOTE QUALITY AND RV BY INTERVAL
# ============================================================

records = []

for (symbol, date, interval), sub in work.groupby(
    ["symbol", "date", "interval"],
    sort=True
):

    duration = next(
        finish - begin
        for name, begin, finish in INTERVALS
        if name == interval
    )

    valid_returns = int(sub["ret"].notna().sum())

    rv = float(sub["ret_sq"].sum())

    n_moves = int(
        sub["nonzero_move"].sum()
    )

    largest_sq = sub["ret_sq"].max()

    if pd.isna(largest_sq):
        largest_sq = np.nan

    expected_returns = duration

    # At 09:00:00, no same-session predecessor exists.
    # Therefore first_30s has at most 29 valid returns.
    if interval == "first_30s":
        expected_returns = 29

    return_coverage = (
        valid_returns / expected_returns
    )

    valid = (
        return_coverage >= MIN_INTERVAL_COVERAGE
    )

    rv_rate = (
        rv / valid_returns
        if valid_returns > 0
        else np.nan
    )

    if valid_returns > 1:
        rv_rate_without_max = (
            (rv - largest_sq) / (valid_returns - 1)
        )
    else:
        rv_rate_without_max = np.nan

    largest_move_share = (
        largest_sq / rv
        if rv > 0
        else np.nan
    )

    records.append({
        "symbol": symbol,
        "date": date,
        "interval": interval,
        "duration": duration,
        "rows": len(sub),
        "expected_returns": expected_returns,
        "valid_returns": valid_returns,
        "return_coverage": return_coverage,
        "valid": valid,
        "valid_quote_fraction": (
            sub["quote_valid"].mean()
        ),
        "crossed_quote_count": int(
            sub["crossed_quote"].sum()
        ),
        "locked_quote_count": int(
            sub["locked_quote"].sum()
        ),
        "median_spread_bps": (
            sub["spread_bps"].median()
        ),
        "max_spread_bps": (
            sub["spread_bps"].max()
        ),
        "median_mid_difference": (
            sub["mid_difference"].median()
        ),
        "max_mid_difference": (
            sub["mid_difference"].max()
        ),
        "n_moves": n_moves,
        "rv": rv,
        "rv_rate": rv_rate,
        "largest_move_sq": largest_sq,
        "largest_move_share": largest_move_share,
        "rv_rate_without_max": rv_rate_without_max
    })

quality = pd.DataFrame(records)

# Invalidate comparisons for incomplete intervals.
for metric in [
    "rv_rate",
    "rv_rate_without_max",
    "largest_move_share"
]:
    quality.loc[
        ~quality["valid"],
        metric
    ] = np.nan

print("\n===== INTERVAL COVERAGE =====")

coverage_summary = (
    quality.groupby("interval")
    .agg(
        available_sessions=("date", "size"),
        valid_sessions=("valid", "sum"),
        median_return_coverage=(
            "return_coverage", "median"
        )
    )
    .reset_index()
)

print(
    coverage_summary.round(6).to_string(index=False)
)


# ============================================================
# 8. OPENING QUOTE QUALITY
# ============================================================

opening_quality = quality.loc[
    quality["interval"] == "first_30s"
].copy()

print("\n===== OPENING QUOTE QUALITY =====")

print(
    opening_quality[
        [
            "symbol",
            "date",
            "rows",
            "valid_returns",
            "valid_quote_fraction",
            "crossed_quote_count",
            "locked_quote_count",
            "median_spread_bps",
            "max_spread_bps",
            "max_mid_difference",
            "largest_move_share"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8g}"
    )
)


# ============================================================
# 9. IDENTIFY LARGEST FIRST-30S RETURN
# ============================================================

opening = work.loc[
    (work["interval"] == "first_30s")
    & work["ret_sq"].notna()
].copy()

if opening.empty:
    jumps = pd.DataFrame()

else:
    idx = (
        opening.groupby(["symbol", "date"])
        ["ret_sq"].idxmax()
    )

    jumps = opening.loc[
        idx,
        [
            "symbol",
            "date",
            "ts",
            "ret",
            "abs_ret",
            "ret_sq",
            "bid_p1",
            "ask_p1",
            "quote_mid",
            "spread_bps"
        ]
    ].sort_values(
        ["symbol", "date"]
    ).reset_index(drop=True)

print("\n===== LARGEST FIRST-30S PRICE MOVEMENT =====")

if not jumps.empty:
    print(
        jumps.to_string(
            index=False,
            float_format=lambda x: f"{x:.10g}"
        )
    )
else:
    print("No valid opening returns found.")


# ============================================================
# 10. ORIGINAL VS ROBUST OPENING RV
#
# For every day and interval:
#
# Original RV rate:
#   sum(r^2) / number of valid returns
#
# Trimmed RV rate:
#   [sum(r^2) - max(r^2)] / (n_valid - 1)
#
# Compare each interval with that day's rest-of-session.
# Both numerator and reference are treated consistently.
# ============================================================

pivot = quality.pivot_table(
    index=["symbol", "date"],
    columns="interval",
    values=["rv_rate", "rv_rate_without_max"]
)

robustness_records = []

for metric in [
    "rv_rate",
    "rv_rate_without_max"
]:

    for interval in [
        "first_30s",
        "next_30s",
        "minute_1_5",
        "minute_5_30"
    ]:

        if (
            (metric, interval) not in pivot.columns
            or (metric, "rest") not in pivot.columns
        ):
            continue

        matched = pd.DataFrame({
            "opening": pivot[(metric, interval)],
            "rest": pivot[(metric, "rest")]
        }).dropna()

        matched = matched.loc[
            (matched["opening"] >= 0)
            & (matched["rest"] > 0)
        ]

        for (symbol, date), row in matched.iterrows():

            robustness_records.append({
                "symbol": symbol,
                "date": date,
                "metric": metric,
                "interval": interval,
                "ratio": (
                    row["opening"] / row["rest"]
                )
            })

robustness = pd.DataFrame(robustness_records)

if robustness.empty:
    raise ValueError(
        "No valid opening/rest comparisons."
    )


# ============================================================
# 11. ROBUSTNESS SUMMARY — ALL DAYS
# ============================================================

summary = (
    robustness.groupby(["metric", "interval"])
    .agg(
        sessions=("ratio", "count"),
        median_ratio=("ratio", "median"),
        mean_ratio=("ratio", "mean"),
        fraction_above_one=(
            "ratio",
            lambda s: (s > 1).mean()
        ),
        min_ratio=("ratio", "min"),
        max_ratio=("ratio", "max")
    )
    .reset_index()
)

print("\n===== ROBUSTNESS: ALL ELIGIBLE DAYS =====")

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 12. EXCLUDE OCTOBER 12 OUTLIER
# ============================================================

without_outlier = robustness.loc[
    robustness["date"].astype(str)
    != "2026-10-12"
].copy()

summary_no_outlier = (
    without_outlier.groupby(["metric", "interval"])
    .agg(
        sessions=("ratio", "count"),
        median_ratio=("ratio", "median"),
        mean_ratio=("ratio", "mean"),
        fraction_above_one=(
            "ratio",
            lambda s: (s > 1).mean()
        ),
        min_ratio=("ratio", "min"),
        max_ratio=("ratio", "max")
    )
    .reset_index()
)

print("\n===== ROBUSTNESS: EXCLUDING 2026-10-12 =====")

print(
    summary_no_outlier.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 13. DAILY OPENING JUMP CONCENTRATION
# ============================================================

print("\n===== FIRST-30S JUMP CONCENTRATION =====")

print(
    opening_quality[
        [
            "symbol",
            "date",
            "rv",
            "largest_move_sq",
            "largest_move_share",
            "rv_rate",
            "rv_rate_without_max"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.10g}"
    )
)


# ============================================================
# 14. SAVE ALL RESULTS
# ============================================================

calendar_exp12_data = work
calendar_exp12_quality = quality
calendar_exp12_jumps = jumps
calendar_exp12_robustness = robustness
calendar_exp12_summary = summary
calendar_exp12_summary_no_outlier = summary_no_outlier

print("\n===== EXPERIMENT 12 COMPLETE =====")
print("Original df_1s remains unchanged.")
print("No calendar features have been validated yet.")



===== DATASET VALIDATION =====
Master shape: (495578, 65)
Working shape: (495578, 5)
Unique columns: True

===== SESSION COVERAGE =====
        symbol       date  rows  first_sec  last_sec  coverage  eligible
USDINR26OCTFUT 2026-09-30 28749      32400     61148  0.998229      True
USDINR26OCTFUT 2026-10-01 28402      32780     61181  0.986181      True
USDINR26OCTFUT 2026-10-05 28767      32400     61166  0.998854      True
USDINR26OCTFUT 2026-10-06 28755      32400     61154  0.998437      True
USDINR26OCTFUT 2026-10-07 28752      32400     61151  0.998333      True
USDINR26OCTFUT 2026-10-08 28749      32400     61148  0.998229      True
USDINR26OCTFUT 2026-10-09 28748      32400     61147  0.998194      True
USDINR26OCTFUT 2026-10-12 28754      32400     61153  0.998403      True
USDINR26OCTFUT 2026-10-13 28793      32400     61192  0.999757      True
USDINR26OCTFUT 2026-10-14 28792      32400     61191  0.999722      True
USDINR26OCTFUT 2026-10-15 28797      32400     61196  0.9998

In [29]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 13
# OPENING LIQUIDITY AND EXECUTION-RISK DIAGNOSTICS
#
# INPUT: df_1s
# DEPENDENCIES: pandas, numpy
#
# OUTPUTS:
# calendar_exp13_data
# calendar_exp13_daily
# calendar_exp13_summary
# calendar_exp13_quality
#
# No modification to df_1s.
# ============================================================

START = 9 * 3600
END = 17 * 3600
MIN_DAY_COVERAGE = 0.97
MIN_INTERVAL_COVERAGE = 0.90
HORIZONS = [30, 60, 300]

INTERVALS = [
    ("first_30s", 0, 30),
    ("next_30s", 30, 60),
    ("minute_1_5", 60, 300),
    ("minute_5_30", 300, 1800),
    ("rest", 1800, 28800)
]

depth_cols = [
    f"{side}_q{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
]

required = [
    "symbol", "ts_sec",
    "bid_p1", "ask_p1"
] + depth_cols

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate master column names")

missing = [c for c in required if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

# ------------------------------------------------------------
# 1. TIMESTAMPS
# ------------------------------------------------------------

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows")

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["eligible"] = (
    (sessions["rows"] / (END - START)
     >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ------------------------------------------------------------
# 2. VALIDATE QUOTES AND DEPTH
# ------------------------------------------------------------

for c in ["bid_p1", "ask_p1"] + depth_cols:
    work[c] = pd.to_numeric(
        work[c], errors="coerce"
    )

work["valid_quote"] = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] >= work["bid_p1"])
)

work["mid"] = (
    work["bid_p1"] + work["ask_p1"]
) / 2

work.loc[~work["valid_quote"], "mid"] = np.nan

work["spread_bps"] = (
    10000
    * (work["ask_p1"] - work["bid_p1"])
    / work["mid"]
)

q = work[depth_cols].where(
    work[depth_cols] >= 0
)

work["depth_l5"] = q.sum(
    axis=1,
    min_count=len(depth_cols)
)

work.loc[
    ~work["valid_quote"],
    ["spread_bps", "depth_l5"]
] = np.nan

# ------------------------------------------------------------
# 3. EXACT FORWARD MID-PRICE MOVEMENTS
#
# Calculate on the complete eligible session grid
# before filtering by interval.
# ------------------------------------------------------------

grid = work[
    ["symbol", "date", "ts_sec", "mid"]
].copy()

for h in HORIZONS:

    future = grid.copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    work[f"abs_forward_{h}"] = (
        np.log(
            work[f"future_mid_{h}"] / work["mid"]
        ).abs()
    )

# ------------------------------------------------------------
# 4. ASSIGN CLOCK INTERVALS
# ------------------------------------------------------------

work["interval"] = "excluded"

for name, begin, finish in INTERVALS:

    mask = (
        (work["sec"] >= START + begin)
        & (work["sec"] < START + finish)
    )

    work.loc[mask, "interval"] = name

work = work[
    work["interval"] != "excluded"
].copy()

# ------------------------------------------------------------
# 5. DAILY INTERVAL STATISTICS
# ------------------------------------------------------------

daily = (
    work.groupby(["symbol", "date", "interval"])
    .agg(
        observations=("ts_sec", "size"),
        valid_quotes=("valid_quote", "sum"),
        median_spread=("spread_bps", "median"),
        p90_spread=(
            "spread_bps",
            lambda s: s.quantile(.90)
        ),
        median_depth=("depth_l5", "median"),
        p10_depth=(
            "depth_l5",
            lambda s: s.quantile(.10)
        ),
        move_30=(
            "abs_forward_30", "median"
        ),
        move_60=(
            "abs_forward_60", "median"
        ),
        move_300=(
            "abs_forward_300", "median"
        )
    )
    .reset_index()
)

duration_map = {
    name: finish - begin
    for name, begin, finish in INTERVALS
}

daily["expected_seconds"] = (
    daily["interval"].map(duration_map)
)

daily["coverage"] = (
    daily["valid_quotes"] / daily["expected_seconds"]
)

daily["valid_interval"] = (
    daily["coverage"] >= MIN_INTERVAL_COVERAGE
)

metrics = [
    "median_spread",
    "p90_spread",
    "median_depth",
    "p10_depth",
    "move_30",
    "move_60",
    "move_300"
]

daily.loc[
    ~daily["valid_interval"],
    metrics
] = np.nan

# ------------------------------------------------------------
# 6. WITHIN-DAY OPENING / REST RATIOS
# ------------------------------------------------------------

records = []

for metric in metrics:

    pivot = daily.pivot_table(
        index=["symbol", "date"],
        columns="interval",
        values=metric
    )

    if "rest" not in pivot.columns:
        continue

    for interval in [
        "first_30s",
        "next_30s",
        "minute_1_5",
        "minute_5_30"
    ]:

        if interval not in pivot.columns:
            continue

        paired = pd.DataFrame({
            "opening": pivot[interval],
            "rest": pivot["rest"]
        }).dropna()

        paired = paired[
            paired["rest"] > 0
        ]

        paired["ratio"] = (
            paired["opening"] / paired["rest"]
        )

        for (symbol, date), row in paired.iterrows():
            records.append({
                "symbol": symbol,
                "date": date,
                "metric": metric,
                "interval": interval,
                "ratio": row["ratio"]
            })

comparison = pd.DataFrame(records)

summary = (
    comparison.groupby(["metric", "interval"])
    .agg(
        sessions=("ratio", "count"),
        median_ratio=("ratio", "median"),
        mean_ratio=("ratio", "mean"),
        fraction_above_one=(
            "ratio",
            lambda s: (s > 1).mean()
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. REPORTS
# ------------------------------------------------------------

print("\n===== INTERVAL QUOTE COVERAGE =====")

quality = (
    daily.groupby("interval")
    .agg(
        sessions=("date", "size"),
        valid_sessions=("valid_interval", "sum"),
        median_quote_coverage=("coverage", "median")
    )
    .reset_index()
)

print(
    quality.round(5).to_string(index=False)
)

print("\n===== OPENING LIQUIDITY RATIOS =====")

liquidity_summary = summary[
    summary["metric"].isin([
        "median_spread",
        "p90_spread",
        "median_depth",
        "p10_depth"
    ])
]

print(
    liquidity_summary.round(5).to_string(index=False)
)

print("\n===== FORWARD MID-MOVEMENT RATIOS =====")

movement_summary = summary[
    summary["metric"].isin([
        "move_30", "move_60", "move_300"
    ])
]

print(
    movement_summary.round(5).to_string(index=False)
)

print("\n===== DAILY FIRST-30S LIQUIDITY =====")

opening = daily[
    daily["interval"] == "first_30s"
][[
    "symbol", "date",
    "median_spread", "p90_spread",
    "median_depth", "p10_depth",
    "move_30", "move_60", "move_300"
]]

print(
    opening.to_string(
        index=False,
        float_format=lambda v: f"{v:.8g}"
    )
)

# ------------------------------------------------------------
# 8. PRESERVE RESULTS
# ------------------------------------------------------------

calendar_exp13_data = work
calendar_exp13_daily = daily
calendar_exp13_summary = summary
calendar_exp13_quality = quality

print("\n===== EXPERIMENT 13 COMPLETE =====")
print("Master df_1s unchanged.")
print("No new feature validated.")



===== INTERVAL QUOTE COVERAGE =====
   interval  sessions  valid_sessions  median_quote_coverage
  first_30s        12              12                 1.0000
 minute_1_5        14              12                 1.0000
minute_5_30        16              16                 1.0000
   next_30s        12              12                 1.0000
       rest        16              16                 0.9987

===== OPENING LIQUIDITY RATIOS =====
       metric    interval  sessions  median_ratio  mean_ratio  fraction_above_one
 median_depth   first_30s        12       0.72420     0.90954             0.16667
 median_depth  minute_1_5        12       0.67262     0.80666             0.25000
 median_depth minute_5_30        16       0.92257     0.97071             0.43750
 median_depth    next_30s        12       0.67886     0.82293             0.16667
median_spread   first_30s        12       2.16629     3.46340             0.83333
median_spread  minute_1_5        12       0.99975     1.08287      

In [30]:

import numpy as np
import pandas as pd

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 14
# CHRONOLOGICAL INCREMENTAL OPENING-TIME VALUE
#
# Question:
# Does opening clock time improve forecasts of future
# absolute mid-price returns beyond causal market state?
#
# Input: df_1s
# Dependencies: pandas, numpy
#
# Outputs:
# calendar_exp14_data
# calendar_exp14_daily
# calendar_exp14_summary
# calendar_exp14_windows
#
# Master df_1s is never modified.
# ============================================================

START = 9 * 3600
END = 17 * 3600
HORIZONS = [30, 60, 300]

MIN_DAY_COVERAGE = 0.97
MIN_TRAIN_DAYS = 5
RIDGE_PENALTY = 0.01

depth_cols = [
    f"{side}_q{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
]

required = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
] + depth_cols

if not df_1s.columns.is_unique:
    raise ValueError("df_1s has duplicate columns")

missing = [c for c in required if c not in df_1s.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

# ============================================================
# 1. TIMESTAMPS AND ELIGIBLE SESSIONS
# ============================================================

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp rows")

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["coverage"] = sessions["rows"] / (END - START)

sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ============================================================
# 2. QUOTE VALIDITY, MID, SPREAD, DEPTH
# ============================================================

numeric_cols = ["bid_p1", "ask_p1"] + depth_cols

for col in numeric_cols:
    work[col] = pd.to_numeric(
        work[col], errors="coerce"
    )

work["valid_quote"] = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] >= work["bid_p1"])
)

work["mid"] = (
    work["bid_p1"] + work["ask_p1"]
) / 2

work.loc[
    ~work["valid_quote"], "mid"
] = np.nan

work["spread_bps"] = (
    10000
    * (work["ask_p1"] - work["bid_p1"])
    / work["mid"]
)

q = work[depth_cols].where(
    work[depth_cols] >= 0
)

work["depth_l5"] = q.sum(
    axis=1, min_count=len(depth_cols)
)

work.loc[
    ~work["valid_quote"],
    ["spread_bps", "depth_l5"]
] = np.nan

work["log_mid"] = np.log(work["mid"])

# ============================================================
# 3. STRICTLY CAUSAL RETURN HISTORY
# ============================================================

group_keys = ["symbol", "date"]

g = work.groupby(group_keys, sort=False)

elapsed = g["ts_sec"].diff()

work["ret_1s"] = g["log_mid"].diff()

work.loc[
    elapsed.ne(1), "ret_1s"
] = np.nan

# Do not roll over gaps.
work["run_id"] = (
    elapsed.ne(1)
    .groupby([work["symbol"], work["date"]])
    .cumsum()
)

run_keys = ["symbol", "date", "run_id"]

# A missing return is not treated as a valid zero return.
# Rolling sums include only available consecutive returns.
work["ret_sq"] = work["ret_1s"] ** 2

work["move"] = (
    work["ret_1s"].ne(0)
    .where(work["ret_1s"].notna(), np.nan)
    .astype(float)
)

runs = work.groupby(run_keys, sort=False)

# Use partial histories near the open.
# Otherwise a 300s warmup would discard the phenomenon
# we are explicitly trying to investigate.
for w in [30, 300]:

    work[f"rv_{w}"] = runs["ret_sq"].transform(
        lambda s: s.rolling(
            w, min_periods=1
        ).sum()
    )

    work[f"n_valid_{w}"] = runs["ret_sq"].transform(
        lambda s: s.rolling(
            w, min_periods=1
        ).count()
    )

work["move_rate_30"] = runs["move"].transform(
    lambda s: s.rolling(
        30, min_periods=1
    ).mean()
)

work["largest_sq_300"] = runs["ret_sq"].transform(
    lambda s: s.rolling(
        300, min_periods=1
    ).max()
)

work["jump_concentration_300"] = (
    work["largest_sq_300"]
    / work["rv_300"].where(work["rv_300"] > 0)
)

# Standardized transformations.
# These scaling constants are fixed, not learned
# from the evaluation period.
work["log_rv_30"] = np.log1p(
    np.maximum(work["rv_30"], 0) * 1e8
)

work["log_rv_300"] = np.log1p(
    np.maximum(work["rv_300"], 0) * 1e8
)

work["log_depth"] = np.log1p(
    work["depth_l5"].clip(lower=0)
)

# Include history length in both models so that the
# calendar model is not merely learning partial-window
# warmup effects.
work["history_300_fraction"] = (
    work["n_valid_300"] / 300.0
)

# ============================================================
# 4. PREDEFINED OPENING FEATURES
# ============================================================

work["seconds_since_open"] = work["sec"] - START

work["open_first_30s"] = (
    work["seconds_since_open"] < 30
).astype(int)

work["open_next_30s"] = (
    (work["seconds_since_open"] >= 30)
    & (work["seconds_since_open"] < 60)
).astype(int)

work["open_1_5m"] = (
    (work["seconds_since_open"] >= 60)
    & (work["seconds_since_open"] < 300)
).astype(int)

work["open_5_30m"] = (
    (work["seconds_since_open"] >= 300)
    & (work["seconds_since_open"] < 1800)
).astype(int)

work["clock_window"] = np.select(
    [
        work["open_first_30s"] == 1,
        work["open_next_30s"] == 1,
        work["open_1_5m"] == 1,
        work["open_5_30m"] == 1
    ],
    [
        "first_30s",
        "next_30s",
        "minute_1_5",
        "minute_5_30"
    ],
    default="rest"
)

base_features = [
    "log_rv_30",
    "log_rv_300",
    "move_rate_30",
    "jump_concentration_300",
    "spread_bps",
    "log_depth",
    "history_300_fraction"
]

clock_features = [
    "open_first_30s",
    "open_next_30s",
    "open_1_5m",
    "open_5_30m"
]

extended_features = base_features + clock_features

# ============================================================
# 5. EXACT FUTURE RETURNS
# ============================================================

price_grid = work[
    ["symbol", "date", "ts_sec", "mid"]
].copy()

for h in HORIZONS:

    future = price_grid.copy()
    future["ts_sec"] -= h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    work[f"target_{h}"] = (
        10000
        * np.log(
            work[f"future_mid_{h}"] / work["mid"]
        ).abs()
    )

# ============================================================
# 6. TRAIN-ONLY RIDGE REGRESSION
#
# All preprocessing (median fill, scaling, coefficients)
# is learned from past training sessions only.
#
# No sklearn dependency.
# ============================================================

def fit_ridge_predict(train, test, features, target):

    x_train = train[features].to_numpy(dtype=float)
    x_test = test[features].to_numpy(dtype=float)

    y_train = train[target].to_numpy(dtype=float)

    x_train[~np.isfinite(x_train)] = np.nan
    x_test[~np.isfinite(x_test)] = np.nan

    # Training-only imputation
    medians = np.nanmedian(x_train, axis=0)
    medians = np.where(np.isfinite(medians), medians, 0.0)

    for j in range(x_train.shape[1]):
        x_train[
            np.isnan(x_train[:, j]), j
        ] = medians[j]

        x_test[
            np.isnan(x_test[:, j]), j
        ] = medians[j]

    # Training-only standardization
    means = x_train.mean(axis=0)
    stds = x_train.std(axis=0)

    stds = np.where(stds > 1e-12, stds, 1.0)

    x_train = (x_train - means) / stds
    x_test = (x_test - means) / stds

    y_mean = y_train.mean()
    y_centered = y_train - y_mean

    n_features = x_train.shape[1]
    n_samples = x_train.shape[0]

    penalty = (
        RIDGE_PENALTY
        * n_samples
        * np.eye(n_features)
    )

    coefficients = np.linalg.solve(
        x_train.T @ x_train + penalty,
        x_train.T @ y_centered
    )

    prediction = (
        y_mean + x_test @ coefficients
    )

    # Absolute-return forecasts cannot be negative.
    return np.maximum(prediction, 0.0)


# ============================================================
# 7. EXPANDING-WINDOW CHRONOLOGICAL EVALUATION
#
# Train on past sessions.
# Predict the next session.
# Refit separately for every evaluation date.
# ============================================================

all_dates = sorted(work["date"].unique())

print("\n===== CHRONOLOGICAL SESSIONS =====")
print("Total eligible sessions:", len(all_dates))
print("Minimum training sessions:", MIN_TRAIN_DAYS)

prediction_records = []

for i in range(MIN_TRAIN_DAYS, len(all_dates)):

    train_dates = all_dates[:i]
    test_date = all_dates[i]

    train_all = work[
        work["date"].isin(train_dates)
    ]

    test_all = work[
        work["date"] == test_date
    ]

    print(
        "Training:", len(train_dates),
        "sessions | Testing:", test_date
    )

    for h in HORIZONS:

        target = f"target_{h}"

        train = train_all[
            train_all[target].notna()
            & np.isfinite(train_all[target])
        ].copy()

        test = test_all[
            test_all[target].notna()
            & np.isfinite(test_all[target])
        ].copy()

        if train.empty or test.empty:
            continue

        pred_base = fit_ridge_predict(
            train, test, base_features, target
        )

        pred_clock = fit_ridge_predict(
            train, test, extended_features, target
        )

        out = test[
            ["symbol", "date", "ts_sec", "clock_window"]
        ].copy()

        out["horizon"] = h
        out["actual_bps"] = test[target].to_numpy()
        out["pred_base"] = pred_base
        out["pred_clock"] = pred_clock

        out["ae_base"] = np.abs(
            out["actual_bps"] - out["pred_base"]
        )

        out["ae_clock"] = np.abs(
            out["actual_bps"] - out["pred_clock"]
        )

        prediction_records.append(out)

if not prediction_records:
    raise ValueError(
        "No valid chronological evaluation sessions."
    )

predictions = pd.concat(
    prediction_records, ignore_index=True
)

# ============================================================
# 8. DAY-LEVEL FORECAST COMPARISON
# ============================================================

daily = (
    predictions.groupby(
        ["symbol", "date", "horizon"]
    )
    .agg(
        observations=("actual_bps", "size"),
        mae_base=("ae_base", "mean"),
        mae_clock=("ae_clock", "mean")
    )
    .reset_index()
)

daily["improvement_pct"] = (
    100
    * (1 - daily["mae_clock"] / daily["mae_base"])
)

print("\n===== DAILY OOS PERFORMANCE =====")

print(
    daily.round(6).to_string(index=False)
)

# ============================================================
# 9. AGGREGATE OOS PERFORMANCE
# ============================================================

summary_rows = []

for h in HORIZONS:

    p = predictions[
        predictions["horizon"] == h
    ]

    d = daily[
        daily["horizon"] == h
    ]

    mae_base = p["ae_base"].mean()
    mae_clock = p["ae_clock"].mean()

    summary_rows.append({
        "horizon": h,
        "sessions": len(d),
        "observations": len(p),
        "base_mae_bps": mae_base,
        "clock_mae_bps": mae_clock,
        "improvement_pct": (
            100 * (1 - mae_clock / mae_base)
        ),
        "winning_days": int(
            (d["improvement_pct"] > 0).sum()
        ),
        "median_daily_improvement_pct": (
            d["improvement_pct"].median()
        )
    })

summary = pd.DataFrame(summary_rows)

print("\n===== AGGREGATE OOS RESULTS =====")
print(summary.round(6).to_string(index=False))

# ============================================================
# 10. CLOCK-WINDOW DIAGNOSTICS
#
# Where is the incremental model helping or hurting?
# ============================================================

window_summary = (
    predictions.groupby(
        ["horizon", "clock_window"]
    )
    .agg(
        observations=("actual_bps", "size"),
        actual_mean_bps=("actual_bps", "mean"),
        base_mae_bps=("ae_base", "mean"),
        clock_mae_bps=("ae_clock", "mean")
    )
    .reset_index()
)

window_summary["improvement_pct"] = (
    100
    * (
        1
        - window_summary["clock_mae_bps"]
        / window_summary["base_mae_bps"]
    )
)

order = [
    "first_30s",
    "next_30s",
    "minute_1_5",
    "minute_5_30",
    "rest"
]

window_summary["clock_window"] = pd.Categorical(
    window_summary["clock_window"],
    categories=order,
    ordered=True
)

window_summary = window_summary.sort_values(
    ["horizon", "clock_window"]
)

print("\n===== OOS PERFORMANCE BY CLOCK WINDOW =====")
print(
    window_summary.round(6).to_string(index=False)
)

# ============================================================
# 11. SAVE OUTPUTS
# ============================================================

calendar_exp14_data = predictions
calendar_exp14_daily = daily
calendar_exp14_summary = summary
calendar_exp14_windows = window_summary

print("\n===== EXPERIMENT 14 COMPLETE =====")
print("Master df_1s remains unchanged.")
print("No calendar features promoted to core.")



===== CHRONOLOGICAL SESSIONS =====
Total eligible sessions: 16
Minimum training sessions: 5
Training: 5 sessions | Testing: 2026-10-05
Training: 6 sessions | Testing: 2026-10-06
Training: 7 sessions | Testing: 2026-10-07
Training: 8 sessions | Testing: 2026-10-08
Training: 9 sessions | Testing: 2026-10-09
Training: 10 sessions | Testing: 2026-10-12
Training: 11 sessions | Testing: 2026-10-13
Training: 12 sessions | Testing: 2026-10-14
Training: 13 sessions | Testing: 2026-10-15
Training: 14 sessions | Testing: 2026-10-16
Training: 15 sessions | Testing: 2026-10-19

===== DAILY OOS PERFORMANCE =====
        symbol       date  horizon  observations  mae_base  mae_clock  improvement_pct
USDINR26OCTFUT 2026-10-05       30         28737  0.235321   0.234633         0.292626
USDINR26OCTFUT 2026-10-05       60         28707  0.296881   0.295743         0.383423
USDINR26OCTFUT 2026-10-05      300         28467  0.551071   0.555245        -0.757430
USDINR26OCTFUT 2026-10-06       30         28

In [31]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 15
# CLOCK FEATURE: OOS STABILITY AND LOSS DIAGNOSTICS
#
# Input:
#   calendar_exp14_data
#
# Output:
#   calendar_exp15_daily
#   calendar_exp15_windows
#   calendar_exp15_stress
#   calendar_exp15_sensitivity
#
# Does not retrain models or modify earlier results.
# ============================================================

required = [
    "symbol", "date", "horizon", "clock_window",
    "actual_bps", "pred_base", "pred_clock",
    "ae_base", "ae_clock"
]

missing = [
    c for c in required
    if c not in calendar_exp14_data.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

z = calendar_exp14_data[required].copy()

z["gain"] = z["ae_base"] - z["ae_clock"]

# Positive gain means adding clock features helped.

z["squared_error_base"] = (
    z["actual_bps"] - z["pred_base"]
) ** 2

z["squared_error_clock"] = (
    z["actual_bps"] - z["pred_clock"]
) ** 2

z["sq_gain"] = (
    z["squared_error_base"]
    - z["squared_error_clock"]
)

# ============================================================
# 1. DAILY PERFORMANCE BY CLOCK WINDOW
# ============================================================

daily = (
    z.groupby(
        ["symbol", "date", "horizon", "clock_window"]
    )
    .agg(
        observations=("gain", "size"),
        mae_base=("ae_base", "mean"),
        mae_clock=("ae_clock", "mean"),
        mean_gain=("gain", "mean"),
        mse_gain=("sq_gain", "mean"),
        actual_mean=("actual_bps", "mean")
    )
    .reset_index()
)

daily["improvement_pct"] = (
    100 * daily["mean_gain"]
    / daily["mae_base"].where(daily["mae_base"] > 0)
)

print("\n===== DAILY OPENING MODEL PERFORMANCE =====")

opening_days = daily[
    daily["clock_window"].isin([
        "first_30s",
        "next_30s",
        "minute_1_5",
        "minute_5_30"
    ])
]

print(
    opening_days[
        [
            "date", "horizon", "clock_window",
            "observations", "mae_base",
            "mae_clock", "improvement_pct"
        ]
    ].round(5).to_string(index=False)
)

# ============================================================
# 2. WINDOW STABILITY AT THE DAY LEVEL
# ============================================================

windows = (
    daily.groupby(["horizon", "clock_window"])
    .agg(
        sessions=("mean_gain", "count"),
        mean_daily_gain=("mean_gain", "mean"),
        median_daily_gain=("mean_gain", "median"),
        winning_fraction=(
            "mean_gain", lambda s: (s > 0).mean()
        ),
        median_daily_improvement_pct=(
            "improvement_pct", "median"
        ),
        worst_day_gain=("mean_gain", "min"),
        best_day_gain=("mean_gain", "max")
    )
    .reset_index()
)

print("\n===== DAY-LEVEL CLOCK WINDOW STABILITY =====")

print(
    windows.round(6).to_string(index=False)
)

# ============================================================
# 3. HIGH-MOVEMENT / STRESS OBSERVATIONS
#
# Threshold fitted from earlier evaluation dates only.
#
# This is retrospective loss stratification using the
# realized target; it is NOT a live trading feature.
# ============================================================

stress_records = []

for h, hdf in z.groupby("horizon"):

    for date in sorted(hdf["date"].unique()):

        current = hdf[hdf["date"] == date].copy()

        prior = hdf[hdf["date"] < date]

        # No prior OOS data for the first evaluated date.
        if prior.empty:
            continue

        threshold = prior["actual_bps"].quantile(0.95)

        current["stress"] = np.where(
            current["actual_bps"] >= threshold,
            "high_movement",
            "ordinary"
        )

        current["threshold_bps"] = threshold

        for regime, sub in current.groupby("stress"):

            stress_records.append({
                "date": date,
                "horizon": h,
                "regime": regime,
                "observations": len(sub),
                "mean_base_mae": sub["ae_base"].mean(),
                "mean_clock_mae": sub["ae_clock"].mean(),
                "mean_gain": sub["gain"].mean(),
                "mean_sq_gain": sub["sq_gain"].mean(),
                "threshold_bps": threshold
            })

stress_daily = pd.DataFrame(stress_records)

stress_summary = (
    stress_daily.groupby(["horizon", "regime"])
    .agg(
        sessions=("mean_gain", "count"),
        mean_gain=("mean_gain", "mean"),
        median_gain=("mean_gain", "median"),
        winning_fraction=(
            "mean_gain", lambda s: (s > 0).mean()
        ),
        mean_sq_gain=("mean_sq_gain", "mean")
    )
    .reset_index()
)

print("\n===== PERFORMANCE DURING LARGE PRICE MOVEMENTS =====")

print(
    stress_summary.round(6).to_string(index=False)
)

# ============================================================
# 4. LEAVE-ONE-SESSION-OUT SENSITIVITY
#
# Recalculate pooled improvement excluding each date.
# This does NOT refit the model.
# ============================================================

sensitivity_records = []

for h, sub in z.groupby("horizon"):

    dates = sorted(sub["date"].unique())

    for excluded_date in dates:

        other = sub[
            sub["date"] != excluded_date
        ]

        baseline_mae = other["ae_base"].mean()
        clock_mae = other["ae_clock"].mean()

        sensitivity_records.append({
            "horizon": h,
            "excluded_date": excluded_date,
            "remaining_sessions": other["date"].nunique(),
            "improvement_pct": (
                100 * (1 - clock_mae / baseline_mae)
            )
        })

sensitivity = pd.DataFrame(sensitivity_records)

print("\n===== LEAVE-ONE-DAY-OUT SENSITIVITY =====")

print(
    sensitivity.round(6).to_string(index=False)
)

print("\n===== SENSITIVITY SUMMARY =====")

print(
    sensitivity.groupby("horizon")["improvement_pct"]
    .agg(["min", "median", "max"])
    .round(6)
    .to_string()
)

# ============================================================
# 5. EXCLUDE OCTOBER 12
# ============================================================

no_oct12 = z[
    z["date"].astype(str) != "2026-10-12"
]

print("\n===== EXCLUDING OCTOBER 12 =====")

for h, sub in no_oct12.groupby("horizon"):

    base = sub["ae_base"].mean()
    clock = sub["ae_clock"].mean()

    print(
        f"{h}s | "
        f"Base MAE={base:.6f} | "
        f"Clock MAE={clock:.6f} | "
        f"Improvement={100 * (1 - clock/base):+.4f}%"
    )

# ============================================================
# 6. SAVE RESULTS
# ============================================================

calendar_exp15_daily = daily
calendar_exp15_windows = windows
calendar_exp15_stress = stress_summary
calendar_exp15_sensitivity = sensitivity

print("\n===== EXPERIMENT 15 COMPLETE =====")
print("No model retraining.")
print("No new features validated.")



===== DAILY OPENING MODEL PERFORMANCE =====
      date  horizon clock_window  observations  mae_base  mae_clock  improvement_pct
2026-10-05       30    first_30s            30   2.56339    2.06779         19.33388
2026-10-05       30   minute_1_5           240   0.42756    0.41683          2.50832
2026-10-05       30  minute_5_30          1500   0.28111    0.28743         -2.24971
2026-10-05       30     next_30s            30   0.59764    0.49457         17.24566
2026-10-05       60    first_30s            30   1.98611    1.44018         27.48754
2026-10-05       60   minute_1_5           240   0.50759    0.46716          7.96483
2026-10-05       60  minute_5_30          1500   0.33398    0.34991         -4.77081
2026-10-05       60     next_30s            30   0.79408    0.58066         26.87689
2026-10-05      300    first_30s            30   1.09381    1.08991          0.35577
2026-10-05      300   minute_1_5           240   1.04483    1.28856        -23.32648
2026-10-05      300 

In [32]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 16
# INTRADAY SPREAD DISTRIBUTION AND PERSISTENCE
#
# Input: df_1s
# Dependencies: pandas, numpy
#
# Outputs:
#   calendar_exp16_data
#   calendar_exp16_bins
#   calendar_exp16_clock_profile
#   calendar_exp16_period_daily
#   calendar_exp16_period_summary
#   calendar_exp16_persistence_daily
#   calendar_exp16_persistence_summary
#
# No new feature is validated by this experiment.
# ============================================================

START = 9 * 3600
END = 17 * 3600
BIN_SECONDS = 300

HORIZONS = [30, 60, 300]

MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95

LOOKBACK = 1800
MIN_TRAILING = 300

# Predefined clock windows
PERIODS = [
    ("opening_0_5m", 0, 300),
    ("opening_5_30m", 300, 1800),
    ("morning", 1800, 7200),
    ("midday", 7200, 14400),
    ("afternoon", 14400, 21600),
    ("late", 21600, 28800)
]

# ============================================================
# 1. LOAD AND VALIDATE MASTER DATA
# ============================================================

required = [
    "symbol",
    "ts_sec",
    "bid_p1",
    "ask_p1"
]

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate columns in df_1s")

missing = [
    c for c in required
    if c not in df_1s.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"],
    errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

work["ts"] = (
    pd.to_datetime(
        work["ts_sec"],
        unit="s",
        utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp rows")

# ============================================================
# 2. SELECT ELIGIBLE SESSIONS
# ============================================================

sessions = (
    work.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_sec=("sec", "min"),
        last_sec=("sec", "max")
    )
    .reset_index()
)

sessions["coverage"] = (
    sessions["rows"] / (END - START)
)

sessions["eligible"] = (
    (sessions["coverage"] >= MIN_DAY_COVERAGE)
    & (
        sessions["date"].astype(str)
        != "2026-09-28"
    )
)

print("\n===== SESSION INVENTORY =====")
print(sessions.to_string(index=False))

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    how="left",
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ============================================================
# 3. RECONSTRUCT L1 SPREAD
# ============================================================

for col in ["bid_p1", "ask_p1"]:
    work[col] = pd.to_numeric(
        work[col],
        errors="coerce"
    )

work["valid_quote"] = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] >= work["bid_p1"])
)

work["locked_quote"] = (
    work["valid_quote"]
    & (work["ask_p1"] == work["bid_p1"])
)

work["crossed_quote"] = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] < work["bid_p1"])
)

mid = (
    work["bid_p1"] + work["ask_p1"]
) / 2

work["spread_bps"] = (
    10000
    * (work["ask_p1"] - work["bid_p1"])
    / mid
)

work.loc[
    ~work["valid_quote"],
    "spread_bps"
] = np.nan

work["bin"] = (
    (work["sec"] - START) // BIN_SECONDS
).astype(int)

work["clock_period"] = "excluded"

for name, begin, finish in PERIODS:

    mask = (
        (work["sec"] >= START + begin)
        & (work["sec"] < START + finish)
    )

    work.loc[mask, "clock_period"] = name

print("\n===== SPREAD DATA QUALITY =====")
print(
    "Valid quote fraction:",
    round(work["valid_quote"].mean(), 6)
)
print(
    "Crossed quotes:",
    int(work["crossed_quote"].sum())
)
print(
    "Locked quotes:",
    int(work["locked_quote"].sum())
)
print(
    "Missing spread fraction:",
    round(work["spread_bps"].isna().mean(), 6)
)

print("\n===== POOLED SPREAD DISTRIBUTION (BPS) =====")
print(
    work["spread_bps"].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.90, 0.95, 0.99
        ]
    ).round(6).to_string()
)

# ============================================================
# 4. FIVE-MINUTE SPREAD DISTRIBUTIONS
# ============================================================

bins = (
    work.groupby(["symbol", "date", "bin"])
    .agg(
        rows=("ts_sec", "size"),
        valid_spreads=("spread_bps", "count"),
        median_spread=("spread_bps", "median"),
        mean_spread=("spread_bps", "mean"),
        p90_spread=(
            "spread_bps",
            lambda s: s.quantile(0.90)
        ),
        p99_spread=(
            "spread_bps",
            lambda s: s.quantile(0.99)
        ),
        max_spread=("spread_bps", "max")
    )
    .reset_index()
)

bins["coverage"] = (
    bins["valid_spreads"] / BIN_SECONDS
)

bins = bins[
    bins["coverage"] >= MIN_BIN_COVERAGE
].copy()

# Normalize within each session only for
# retrospective comparison of intraday shape.

for metric in [
    "median_spread",
    "p90_spread",
    "p99_spread"
]:

    daily_reference = (
        bins.groupby(["symbol", "date"])[metric]
        .transform("median")
    )

    bins[f"{metric}_relative"] = (
        bins[metric]
        / daily_reference.where(daily_reference > 0)
    )

profile = (
    bins.groupby("bin")
    .agg(
        sessions=("date", "size"),
        median_spread_bps=("median_spread", "median"),
        median_p90_bps=("p90_spread", "median"),
        median_p99_bps=("p99_spread", "median"),
        relative_median_spread=(
            "median_spread_relative", "median"
        ),
        relative_p90_spread=(
            "p90_spread_relative", "median"
        ),
        relative_p99_spread=(
            "p99_spread_relative", "median"
        )
    )
    .reset_index()
)

def clock_label(b):
    seconds = START + int(b) * BIN_SECONDS
    return (
        f"{seconds // 3600:02d}:"
        f"{(seconds % 3600) // 60:02d}"
    )

profile["clock"] = profile["bin"].map(clock_label)

print("\n===== FIVE-MINUTE CLOCK PROFILE =====")

selected_bins = list(range(0, 96, 6))

print(
    profile[
        profile["bin"].isin(selected_bins)
    ][[
        "clock",
        "sessions",
        "median_spread_bps",
        "median_p90_bps",
        "median_p99_bps",
        "relative_median_spread",
        "relative_p90_spread"
    ]].round(5).to_string(index=False)
)

# ============================================================
# 5. DAILY SPREAD STATISTICS BY CLOCK PERIOD
# ============================================================

period_daily = (
    work.groupby(
        ["symbol", "date", "clock_period"]
    )
    .agg(
        observations=("ts_sec", "size"),
        valid_spreads=("spread_bps", "count"),
        median_spread=("spread_bps", "median"),
        p90_spread=(
            "spread_bps",
            lambda s: s.quantile(0.90)
        ),
        p99_spread=(
            "spread_bps",
            lambda s: s.quantile(0.99)
        ),
        mean_spread=("spread_bps", "mean")
    )
    .reset_index()
)

duration_map = {
    name: finish - begin
    for name, begin, finish in PERIODS
}

period_daily["expected_seconds"] = (
    period_daily["clock_period"].map(duration_map)
)

period_daily["coverage"] = (
    period_daily["valid_spreads"]
    / period_daily["expected_seconds"]
)

period_daily["valid"] = (
    period_daily["coverage"] >= 0.90
)

metrics = [
    "median_spread",
    "p90_spread",
    "p99_spread",
    "mean_spread"
]

period_daily.loc[
    ~period_daily["valid"],
    metrics
] = np.nan

period_summary = (
    period_daily.groupby("clock_period")
    .agg(
        sessions=("median_spread", "count"),
        median_daily_spread=("median_spread", "median"),
        median_daily_p90=("p90_spread", "median"),
        median_daily_p99=("p99_spread", "median"),
        mean_daily_spread=("mean_spread", "mean")
    )
    .reset_index()
)

print("\n===== SPREAD BY CLOCK PERIOD =====")
print(
    period_summary.round(6).to_string(index=False)
)

# Same-session comparison avoids pooling
# different contract-level spread regimes.

print("\n===== OPENING VS MIDDAY: PAIRED COMPARISON =====")

for metric in ["median_spread", "p90_spread", "p99_spread"]:

    p = period_daily.pivot_table(
        index=["symbol", "date"],
        columns="clock_period",
        values=metric
    )

    if not {"opening_0_5m", "midday"}.issubset(p.columns):
        continue

    paired = p[
        ["opening_0_5m", "midday"]
    ].dropna()

    if paired.empty:
        continue

    diff = (
        paired["opening_0_5m"]
        - paired["midday"]
    )

    ratio = (
        paired["opening_0_5m"]
        / paired["midday"].where(
            paired["midday"] > 0
        )
    )

    print(f"\nMetric: {metric}")
    print("Sessions:", len(paired))
    print("Median difference (bps):", round(diff.median(), 6))
    print("Median ratio:", round(ratio.median(), 6))
    print(
        "Fraction opening higher:",
        round((diff > 0).mean(), 4)
    )

# ============================================================
# 6. EXACT FUTURE SPREAD MATCHING
#
# Use spread(t+h), same contract and date.
# Never shift rows to estimate a time horizon.
# ============================================================

spread_grid = work[
    ["symbol", "date", "ts_sec", "spread_bps"]
].copy()

for h in HORIZONS:

    future = spread_grid.copy()
    future["ts_sec"] -= h

    future = future.rename(
        columns={
            "spread_bps": f"future_spread_{h}"
        }
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

# ============================================================
# 7. RETROSPECTIVE DAILY WIDE-SPREAD THRESHOLD
#
# Diagnostic only, NOT causal.
# ============================================================

daily_p90 = (
    work.groupby(["symbol", "date"])["spread_bps"]
    .transform(lambda s: s.quantile(0.90))
)

work["daily_wide_spread"] = (
    work["spread_bps"] > daily_p90
)

work.loc[
    work["spread_bps"].isna(),
    "daily_wide_spread"
] = False

# ============================================================
# 8. CAUSAL TRAILING WIDE-SPREAD THRESHOLD
#
# Compute percentile from strictly preceding observations.
# Separate timestamp-contiguous runs.
#
# A spread at time t is compared with historical
# spreads strictly before t.
#
# This threshold is causally constructed, provided
# end-of-second quotes are available at decision time.
# ============================================================

g = work.groupby(
    ["symbol", "date"],
    sort=False
)

gap = g["ts_sec"].diff().ne(1)

work["run_id"] = gap.groupby(
    [work["symbol"], work["date"]]
).cumsum()

runs = work.groupby(
    ["symbol", "date", "run_id"],
    sort=False
)

work["trailing_spread_p90"] = runs[
    "spread_bps"
].transform(
    lambda s: s.shift(1).rolling(
        LOOKBACK,
        min_periods=MIN_TRAILING
    ).quantile(0.90)
)

work["causal_wide_spread"] = (
    work["spread_bps"]
    > work["trailing_spread_p90"]
)

# Explicitly mark periods without historical context.
work.loc[
    work["trailing_spread_p90"].isna()
    | work["spread_bps"].isna(),
    "causal_wide_spread"
] = pd.NA

work["causal_wide_spread"] = (
    work["causal_wide_spread"].astype("boolean")
)

print("\n===== CAUSAL THRESHOLD COVERAGE =====")

print(
    "Availability fraction:",
    round(
        work["trailing_spread_p90"].notna().mean(),
        5
    )
)

print(
    "Wide-spread fraction:",
    round(
        work["causal_wide_spread"]
        .astype(float).mean(),
        5
    )
)

# ============================================================
# 9. DAILY PERSISTENCE / AUTOCORRELATION
# ============================================================

persistence_records = []

for (symbol, date), day in work.groupby(
    ["symbol", "date"],
    sort=True
):

    for h in HORIZONS:

        future_col = f"future_spread_{h}"

        # A. Spread autocorrelation at exact time lag.
        pairs = day[
            ["spread_bps", future_col]
        ].dropna()

        corr = (
            pairs["spread_bps"].corr(pairs[future_col])
            if (
                len(pairs) >= 300
                and pairs["spread_bps"].nunique() >= 3
                and pairs[future_col].nunique() >= 3
            )
            else np.nan
        )

        # B. Retrospective same-day wide-spread persistence.
        threshold = day["spread_bps"].quantile(0.90)

        valid = day[
            ["spread_bps", future_col]
        ].dropna()

        currently_wide = (
            valid["spread_bps"] > threshold
        )

        future_wide = (
            valid[future_col] > threshold
        )

        baseline_wide_rate = future_wide.mean()

        n_wide = int(currently_wide.sum())

        conditional_wide_rate = (
            future_wide[currently_wide].mean()
            if n_wide >= 30 else np.nan
        )

        lift = (
            conditional_wide_rate / baseline_wide_rate
            if (
                pd.notna(conditional_wide_rate)
                and baseline_wide_rate > 0
            )
            else np.nan
        )

        # C. Causal threshold evaluation.
        #
        # Use t's past-only threshold for both current
        # and future-spread comparisons.
        # Future spread is used only as a label.
        #
        # Do not treat t+h information as a feature.

        causal = day[
            [
                "spread_bps",
                "trailing_spread_p90",
                future_col
            ]
        ].dropna()

        causal_wide = (
            causal["spread_bps"]
            > causal["trailing_spread_p90"]
        )

        causal_future_wide = (
            causal[future_col]
            > causal["trailing_spread_p90"]
        )

        n_causal_wide = int(causal_wide.sum())

        causal_baseline = causal_future_wide.mean()

        causal_conditional = (
            causal_future_wide[causal_wide].mean()
            if n_causal_wide >= 30
            else np.nan
        )

        causal_lift = (
            causal_conditional / causal_baseline
            if (
                pd.notna(causal_conditional)
                and causal_baseline > 0
            )
            else np.nan
        )

        persistence_records.append({
            "symbol": symbol,
            "date": date,
            "horizon": h,
            "paired_observations": len(pairs),
            "spread_autocorr": corr,
            "daily_threshold_wide_n": n_wide,
            "daily_threshold_lift": lift,
            "causal_wide_n": n_causal_wide,
            "causal_conditional_rate": causal_conditional,
            "causal_baseline_rate": causal_baseline,
            "causal_persistence_lift": causal_lift
        })

persistence_daily = pd.DataFrame(
    persistence_records
)

print("\n===== DAILY SPREAD PERSISTENCE =====")

print(
    persistence_daily[
        [
            "symbol",
            "date",
            "horizon",
            "spread_autocorr",
            "daily_threshold_lift",
            "causal_persistence_lift"
        ]
    ].round(5).to_string(index=False)
)

# ============================================================
# 10. SESSION-LEVEL PERSISTENCE SUMMARY
# ============================================================

persistence_summary = (
    persistence_daily.groupby("horizon")
    .agg(
        sessions=("spread_autocorr", "count"),
        mean_autocorr=("spread_autocorr", "mean"),
        median_autocorr=("spread_autocorr", "median"),
        positive_autocorr_fraction=(
            "spread_autocorr",
            lambda s: (s.dropna() > 0).mean()
        ),
        median_daily_threshold_lift=(
            "daily_threshold_lift", "median"
        ),
        median_causal_lift=(
            "causal_persistence_lift", "median"
        ),
        causal_lift_above_one_fraction=(
            "causal_persistence_lift",
            lambda s: (s.dropna() > 1).mean()
        )
    )
    .reset_index()
)

print("\n===== SPREAD PERSISTENCE SUMMARY =====")

print(
    persistence_summary.round(6).to_string(index=False)
)

# ============================================================
# 11. CAUSAL WIDE-SPREAD FREQUENCY BY CLOCK PERIOD
#
# Descriptive only; the threshold itself is past-only.
# ============================================================

wide_by_period = (
    work.groupby(
        ["symbol", "date", "clock_period"]
    )["causal_wide_spread"]
    .agg(
        available="count",
        wide="sum",
        rate="mean"
    )
    .reset_index()
)

print("\n===== CAUSAL WIDE-SPREAD FREQUENCY =====")

print(
    wide_by_period.groupby("clock_period")
    .agg(
        sessions=("rate", "count"),
        median_daily_rate=("rate", "median"),
        mean_daily_rate=("rate", "mean")
    )
    .round(6)
    .to_string()
)

# ============================================================
# 12. SAVE EXPLORATORY RESULTS
# ============================================================

calendar_exp16_data = work
calendar_exp16_bins = bins
calendar_exp16_clock_profile = profile
calendar_exp16_period_daily = period_daily
calendar_exp16_period_summary = period_summary
calendar_exp16_persistence_daily = persistence_daily
calendar_exp16_persistence_summary = persistence_summary
calendar_exp16_wide_by_period = wide_by_period

print("\n===== EXPERIMENT 16 COMPLETE =====")
print("Master df_1s remains unchanged.")
print("No features added to validated core.")



===== SESSION INVENTORY =====
        symbol       date  rows  first_sec  last_sec  coverage  eligible
USDINR26OCTFUT 2026-09-30 28749      32400     61148  0.998229      True
USDINR26OCTFUT 2026-10-01 28402      32780     61181  0.986181      True
USDINR26OCTFUT 2026-10-05 28767      32400     61166  0.998854      True
USDINR26OCTFUT 2026-10-06 28755      32400     61154  0.998437      True
USDINR26OCTFUT 2026-10-07 28752      32400     61151  0.998333      True
USDINR26OCTFUT 2026-10-08 28749      32400     61148  0.998229      True
USDINR26OCTFUT 2026-10-09 28748      32400     61147  0.998194      True
USDINR26OCTFUT 2026-10-12 28754      32400     61153  0.998403      True
USDINR26OCTFUT 2026-10-13 28793      32400     61192  0.999757      True
USDINR26OCTFUT 2026-10-14 28792      32400     61191  0.999722      True
USDINR26OCTFUT 2026-10-15 28797      32400     61196  0.999896      True
USDINR26OCTFUT 2026-10-16 28796      32400     61195  0.999861      True
USDINR26OCTFUT 2026-

In [33]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 17
# HISTORICAL CLOCK SPREAD VS CURRENT LIQUIDITY STATE
#
# INPUT: df_1s
# Dependencies: pandas, numpy
#
# Outputs:
#   calendar_exp17_data
#   calendar_exp17_daily
#   calendar_exp17_summary
#   calendar_exp17_winners
#
# Master df_1s remains unchanged.
# ============================================================

START = 9 * 3600
END = 17 * 3600
BIN_SECONDS = 300

HORIZONS = [30, 60, 300]
HISTORY_SESSIONS = 3

MIN_DAY_COVERAGE = 0.97
MIN_BIN_COVERAGE = 0.95
RECENT_WINDOW = 300
MIN_RECENT = 60

required = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
]

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate columns in df_1s")

missing = [
    c for c in required
    if c not in df_1s.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

# ============================================================
# 1. TIMESTAMPS AND SESSION ELIGIBILITY
# ============================================================

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

work["ts"] = (
    pd.to_datetime(
        work["ts_sec"], unit="s", utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp rows")

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["eligible"] = (
    (sessions["rows"] / (END - START)
     >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

work["bin"] = (
    (work["sec"] - START) // BIN_SECONDS
).astype(int)

# ============================================================
# 2. RECONSTRUCT SPREAD
# ============================================================

for col in ["bid_p1", "ask_p1"]:
    work[col] = pd.to_numeric(
        work[col], errors="coerce"
    )

valid_quote = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] >= work["bid_p1"])
)

mid = (
    work["bid_p1"] + work["ask_p1"]
) / 2

work["spread_bps"] = (
    10000
    * (work["ask_p1"] - work["bid_p1"])
    / mid
)

work.loc[
    ~valid_quote, "spread_bps"
] = np.nan

# ============================================================
# 3. CAUSAL RECENT SPREAD MEDIAN
#
# Do not roll across timestamp gaps or sessions.
# Exclude current second from the recent median.
# ============================================================

g = work.groupby(
    ["symbol", "date"], sort=False
)

gap = g["ts_sec"].diff().ne(1)

work["run_id"] = gap.groupby(
    [work["symbol"], work["date"]]
).cumsum()

work["pred_recent"] = (
    work.groupby(
        ["symbol", "date", "run_id"],
        sort=False
    )["spread_bps"]
    .transform(
        lambda s: s.shift(1).rolling(
            RECENT_WINDOW,
            min_periods=MIN_RECENT
        ).median()
    )
)

work["pred_current"] = work["spread_bps"]

# ============================================================
# 4. EXACT FUTURE SPREAD LABELS
# ============================================================

grid = work[
    ["symbol", "date", "ts_sec", "spread_bps"]
].copy()

for h in HORIZONS:

    future = grid.copy()
    future["ts_sec"] -= h

    future = future.rename(
        columns={
            "spread_bps": f"future_spread_{h}"
        }
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

# ============================================================
# 5. HISTORICAL FIVE-MINUTE SPREAD LEVELS
#
# Only completed, sufficiently covered bins are included
# in historical clock predictions.
# ============================================================

bin_stats = (
    work.groupby(["symbol", "date", "bin"])
    .agg(
        count=("spread_bps", "count"),
        median_spread=("spread_bps", "median")
    )
    .reset_index()
)

bin_stats.loc[
    bin_stats["count"]
    < BIN_SECONDS * MIN_BIN_COVERAGE,
    "median_spread"
] = np.nan

# ============================================================
# 6. CONSTRUCT PAST-ONLY HISTORICAL FORECASTS
# ============================================================

baseline_frames = []

for symbol, contract in bin_stats.groupby("symbol"):

    dates = sorted(contract["date"].unique())

    for i in range(
        HISTORY_SESSIONS, len(dates)
    ):

        test_date = dates[i]

        history_dates = dates[
            i-HISTORY_SESSIONS:i
        ]

        history = contract[
            contract["date"].isin(history_dates)
        ].dropna(subset=["median_spread"])

        test = contract[
            contract["date"] == test_date
        ][["symbol", "date", "bin"]].copy()

        # Historical constant: median of available
        # five-minute median spreads.
        constant = history["median_spread"].median()

        clock = (
            history.groupby("bin")
            .agg(
                pred_clock=("median_spread", "median"),
                history_days=("date", "nunique")
            )
            .reset_index()
        )

        clock = clock[
            clock["history_days"] >= HISTORY_SESSIONS
        ]

        test["pred_constant"] = constant

        test = test.merge(
            clock[["bin", "pred_clock"]],
            on="bin",
            how="left",
            validate="one_to_one"
        )

        baseline_frames.append(test)

if not baseline_frames:
    raise ValueError("Insufficient historical sessions")

baselines = pd.concat(
    baseline_frames,
    ignore_index=True
)

work = work.merge(
    baselines,
    on=["symbol", "date", "bin"],
    how="inner",
    validate="many_to_one"
)

# ============================================================
# 7. COMMON EVALUATION SAMPLE
#
# All candidate forecasts must be available on the same
# exact set of timestamps.
# ============================================================

prediction_cols = [
    "pred_constant",
    "pred_clock",
    "pred_current",
    "pred_recent"
]

records = []

for h in HORIZONS:

    target = f"future_spread_{h}"

    cols = [
        "symbol", "date", "sec", "ts_sec", "bin"
    ] + prediction_cols + [target]

    z = work[cols].copy()

    z = z.replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if z.empty:
        continue

    z["horizon"] = h
    z["actual"] = z[target]

    for col in prediction_cols:
        z[f"ae_{col}"] = (
            z["actual"] - z[col]
        ).abs()

    records.append(z.drop(columns=[target]))

if not records:
    raise ValueError("No common evaluation observations")

results = pd.concat(
    records, ignore_index=True
)

# ============================================================
# 8. DAILY ERROR COMPARISON
# ============================================================

error_cols = [
    f"ae_{col}"
    for col in prediction_cols
]

daily = (
    results.groupby(
        ["symbol", "date", "horizon"]
    )[error_cols]
    .mean()
    .reset_index()
)

daily["clock_vs_constant_pct"] = (
    100 * (
        1
        - daily["ae_pred_clock"]
        / daily["ae_pred_constant"]
    )
)

daily["clock_vs_current_pct"] = (
    100 * (
        1
        - daily["ae_pred_clock"]
        / daily["ae_pred_current"]
    )
)

daily["recent_vs_current_pct"] = (
    100 * (
        1
        - daily["ae_pred_recent"]
        / daily["ae_pred_current"]
    )
)

print("\n===== DAILY FORECAST COMPARISON =====")

print(
    daily.round(6).to_string(index=False)
)

# ============================================================
# 9. AGGREGATE PERFORMANCE
# ============================================================

summary_rows = []

for h, group in results.groupby("horizon"):

    daily_h = daily[
        daily["horizon"] == h
    ]

    errors = {
        col: group[f"ae_{col}"].mean()
        for col in prediction_cols
    }

    summary_rows.append({
        "horizon": h,
        "sessions": len(daily_h),
        "observations": len(group),
        "mae_constant": errors["pred_constant"],
        "mae_clock": errors["pred_clock"],
        "mae_current": errors["pred_current"],
        "mae_recent": errors["pred_recent"],
        "clock_vs_constant_pct": (
            100 * (
                1 - errors["pred_clock"]
                / errors["pred_constant"]
            )
        ),
        "clock_vs_current_pct": (
            100 * (
                1 - errors["pred_clock"]
                / errors["pred_current"]
            )
        ),
        "recent_vs_current_pct": (
            100 * (
                1 - errors["pred_recent"]
                / errors["pred_current"]
            )
        ),
        "clock_wins_vs_constant": int(
            (daily_h["clock_vs_constant_pct"] > 0).sum()
        ),
        "clock_wins_vs_current": int(
            (daily_h["clock_vs_current_pct"] > 0).sum()
        ),
        "recent_wins_vs_current": int(
            (daily_h["recent_vs_current_pct"] > 0).sum()
        )
    })

summary = pd.DataFrame(summary_rows)

print("\n===== AGGREGATE SPREAD FORECASTS =====")
print(summary.round(6).to_string(index=False))

# ============================================================
# 10. DAILY WINNING FORECAST
# ============================================================

daily["best_model"] = (
    daily[error_cols]
    .idxmin(axis=1)
    .str.replace("ae_pred_", "", regex=False)
)

winners = (
    daily.groupby(["horizon", "best_model"])
    .size()
    .rename("winning_sessions")
    .reset_index()
)

print("\n===== WINNING FORECAST BY SESSION =====")
print(winners.to_string(index=False))

# ============================================================
# 11. SESSION-LEVEL STABILITY
# ============================================================

print("\n===== MEDIAN DAILY RELATIVE IMPROVEMENT =====")

print(
    daily.groupby("horizon")[
        [
            "clock_vs_constant_pct",
            "clock_vs_current_pct",
            "recent_vs_current_pct"
        ]
    ]
    .median()
    .round(5)
    .to_string()
)

# ============================================================
# 12. SAVE RESULTS
# ============================================================

calendar_exp17_data = results
calendar_exp17_daily = daily
calendar_exp17_summary = summary
calendar_exp17_winners = winners

print("\n===== EXPERIMENT 17 COMPLETE =====")
print("df_1s remains unchanged.")
print("No feature added to the validated core.")



===== DAILY FORECAST COMPARISON =====
        symbol       date  horizon  ae_pred_constant  ae_pred_clock  ae_pred_current  ae_pred_recent  clock_vs_constant_pct  clock_vs_current_pct  recent_vs_current_pct
USDINR26OCTFUT 2026-10-06       30          0.363742       0.418671         0.247048        0.344810             -15.101104            -69.469685             -39.572307
USDINR26OCTFUT 2026-10-06       60          0.363677       0.418768         0.329223        0.367492             -15.148540            -27.198841             -11.623875
USDINR26OCTFUT 2026-10-06      300          0.363857       0.442971         0.476496        0.423024             -21.743096              7.035631              11.221804
USDINR26OCTFUT 2026-10-07       30          0.490527       0.533051         0.411664        0.475736              -8.669099            -29.487043             -15.564155
USDINR26OCTFUT 2026-10-07       60          0.490345       0.533296         0.495844        0.493579              -8

In [34]:

import pandas as pd
import numpy as np

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 18
# DOES CLOCK TIME ADD SPREAD-TAIL PREDICTIVE INFORMATION?
#
# Input: df_1s
# Dependencies: pandas, numpy
#
# Outputs:
#   calendar_exp18_predictions
#   calendar_exp18_daily
#   calendar_exp18_summary
#   calendar_exp18_windows
#   calendar_exp18_thresholds
#
# No modification to df_1s.
# ============================================================

START = 9 * 3600
END = 17 * 3600

HORIZONS = [30, 60, 300]
MIN_DAY_COVERAGE = 0.97
MIN_TRAIN_SESSIONS = 3
RIDGE_ALPHA = 0.01

required = ["symbol", "ts_sec", "bid_p1", "ask_p1"]

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate master columns")

missing = [c for c in required if c not in df_1s.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

work = df_1s[required].copy()

# ------------------------------------------------------------
# 1. TIMESTAMPS AND SESSION FILTER
# ------------------------------------------------------------

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="coerce"
)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

work["ts"] = (
    pd.to_datetime(work["ts_sec"], unit="s", utc=True)
    .dt.tz_convert("Asia/Kolkata")
)

work["date"] = work["ts"].dt.date

work["sec"] = (
    work["ts"].dt.hour * 3600
    + work["ts"].dt.minute * 60
    + work["ts"].dt.second
)

work = work[
    (work["sec"] >= START)
    & (work["sec"] < END)
].copy()

work = work.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp rows")

sessions = (
    work.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "size"))
    .reset_index()
)

sessions["eligible"] = (
    (sessions["rows"] / (END - START) >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

work = work.merge(
    sessions[["symbol", "date", "eligible"]],
    on=["symbol", "date"],
    validate="many_to_one"
)

work = work[work["eligible"]].copy()

# ------------------------------------------------------------
# 2. RECONSTRUCT SPREAD
# ------------------------------------------------------------

for col in ["bid_p1", "ask_p1"]:
    work[col] = pd.to_numeric(work[col], errors="coerce")

valid = (
    (work["bid_p1"] > 0)
    & (work["ask_p1"] > 0)
    & (work["ask_p1"] >= work["bid_p1"])
)

mid = (work["bid_p1"] + work["ask_p1"]) / 2

work["spread_bps"] = (
    10000 * (work["ask_p1"] - work["bid_p1"]) / mid
)

work.loc[~valid, "spread_bps"] = np.nan

# ------------------------------------------------------------
# 3. CAUSAL TRAILING SPREAD STATE
# ------------------------------------------------------------

g = work.groupby(["symbol", "date"], sort=False)

gaps = g["ts_sec"].diff().ne(1)

work["run_id"] = gaps.groupby(
    [work["symbol"], work["date"]]
).cumsum()

runs = work.groupby(
    ["symbol", "date", "run_id"], sort=False
)

# Include current quote: available at end of second t.
# Partial history is allowed, and explicitly measured.

for w in [60, 300]:

    work[f"spread_median_{w}"] = (
        runs["spread_bps"].transform(
            lambda s: s.rolling(
                w, min_periods=1
            ).median()
        )
    )

    work[f"spread_count_{w}"] = (
        runs["spread_bps"].transform(
            lambda s: s.rolling(
                w, min_periods=1
            ).count()
        )
    )

work["spread_p90_300"] = (
    runs["spread_bps"].transform(
        lambda s: s.rolling(
            300, min_periods=1
        ).quantile(0.90)
    )
)

work["spread_relative_300"] = (
    work["spread_bps"]
    - work["spread_median_300"]
)

work["spread_history_fraction"] = (
    work["spread_count_300"] / 300
)

# ------------------------------------------------------------
# 4. PREDEFINED CLOCK REPRESENTATION
# ------------------------------------------------------------

elapsed = work["sec"] - START

clock_intervals = [
    ("first_30s", 0, 30),
    ("next_30s", 30, 60),
    ("minute_1_5", 60, 300),
    ("minute_5_30", 300, 1800)
]

work["clock_window"] = "rest"

clock_features = []

for name, begin, finish in clock_intervals:

    col = f"clock_{name}"
    clock_features.append(col)

    mask = (elapsed >= begin) & (elapsed < finish)

    work[col] = mask.astype(int)
    work.loc[mask, "clock_window"] = name

base_features = [
    "spread_bps",
    "spread_median_60",
    "spread_median_300",
    "spread_p90_300",
    "spread_relative_300",
    "spread_history_fraction"
]

extended_features = base_features + clock_features

# ------------------------------------------------------------
# 5. EXACT FUTURE SPREAD LABELS
# ------------------------------------------------------------

price_grid = work[
    ["symbol", "date", "ts_sec", "spread_bps"]
].copy()

for h in HORIZONS:

    future = price_grid.copy()
    future["ts_sec"] = future["ts_sec"] - h

    future = future.rename(
        columns={"spread_bps": f"future_spread_{h}"}
    )

    work = work.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# 6. RIDGE LINEAR-PROBABILITY MODEL
#
# Simple benchmark. Train-only imputation and scaling.
# Predicted probabilities clipped to [0,1].
# No sklearn required.
# ------------------------------------------------------------

def predict_probability(train, test, features, target):

    X = train[features].to_numpy(dtype=float).copy()
    T = test[features].to_numpy(dtype=float).copy()
    y = train[target].to_numpy(dtype=float)

    X[~np.isfinite(X)] = np.nan
    T[~np.isfinite(T)] = np.nan

    med = np.nanmedian(X, axis=0)
    med = np.where(np.isfinite(med), med, 0)

    X = np.where(np.isnan(X), med, X)
    T = np.where(np.isnan(T), med, T)

    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std > 1e-12, std, 1.0)

    X = (X - mean) / std
    T = (T - mean) / std

    y_mean = y.mean()
    yc = y - y_mean

    n, k = X.shape

    beta = np.linalg.solve(
        X.T @ X + RIDGE_ALPHA * n * np.eye(k),
        X.T @ yc
    )

    p = y_mean + T @ beta

    return np.clip(p, 0, 1)

# ------------------------------------------------------------
# 7. CHRONOLOGICAL SAME-CONTRACT EVALUATION
#
# Thresholds and coefficients trained on previous sessions.
# Current evaluation day never influences fitting.
# ------------------------------------------------------------

prediction_frames = []
threshold_records = []

for symbol, contract in work.groupby("symbol"):

    dates = sorted(contract["date"].unique())

    for i in range(MIN_TRAIN_SESSIONS, len(dates)):

        train_dates = dates[:i]
        test_date = dates[i]

        train_all = contract[
            contract["date"].isin(train_dates)
        ].copy()

        test_all = contract[
            contract["date"] == test_date
        ].copy()

        # Fixed spread threshold from past sessions ONLY.
        threshold = train_all["spread_bps"].quantile(0.90)

        if not np.isfinite(threshold):
            continue

        threshold_records.append({
            "symbol": symbol,
            "test_date": test_date,
            "train_sessions": len(train_dates),
            "train_spread_p90": threshold
        })

        print(
            f"Testing {symbol} {test_date} | "
            f"Train days={len(train_dates)} | "
            f"Wide threshold={threshold:.5f} bps"
        )

        for h in HORIZONS:

            future_col = f"future_spread_{h}"

            train = train_all[
                train_all[future_col].notna()
            ].copy()

            test = test_all[
                test_all[future_col].notna()
            ].copy()

            if train.empty or test.empty:
                continue

            train["target"] = (
                train[future_col] > threshold
            ).astype(float)

            test["target"] = (
                test[future_col] > threshold
            ).astype(float)

            # Historical event rate: train data only.
            constant_p = train["target"].mean()

            p_state = predict_probability(
                train, test, base_features, "target"
            )

            p_clock = predict_probability(
                train, test, extended_features, "target"
            )

            p_constant = np.full(
                len(test), constant_p
            )

            out = test[
                [
                    "symbol", "date", "ts_sec",
                    "clock_window", "target"
                ]
            ].copy()

            out["horizon"] = h
            out["p_constant"] = p_constant
            out["p_state"] = p_state
            out["p_clock"] = p_clock

            for name in [
                "constant", "state", "clock"
            ]:
                out[f"brier_{name}"] = (
                    out[f"p_{name}"] - out["target"]
                ) ** 2

            prediction_frames.append(out)

if not prediction_frames:
    raise ValueError(
        "Insufficient history for chronological evaluation"
    )

predictions = pd.concat(
    prediction_frames, ignore_index=True
)

thresholds = pd.DataFrame(threshold_records)

# ------------------------------------------------------------
# 8. DAILY OUT-OF-SAMPLE COMPARISON
# ------------------------------------------------------------

daily = (
    predictions.groupby(
        ["symbol", "date", "horizon"]
    )
    .agg(
        observations=("target", "size"),
        event_rate=("target", "mean"),
        brier_constant=("brier_constant", "mean"),
        brier_state=("brier_state", "mean"),
        brier_clock=("brier_clock", "mean")
    )
    .reset_index()
)

daily["clock_vs_state_pct"] = (
    100 * (
        1 - daily["brier_clock"]
        / daily["brier_state"]
    )
)

daily["state_vs_constant_pct"] = (
    100 * (
        1 - daily["brier_state"]
        / daily["brier_constant"]
    )
)

print("\n===== DAILY OOS SPREAD-TAIL FORECASTS =====")
print(daily.round(6).to_string(index=False))

# ------------------------------------------------------------
# 9. AGGREGATE PERFORMANCE
# ------------------------------------------------------------

summary_records = []

for h, sample in predictions.groupby("horizon"):

    days = daily[daily["horizon"] == h]

    base = sample["brier_constant"].mean()
    state = sample["brier_state"].mean()
    clock = sample["brier_clock"].mean()

    summary_records.append({
        "horizon": h,
        "sessions": len(days),
        "observations": len(sample),
        "event_rate": sample["target"].mean(),
        "brier_constant": base,
        "brier_state": state,
        "brier_clock": clock,
        "state_vs_constant_pct": (
            100 * (1 - state / base)
        ),
        "clock_vs_state_pct": (
            100 * (1 - clock / state)
        ),
        "clock_winning_days": int(
            (days["clock_vs_state_pct"] > 0).sum()
        ),
        "median_daily_clock_gain_pct": (
            days["clock_vs_state_pct"].median()
        )
    })

summary = pd.DataFrame(summary_records)

print("\n===== AGGREGATE OOS SPREAD-TAIL RESULTS =====")
print(summary.round(6).to_string(index=False))

# ------------------------------------------------------------
# 10. CLOCK-WINDOW PERFORMANCE
# ------------------------------------------------------------

windows = (
    predictions.groupby(
        ["horizon", "clock_window"]
    )
    .agg(
        observations=("target", "size"),
        event_rate=("target", "mean"),
        brier_state=("brier_state", "mean"),
        brier_clock=("brier_clock", "mean")
    )
    .reset_index()
)

windows["clock_vs_state_pct"] = (
    100 * (
        1 - windows["brier_clock"]
        / windows["brier_state"]
    )
)

window_order = [
    "first_30s",
    "next_30s",
    "minute_1_5",
    "minute_5_30",
    "rest"
]

windows["clock_window"] = pd.Categorical(
    windows["clock_window"],
    categories=window_order,
    ordered=True
)

windows = windows.sort_values(
    ["horizon", "clock_window"]
)

print("\n===== SPREAD-TAIL PERFORMANCE BY CLOCK WINDOW =====")
print(windows.round(6).to_string(index=False))

# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

calendar_exp18_predictions = predictions
calendar_exp18_daily = daily
calendar_exp18_summary = summary
calendar_exp18_windows = windows
calendar_exp18_thresholds = thresholds

print("\n===== EXPERIMENT 18 COMPLETE =====")
print("Master df_1s remains unchanged.")
print("No features added to validated core.")


Testing USDINR26OCTFUT 2026-10-06 | Train days=3 | Wide threshold=1.56035 bps
Testing USDINR26OCTFUT 2026-10-07 | Train days=4 | Wide threshold=1.55557 bps
Testing USDINR26OCTFUT 2026-10-08 | Train days=5 | Wide threshold=1.55909 bps
Testing USDINR26OCTFUT 2026-10-09 | Train days=6 | Wide threshold=1.55501 bps
Testing USDINR26OCTFUT 2026-10-12 | Train days=7 | Wide threshold=1.55501 bps
Testing USDINR26OCTFUT 2026-10-13 | Train days=8 | Wide threshold=1.56287 bps
Testing USDINR26OCTFUT 2026-10-14 | Train days=9 | Wide threshold=1.80396 bps
Testing USDINR26OCTFUT 2026-10-15 | Train days=10 | Wide threshold=1.82161 bps
Testing USDINR26OCTFUT 2026-10-16 | Train days=11 | Wide threshold=1.83884 bps
Testing USDINR26OCTFUT 2026-10-19 | Train days=12 | Wide threshold=1.84039 bps

===== DAILY OOS SPREAD-TAIL FORECASTS =====
        symbol       date  horizon  observations  event_rate  brier_constant  brier_state  brier_clock  clock_vs_state_pct  state_vs_constant_pct
USDINR26OCTFUT 2026-10-06 

In [35]:

import numpy as np
import pandas as pd

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 19
# SPREAD-TAIL MODEL ROBUSTNESS AND CALIBRATION
#
# Input: calendar_exp18_predictions
#
# Outputs:
#   calendar_exp19_daily_windows
#   calendar_exp19_window_summary
#   calendar_exp19_calibration
#   calendar_exp19_reliability
#   calendar_exp19_sensitivity
#
# No retraining or changes to master data.
# ============================================================

required = [
    "symbol", "date", "horizon", "clock_window",
    "target", "p_constant", "p_state", "p_clock",
    "brier_constant", "brier_state", "brier_clock"
]

missing = [
    c for c in required
    if c not in calendar_exp18_predictions.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

z = calendar_exp18_predictions[required].copy()

z["gain"] = z["brier_state"] - z["brier_clock"]

# ============================================================
# 1. DAY-LEVEL WINDOW PERFORMANCE
# ============================================================

daily_windows = (
    z.groupby(
        ["symbol", "date", "horizon", "clock_window"]
    )
    .agg(
        observations=("target", "size"),
        actual_rate=("target", "mean"),
        mean_p_state=("p_state", "mean"),
        mean_p_clock=("p_clock", "mean"),
        brier_state=("brier_state", "mean"),
        brier_clock=("brier_clock", "mean"),
        mean_gain=("gain", "mean")
    )
    .reset_index()
)

daily_windows["improvement_pct"] = (
    100 * daily_windows["mean_gain"]
    / daily_windows["brier_state"].where(
        daily_windows["brier_state"] > 0
    )
)

window_summary = (
    daily_windows.groupby(["horizon", "clock_window"])
    .agg(
        days=("mean_gain", "count"),
        winning_days=(
            "mean_gain", lambda s: int((s > 0).sum())
        ),
        median_daily_gain=("mean_gain", "median"),
        mean_daily_gain=("mean_gain", "mean"),
        median_improvement_pct=(
            "improvement_pct", "median"
        ),
        median_event_rate=("actual_rate", "median"),
        worst_daily_gain=("mean_gain", "min"),
        best_daily_gain=("mean_gain", "max")
    )
    .reset_index()
)

order = [
    "first_30s",
    "next_30s",
    "minute_1_5",
    "minute_5_30",
    "rest"
]

window_summary["clock_window"] = pd.Categorical(
    window_summary["clock_window"],
    categories=order,
    ordered=True
)

window_summary = window_summary.sort_values(
    ["horizon", "clock_window"]
)

print("\n===== DAILY WINDOW ROBUSTNESS =====")
print(
    window_summary.round(6).to_string(index=False)
)

# ============================================================
# 2. DAY-LEVEL CALIBRATION
#
# Compare mean forecast probability with realized event rate.
# This is calibration-in-the-large, not full calibration.
# ============================================================

calibration_records = []

for (symbol, date, horizon), day in z.groupby(
    ["symbol", "date", "horizon"]
):

    actual = day["target"].mean()

    for model in ["constant", "state", "clock"]:

        predicted = day[f"p_{model}"].mean()

        calibration_records.append({
            "symbol": symbol,
            "date": date,
            "horizon": horizon,
            "model": model,
            "actual_rate": actual,
            "predicted_rate": predicted,
            "bias": predicted - actual,
            "absolute_bias": abs(predicted - actual)
        })

calibration_daily = pd.DataFrame(calibration_records)

calibration_summary = (
    calibration_daily.groupby(["horizon", "model"])
    .agg(
        days=("date", "count"),
        mean_bias=("bias", "mean"),
        median_bias=("bias", "median"),
        mean_abs_bias=("absolute_bias", "mean"),
        median_abs_bias=("absolute_bias", "median")
    )
    .reset_index()
)

print("\n===== DAILY CALIBRATION-IN-THE-LARGE =====")
print(
    calibration_summary.round(6).to_string(index=False)
)

# ============================================================
# 3. RELIABILITY BY PREDICTED-PROBABILITY BIN
#
# Predetermined probability ranges.
# Diagnostic only; bins are not optimized.
# ============================================================

edges = [-0.000001, 0.05, 0.10, 0.20, 0.40, 1.000001]

reliability_records = []

for horizon, dayset in z.groupby("horizon"):

    for model in ["state", "clock"]:

        temp = dayset[
            ["target", f"p_{model}"]
        ].copy()

        temp["prob_bin"] = pd.cut(
            temp[f"p_{model}"],
            bins=edges,
            include_lowest=True
        )

        grouped = (
            temp.groupby(
                "prob_bin", observed=True
            )
            .agg(
                observations=("target", "size"),
                predicted_rate=(f"p_{model}", "mean"),
                realized_rate=("target", "mean")
            )
            .reset_index()
        )

        grouped["horizon"] = horizon
        grouped["model"] = model
        grouped["calibration_error"] = (
            grouped["predicted_rate"]
            - grouped["realized_rate"]
        )

        reliability_records.append(grouped)

reliability = pd.concat(
    reliability_records, ignore_index=True
)

print("\n===== PROBABILITY RELIABILITY =====")
print(
    reliability.round(6).to_string(index=False)
)

# ============================================================
# 4. LEAVE-ONE-EVALUATION-DAY-OUT SENSITIVITY
#
# No refitting. Only measures dependence of the
# observed aggregate result on each evaluation day.
# ============================================================

sensitivity_records = []

for horizon, sample in z.groupby("horizon"):

    for excluded_date in sorted(sample["date"].unique()):

        other = sample[
            sample["date"] != excluded_date
        ]

        state_bs = other["brier_state"].mean()
        clock_bs = other["brier_clock"].mean()

        sensitivity_records.append({
            "horizon": horizon,
            "excluded_date": excluded_date,
            "remaining_days": other["date"].nunique(),
            "gain_pct": (
                100 * (1 - clock_bs / state_bs)
            )
        })

sensitivity = pd.DataFrame(sensitivity_records)

print("\n===== LEAVE-ONE-DAY-OUT SENSITIVITY =====")
print(
    sensitivity.round(6).to_string(index=False)
)

print("\n===== SENSITIVITY RANGE =====")
print(
    sensitivity.groupby("horizon")["gain_pct"]
    .agg(["min", "median", "max"])
    .round(6)
    .to_string()
)

# ============================================================
# 5. EXCLUDING OCTOBER 12
# ============================================================

without_oct12 = z[
    z["date"].astype(str) != "2026-10-12"
]

print("\n===== EXCLUDING OCTOBER 12 =====")

for h, sample in without_oct12.groupby("horizon"):

    state_bs = sample["brier_state"].mean()
    clock_bs = sample["brier_clock"].mean()

    print(
        f"{h}s | "
        f"State={state_bs:.6f} | "
        f"Clock={clock_bs:.6f} | "
        f"Gain={100 * (1 - clock_bs/state_bs):+.5f}%"
    )

# ============================================================
# 6. PRESERVE OUTPUTS
# ============================================================

calendar_exp19_daily_windows = daily_windows
calendar_exp19_window_summary = window_summary
calendar_exp19_calibration = calibration_summary
calendar_exp19_reliability = reliability
calendar_exp19_sensitivity = sensitivity

print("\n===== EXPERIMENT 19 COMPLETE =====")
print("No retraining.")
print("No additional feature promoted.")



===== DAILY WINDOW ROBUSTNESS =====
 horizon clock_window  days  winning_days  median_daily_gain  mean_daily_gain  median_improvement_pct  median_event_rate  worst_daily_gain  best_daily_gain
      30    first_30s    10             9           0.057478         0.057897               31.932760           0.083333         -0.234338         0.215454
      30     next_30s    10             3          -0.006578        -0.042975              -11.223222           0.116667         -0.339885         0.088419
      30   minute_1_5    10             8           0.002742         0.009221                4.800094           0.137500         -0.009850         0.042635
      30  minute_5_30    10             3          -0.000048        -0.000405               -0.049595           0.111667         -0.002105         0.000343
      30         rest    10             8           0.000194         0.000392                0.183322           0.176442         -0.000022         0.001126
      60    first_30s    10

In [36]:

import pandas as pd
import numpy as np
import re

# ============================================================
# USDINR CALENDAR RESEARCH — EXPERIMENT 20
# EVENT TIMESTAMP, MACRO DATA AND EXPIRY METADATA AUDIT
#
# Required input:
#   df_1s
#
# Optional input:
#   Existing macro/event DataFrame in notebook globals.
#
# Outputs:
#   calendar_exp20_sessions
#   calendar_exp20_contracts
#   calendar_exp20_macro_inventory
#   calendar_exp20_macro_samples
#   calendar_exp20_event_schema
#   calendar_exp20_audit
#
# Important:
#   - NO event-to-price merge
#   - NO invented publication timestamps
#   - NO inferred expiry dates
#   - NO modifications to master data
# ============================================================

IST = "Asia/Kolkata"

START_SEC = 9 * 3600
END_SEC = 17 * 3600

MIN_DAY_COVERAGE = 0.97

# Set this to an explicit DataFrame name if automatic
# discovery identifies the wrong dataset.
#
# Example:
# MACRO_DF_NAME = "macro_df"
#
MACRO_DF_NAME = None

# ============================================================
# 1. VALIDATE FUTURES MASTER
# ============================================================

required = ["symbol", "ts_sec"]

if not isinstance(df_1s, pd.DataFrame):
    raise TypeError("df_1s must be a DataFrame")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate df_1s column names")

missing = [c for c in required if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

fut = df_1s[required].copy()

fut["ts_sec"] = pd.to_numeric(
    fut["ts_sec"], errors="coerce"
)

if fut["ts_sec"].isna().any():
    raise ValueError("Invalid ts_sec values")

fut["ts_ist"] = (
    pd.to_datetime(
        fut["ts_sec"],
        unit="s",
        utc=True
    ).dt.tz_convert(IST)
)

fut["date"] = fut["ts_ist"].dt.date

fut["sec"] = (
    fut["ts_ist"].dt.hour * 3600
    + fut["ts_ist"].dt.minute * 60
    + fut["ts_ist"].dt.second
)

fut = fut[
    (fut["sec"] >= START_SEC)
    & (fut["sec"] < END_SEC)
].copy()

fut = fut.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if fut.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError(
        "Duplicate symbol and timestamp rows"
    )

# ============================================================
# 2. FUTURES SESSION INVENTORY
# ============================================================

sessions = (
    fut.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_timestamp=("ts_ist", "min"),
        last_timestamp=("ts_ist", "max"),
        first_sec=("sec", "min"),
        last_sec=("sec", "max")
    )
    .reset_index()
)

sessions["day_coverage"] = (
    sessions["rows"] / (END_SEC - START_SEC)
)

sessions["starts_at_0900"] = (
    sessions["first_sec"] == START_SEC
)

sessions["eligible"] = (
    (sessions["day_coverage"] >= MIN_DAY_COVERAGE)
    & (sessions["date"].astype(str) != "2026-09-28")
)

sessions["weekday"] = (
    pd.to_datetime(sessions["date"])
    .dt.day_name()
)

print("\n===== FUTURES SESSION AUDIT =====")

print(
    sessions[
        [
            "symbol", "date",
            "rows", "day_coverage",
            "starts_at_0900",
            "eligible", "weekday"
        ]
    ].to_string(
        index=False,
        float_format=lambda v: f"{v:.6f}"
    )
)

print("\nTotal eligible sessions:",
      int(sessions["eligible"].sum()))

# ============================================================
# 3. CONTRACT METADATA INVENTORY
#
# Symbol month is NOT an expiry date.
# ============================================================

contracts = (
    sessions.groupby("symbol")
    .agg(
        available_sessions=("date", "count"),
        eligible_sessions=("eligible", "sum"),
        first_available=("date", "min"),
        last_available=("date", "max")
    )
    .reset_index()
)

contracts["contract_month_label"] = (
    contracts["symbol"].str.extract(
        r"(\d{2}[A-Z]{3})FUT$",
        expand=False
    )
)

contracts["verified_expiry_date"] = pd.NaT

contracts["expiry_source"] = pd.NA

contracts["expiry_verified"] = False

print("\n===== CONTRACT / EXPIRY INVENTORY =====")

print(
    contracts[
        [
            "symbol",
            "contract_month_label",
            "first_available",
            "last_available",
            "eligible_sessions",
            "expiry_verified"
        ]
    ].to_string(index=False)
)

print(
    "\nExpiry dates are intentionally unpopulated.",
    "\nA contract's month label does not verify",
    "its actual exchange expiry date."
)

# ============================================================
# 4. DISCOVER MACRO / EVENT DATAFRAMES
#
# Avoid selecting the futures master or calendar
# experiment outputs as macro data.
# ============================================================

def macro_candidate_score(name, frame):

    n = name.lower()

    if name == "df_1s":
        return -100

    if n.startswith("calendar_exp"):
        return -100

    if frame is df_1s:
        return -100

    cols = " ".join(
        str(c).lower() for c in frame.columns
    )

    score = 0

    for word in [
        "macro", "economic", "calendar",
        "event", "release"
    ]:
        if word in n:
            score += 4

    for word in [
        "actual", "forecast", "consensus",
        "importance", "impact",
        "release", "event", "country"
    ]:
        if word in cols:
            score += 2

    for word in ["date", "time", "timestamp"]:
        if word in cols:
            score += 1

    if (
        "bid_p1" in frame.columns
        or "ask_p1" in frame.columns
    ):
        score -= 20

    return score


candidates = []

if MACRO_DF_NAME is not None:

    obj = globals().get(MACRO_DF_NAME)

    if not isinstance(obj, pd.DataFrame):
        raise ValueError(
            f"{MACRO_DF_NAME} is not a DataFrame"
        )

    candidates = [
        (MACRO_DF_NAME, obj, 1000)
    ]

else:

    for name, obj in list(globals().items()):

        if not isinstance(obj, pd.DataFrame):
            continue

        score = macro_candidate_score(name, obj)

        if score > 0:
            candidates.append((name, obj, score))

    candidates = sorted(
        candidates,
        key=lambda item: (-item[2], item[0])
    )

print("\n===== MACRO DATAFRAME DISCOVERY =====")

if candidates:

    for name, frame, score in candidates[:10]:

        print(
            f"Name={name} | "
            f"Shape={frame.shape} | "
            f"Score={score}"
        )

    macro_name, macro_source, _ = candidates[0]

    macro = macro_source.copy()

    print("\nSelected macro dataset:", macro_name)

else:

    macro_name = None
    macro = None

    print(
        "No macro DataFrame detected.",
        "\nThe futures audit can still complete."
    )

# ============================================================
# 5. SCHEMA / FIELD-TYPE CLASSIFICATION
#
# These are candidate classifications, not verified
# publication-time semantics.
# ============================================================

def classify_field(col):

    s = str(col).lower()

    if any(x in s for x in [
        "publish", "released_at",
        "release_timestamp",
        "available_at", "received_at"
    ]):
        return "POSSIBLE_PUBLICATION_TIMESTAMP"

    if any(x in s for x in [
        "scheduled", "event_time",
        "release_time"
    ]):
        return "POSSIBLE_SCHEDULED_TIME"

    if any(x in s for x in [
        "timestamp", "datetime", "created_at",
        "updated_at"
    ]):
        return "TIMESTAMP_SEMANTICS_UNVERIFIED"

    if "date" in s or s == "time":
        return "DATE_OR_TIME_SEMANTICS_UNVERIFIED"

    if any(x in s for x in [
        "forecast", "consensus", "estimate"
    ]):
        return "POSSIBLE_PRE_RELEASE_EXPECTATION"

    if "previous" in s or "prior" in s:
        return "PREVIOUS_VALUE_REVISION_RISK"

    if "actual" in s:
        return "POSSIBLE_RELEASED_ACTUAL"

    if any(x in s for x in [
        "event", "indicator", "title",
        "country", "currency", "importance",
        "impact"
    ]):
        return "EVENT_METADATA"

    return "OTHER"


def safe_examples(series, n=3):

    values = series.dropna().head(n).tolist()

    return " | ".join(
        str(v)[:90] for v in values
    )


inventory_records = []

if macro is not None:

    for col in macro.columns:

        s = macro[col]

        inventory_records.append({
            "column": str(col),
            "dtype": str(s.dtype),
            "non_null": int(s.notna().sum()),
            "missing_fraction": float(s.isna().mean()),
            "unique_values": int(
                s.nunique(dropna=True)
            ),
            "classification": classify_field(col),
            "examples": safe_examples(s)
        })

inventory = pd.DataFrame(inventory_records)

print("\n===== MACRO COLUMN INVENTORY =====")

if inventory.empty:
    print("No macro column inventory available.")
else:
    print(
        inventory.to_string(
            index=False,
            max_colwidth=90,
            float_format=lambda v: f"{v:.5f}"
        )
    )

# ============================================================
# 6. DATE / TIMESTAMP SEMANTICS AUDIT
#
# CRITICAL:
#
# We do not assume naive timestamps are UTC or IST.
# We do not convert date-only fields into event times.
# We do not interpret collection timestamps as
# publication timestamps.
# ============================================================

audit_records = []

timestamp_classes = {
    "POSSIBLE_PUBLICATION_TIMESTAMP",
    "POSSIBLE_SCHEDULED_TIME",
    "TIMESTAMP_SEMANTICS_UNVERIFIED",
    "DATE_OR_TIME_SEMANTICS_UNVERIFIED"
}

if macro is not None:

    for col in macro.columns:

        classification = classify_field(col)

        if classification not in timestamp_classes:
            continue

        s = macro[col]
        nonnull = s.dropna()

        sample = nonnull.astype(str)

        # Detect explicit timezone offsets / Z suffix
        # in textual values only.
        explicit_zone = sample.str.contains(
            r"(Z|[+-]\d{2}:?\d{2})$",
            regex=True,
            case=False,
            na=False
        )

        date_only = sample.str.fullmatch(
            r"\d{4}-\d{2}-\d{2}",
            na=False
        )

        audit_records.append({
            "column": str(col),
            "classification": classification,
            "non_null": len(nonnull),
            "date_only_fraction": (
                float(date_only.mean())
                if len(nonnull) else np.nan
            ),
            "explicit_timezone_fraction": (
                float(explicit_zone.mean())
                if len(nonnull) else np.nan
            ),
            "dtype_timezone": (
                str(s.dt.tz)
                if isinstance(s.dtype, pd.DatetimeTZDtype)
                else "NONE_OR_UNVERIFIED"
            ),
            "examples": safe_examples(s, 4)
        })

timestamp_audit = pd.DataFrame(audit_records)

print("\n===== EVENT TIMESTAMP SEMANTICS =====")

if timestamp_audit.empty:

    print("No timestamp-like fields detected.")

else:

    print(
        timestamp_audit.to_string(
            index=False,
            max_colwidth=95,
            float_format=lambda v: f"{v:.5f}"
        )
    )

# ============================================================
# 7. MACRO DATA QUALITY
#
# This is only an observational audit.
# Do not invent event identifiers by grouping on title
# or treat revised values as original releases.
# ============================================================

if macro is not None:

    print("\n===== MACRO DATA QUALITY =====")

    print("Rows:", len(macro))
    print("Columns:", len(macro.columns))
    print(
        "Fully duplicated rows:",
        int(macro.duplicated().sum())
        if macro.columns.is_unique
        else "Cannot test: duplicate columns"
    )

    print("\nPossible value fields:")

    value_fields = inventory[
        inventory["classification"].isin([
            "POSSIBLE_RELEASED_ACTUAL",
            "POSSIBLE_PRE_RELEASE_EXPECTATION",
            "PREVIOUS_VALUE_REVISION_RISK"
        ])
    ]

    if value_fields.empty:
        print("No obvious actual/forecast fields.")
    else:
        print(
            value_fields[
                [
                    "column", "classification",
                    "missing_fraction", "examples"
                ]
            ].to_string(
                index=False,
                max_colwidth=95
            )
        )

    print("\nFirst three records (inspection only):")

    print(
        macro.head(3).to_string(
            index=False,
            max_colwidth=80
        )
    )

# ============================================================
# 8. IDENTIFY WHAT IS STILL REQUIRED FOR A
#    LEAKAGE-FREE EVENT JOIN
# ============================================================

event_schema = pd.DataFrame([
    {
        "field": "event_id",
        "requirement": "Stable unique event identifier",
        "availability_rule": "Metadata"
    },
    {
        "field": "event_type",
        "requirement": "RBI/CPI/NFP/FOMC/etc.",
        "availability_rule": "Metadata"
    },
    {
        "field": "scheduled_at_utc",
        "requirement": "Verified timezone-aware planned time",
        "availability_rule": "Only use after schedule known"
    },
    {
        "field": "schedule_known_at_utc",
        "requirement": "Time schedule became public",
        "availability_rule": "Must precede feature observation"
    },
    {
        "field": "actual_published_at_utc",
        "requirement": "Verified actual release timestamp",
        "availability_rule": "Actual value usable afterward"
    },
    {
        "field": "actual",
        "requirement": "First-published actual, not a later revision",
        "availability_rule": "After actual_published_at_utc"
    },
    {
        "field": "consensus",
        "requirement": "Pre-release forecast snapshot",
        "availability_rule": "As known before publication"
    },
    {
        "field": "consensus_asof_utc",
        "requirement": "Timestamp for forecast availability",
        "availability_rule": "Must not use later forecasts"
    },
    {
        "field": "source",
        "requirement": "Original provider/exchange source",
        "availability_rule": "Audit trail"
    },
    {
        "field": "source_timezone",
        "requirement": "Explicit original timezone",
        "availability_rule": "Required for naive source times"
    },
    {
        "field": "revision_flag",
        "requirement": "Whether a figure was subsequently revised",
        "availability_rule": "Prevent revised-data leakage"
    }
])

print("\n===== REQUIRED EVENT DATA SCHEMA =====")

print(event_schema.to_string(index=False))

# ============================================================
# 9. RESEARCH AUDIT STATUS
# ============================================================

has_pub_candidate = (
    not inventory.empty
    and (
        inventory["classification"]
        == "POSSIBLE_PUBLICATION_TIMESTAMP"
    ).any()
)

has_schedule_candidate = (
    not inventory.empty
    and (
        inventory["classification"]
        == "POSSIBLE_SCHEDULED_TIME"
    ).any()
)

has_actual_candidate = (
    not inventory.empty
    and (
        inventory["classification"]
        == "POSSIBLE_RELEASED_ACTUAL"
    ).any()
)

has_consensus_candidate = (
    not inventory.empty
    and (
        inventory["classification"]
        == "POSSIBLE_PRE_RELEASE_EXPECTATION"
    ).any()
)

audit = pd.DataFrame([
    {
        "check": "Futures timestamps in IST",
        "status": "PASS",
        "detail": "Derived from Unix seconds"
    },
    {
        "check": "Eligible session filtering",
        "status": "PASS",
        "detail": (
            f"{int(sessions['eligible'].sum())} "
            "eligible sessions"
        )
    },
    {
        "check": "Macro dataset detected",
        "status": (
            "CANDIDATE_FOUND"
            if macro is not None
            else "NOT_FOUND"
        ),
        "detail": str(macro_name)
    },
    {
        "check": "Publication timestamp field",
        "status": (
            "CANDIDATE_UNVERIFIED"
            if has_pub_candidate
            else "NOT_ESTABLISHED"
        ),
        "detail": "Source semantics must be confirmed"
    },
    {
        "check": "Scheduled release time field",
        "status": (
            "CANDIDATE_UNVERIFIED"
            if has_schedule_candidate
            else "NOT_ESTABLISHED"
        ),
        "detail": "Schedule must be known in advance"
    },
    {
        "check": "First-release actual values",
        "status": (
            "CANDIDATE_UNVERIFIED"
            if has_actual_candidate
            else "NOT_ESTABLISHED"
        ),
        "detail": "Revision history must be checked"
    },
    {
        "check": "Pre-release consensus snapshots",
        "status": (
            "CANDIDATE_UNVERIFIED"
            if has_consensus_candidate
            else "NOT_ESTABLISHED"
        ),
        "detail": "Consensus as-of time required"
    },
    {
        "check": "Official futures expiry dates",
        "status": "NOT_ESTABLISHED",
        "detail": "Exchange contract metadata required"
    },
    {
        "check": "Leakage-free event merge",
        "status": "NOT_ATTEMPTED",
        "detail": "Blocked pending timestamp verification"
    }
])

print("\n===== EXPERIMENT 20 AUDIT SUMMARY =====")

print(audit.to_string(index=False))

# ============================================================
# 10. SAVE RESULTS
# ============================================================

calendar_exp20_sessions = sessions
calendar_exp20_contracts = contracts
calendar_exp20_macro_inventory = inventory
calendar_exp20_macro_samples = (
    macro.head(20).copy()
    if macro is not None
    else pd.DataFrame()
)
calendar_exp20_timestamp_audit = timestamp_audit
calendar_exp20_event_schema = event_schema
calendar_exp20_audit = audit

print("\n===== EXPERIMENT 20 COMPLETE =====")
print("Master df_1s unchanged.")
print("No macro data merged.")
print("No expiry dates assumed.")
print("No new calendar feature validated.")



===== FUTURES SESSION AUDIT =====
        symbol       date  rows  day_coverage  starts_at_0900  eligible   weekday
USDINR26OCTFUT 2026-09-30 28749      0.998229            True      True Wednesday
USDINR26OCTFUT 2026-10-01 28402      0.986181           False      True  Thursday
USDINR26OCTFUT 2026-10-05 28767      0.998854            True      True    Monday
USDINR26OCTFUT 2026-10-06 28755      0.998437            True      True   Tuesday
USDINR26OCTFUT 2026-10-07 28752      0.998333            True      True Wednesday
USDINR26OCTFUT 2026-10-08 28749      0.998229            True      True  Thursday
USDINR26OCTFUT 2026-10-09 28748      0.998194            True      True    Friday
USDINR26OCTFUT 2026-10-12 28754      0.998403            True      True    Monday
USDINR26OCTFUT 2026-10-13 28793      0.999757            True      True   Tuesday
USDINR26OCTFUT 2026-10-14 28792      0.999722            True      True Wednesday
USDINR26OCTFUT 2026-10-15 28797      0.999896            True  

In [37]:

import pandas as pd
import numpy as np
import re

# ============================================================
# EXPERIMENT 20B — MACRO SCHEMA AND TIMESTAMP INSPECTION
# INPUT: df_macro
# NO joins, conversions, or master-data modifications.
# ============================================================

if "df_macro" not in globals():
    raise ValueError("df_macro is not loaded")

m = df_macro.copy()

print("\n===== MACRO DATASET OVERVIEW =====")
print("Shape:", m.shape)
print("Column names unique:", m.columns.is_unique)

print("\n===== ALL MACRO COLUMNS =====")
for i, col in enumerate(m.columns):
    print(f"{i:02d} | {col} | {m[col].dtype}")

# Detect likely research-relevant columns, preserving
# their exact names and original string representations.
patterns = [
    r"date", r"time", r"stamp", r"event",
    r"release", r"publish", r"scheduled",
    r"actual", r"forecast", r"consensus",
    r"previous", r"prior", r"country",
    r"currency", r"importance", r"impact",
    r"source", r"provider", r"revis"
]

relevant = [
    col for col in m.columns
    if any(
        re.search(p, str(col), flags=re.IGNORECASE)
        for p in patterns
    )
]

print("\n===== RELEVANT FIELD INVENTORY =====")

inventory = []

for col in relevant:
    s = m[col]
    inventory.append({
        "column": str(col),
        "dtype": str(s.dtype),
        "nonnull": int(s.notna().sum()),
        "missing_pct": round(100 * s.isna().mean(), 2),
        "distinct": int(s.nunique(dropna=True)),
        "sample_values": " | ".join(
            s.dropna().astype(str).head(3).tolist()
        )[:180]
    })

if inventory:
    print(pd.DataFrame(inventory).to_string(index=False))
else:
    print("No recognizable event fields found.")

print("\n===== FIRST 5 RECORDS: RELEVANT FIELDS =====")

if relevant:
    print(
        m[relevant].head(5).to_string(
            index=False, max_colwidth=55
        )
    )
else:
    print(m.head(5).to_string(index=False))

print("\n===== DATE/TIMESTAMP VALUE SAMPLES =====")

time_cols = [
    col for col in m.columns
    if any(
        x in str(col).lower()
        for x in [
            "date", "time", "timestamp",
            "release", "publish", "scheduled",
            "created", "updated"
        ]
    )
]

for col in time_cols:
    print(f"\nCOLUMN: {col}")
    print("Dtype:", m[col].dtype)
    print("First 10 distinct non-null values:")
    print(
        m[col].dropna()
        .astype(str).drop_duplicates()
        .head(10).to_list()
    )

print("\n===== MACRO RECORD COUNTS BY DATE =====")

# Date-like fields are examined separately.
# Parsing here is for diagnostics ONLY:
# no timezone localization and no feature availability
# is inferred from these values.

for col in time_cols:
    try:
        parsed = pd.to_datetime(
            m[col], errors="coerce", utc=False
        )
        if not isinstance(parsed.dtype, pd.DatetimeTZDtype) \
                and not pd.api.types.is_datetime64_any_dtype(parsed):
            print(f"{col}: mixed/ambiguous timestamps; skipped")
            continue

        dates = parsed.dt.date

        if dates.notna().any():
            print(f"\nColumn: {col}")
            print("Earliest:", dates.min())
            print("Latest:", dates.max())
            print("Parseable:", int(dates.notna().sum()))
            print(
                dates.value_counts()
                .sort_index().tail(20).to_string()
            )

    except (ValueError, TypeError, AttributeError) as e:
        print(f"{col}: unable to parse consistently: {e}")

print("\n===== EXPERIMENT 20B COMPLETE =====")
print("Original df_macro and df_1s unchanged.")
print("No release timestamp verified or inferred.")



===== MACRO DATASET OVERVIEW =====
Shape: (581, 84)
Column names unique: True

===== ALL MACRO COLUMNS =====
00 | ts_ist | datetime64[ns, UTC+05:30]
01 | ist_sec | int64
02 | vix | float64
03 | vix_chg | float64
04 | vix_chg_pct | float64
05 | vix_prev | float64
06 | vix_1w_ago | float64
07 | vix_1m_ago | float64
08 | vix_regime | object
09 | spot_usdinr | float64
10 | spot_eurusd | float64
11 | spot_gbpusd | float64
12 | spot_date | object
13 | dxy | float64
14 | dxy_prev | float64
15 | dxy_chg | float64
16 | dxy_chg_pct | float64
17 | dxy_regime | object
18 | days_to_rbi | int64
19 | days_to_fomc | int64
20 | days_to_expiry | int64
21 | near_rbi | int64
22 | near_fomc | int64
23 | near_expiry | int64
24 | day_of_week | int64
25 | week_of_month | int64
26 | usdinr_dxy_corr_proxy | float64
27 | date | object
28 | spot_jpyusd | float64
29 | crudeoil_price | float64
30 | crudeoil_chg | float64
31 | crudeoil_chg_pct | float64
32 | crudeoil_oi | float64
33 | gold_price | float64
34 | gold

In [39]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 21A — LIQUIDITY FEATURE DISCOVERY
#
# Three feature families x three lookback windows
# Targets: future absolute returns and future spread
#
# No modification to df_1s
# No sklearn dependency
# ============================================================

WINDOWS = [60, 300, 900]
HORIZONS = [30, 60, 300]

SESSION_START = 9 * 3600
SESSION_END = 17 * 3600

MIN_SESSION_COVERAGE = 0.97
MIN_WINDOW_COVERAGE = 0.90
MIN_IC_ROWS = 200
EPS = 1e-12

required = [
    "symbol", "ts_sec", "bid_p1", "ask_p1",
    "bid_q1", "bid_q2", "bid_q3", "bid_q4", "bid_q5",
    "ask_q1", "ask_q2", "ask_q3", "ask_q4", "ask_q5"
]

missing = [c for c in required if c not in df_1s.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("df_1s contains duplicate columns")

# ------------------------------------------------------------
# 1. Independent working copy
# ------------------------------------------------------------

x = df_1s[required].copy()

for c in required:
    if c != "symbol":
        x[c] = pd.to_numeric(x[c], errors="coerce")

if x["ts_sec"].isna().any():
    raise ValueError("Invalid timestamp values")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp pairs")

x["ts_ist"] = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = x["ts_ist"].dt.date

x["ist_sec"] = (
    x["ts_ist"].dt.hour * 3600
    + x["ts_ist"].dt.minute * 60
    + x["ts_ist"].dt.second
)

x = x[
    (x["ist_sec"] >= SESSION_START)
    & (x["ist_sec"] < SESSION_END)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Session eligibility
# ------------------------------------------------------------

session_sizes = (
    x.groupby(["symbol", "date"])
     .size()
     .rename("rows")
     .reset_index()
)

eligible = session_sizes[
    session_sizes["rows"] >= (
        MIN_SESSION_COVERAGE
        * (SESSION_END - SESSION_START)
    )
].copy()

# Previously identified truncated session
eligible = eligible[
    eligible["date"].astype(str) != "2026-09-28"
]

x = x.merge(
    eligible[["symbol", "date"]].assign(keep=True),
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
).drop(columns="keep")

print("Eligible sessions:", len(eligible))
print("Working rows:", len(x))

# ------------------------------------------------------------
# 3. Reconstruct causal book variables
# ------------------------------------------------------------

x["mid"] = (
    x["bid_p1"] + x["ask_p1"]
) / 2.0

x["spread_bps"] = (
    10000.0
    * (x["ask_p1"] - x["bid_p1"])
    / x["mid"]
)

qty_cols = [
    "bid_q1", "bid_q2", "bid_q3", "bid_q4", "bid_q5",
    "ask_q1", "ask_q2", "ask_q3", "ask_q4", "ask_q5"
]

bad_quote = (
    (x["bid_p1"] <= 0)
    | (x["ask_p1"] <= 0)
    | (x["ask_p1"] < x["bid_p1"])
)

bad_depth = (
    (x[qty_cols] < 0).any(axis=1)
    | x[qty_cols].isna().any(axis=1)
)

x["depth_l5"] = x[qty_cols].sum(axis=1)

x.loc[bad_quote, ["mid", "spread_bps"]] = np.nan
x.loc[bad_depth, "depth_l5"] = np.nan

# ------------------------------------------------------------
# 4. One-second returns
#
# No return across gaps or across sessions.
# ------------------------------------------------------------

g = x.groupby(["symbol", "date"], sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

consecutive = (
    x["ts_sec"] - prev_ts == 1
)

x["ret_1s"] = np.log(
    x["mid"] / prev_mid
)

x.loc[~consecutive, "ret_1s"] = np.nan

x["ret_sq"] = x["ret_1s"] ** 2

# ------------------------------------------------------------
# 5. Candidate feature construction
#
# Rolling values include the present observation,
# never future observations.
# ------------------------------------------------------------

candidate_names = []
component_names = []

for w in WINDOWS:

    minp = int(np.ceil(
        w * MIN_WINDOW_COVERAGE
    ))

    grp = x.groupby(["symbol", "date"], sort=False)

    x[f"rv_{w}"] = (
        grp["ret_sq"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 10000.0
    )

    x[f"spread_med_{w}"] = (
        grp["spread_bps"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).median()
        )
    )

    x[f"depth_med_{w}"] = (
        grp["depth_l5"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).median()
        )
    )

    # Current depth is used with trailing volatility.
    # A log transform makes extreme values less dominant.
    #
    # The constant +1 in the denominator prevents
    # undefined values when book depth is zero.
    #
    # Note: values are exchange-quantity dependent.

    a = f"vol_depth_stress_{w}"

    x[a] = np.log1p(
        x[f"rv_{w}"]
        / np.sqrt(
            x["depth_l5"].clip(lower=0) + 1.0
        )
    )

    # Spread relative to observed trailing volatility.
    # Units: spread in bps, volatility in bps.
    # The volatility floor prevents exploding ratios.
    #
    # This is an exploratory definition.

    b = f"spread_vol_ratio_{w}"

    x[b] = np.log1p(
        x["spread_bps"].clip(lower=0)
        / (x[f"rv_{w}"] + 0.1)
    )

    # Current spread jointly with recent typical depth.

    c = f"spread_depth_stress_{w}"

    x[c] = np.log1p(
        x["spread_bps"].clip(lower=0)
        / np.sqrt(
            x[f"depth_med_{w}"].clip(lower=0) + 1.0
        )
    )

    candidate_names += [a, b, c]

    component_names += [
        f"rv_{w}",
        f"spread_med_{w}",
        f"depth_med_{w}"
    ]

# ------------------------------------------------------------
# 6. Exact future targets
#
# Shift is allowed only when:
#   timestamp(t+h) - timestamp(t) == h
#   symbol/date do not change
# ------------------------------------------------------------

for h in HORIZONS:

    grp = x.groupby(["symbol", "date"], sort=False)

    future_ts = grp["ts_sec"].shift(-h)
    future_mid = grp["mid"].shift(-h)
    future_spread = grp["spread_bps"].shift(-h)

    exact = (
        future_ts - x["ts_sec"] == h
    )

    forward_return = (
        10000.0
        * np.log(future_mid / x["mid"])
    )

    x[f"future_absret_{h}"] = (
        forward_return.abs().where(exact)
    )

    x[f"future_spread_{h}"] = (
        future_spread.where(exact)
    )

# ------------------------------------------------------------
# 7. Session-level Spearman IC
#
# A daily IC counts one session as one observation
# for stability assessment.
# ------------------------------------------------------------

def safe_spearman(a, b):
    mask = np.isfinite(a) & np.isfinite(b)

    if mask.sum() < MIN_IC_ROWS:
        return np.nan

    aa = pd.Series(a[mask]).rank().to_numpy()
    bb = pd.Series(b[mask]).rank().to_numpy()

    if aa.std() < EPS or bb.std() < EPS:
        return np.nan

    return float(np.corrcoef(aa, bb)[0, 1])


features_to_test = (
    candidate_names
    + component_names
    + ["spread_bps", "depth_l5"]
)

records = []

for (sym, date), day in x.groupby(
    ["symbol", "date"], sort=False
):

    for h in HORIZONS:

        targets = [
            f"future_absret_{h}",
            f"future_spread_{h}"
        ]

        for target in targets:

            y = day[target].to_numpy(dtype=float)

            for feat in features_to_test:

                z = day[feat].to_numpy(dtype=float)

                ic = safe_spearman(z, y)

                records.append({
                    "symbol": sym,
                    "date": date,
                    "horizon": h,
                    "target": target.split("_")[1],
                    "feature": feat,
                    "ic": ic
                })

calendar_exp21_daily_ic = pd.DataFrame(records)

# ------------------------------------------------------------
# 8. Summarize across sessions
# ------------------------------------------------------------

summary = (
    calendar_exp21_daily_ic
    .groupby(
        ["target", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda s: int((s > 0).sum())),
        negative_days=("ic", lambda s: int((s < 0).sum()))
    )
    .reset_index()
)

summary["sign_consistency"] = (
    summary[
        ["positive_days", "negative_days"]
    ].max(axis=1) / summary["days"].replace(0, np.nan)
)

summary["abs_mean_ic"] = summary["mean_ic"].abs()

calendar_exp21_summary = summary

# ------------------------------------------------------------
# 9. Candidate-only ranking
#
# Ranks are for exploration, not automated selection.
# Avoid treating repeated horizon/window searches
# as independent evidence.
# ------------------------------------------------------------

candidate_results = summary[
    summary["feature"].isin(candidate_names)
].copy()

candidate_results = candidate_results.sort_values(
    ["target", "horizon", "abs_mean_ic"],
    ascending=[True, True, False]
)

print("\n===== CANDIDATE FEATURE RESULTS =====")

print(
    candidate_results[
        [
            "target", "horizon", "feature",
            "days", "mean_ic", "median_ic",
            "std_ic", "sign_consistency"
        ]
    ].round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 10. Compare against individual components
#
# This comparison helps identify obvious redundancy.
# It is NOT yet a conditional predictive ablation.
# ------------------------------------------------------------

print("\n===== RAW COMPONENT BENCHMARKS =====")

benchmark = summary[
    summary["feature"].isin([
        "spread_bps",
        "depth_l5",
        "rv_60", "rv_300", "rv_900",
        "spread_med_60", "spread_med_300",
        "spread_med_900",
        "depth_med_60", "depth_med_300",
        "depth_med_900"
    ])
]

print(
    benchmark[
        [
            "target", "horizon", "feature",
            "mean_ic", "median_ic",
            "sign_consistency"
        ]
    ].round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 11. Same-time redundancy of candidate vs components
# ------------------------------------------------------------

redundancy = []

for feat in candidate_names:

    for component in (
        ["spread_bps", "depth_l5"]
        + component_names
    ):

        rho = x[[feat, component]].corr(
            method="spearman"
        ).iloc[0, 1]

        redundancy.append({
            "candidate": feat,
            "component": component,
            "spearman": rho
        })

calendar_exp21_redundancy = pd.DataFrame(redundancy)

top_redundancy = (
    calendar_exp21_redundancy
    .assign(
        absolute=lambda d: d["spearman"].abs()
    )
    .sort_values(
        ["candidate", "absolute"],
        ascending=[True, False]
    )
    .groupby("candidate")
    .head(3)
)

print("\n===== TOP 3 COMPONENT CORRELATIONS =====")

print(
    top_redundancy[
        ["candidate", "component", "spearman"]
    ].round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 12. Final diagnostics
# ------------------------------------------------------------

calendar_exp21_features = x[
    [
        "symbol", "date", "ts_sec",
        "mid", "spread_bps", "depth_l5"
    ]
    + component_names
    + candidate_names
    + [
        f"future_absret_{h}"
        for h in HORIZONS
    ]
    + [
        f"future_spread_{h}"
        for h in HORIZONS
    ]
].copy()

print("\n===== FEATURE AVAILABILITY =====")

availability = pd.DataFrame({
    "feature": candidate_names,
    "missing_pct": [
        100 * x[c].isna().mean()
        for c in candidate_names
    ],
    "n_unique": [
        x[c].nunique()
        for c in candidate_names
    ]
})

print(
    availability.round(4).to_string(index=False)
)

print("\n===== COMPLETE =====")
print("Candidate feature variants:", len(candidate_names))
print("Original df_1s unchanged.")
print("No feature promoted yet.")


Eligible sessions: 16
Working rows: 459122

===== CANDIDATE FEATURE RESULTS =====
target  horizon                 feature  days  mean_ic  median_ic  std_ic  sign_consistency
absret       30     vol_depth_stress_60    16  0.24239    0.23319 0.06659            1.0000
absret       30     spread_vol_ratio_60    16 -0.23611   -0.25500 0.06008            1.0000
absret       30    spread_vol_ratio_300    16 -0.18629   -0.19219 0.05595            1.0000
absret       30    vol_depth_stress_300    16  0.18585    0.16847 0.07845            1.0000
absret       30    spread_vol_ratio_900    16 -0.14065   -0.15296 0.05727            1.0000
absret       30    vol_depth_stress_900    16  0.13858    0.11423 0.08837            1.0000
absret       30 spread_depth_stress_900    16 -0.05016   -0.03841 0.06503            0.8125
absret       30  spread_depth_stress_60    16 -0.04276   -0.03848 0.06905            0.8125
absret       30 spread_depth_stress_300    16 -0.04068   -0.03535 0.06588            0.812

In [40]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 21B — CONDITIONAL LIQUIDITY FEATURE ABLATION
#
# Run AFTER Experiment 21A.
#
# Inputs:
#   calendar_exp21_features
#
# Outputs:
#   calendar_exp21b_daily
#   calendar_exp21b_summary
#   calendar_exp21b_session_audit
#
# Compares:
#   Base: raw spread, depth, volatility, clock
#   Plus: one candidate at a time
#
# No changes to df_1s.
# ============================================================

if "calendar_exp21_features" not in globals():
    raise ValueError("Run Experiment 21A first.")

x = calendar_exp21_features.copy()

HORIZONS = [30, 60, 300]
WINDOWS = [60, 300, 900]
MIN_TRAIN_DATES = 5
TRAIN_STRIDE = 10
RIDGE_ALPHA = 0.01

CANDIDATES = [
    f"{family}_{w}"
    for family in [
        "vol_depth_stress",
        "spread_vol_ratio",
        "spread_depth_stress"
    ]
    for w in WINDOWS
]

# ------------------------------------------------------------
# 1. Confirm data integrity
# ------------------------------------------------------------

required = [
    "symbol", "date", "ts_sec",
    "spread_bps", "depth_l5"
] + CANDIDATES

required += [
    f"rv_{w}" for w in WINDOWS
]

required += [
    f"depth_med_{w}" for w in WINDOWS
]

required += [
    f"future_absret_{h}" for h in HORIZONS
]

required += [
    f"future_spread_{h}" for h in HORIZONS
]

missing = [c for c in required if c not in x.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp records.")

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

x["clock_seconds"] = (
    x["ts_sec"] % 86400 + 19800
) % 86400 - 9 * 3600

# Check clock derived from UTC timestamp in IST.
if not x["clock_seconds"].between(0, 28800).all():
    raise ValueError("Unexpected IST session clock values.")

x["open_first_30s"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m"] = (
    (x["clock_seconds"] >= 60)
    & (x["clock_seconds"] < 300)
).astype(float)

# ------------------------------------------------------------
# 2. Audit eligible sessions
# ------------------------------------------------------------

calendar_exp21b_session_audit = (
    x.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        first_timestamp=("ts_sec", "min"),
        last_timestamp=("ts_sec", "max")
    )
    .reset_index()
)

print("\n===== SESSION AUDIT =====")
print(
    calendar_exp21b_session_audit.to_string(
        index=False
    )
)

print(
    "\nUnique eligible sessions:",
    len(calendar_exp21b_session_audit)
)

print(
    "Unique evaluation dates available:",
    x["date"].nunique()
)

# ------------------------------------------------------------
# 3. Common component baseline
# ------------------------------------------------------------

BASE = [
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "depth_med_60",
    "depth_med_300",
    "depth_med_900",
    "clock_seconds",
    "open_first_30s",
    "open_1_5m"
]

ALL_FEATURES = BASE + CANDIDATES

for col in ALL_FEATURES:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# 4. Ridge regression with train-only preprocessing
# ------------------------------------------------------------

def ridge_predict(train, test, features, target):

    A = train[features].to_numpy(dtype=float)
    B = test[features].to_numpy(dtype=float)

    y = train[target].to_numpy(dtype=float)

    A[~np.isfinite(A)] = np.nan
    B[~np.isfinite(B)] = np.nan

    # Median imputation from training only.
    with np.errstate(all="ignore"):
        med = np.nanmedian(A, axis=0)

    med[~np.isfinite(med)] = 0.0

    A = np.where(np.isfinite(A), A, med)
    B = np.where(np.isfinite(B), B, med)

    mu = A.mean(axis=0)
    sd = A.std(axis=0)

    sd[sd < 1e-12] = 1.0

    A = (A - mu) / sd
    B = (B - mu) / sd

    ybar = y.mean()
    yy = y - ybar

    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A
        + RIDGE_ALPHA * n * np.eye(k),
        A.T @ yy
    )

    prediction = ybar + B @ beta

    # Nonnegative outcome forecasts.
    return np.maximum(prediction, 0.0)

# ------------------------------------------------------------
# 5. Chronological expanding-window evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DATES:
    raise ValueError("Too few evaluation dates.")

records = []

for i in range(MIN_TRAIN_DATES, len(dates)):

    test_date = dates[i]
    training_dates = dates[:i]

    train_all = x[
        x["date"].isin(training_dates)
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test_all = x[x["date"] == test_date]

    print(
        f"Evaluating {test_date} | "
        f"train dates={len(training_dates)}"
    )

    for h in HORIZONS:

        for task in ["absret", "spread"]:

            target = f"future_{task}_{h}"

            # Common outcome-valid rows.
            train = train_all[
                np.isfinite(train_all[target])
            ]

            test = test_all[
                np.isfinite(test_all[target])
            ]

            if len(train) < 1000 or len(test) < 200:
                continue

            y = test[target].to_numpy(dtype=float)

            base_prediction = ridge_predict(
                train, test, BASE, target
            )

            base_error = np.abs(y - base_prediction)

            base_mae = float(base_error.mean())

            for candidate in CANDIDATES:

                prediction = ridge_predict(
                    train,
                    test,
                    BASE + [candidate],
                    target
                )

                candidate_mae = float(
                    np.abs(y - prediction).mean()
                )

                gain_pct = (
                    100 * (base_mae - candidate_mae)
                    / base_mae
                    if base_mae > 0
                    else np.nan
                )

                records.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "candidate": candidate,
                    "test_rows": len(test),
                    "base_mae": base_mae,
                    "candidate_mae": candidate_mae,
                    "gain_pct": gain_pct
                })

calendar_exp21b_daily = pd.DataFrame(records)

if calendar_exp21b_daily.empty:
    raise ValueError("No valid evaluations.")

# ------------------------------------------------------------
# 6. Session-level results
# ------------------------------------------------------------

calendar_exp21b_summary = (
    calendar_exp21b_daily
    .groupby(["task", "horizon", "candidate"])
    .agg(
        days=("date", "nunique"),
        mean_gain_pct=("gain_pct", "mean"),
        median_gain_pct=("gain_pct", "median"),
        winning_days=(
            "gain_pct",
            lambda z: int((z > 0).sum())
        ),
        worst_day_pct=("gain_pct", "min"),
        best_day_pct=("gain_pct", "max")
    )
    .reset_index()
)

calendar_exp21b_summary["win_fraction"] = (
    calendar_exp21b_summary["winning_days"]
    / calendar_exp21b_summary["days"]
)

calendar_exp21b_summary["positive_screen"] = (
    (calendar_exp21b_summary["median_gain_pct"] > 0)
    & (calendar_exp21b_summary["win_fraction"] >= 0.7)
)

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

print("\n===== EXPERIMENT 21B SUMMARY =====")

print(
    calendar_exp21b_summary[
        [
            "task",
            "horizon",
            "candidate",
            "days",
            "mean_gain_pct",
            "median_gain_pct",
            "win_fraction",
            "worst_day_pct",
            "positive_screen"
        ]
    ].round(5).to_string(index=False)
)

print("\n===== POSITIVE CANDIDATES =====")

positive = calendar_exp21b_summary[
    calendar_exp21b_summary["positive_screen"]
]

if positive.empty:
    print("None passed the preliminary screen.")
else:
    print(
        positive[
            [
                "task", "horizon", "candidate",
                "median_gain_pct", "win_fraction"
            ]
        ].round(5).to_string(index=False)
    )

print("\n===== EXPERIMENT 21B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")



===== SESSION AUDIT =====
        symbol       date  rows  first_timestamp  last_timestamp
USDINR26OCTFUT 2026-09-30 28749       1790739000      1790767748
USDINR26OCTFUT 2026-10-01 28402       1790825780      1790854181
USDINR26OCTFUT 2026-10-05 28767       1791171000      1791199766
USDINR26OCTFUT 2026-10-06 28755       1791257400      1791286154
USDINR26OCTFUT 2026-10-07 28752       1791343800      1791372551
USDINR26OCTFUT 2026-10-08 28749       1791430200      1791458948
USDINR26OCTFUT 2026-10-09 28748       1791516600      1791545347
USDINR26OCTFUT 2026-10-12 28754       1791775800      1791804553
USDINR26OCTFUT 2026-10-13 28793       1791862200      1791890992
USDINR26OCTFUT 2026-10-14 28792       1791948600      1791977391
USDINR26OCTFUT 2026-10-15 28797       1792035000      1792063796
USDINR26OCTFUT 2026-10-16 28796       1792121400      1792150195
USDINR26OCTFUT 2026-10-19 28784       1792380600      1792409383
USDINR26SEPFUT 2026-09-23 28437       1790134435      179016287

In [41]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 21C — ROBUSTNESS OF SPREAD-DEPTH STRESS
#
# Inputs: calendar_exp21b_daily
# No changes to original data
# ============================================================

d = calendar_exp21b_daily.copy()
d["date"] = pd.to_datetime(d["date"]).dt.date

selected = [
    "spread_depth_stress_60",
    "spread_depth_stress_300",
    "spread_depth_stress_900"
]

d = d[d["candidate"].isin(selected)].copy()

# 1. Detailed per-day gains at key horizons
print("\n===== DAILY ABSOLUTE-RETURN GAINS =====")

sub = d[
    (d["task"] == "absret")
    & (d["horizon"].isin([30, 60]))
]

daily_table = sub.pivot_table(
    index=["date", "horizon"],
    columns="candidate",
    values="gain_pct"
)

print(daily_table.round(4).to_string())

# 2. Pooled MAE gain, weighting by number of rows
#    and the underlying baseline error.
print("\n===== POOLED ERROR IMPROVEMENTS =====")

records = []

for (task, h, feat), g in d.groupby(
    ["task", "horizon", "candidate"]
):
    weighted_base = (
        g["base_mae"] * g["test_rows"]
    ).sum()

    weighted_new = (
        g["candidate_mae"] * g["test_rows"]
    ).sum()

    pooled_gain = (
        100 * (1 - weighted_new / weighted_base)
        if weighted_base > 0 else np.nan
    )

    records.append({
        "task": task,
        "horizon": h,
        "candidate": feat,
        "days": len(g),
        "pooled_gain_pct": pooled_gain,
        "positive_days": int((g["gain_pct"] > 0).sum())
    })

calendar_exp21c_pooled = pd.DataFrame(records)

print(
    calendar_exp21c_pooled
    .round(5).to_string(index=False)
)

# 3. Leave-one-evaluation-day-out pooled improvements
print("\n===== LEAVE-ONE-DAY-OUT SENSITIVITY =====")

loo_records = []

for (task, h, feat), g in d.groupby(
    ["task", "horizon", "candidate"]
):
    for excluded in sorted(g["date"].unique()):

        remaining = g[g["date"] != excluded]

        base = (
            remaining["base_mae"]
            * remaining["test_rows"]
        ).sum()

        new = (
            remaining["candidate_mae"]
            * remaining["test_rows"]
        ).sum()

        gain = 100 * (1 - new / base)

        loo_records.append({
            "task": task,
            "horizon": h,
            "candidate": feat,
            "excluded": excluded,
            "pooled_gain_pct": gain
        })

calendar_exp21c_loo = pd.DataFrame(loo_records)

loo_summary = (
    calendar_exp21c_loo
    .groupby(["task", "horizon", "candidate"])
    .agg(
        minimum_gain=("pooled_gain_pct", "min"),
        median_gain=("pooled_gain_pct", "median"),
        maximum_gain=("pooled_gain_pct", "max")
    )
    .reset_index()
)

print(loo_summary.round(5).to_string(index=False))

# 4. Direct paired comparison: 900s vs 300s
#    Both evaluated on the same dates.
print("\n===== 900s VS 300s PARAMETER COMPARISON =====")

pair = d[
    d["candidate"].isin([
        "spread_depth_stress_300",
        "spread_depth_stress_900"
    ])
].pivot(
    index=["date", "task", "horizon"],
    columns="candidate",
    values="candidate_mae"
).dropna()

pair["gain_900_vs_300_pct"] = 100 * (
    1 - pair["spread_depth_stress_900"]
    / pair["spread_depth_stress_300"]
)

calendar_exp21c_pair = pair.reset_index()

pair_summary = (
    calendar_exp21c_pair
    .groupby(["task", "horizon"])
    .agg(
        days=("date", "nunique"),
        mean_gain=("gain_900_vs_300_pct", "mean"),
        median_gain=("gain_900_vs_300_pct", "median"),
        days_900_wins=(
            "gain_900_vs_300_pct",
            lambda s: int((s > 0).sum())
        )
    )
    .reset_index()
)

print(pair_summary.round(5).to_string(index=False))

# 5. Exclude notable sessions
print("\n===== EXCLUSION SENSITIVITY =====")

for exclude in [
    ["2026-10-12"],
    ["2026-10-19"],
    ["2026-10-12", "2026-10-19"]
]:
    remaining = d[
        ~d["date"].astype(str).isin(exclude)
    ]

    results = []

    for (task, h, feat), g in remaining.groupby(
        ["task", "horizon", "candidate"]
    ):
        base = (g["base_mae"] * g["test_rows"]).sum()
        new = (g["candidate_mae"] * g["test_rows"]).sum()

        results.append({
            "task": task,
            "horizon": h,
            "candidate": feat,
            "pooled_gain_pct": 100 * (1 - new / base)
        })

    print("\nExcluded:", exclude)
    print(
        pd.DataFrame(results)
        .round(5).to_string(index=False)
    )

print("\n===== EXPERIMENT 21C COMPLETE =====")
print("No feature added to the main 28.")



===== DAILY ABSOLUTE-RETURN GAINS =====
candidate           spread_depth_stress_300  spread_depth_stress_60  spread_depth_stress_900
date       horizon                                                                          
2026-10-05 30                        0.0021                 -0.0366                   0.0068
           60                       -0.0214                 -0.0246                   0.0074
2026-10-06 30                        0.0066                 -0.1662                  -0.0737
           60                       -0.0683                 -0.1358                  -0.0241
2026-10-07 30                        0.0153                  0.0516                   0.0484
           60                        0.0276                  0.0457                   0.0341
2026-10-08 30                        0.5448                  0.5701                   0.8792
           60                        0.6374                  0.5228                   1.0107
2026-10-09 30                

In [42]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 21D — RECENT-SESSION STABILITY
#
# Uses already-computed 21A and 21B outputs.
# No model retraining and no df_1s modification.
# ============================================================

assert "calendar_exp21b_daily" in globals()
assert "calendar_exp21_features" in globals()

daily = calendar_exp21b_daily.copy()
features = calendar_exp21_features.copy()

daily["date"] = pd.to_datetime(daily["date"]).dt.date
features["date"] = pd.to_datetime(
    features["date"]
).dt.date

candidate = "spread_depth_stress_900"

# ------------------------------------------------------------
# 1. Chronological change in incremental model performance
# ------------------------------------------------------------

d = daily[
    (daily["candidate"] == candidate)
    & (daily["task"].isin(["absret", "spread"]))
    & (daily["horizon"].isin([30, 60]))
].copy()

dates = sorted(d["date"].unique())

# Purely diagnostic split, not a new training rule.
midpoint = len(dates) // 2

early_dates = set(dates[:midpoint])
late_dates = set(dates[midpoint:])

d["period"] = np.where(
    d["date"].isin(early_dates),
    "Earlier evaluation days",
    "Later evaluation days"
)

records = []

for (task, horizon, period), g in d.groupby(
    ["task", "horizon", "period"]
):

    base_error = (
        g["base_mae"] * g["test_rows"]
    ).sum()

    new_error = (
        g["candidate_mae"] * g["test_rows"]
    ).sum()

    records.append({
        "task": task,
        "horizon": horizon,
        "period": period,
        "days": g["date"].nunique(),
        "pooled_gain_pct": (
            100 * (1 - new_error / base_error)
            if base_error > 0 else np.nan
        ),
        "winning_days": int(
            (g["gain_pct"] > 0).sum()
        )
    })

calendar_exp21d_periods = pd.DataFrame(records)

print("\n===== EARLY VS LATE EVALUATION =====")
print(
    calendar_exp21d_periods.round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 2. Characterize recent market conditions
# ------------------------------------------------------------

feature_columns = [
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "spread_depth_stress_900"
]

for col in feature_columns:
    features[col] = pd.to_numeric(
        features[col],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

diagnostic_records = []

for date, g in features.groupby("date"):

    row = {
        "date": date,
        "rows": len(g),
        "spread_median": g["spread_bps"].median(),
        "spread_p90": g["spread_bps"].quantile(0.90),
        "depth_median": g["depth_l5"].median(),
        "rv60_median": g["rv_60"].median(),
        "rv300_median": g["rv_300"].median(),
        "stress900_median":
            g["spread_depth_stress_900"].median(),
        "stress900_p90":
            g["spread_depth_stress_900"].quantile(0.90)
    }

    diagnostic_records.append(row)

calendar_exp21d_market = (
    pd.DataFrame(diagnostic_records)
    .sort_values("date")
)

print("\n===== DAILY MARKET CONDITIONS =====")
print(
    calendar_exp21d_market.round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 3. October 19 versus preceding October sessions
# ------------------------------------------------------------

reference = calendar_exp21d_market[
    (calendar_exp21d_market["date"].astype(str)
     >= "2026-10-05")
    & (calendar_exp21d_market["date"].astype(str)
       < "2026-10-19")
]

recent = calendar_exp21d_market[
    calendar_exp21d_market["date"].astype(str)
    == "2026-10-19"
]

if not reference.empty and not recent.empty:

    print("\n===== OCT 19 VS PREVIOUS OCT SESSIONS =====")

    for col in [
        "spread_median",
        "spread_p90",
        "depth_median",
        "rv60_median",
        "rv300_median",
        "stress900_median",
        "stress900_p90"
    ]:

        historical = reference[col].median()
        current = recent[col].iloc[0]

        ratio = (
            current / historical
            if historical != 0 and np.isfinite(historical)
            else np.nan
        )

        print(
            f"{col:22s} "
            f"historical={historical:.6f} "
            f"oct19={current:.6f} "
            f"ratio={ratio:.3f}"
        )

print("\n===== EXPERIMENT 21D COMPLETE =====")
print("Original df_1s unchanged.")
print("No features promoted.")



===== EARLY VS LATE EVALUATION =====
  task  horizon                  period  days  pooled_gain_pct  winning_days
absret       30 Earlier evaluation days     5          0.24000             4
absret       30   Later evaluation days     6          1.21717             5
absret       60 Earlier evaluation days     5          0.25347             4
absret       60   Later evaluation days     6          0.84035             5
spread       30 Earlier evaluation days     5         -0.18661             2
spread       30   Later evaluation days     6          1.95310             6
spread       60 Earlier evaluation days     5         -0.17936             2
spread       60   Later evaluation days     6          0.88087             6

===== DAILY MARKET CONDITIONS =====
      date  rows  spread_median  spread_p90  depth_median  rv60_median  rv300_median  stress900_median  stress900_p90
2026-09-23 28437        0.78383     1.56855        2779.0      0.26149       1.23174           0.01441        0.03

In [43]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 21E
# CONDITIONAL VALUE OF SPREAD-DEPTH STRESS
#
# Baseline vs +300s vs +900s
# Targets: future absolute returns / future spread
# Horizons: 30s and 60s
#
# Chronological expanding window.
# Regime thresholds fitted on training dates only.
# No change to df_1s or previous experiment outputs.
# ============================================================

if "calendar_exp21_features" not in globals():
    raise ValueError("Run Experiment 21A first.")

calendar_exp21e_work = calendar_exp21_features.copy()

HORIZONS_21E = [30, 60]
TRAIN_DAYS_21E = 5
TRAIN_STRIDE_21E = 10
ALPHA_21E = 0.01

BASE_21E = [
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "depth_med_60",
    "depth_med_300",
    "depth_med_900",
    "clock_seconds",
    "open_first_30s",
    "open_1_5m"
]

CANDIDATES_21E = [
    "spread_depth_stress_300",
    "spread_depth_stress_900"
]

x = calendar_exp21e_work

x["date"] = pd.to_datetime(x["date"]).dt.date
x["ts_sec"] = pd.to_numeric(x["ts_sec"])

# Exact IST clock
x["clock_seconds"] = (
    (x["ts_sec"] + 19800) % 86400
) - 9 * 3600

if not x["clock_seconds"].between(
    0, 28800, inclusive="left"
).all():
    raise ValueError("Invalid session clock.")

x["open_first_30s"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m"] = (
    (x["clock_seconds"] >= 60)
    & (x["clock_seconds"] < 300)
).astype(float)

REQUIRED_21E = (
    ["date", "symbol", "ts_sec", "spread_bps", "depth_l5"]
    + BASE_21E
    + CANDIDATES_21E
    + [
        f"future_{task}_{h}"
        for task in ["absret", "spread"]
        for h in HORIZONS_21E
    ]
)

missing = [
    c for c in set(REQUIRED_21E)
    if c not in x.columns
]

if missing:
    raise ValueError(f"Missing: {missing}")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp records.")

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

for c in set(BASE_21E + CANDIDATES_21E):
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# Ridge with training-only preprocessing
# ------------------------------------------------------------

def predict_21e(train, test, features, target):

    A = train[features].to_numpy(
        dtype=float, copy=True
    )
    B = test[features].to_numpy(
        dtype=float, copy=True
    )
    y = train[target].to_numpy(dtype=float)

    A[~np.isfinite(A)] = np.nan
    B[~np.isfinite(B)] = np.nan

    # Training-only median imputation
    med = np.array([
        np.nanmedian(v) if np.isfinite(v).any()
        else 0.0
        for v in A.T
    ])

    A = np.where(np.isfinite(A), A, med)
    B = np.where(np.isfinite(B), B, med)

    mean = A.mean(axis=0)
    std = A.std(axis=0)
    std = np.where(std > 1e-12, std, 1.0)

    A = (A - mean) / std
    B = (B - mean) / std

    ymean = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A
        + ALPHA_21E * n * np.eye(k),
        A.T @ (y - ymean)
    )

    return np.maximum(
        ymean + B @ beta, 0.0
    )

# ------------------------------------------------------------
# Chronological evaluation
# ------------------------------------------------------------

all_dates = sorted(x["date"].unique())

if len(all_dates) <= TRAIN_DAYS_21E:
    raise ValueError("Not enough trading dates.")

records = []
threshold_records = []

for i in range(TRAIN_DAYS_21E, len(all_dates)):

    test_date = all_dates[i]

    train_all = x[
        x["date"].isin(all_dates[:i])
        & (x["ts_sec"] % TRAIN_STRIDE_21E == 0)
    ]

    test_all = x[x["date"] == test_date]

    # Regime thresholds based on training data only.
    train_spread = train_all["spread_bps"].dropna()
    train_depth = train_all["depth_l5"].dropna()

    if train_spread.empty or train_depth.empty:
        continue

    spread_q75 = train_spread.quantile(0.75)
    depth_q25 = train_depth.quantile(0.25)

    threshold_records.append({
        "date": test_date,
        "spread_q75": spread_q75,
        "depth_q25": depth_q25
    })

    print("Evaluating:", test_date)

    for h in HORIZONS_21E:
        for task in ["absret", "spread"]:

            target = f"future_{task}_{h}"

            train = train_all[
                np.isfinite(train_all[target])
            ]

            test = test_all[
                np.isfinite(test_all[target])
            ].copy()

            if len(train) < 1000 or len(test) < 200:
                continue

            actual = test[target].to_numpy(dtype=float)

            preds = {
                "baseline": predict_21e(
                    train, test, BASE_21E, target
                )
            }

            for candidate in CANDIDATES_21E:
                preds[candidate] = predict_21e(
                    train,
                    test,
                    BASE_21E + [candidate],
                    target
                )

            # Regimes assigned solely using live-time
            # observable quotes and historical thresholds.
            high_spread = (
                test["spread_bps"] >= spread_q75
            ).fillna(False).to_numpy()

            low_depth = (
                test["depth_l5"] <= depth_q25
            ).fillna(False).to_numpy()

            regimes = {
                "all": np.ones(
                    len(test), dtype=bool
                ),
                "high_spread": high_spread,
                "low_depth": low_depth,
                "either_stressed": (
                    high_spread | low_depth
                ),
                "both_stressed": (
                    high_spread & low_depth
                ),
                "neither_stressed": (
                    ~high_spread & ~low_depth
                )
            }

            base_error = np.abs(
                actual - preds["baseline"]
            )

            for regime_name, mask in regimes.items():

                n = int(mask.sum())

                if n < 100:
                    continue

                base_mae = base_error[mask].mean()

                if not np.isfinite(base_mae) or base_mae <= 0:
                    continue

                for candidate in CANDIDATES_21E:

                    new_error = np.abs(
                        actual - preds[candidate]
                    )

                    new_mae = new_error[mask].mean()

                    records.append({
                        "date": test_date,
                        "task": task,
                        "horizon": h,
                        "regime": regime_name,
                        "candidate": candidate,
                        "rows": n,
                        "base_mae": base_mae,
                        "new_mae": new_mae,
                        "gain_pct": (
                            100 * (1 - new_mae / base_mae)
                        )
                    })

calendar_exp21e_daily = pd.DataFrame(records)

calendar_exp21e_thresholds = pd.DataFrame(
    threshold_records
)

if calendar_exp21e_daily.empty:
    raise ValueError("No valid evaluations.")

# ------------------------------------------------------------
# Aggregate exact paired errors within regimes
# ------------------------------------------------------------

result_rows = []

for keys, g in calendar_exp21e_daily.groupby(
    ["task", "horizon", "regime", "candidate"]
):

    base_total = (
        g["base_mae"] * g["rows"]
    ).sum()

    new_total = (
        g["new_mae"] * g["rows"]
    ).sum()

    result_rows.append({
        "task": keys[0],
        "horizon": keys[1],
        "regime": keys[2],
        "candidate": keys[3],
        "days": g["date"].nunique(),
        "rows": int(g["rows"].sum()),
        "pooled_gain_pct": (
            100 * (1 - new_total / base_total)
            if base_total > 0 else np.nan
        ),
        "median_daily_gain": g["gain_pct"].median(),
        "winning_days": int(
            (g["gain_pct"] > 0).sum()
        )
    })

calendar_exp21e_summary = pd.DataFrame(
    result_rows
)

calendar_exp21e_summary["win_fraction"] = (
    calendar_exp21e_summary["winning_days"]
    / calendar_exp21e_summary["days"]
)

# ------------------------------------------------------------
# Print manageable targeted reports
# ------------------------------------------------------------

print("\n===== TRAINING-ONLY REGIME THRESHOLDS =====")

print(
    calendar_exp21e_thresholds
    .round(5).to_string(index=False)
)

for task in ["absret", "spread"]:
    for h in HORIZONS_21E:

        sub = calendar_exp21e_summary[
            (calendar_exp21e_summary["task"] == task)
            & (calendar_exp21e_summary["horizon"] == h)
        ]

        print(
            f"\n===== {task.upper()} | {h}s | "
            "CONDITIONAL RESULTS ====="
        )

        print(
            sub[
                [
                    "regime",
                    "candidate",
                    "days",
                    "rows",
                    "pooled_gain_pct",
                    "median_daily_gain",
                    "win_fraction"
                ]
            ].round(5).to_string(index=False)
        )

print("\n===== OCTOBER 19 DIAGNOSTIC =====")

oct19 = calendar_exp21e_daily[
    calendar_exp21e_daily["date"].astype(str)
    == "2026-10-19"
]

print(
    oct19[
        [
            "task", "horizon", "regime",
            "candidate", "rows", "gain_pct"
        ]
    ].round(5).to_string(index=False)
)

print("\n===== EXPERIMENT 21E COMPLETE =====")
print("No master data modified.")
print("No candidate promoted.")


Evaluating: 2026-10-05
Evaluating: 2026-10-06
Evaluating: 2026-10-07
Evaluating: 2026-10-08
Evaluating: 2026-10-09
Evaluating: 2026-10-12
Evaluating: 2026-10-13
Evaluating: 2026-10-14
Evaluating: 2026-10-15
Evaluating: 2026-10-16
Evaluating: 2026-10-19

===== TRAINING-ONLY REGIME THRESHOLDS =====
      date  spread_q75  depth_q25
2026-10-05     1.29943     2484.0
2026-10-06     1.29531     2645.0
2026-10-07     1.04570     2691.0
2026-10-08     1.28821     2703.0
2026-10-09     1.04466     2756.0
2026-10-12     1.04466     2785.0
2026-10-13     1.28748     2700.0
2026-10-14     1.28954     2585.0
2026-10-15     1.29539     2365.0
2026-10-16     1.29980     2267.0
2026-10-19     1.30247     2202.0

===== ABSRET | 30s | CONDITIONAL RESULTS =====
          regime               candidate  days   rows  pooled_gain_pct  median_daily_gain  win_fraction
             all spread_depth_stress_300    11 316157          0.67497            0.28097       0.90909
             all spread_depth_stress_9

In [44]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 21F
# GLOBAL VS REGIME-GATED LIQUIDITY STRESS
#
# Requires calendar_exp21_features from Experiment 21A
#
# Compares:
#  1. Nonlinear component baseline
#  2. Baseline + global stress900
#  3. Baseline + stress900 gated to either stressed
#  4. Baseline + stress900 gated to both stressed
#  5. Baseline + global stress300
#
# Thresholds and preprocessing fitted using past days.
# No changes to df_1s.
# ============================================================

if "calendar_exp21_features" not in globals():
    raise ValueError("Run Experiment 21A first")

calendar_exp21f_work = calendar_exp21_features.copy()

x = calendar_exp21f_work

HORIZONS = [30, 60]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10
RIDGE_ALPHA = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date

x["clock_seconds"] = (
    (pd.to_numeric(x["ts_sec"]) + 19800) % 86400
) - 9 * 3600

x["open_first_30s"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m"] = (
    (x["clock_seconds"] >= 60)
    & (x["clock_seconds"] < 300)
).astype(float)

# ------------------------------------------------------------
# 1. Nonlinear component baseline
# ------------------------------------------------------------

spread = pd.to_numeric(
    x["spread_bps"], errors="coerce"
).clip(lower=0)

depth900 = pd.to_numeric(
    x["depth_med_900"], errors="coerce"
).clip(lower=0)

x["log_spread"] = np.log1p(spread)
x["log_depth900"] = np.log1p(depth900)

# An unlogged spread-depth interaction:
x["spread_depth_interaction"] = (
    spread / np.sqrt(depth900 + 1.0)
)

BASE = [
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "depth_med_60",
    "depth_med_300",
    "depth_med_900",
    "clock_seconds",
    "open_first_30s",
    "open_1_5m",
    "log_spread",
    "log_depth900",
    "spread_depth_interaction",
    "high_spread_flag",
    "low_depth_flag",
    "both_stressed_flag"
]

MODEL_ADDITIONS = {
    "baseline": [],
    "global_900": ["spread_depth_stress_900"],
    "gated_either_900": ["gated_either_900"],
    "gated_both_900": ["gated_both_900"],
    "global_300": ["spread_depth_stress_300"]
}

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

dates = sorted(x["date"].unique())

# ------------------------------------------------------------
# 2. Ridge fit: all preprocessing from training only
# ------------------------------------------------------------

def ridge_21f(train, test, cols, target):

    A = train[cols].to_numpy(dtype=float, copy=True)
    B = test[cols].to_numpy(dtype=float, copy=True)
    y = train[target].to_numpy(dtype=float)

    A[~np.isfinite(A)] = np.nan
    B[~np.isfinite(B)] = np.nan

    med = np.array([
        np.nanmedian(v)
        if np.isfinite(v).any() else 0.0
        for v in A.T
    ])

    A = np.where(np.isfinite(A), A, med)
    B = np.where(np.isfinite(B), B, med)

    mean = A.mean(axis=0)
    std = A.std(axis=0)

    std = np.where(std > 1e-12, std, 1.0)

    A = (A - mean) / std
    B = (B - mean) / std

    ymean = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A + RIDGE_ALPHA * n * np.eye(k),
        A.T @ (y - ymean)
    )

    return np.maximum(ymean + B @ beta, 0.0)

# ------------------------------------------------------------
# 3. Expanding-window model comparison
# ------------------------------------------------------------

records = []
regime_records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    day = dates[i]

    train = x[
        x["date"].isin(dates[:i])
    ].copy()

    test = x[x["date"] == day].copy()

    # Thresholds based on PREVIOUS days only.
    sampled_train = train[
        train["ts_sec"] % TRAIN_STRIDE == 0
    ]

    spread_q75 = sampled_train["spread_bps"].quantile(0.75)
    depth_q25 = sampled_train["depth_l5"].quantile(0.25)

    if not np.isfinite(spread_q75) or not np.isfinite(depth_q25):
        continue

    for frame in (train, test):

        frame["high_spread_flag"] = (
            frame["spread_bps"] >= spread_q75
        ).astype(float)

        frame["low_depth_flag"] = (
            frame["depth_l5"] <= depth_q25
        ).astype(float)

        frame["both_stressed_flag"] = (
            (frame["high_spread_flag"] == 1)
            & (frame["low_depth_flag"] == 1)
        ).astype(float)

        frame["either_stressed_flag"] = (
            (frame["high_spread_flag"] == 1)
            | (frame["low_depth_flag"] == 1)
        ).astype(float)

        frame["gated_either_900"] = (
            frame["spread_depth_stress_900"]
            * frame["either_stressed_flag"]
        )

        frame["gated_both_900"] = (
            frame["spread_depth_stress_900"]
            * frame["both_stressed_flag"]
        )

    train = train[
        train["ts_sec"] % TRAIN_STRIDE == 0
    ]

    for h in HORIZONS:

        for task in ["absret", "spread"]:

            target = f"future_{task}_{h}"

            tr = train[
                np.isfinite(train[target])
            ]

            te = test[
                np.isfinite(test[target])
            ]

            if len(tr) < 1000 or len(te) < 200:
                continue

            y = te[target].to_numpy(dtype=float)

            predictions = {}

            for model, additions in MODEL_ADDITIONS.items():

                predictions[model] = ridge_21f(
                    tr,
                    te,
                    BASE + additions,
                    target
                )

            base_err = np.abs(
                y - predictions["baseline"]
            )

            # Overall results.
            for model in MODEL_ADDITIONS:

                err = np.abs(y - predictions[model])

                records.append({
                    "date": day,
                    "task": task,
                    "horizon": h,
                    "model": model,
                    "rows": len(te),
                    "base_error_sum": base_err.sum(),
                    "model_error_sum": err.sum(),
                    "base_mae": base_err.mean(),
                    "model_mae": err.mean()
                })

            high = (
                te["high_spread_flag"].to_numpy()
                == 1
            )

            low = (
                te["low_depth_flag"].to_numpy()
                == 1
            )

            masks = {
                "both_stressed": high & low,
                "either_stressed": high | low,
                "neither_stressed": ~high & ~low
            }

            # Within-regime predictions use the
            # same fitted models as the overall test.
            for regime, mask in masks.items():

                if mask.sum() < 100:
                    continue

                for model in MODEL_ADDITIONS:

                    err = np.abs(
                        y - predictions[model]
                    )

                    regime_records.append({
                        "date": day,
                        "task": task,
                        "horizon": h,
                        "regime": regime,
                        "model": model,
                        "rows": int(mask.sum()),
                        "base_error_sum": base_err[mask].sum(),
                        "model_error_sum": err[mask].sum()
                    })

    print("Completed:", day)

calendar_exp21f_daily = pd.DataFrame(records)
calendar_exp21f_regimes = pd.DataFrame(regime_records)

# ------------------------------------------------------------
# 4. Summarize results
# ------------------------------------------------------------

if calendar_exp21f_daily.empty:
    raise ValueError("No valid model comparisons")

def summarize_21f(frame, keys):

    z = frame.copy()

    if "base_mae" in z.columns:
        z["daily_gain_pct"] = (
            100 * (
                1 - z["model_mae"] / z["base_mae"]
            )
        )
    else:
        z["daily_gain_pct"] = (
            100 * (
                1 - z["model_error_sum"]
                / z["base_error_sum"]
            )
        )

    out = (
        z.groupby(keys)
        .agg(
            days=("date", "nunique"),
            winning_days=(
                "daily_gain_pct",
                lambda s: int((s > 0).sum())
            ),
            base_total=("base_error_sum", "sum"),
            model_total=("model_error_sum", "sum")
        )
        .reset_index()
    )

    out["pooled_gain_pct"] = (
        100 * (
            1 - out["model_total"]
            / out["base_total"]
        )
    )

    out["win_fraction"] = (
        out["winning_days"] / out["days"]
    )

    return out.drop(
        columns=["base_total", "model_total"]
    )

calendar_exp21f_summary = summarize_21f(
    calendar_exp21f_daily,
    ["task", "horizon", "model"]
)

calendar_exp21f_regime_summary = summarize_21f(
    calendar_exp21f_regimes,
    ["task", "horizon", "regime", "model"]
)

print("\n===== OVERALL MODEL COMPARISON =====")

print(
    calendar_exp21f_summary[
        calendar_exp21f_summary["model"] != "baseline"
    ].round(5).to_string(index=False)
)

print("\n===== CONDITIONAL ABSOLUTE-RETURN RESULTS =====")

conditional = calendar_exp21f_regime_summary[
    (calendar_exp21f_regime_summary["task"] == "absret")
    & (calendar_exp21f_regime_summary["model"] != "baseline")
]

print(
    conditional.round(5).to_string(index=False)
)

print("\n===== OCTOBER 19 RESULTS =====")

oct19 = calendar_exp21f_daily[
    calendar_exp21f_daily["date"].astype(str)
    == "2026-10-19"
].copy()

oct19["gain_pct"] = (
    100 * (
        1 - oct19["model_mae"] / oct19["base_mae"]
    )
)

print(
    oct19[
        ["task", "horizon", "model", "gain_pct"]
    ].round(5).to_string(index=False)
)

print("\n===== EXPERIMENT 21F COMPLETE =====")
print("No original dataframe modified.")
print("No feature promoted.")


Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== OVERALL MODEL COMPARISON =====
  task  horizon            model  days  winning_days  pooled_gain_pct  win_fraction
absret       30   gated_both_900    11             5         -0.00306       0.45455
absret       30 gated_either_900    11             5         -0.00978       0.45455
absret       30       global_300    11             6         -0.00922       0.54545
absret       30       global_900    11             9          0.01053       0.81818
absret       60   gated_both_900    11             5         -0.01383       0.45455
absret       60 gated_either_900    11             6         -0.00008       0.54545
absret       60       global_300    11             2         -0.01951       0.18182
absret       60       global_900    11          

In [45]:

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except ImportError:
    raise ImportError(
        "LightGBM is not installed. "
        "Run %pip install lightgbm in a notebook cell."
    )

# ============================================================
# EXPERIMENT 21G
# NONLINEAR LIQUIDITY FEATURE ABLATION
#
# Requires:
#   calendar_exp21_features (from Experiment 21A)
#
# Models:
#   A: Component baseline
#   B: Baseline + log spread/depth stress (900s)
#   C: Baseline + raw spread/depth interaction
#   D: Baseline + both
#
# Targets:
#   Future absolute mid-price return (bps)
#   Future spread (bps)
#
# Horizons:
#   30s, 60s
#
# No df_1s modification.
# ============================================================

if "calendar_exp21_features" not in globals():
    raise ValueError(
        "calendar_exp21_features missing. "
        "Run Experiment 21A first."
    )

calendar_exp21g_work = calendar_exp21_features.copy()

HORIZONS_21G = [30, 60]
MIN_TRAIN_DAYS_21G = 5
TRAIN_STRIDE_21G = 10

# Fixed BEFORE this experiment.
# No hyperparameter search or early stopping.
PARAMS_21G = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

x = calendar_exp21g_work

# ------------------------------------------------------------
# 1. DATA INTEGRITY AND CLOCK FEATURES
# ------------------------------------------------------------

if not x.columns.is_unique:
    raise ValueError("Duplicate columns in source dataframe")

required_base = [
    "symbol",
    "date",
    "ts_sec",
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "depth_med_60",
    "depth_med_300",
    "depth_med_900",
    "spread_depth_stress_900"
]

required_targets = [
    f"future_{task}_{h}"
    for task in ["absret", "spread"]
    for h in HORIZONS_21G
]

missing = [
    c for c in required_base + required_targets
    if c not in x.columns
]

if missing:
    raise ValueError(
        f"Missing required fields: {missing}"
    )

x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
)

x["date"] = pd.to_datetime(x["date"]).dt.date

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp records")

x["clock_seconds"] = (
    (x["ts_sec"] + 19800) % 86400
) - 9 * 3600

if not x["clock_seconds"].between(
    0, 28800, inclusive="left"
).all():
    raise ValueError(
        "Unexpected timestamp outside IST session"
    )

x["open_first_30s"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m"] = (
    (x["clock_seconds"] >= 60)
    & (x["clock_seconds"] < 300)
).astype(float)

# ------------------------------------------------------------
# 2. RECONSTRUCT BOTH STRESS TRANSFORMATIONS
#
# Existing Experiment 21A:
# stress900 = log1p(
#     current_spread_bps /
#     sqrt(trailing_depth_median_900s + 1)
# )
#
# The new raw interaction uses EXACTLY the same inputs.
# ------------------------------------------------------------

spread = pd.to_numeric(
    x["spread_bps"], errors="coerce"
)

depth_900 = pd.to_numeric(
    x["depth_med_900"], errors="coerce"
)

x["raw_spread_depth_900"] = (
    spread.clip(lower=0)
    / np.sqrt(depth_900.clip(lower=0) + 1.0)
)

x["stress900_reconstructed"] = np.log1p(
    x["raw_spread_depth_900"]
)

original_stress = pd.to_numeric(
    x["spread_depth_stress_900"],
    errors="coerce"
)

valid = (
    np.isfinite(original_stress)
    & np.isfinite(x["stress900_reconstructed"])
)

if valid.any():

    max_difference = np.max(np.abs(
        original_stress.loc[valid].to_numpy()
        - x.loc[
            valid, "stress900_reconstructed"
        ].to_numpy()
    ))

    print(
        "Maximum feature reconstruction difference:",
        max_difference
    )

    if max_difference > 1e-9:
        raise ValueError(
            "Stress feature does not match its "
            "previously tested construction."
        )

# Also verify missing masks match.
if not np.array_equal(
    np.isfinite(original_stress),
    np.isfinite(x["stress900_reconstructed"])
):
    raise ValueError(
        "Mismatch in stress feature availability"
    )

# ------------------------------------------------------------
# 3. DEFINE FOUR MODEL CONFIGURATIONS
# ------------------------------------------------------------

BASE_21G = [
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "depth_med_60",
    "depth_med_300",
    "depth_med_900",
    "clock_seconds",
    "open_first_30s",
    "open_1_5m"
]

MODEL_FEATURES_21G = {
    "A_components": BASE_21G,

    "B_plus_stress900": (
        BASE_21G
        + ["spread_depth_stress_900"]
    ),

    "C_plus_raw_interaction": (
        BASE_21G
        + ["raw_spread_depth_900"]
    ),

    "D_plus_both": (
        BASE_21G
        + [
            "raw_spread_depth_900",
            "spread_depth_stress_900"
        ]
    )
}

for model, features in MODEL_FEATURES_21G.items():

    if len(features) != len(set(features)):
        raise ValueError(
            f"Duplicate features in {model}"
        )

ALL_INPUTS = sorted(set(
    f
    for features in MODEL_FEATURES_21G.values()
    for f in features
))

for c in ALL_INPUTS + required_targets:

    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace(
        [np.inf, -np.inf], np.nan
    )

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

print("\n===== MODEL CONFIGURATIONS =====")

for name, features in MODEL_FEATURES_21G.items():
    print(
        f"{name:26s}: {len(features)} features"
    )

# ------------------------------------------------------------
# 4. STRICT COMMON SAMPLE
#
# For each horizon/target, train and evaluate
# all models on the same rows.
#
# Require nonmissing underlying component features
# so warm-up differences cannot create artificial
# gains from different usable sample sizes.
# ------------------------------------------------------------

sample_required = sorted(set(
    BASE_21G
    + [
        "raw_spread_depth_900",
        "spread_depth_stress_900"
    ]
))

complete_inputs = np.isfinite(
    x[sample_required].to_numpy(
        dtype=np.float64
    )
).all(axis=1)

x["_inputs_complete"] = complete_inputs

print("\n===== COMMON SAMPLE AVAILABILITY =====")
print("Total rows:", len(x))
print(
    "Rows with all inputs available:",
    int(complete_inputs.sum())
)
print(
    "Coverage percentage:",
    round(100 * complete_inputs.mean(), 3)
)

# ------------------------------------------------------------
# 5. CHRONOLOGICAL EXPANDING-WINDOW TEST
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS_21G:
    raise ValueError(
        "Not enough eligible trading dates"
    )

records = []
importance_records = []

for i in range(
    MIN_TRAIN_DAYS_21G,
    len(dates)
):

    test_date = dates[i]
    train_dates = dates[:i]

    train_pool = x[
        x["date"].isin(train_dates)
        & (
            x["ts_sec"] % TRAIN_STRIDE_21G == 0
        )
        & x["_inputs_complete"]
    ]

    test_pool = x[
        (x["date"] == test_date)
        & x["_inputs_complete"]
    ]

    print(
        f"\nEvaluating {test_date}"
        f" | Previous dates: {len(train_dates)}"
    )

    for h in HORIZONS_21G:

        for task in ["absret", "spread"]:

            target = f"future_{task}_{h}"

            train = train_pool[
                np.isfinite(train_pool[target])
            ]

            test = test_pool[
                np.isfinite(test_pool[target])
            ]

            if len(train) < 1000 or len(test) < 200:
                continue

            y_train = train[target].to_numpy(
                dtype=np.float64
            )

            y_test = test[target].to_numpy(
                dtype=np.float64
            )

            predictions = {}

            for model_name, features in (
                MODEL_FEATURES_21G.items()
            ):

                model = lgb.LGBMRegressor(
                    **PARAMS_21G
                )

                model.fit(
                    train[features],
                    y_train
                )

                pred = model.predict(
                    test[features]
                )

                pred = np.maximum(
                    pred, 0.0
                )

                predictions[model_name] = pred

                # Gain-based importance is diagnostic,
                # not proof of incremental predictiveness.
                for feature, importance in zip(
                    features,
                    model.booster_.feature_importance(
                        importance_type="gain"
                    )
                ):

                    importance_records.append({
                        "date": test_date,
                        "task": task,
                        "horizon": h,
                        "model": model_name,
                        "feature": feature,
                        "gain_importance": importance
                    })

            base_prediction = predictions[
                "A_components"
            ]

            base_error = np.abs(
                y_test - base_prediction
            )

            base_mae = base_error.mean()

            for model_name, pred in (
                predictions.items()
            ):

                error = np.abs(y_test - pred)

                mae = error.mean()
                rmse = np.sqrt(
                    np.mean(
                        (y_test - pred) ** 2
                    )
                )

                records.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "train_rows": len(train),
                    "test_rows": len(test),
                    "mae": float(mae),
                    "rmse": float(rmse),
                    "error_sum": float(error.sum()),
                    "base_error_sum": float(
                        base_error.sum()
                    ),
                    "daily_gain_pct": (
                        100 * (1 - mae / base_mae)
                        if base_mae > 0
                        else np.nan
                    )
                })

# ------------------------------------------------------------
# 6. DAILY AND POOLED SUMMARY
# ------------------------------------------------------------

calendar_exp21g_daily = pd.DataFrame(records)

calendar_exp21g_importance = pd.DataFrame(
    importance_records
)

if calendar_exp21g_daily.empty:
    raise ValueError(
        "No valid model evaluations"
    )

summary_rows = []

for (task, horizon, model), g in (
    calendar_exp21g_daily.groupby(
        ["task", "horizon", "model"]
    )
):

    base_total = g["base_error_sum"].sum()
    new_total = g["error_sum"].sum()

    summary_rows.append({
        "task": task,
        "horizon": horizon,
        "model": model,
        "days": g["date"].nunique(),
        "pooled_gain_pct": (
            100 * (
                1 - new_total / base_total
            )
            if base_total > 0
            else np.nan
        ),
        "mean_daily_gain_pct": (
            g["daily_gain_pct"].mean()
        ),
        "median_daily_gain_pct": (
            g["daily_gain_pct"].median()
        ),
        "winning_days": int(
            (g["daily_gain_pct"] > 0).sum()
        ),
        "worst_day_pct": (
            g["daily_gain_pct"].min()
        )
    })

calendar_exp21g_summary = pd.DataFrame(
    summary_rows
)

calendar_exp21g_summary["win_fraction"] = (
    calendar_exp21g_summary["winning_days"]
    / calendar_exp21g_summary["days"]
)

print("\n===== LIGHTGBM OVERALL RESULTS =====")

print(
    calendar_exp21g_summary[
        calendar_exp21g_summary["model"]
        != "A_components"
    ].round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 7. DIRECT B VS C VS D COMPARISON
# ------------------------------------------------------------

comparison = calendar_exp21g_daily.pivot_table(
    index=["date", "task", "horizon"],
    columns="model",
    values="mae",
    aggfunc="first"
)

comparison["B_vs_C_pct"] = (
    100 * (
        1 - comparison["B_plus_stress900"]
        / comparison["C_plus_raw_interaction"]
    )
)

comparison["D_vs_C_pct"] = (
    100 * (
        1 - comparison["D_plus_both"]
        / comparison["C_plus_raw_interaction"]
    )
)

calendar_exp21g_direct = (
    comparison.reset_index()
)

print("\n===== TRANSFORMATION VS RAW INTERACTION =====")

direct_summary = (
    calendar_exp21g_direct.groupby(
        ["task", "horizon"]
    )
    .agg(
        days=("date", "nunique"),
        B_vs_C_mean=("B_vs_C_pct", "mean"),
        B_vs_C_median=("B_vs_C_pct", "median"),
        B_beats_C_days=(
            "B_vs_C_pct",
            lambda s: int((s > 0).sum())
        ),
        D_vs_C_mean=("D_vs_C_pct", "mean"),
        D_vs_C_median=("D_vs_C_pct", "median"),
        D_beats_C_days=(
            "D_vs_C_pct",
            lambda s: int((s > 0).sum())
        )
    )
    .reset_index()
)

print(
    direct_summary.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 8. OCTOBER 19 STABILITY
# ------------------------------------------------------------

print("\n===== OCTOBER 19 LIGHTGBM RESULTS =====")

oct19 = calendar_exp21g_daily[
    calendar_exp21g_daily["date"].astype(str)
    == "2026-10-19"
]

print(
    oct19[
        [
            "task",
            "horizon",
            "model",
            "daily_gain_pct"
        ]
    ].round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 9. FEATURE IMPORTANCE DIAGNOSTICS
# ------------------------------------------------------------

print("\n===== ENGINEERED FEATURE IMPORTANCE =====")

if not calendar_exp21g_importance.empty:

    relevant = calendar_exp21g_importance[
        calendar_exp21g_importance["feature"].isin([
            "spread_depth_stress_900",
            "raw_spread_depth_900"
        ])
    ]

    importance_summary = (
        relevant.groupby(
            [
                "task",
                "horizon",
                "model",
                "feature"
            ]
        )["gain_importance"]
        .median()
        .reset_index()
    )

    print(
        importance_summary
        .round(4).to_string(index=False)
    )

# ------------------------------------------------------------
# 10. RESEARCH CONCLUSIONS — NO AUTOMATIC PROMOTION
# ------------------------------------------------------------

print("\n===== EXPERIMENT 21G COMPLETE =====")
print("Original df_1s unchanged.")
print("No new features automatically promoted.")
print("Fixed LightGBM parameters used.")
print("Identical evaluation rows across models.")
print("Chronological expanding-window validation.")
print("Results remain exploratory until fresh holdout.")


Maximum feature reconstruction difference: 0.0

===== MODEL CONFIGURATIONS =====
A_components              : 11 features
B_plus_stress900          : 12 features
C_plus_raw_interaction    : 12 features
D_plus_both               : 13 features

===== COMMON SAMPLE AVAILABILITY =====
Total rows: 459122
Rows with all inputs available: 446162
Coverage percentage: 97.177

Evaluating 2026-10-05 | Previous dates: 5

Evaluating 2026-10-06 | Previous dates: 6

Evaluating 2026-10-07 | Previous dates: 7

Evaluating 2026-10-08 | Previous dates: 8

Evaluating 2026-10-09 | Previous dates: 9

Evaluating 2026-10-12 | Previous dates: 10

Evaluating 2026-10-13 | Previous dates: 11

Evaluating 2026-10-14 | Previous dates: 12

Evaluating 2026-10-15 | Previous dates: 13

Evaluating 2026-10-16 | Previous dates: 14

Evaluating 2026-10-19 | Previous dates: 15

===== LIGHTGBM OVERALL RESULTS =====
  task  horizon                  model  days  pooled_gain_pct  mean_daily_gain_pct  median_daily_gain_pct  winning_d

In [46]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 21H — FROZEN FORWARD VALIDATION
#
# Requires refreshed calendar_exp21_features.
# Frozen research cutoff: 2026-10-19 inclusive.
# Tests only later eligible sessions.
#
# No feature engineering changes.
# No hyperparameter tuning.
# No modification of df_1s.
# ============================================================

CUTOFF = pd.Timestamp("2026-10-19").date()
HORIZONS = [30, 60]
TRAIN_STRIDE = 10

BASE = [
    "spread_bps",
    "depth_l5",
    "rv_60",
    "rv_300",
    "rv_900",
    "depth_med_60",
    "depth_med_300",
    "depth_med_900",
    "clock_seconds",
    "open_first_30s",
    "open_1_5m"
]

FEATURE = "spread_depth_stress_900"

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

if "calendar_exp21_features" not in globals():
    raise ValueError(
        "Refresh df_1s and rerun Experiment 21A first."
    )

x = calendar_exp21_features.copy()

x["date"] = pd.to_datetime(x["date"]).dt.date

x["clock_seconds"] = (
    (pd.to_numeric(x["ts_sec"]) + 19800) % 86400
) - 9 * 3600

x["open_first_30s"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m"] = (
    (x["clock_seconds"] >= 60)
    & (x["clock_seconds"] < 300)
).astype(float)

required = (
    ["symbol", "date", "ts_sec"]
    + BASE
    + [FEATURE]
    + [
        f"future_{task}_{h}"
        for task in ["absret", "spread"]
        for h in HORIZONS
    ]
)

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows.")

for col in BASE + [FEATURE]:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Match 21G's strict common input sample.
complete = np.isfinite(
    x[BASE + [FEATURE]].to_numpy(dtype=float)
).all(axis=1)

x = x.loc[complete].copy()

train_pool = x[
    (x["date"] <= CUTOFF)
    & (x["ts_sec"] % TRAIN_STRIDE == 0)
].copy()

test_pool = x[x["date"] > CUTOFF].copy()

test_dates = sorted(test_pool["date"].unique())

print("Frozen training rows:", len(train_pool))
print("Post-cutoff sessions:", len(test_dates))
print("Evaluation dates:", test_dates)

if not len(test_dates):
    print(
        "\nNo post-cutoff eligible sessions available yet. "
        "Refresh the data and run this cell again."
    )

records = []

for task in ["absret", "spread"]:
    for h in HORIZONS:

        target = f"future_{task}_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ].copy()

        if len(train) < 1000:
            continue

        predictions = {}

        for model_name, features in {
            "baseline": BASE,
            "plus_stress900": BASE + [FEATURE]
        }.items():

            model = lgb.LGBMRegressor(**PARAMS)

            model.fit(
                train[features],
                train[target]
            )

            predictions[model_name] = model

        for date in test_dates:

            test = test_pool[
                (test_pool["date"] == date)
                & np.isfinite(test_pool[target])
            ].copy()

            if len(test) < 200:
                continue

            y = test[target].to_numpy(dtype=float)

            pred_base = np.maximum(
                predictions["baseline"].predict(
                    test[BASE]
                ), 0.0
            )

            pred_new = np.maximum(
                predictions["plus_stress900"].predict(
                    test[BASE + [FEATURE]]
                ), 0.0
            )

            baseline_mae = np.mean(
                np.abs(y - pred_base)
            )

            candidate_mae = np.mean(
                np.abs(y - pred_new)
            )

            records.append({
                "date": date,
                "task": task,
                "horizon": h,
                "rows": len(test),
                "baseline_mae": baseline_mae,
                "candidate_mae": candidate_mae,
                "gain_pct": 100 * (
                    1 - candidate_mae / baseline_mae
                ) if baseline_mae > 0 else np.nan
            })

calendar_exp21h_daily = pd.DataFrame(records)

print("\n===== FROZEN FORWARD DAILY RESULTS =====")

if calendar_exp21h_daily.empty:
    print("No valid post-cutoff evaluations.")
else:
    print(
        calendar_exp21h_daily.round(6)
        .to_string(index=False)
    )

    summary_records = []

    for (task, h), g in calendar_exp21h_daily.groupby(
        ["task", "horizon"]
    ):

        baseline_error = (
            g["baseline_mae"] * g["rows"]
        ).sum()

        candidate_error = (
            g["candidate_mae"] * g["rows"]
        ).sum()

        summary_records.append({
            "task": task,
            "horizon": h,
            "days": len(g),
            "winning_days": int(
                (g["gain_pct"] > 0).sum()
            ),
            "pooled_gain_pct": (
                100 * (
                    1 - candidate_error / baseline_error
                )
            )
        })

    calendar_exp21h_summary = pd.DataFrame(
        summary_records
    )

    print("\n===== FROZEN FORWARD SUMMARY =====")

    print(
        calendar_exp21h_summary.round(6)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 21H COMPLETE =====")
print("Training cutoff fixed at October 19, 2026.")
print("Model hyperparameters frozen.")
print("Original df_1s unchanged.")
print("No feature promoted.")


Frozen training rows: 44623
Post-cutoff sessions: 0
Evaluation dates: []

No post-cutoff eligible sessions available yet. Refresh the data and run this cell again.

===== FROZEN FORWARD DAILY RESULTS =====
No valid post-cutoff evaluations.

===== EXPERIMENT 21H COMPLETE =====
Training cutoff fixed at October 19, 2026.
Model hyperparameters frozen.
Original df_1s unchanged.
No feature promoted.


In [47]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22A — DEPTH DEPLETION PERSISTENCE
#
# Uses original df_1s in calendar research notebook.
#
# Candidate:
# Fraction of the past W seconds during which
# displayed L5 depth was below its causal,
# previous-300-second reference level.
#
# Windows: 60, 300, 900 seconds
# Outcomes: future absolute returns, future spread
# Horizons: 30, 60, 300 seconds
#
# No modification to df_1s.
# ============================================================

WINDOWS = [60, 300, 900]
HORIZONS = [30, 60, 300]
REFERENCE_WINDOW = 300
MIN_SESSION_COVERAGE = 0.97

qty_cols = [
    f"{side}_q{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
]

required = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
] + qty_cols

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate dataframe columns")

x = df_1s[required].copy()

for c in required:
    if c != "symbol":
        x[c] = pd.to_numeric(x[c], errors="coerce")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp records")

ts = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ts.dt.date
x["sec"] = (
    ts.dt.hour * 3600
    + ts.dt.minute * 60
    + ts.dt.second
)

x = x[
    x["sec"].between(32400, 61200, inclusive="left")
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage[
    coverage["rows"] >= 28800 * MIN_SESSION_COVERAGE
]

eligible = eligible[
    eligible["date"].astype(str) != "2026-09-28"
]

x = x.merge(
    eligible[["symbol", "date"]],
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

print("Eligible sessions:", len(eligible))

# ------------------------------------------------------------
# 1. Valid current quotes and displayed depth
# ------------------------------------------------------------

bid = x["bid_p1"]
ask = x["ask_p1"]

valid_quote = (
    bid.gt(0) & ask.gt(0) & ask.ge(bid)
)

x["mid"] = ((bid + ask) / 2).where(valid_quote)

x["spread_bps"] = (
    1e4 * (ask - bid) / x["mid"]
).where(valid_quote)

valid_depth = (
    x[qty_cols].notna().all(axis=1)
    & np.isfinite(x[qty_cols]).all(axis=1)
    & x[qty_cols].ge(0).all(axis=1)
)

x["depth_l5"] = (
    x[qty_cols].sum(axis=1)
).where(valid_depth)

# ------------------------------------------------------------
# 2. Causal reference depth
#
# Reference uses observations strictly before t.
# We mark the current book as depleted when its
# depth is less than 75% of the historical median.
#
# The 75% threshold is a prespecified hypothesis,
# not an optimized parameter.
# ------------------------------------------------------------

g = x.groupby(["symbol", "date"], sort=False)

x["reference_depth_300"] = (
    g["depth_l5"].transform(
        lambda s: s.shift(1).rolling(
            REFERENCE_WINDOW,
            min_periods=270
        ).median()
    )
)

x["depth_ratio"] = (
    x["depth_l5"]
    / x["reference_depth_300"].where(
        x["reference_depth_300"] > 0
    )
)

valid_ratio = (
    np.isfinite(x["depth_ratio"])
    & x["depth_ratio"].ge(0)
)

x["depleted"] = (
    x["depth_ratio"] < 0.75
).astype(float)

x.loc[~valid_ratio, "depleted"] = np.nan

# ------------------------------------------------------------
# 3. Persistence features
#
# Require 90% valid observations in each window.
# Rolling proportion uses only present/past values.
# ------------------------------------------------------------

CANDIDATES = []

for w in WINDOWS:
    name = f"depth_depletion_fraction_{w}"

    x[name] = (
        x.groupby(["symbol", "date"], sort=False)
        ["depleted"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.9 * w)
            ).mean()
        )
    )

    CANDIDATES.append(name)

# ------------------------------------------------------------
# 4. Exact future timestamp labels
#
# Using merge, not row shift, so the target never
# crosses a gap, session or contract boundary.
# ------------------------------------------------------------

for h in HORIZONS:
    future = x[
        ["symbol", "date", "ts_sec", "mid", "spread_bps"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(columns={
        "mid": f"future_mid_{h}",
        "spread_bps": f"future_spread_{h}"
    })

    x = x.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    x[f"future_absret_{h}"] = (
        1e4 * np.log(
            x[f"future_mid_{h}"] / x["mid"]
        )
    ).abs()

    x[f"future_spread_change_{h}"] = (
        x[f"future_spread_{h}"] - x["spread_bps"]
    )

# ------------------------------------------------------------
# 5. Per-session Spearman IC
# ------------------------------------------------------------

def daily_ic(a, b):
    mask = np.isfinite(a) & np.isfinite(b)

    if mask.sum() < 200:
        return np.nan

    aa = pd.Series(a[mask]).rank().to_numpy()
    bb = pd.Series(b[mask]).rank().to_numpy()

    if aa.std() == 0 or bb.std() == 0:
        return np.nan

    return float(np.corrcoef(aa, bb)[0, 1])

FEATURES = CANDIDATES + [
    "depth_ratio", "depth_l5",
    "reference_depth_300", "spread_bps"
]

records = []

for (symbol, date), day in x.groupby(
    ["symbol", "date"], sort=False
):
    for h in HORIZONS:
        for target in [
            f"future_absret_{h}",
            f"future_spread_change_{h}"
        ]:
            y = day[target].to_numpy(dtype=float)

            for feature in FEATURES:
                a = day[feature].to_numpy(dtype=float)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "horizon": h,
                    "target": (
                        "absret" if "absret" in target
                        else "spread_change"
                    ),
                    "feature": feature,
                    "ic": daily_ic(a, y)
                })

calendar_exp22a_daily = pd.DataFrame(records)

calendar_exp22a_summary = (
    calendar_exp22a_daily
    .groupby(["target", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=("ic", lambda z: int((z > 0).sum())),
        negative_days=("ic", lambda z: int((z < 0).sum()))
    )
    .reset_index()
)

calendar_exp22a_summary["sign_consistency"] = (
    calendar_exp22a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp22a_summary["days"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 6. Redundancy against existing measurements
# ------------------------------------------------------------

calendar_exp22a_redundancy = (
    x[
        CANDIDATES
        + ["depth_ratio", "depth_l5", "spread_bps"]
    ].corr(method="spearman")
)

# ------------------------------------------------------------
# 7. Outputs
# ------------------------------------------------------------

print("\n===== CANDIDATE IC RESULTS =====")

print(
    calendar_exp22a_summary[
        calendar_exp22a_summary["feature"].isin(CANDIDATES)
    ].round(5).to_string(index=False)
)

print("\n===== COMPONENT BASELINES =====")

print(
    calendar_exp22a_summary[
        ~calendar_exp22a_summary["feature"].isin(CANDIDATES)
    ].round(5).to_string(index=False)
)

print("\n===== CANDIDATE REDUNDANCY =====")

print(
    calendar_exp22a_redundancy.loc[
        CANDIDATES,
        ["depth_ratio", "depth_l5", "spread_bps"]
    ].round(5).to_string()
)

print("\n===== FEATURE AVAILABILITY =====")

for name in CANDIDATES:
    print(
        name,
        "missing_pct:",
        round(100 * x[name].isna().mean(), 4),
        "unique:",
        x[name].nunique()
    )

calendar_exp22a_features = x

print("\n===== EXPERIMENT 22A COMPLETE =====")
print("Original df_1s unchanged")
print("No feature promoted")


Eligible sessions: 16

===== CANDIDATE IC RESULTS =====
       target  horizon                      feature  days  mean_ic  median_ic  positive_days  negative_days  sign_consistency
       absret       30 depth_depletion_fraction_300    16  0.06473    0.04569             15              1            0.9375
       absret       30  depth_depletion_fraction_60    16  0.05723    0.05364             14              2            0.8750
       absret       30 depth_depletion_fraction_900    16  0.05745    0.05358             12              4            0.7500
       absret       60 depth_depletion_fraction_300    16  0.07363    0.05998             16              0            1.0000
       absret       60  depth_depletion_fraction_60    16  0.05576    0.04708             12              4            0.7500
       absret       60 depth_depletion_fraction_900    16  0.05863    0.05500             13              3            0.8125
       absret      300 depth_depletion_fraction_300    16  0.0

In [48]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22B
# INCREMENTAL DEPTH-DEPLETION FEATURE TEST
#
# Input:
#   calendar_exp22a_features
#
# Models:
#   Base liquidity and volatility components
#   Base + depletion_60
#   Base + depletion_300
#   Base + both
#
# Targets: future absolute mid returns (bps)
# Horizons: 30, 60, 300 seconds
#
# Chronological expanding window
# No sklearn required
# No modification to df_1s
# ============================================================

if "calendar_exp22a_features" not in globals():
    raise ValueError("Run Experiment 22A first.")

x = calendar_exp22a_features.copy()

HORIZONS_22B = [30, 60, 300]
MIN_TRAIN_DAYS_22B = 5
TRAIN_STRIDE_22B = 10
RIDGE_ALPHA_22B = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate timestamp records.")

# ------------------------------------------------------------
# 1. Causal realized volatility controls
# ------------------------------------------------------------

g = x.groupby(["symbol", "date"], sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

x["ret_1s_22b"] = np.log(
    x["mid"] / prev_mid
)

x.loc[
    x["ts_sec"] - prev_ts != 1,
    "ret_1s_22b"
] = np.nan

x["ret_sq_22b"] = x["ret_1s_22b"] ** 2

for w in [60, 300]:

    x[f"rv_bps_{w}_22b"] = (
        x.groupby(
            ["symbol", "date"], sort=False
        )["ret_sq_22b"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 1e4
    )

# ------------------------------------------------------------
# 2. Clock controls
# ------------------------------------------------------------

x["clock_seconds_22b"] = (
    (x["ts_sec"] + 19800) % 86400
) - 9 * 3600

x["open_1_5m_22b"] = (
    (x["clock_seconds_22b"] >= 60)
    & (x["clock_seconds_22b"] < 300)
).astype(float)

# ------------------------------------------------------------
# 3. Define feature sets
# ------------------------------------------------------------

BASE_22B = [
    "depth_l5",
    "depth_ratio",
    "reference_depth_300",
    "spread_bps",
    "rv_bps_60_22b",
    "rv_bps_300_22b",
    "clock_seconds_22b",
    "open_1_5m_22b"
]

FEATURE_SETS_22B = {
    "baseline": BASE_22B,
    "plus_60": (
        BASE_22B
        + ["depth_depletion_fraction_60"]
    ),
    "plus_300": (
        BASE_22B
        + ["depth_depletion_fraction_300"]
    ),
    "plus_both": (
        BASE_22B
        + [
            "depth_depletion_fraction_60",
            "depth_depletion_fraction_300"
        ]
    )
}

ALL_INPUTS_22B = sorted(set(
    f
    for features in FEATURE_SETS_22B.values()
    for f in features
))

targets = [
    f"future_absret_{h}"
    for h in HORIZONS_22B
]

required = (
    ["symbol", "date", "ts_sec", "mid"]
    + ALL_INPUTS_22B
    + targets
)

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing fields: {missing}")

for col in ALL_INPUTS_22B + targets:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Identical warm-up and feature availability for all models.
x["_common_22b"] = np.isfinite(
    x[ALL_INPUTS_22B].to_numpy(dtype=float)
).all(axis=1)

print("\n===== COMMON FEATURE AVAILABILITY =====")
print("Total rows:", len(x))
print("Complete rows:", int(x["_common_22b"].sum()))
print(
    "Complete fraction:",
    round(x["_common_22b"].mean(), 5)
)

# ------------------------------------------------------------
# 4. Fixed ridge model
#
# Median/mean/std estimated only on training dates.
# Same train/test observations for each model.
# ------------------------------------------------------------

def ridge_predict_22b(train, test, features, target):

    A = train[features].to_numpy(
        dtype=np.float64, copy=True
    )
    B = test[features].to_numpy(
        dtype=np.float64, copy=True
    )
    y = train[target].to_numpy(dtype=np.float64)

    med = np.array([
        np.nanmedian(v) if np.isfinite(v).any()
        else 0.0
        for v in A.T
    ])

    A = np.where(np.isfinite(A), A, med)
    B = np.where(np.isfinite(B), B, med)

    mean = A.mean(axis=0)
    std = A.std(axis=0)
    std = np.where(std > 1e-12, std, 1.0)

    A = (A - mean) / std
    B = (B - mean) / std

    ymean = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A
        + RIDGE_ALPHA_22B * n * np.eye(k),
        A.T @ (y - ymean)
    )

    return np.maximum(
        ymean + B @ beta, 0.0
    )

# ------------------------------------------------------------
# 5. Expanding-window testing
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS_22B:
    raise ValueError("Insufficient trading sessions.")

records = []

for i in range(MIN_TRAIN_DAYS_22B, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_22b"]
        & (x["ts_sec"] % TRAIN_STRIDE_22B == 0)
    ]

    test_pool = x[
        (x["date"] == test_date)
        & x["_common_22b"]
    ]

    for h in HORIZONS_22B:

        target = f"future_absret_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        actual = test[target].to_numpy(
            dtype=np.float64
        )

        predictions = {}

        for name, features in FEATURE_SETS_22B.items():
            predictions[name] = ridge_predict_22b(
                train, test, features, target
            )

        base_error = np.abs(
            actual - predictions["baseline"]
        )

        base_mae = base_error.mean()

        for name, pred in predictions.items():

            mae = np.abs(actual - pred).mean()

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "train_rows": len(train),
                "test_rows": len(test),
                "base_mae": float(base_mae),
                "model_mae": float(mae),
                "gain_pct": (
                    100 * (1 - mae / base_mae)
                    if base_mae > 0 else np.nan
                )
            })

    print("Completed:", test_date)

calendar_exp22b_daily = pd.DataFrame(records)

if calendar_exp22b_daily.empty:
    raise ValueError("No valid model evaluations.")

# ------------------------------------------------------------
# 6. Pooled and per-day summary
# ------------------------------------------------------------

summary_rows = []

for (h, name), day in calendar_exp22b_daily.groupby(
    ["horizon", "model"]
):

    total_base = (
        day["base_mae"] * day["test_rows"]
    ).sum()

    total_model = (
        day["model_mae"] * day["test_rows"]
    ).sum()

    summary_rows.append({
        "horizon": h,
        "model": name,
        "days": day["date"].nunique(),
        "pooled_gain_pct": (
            100 * (1 - total_model / total_base)
            if total_base > 0 else np.nan
        ),
        "median_daily_gain_pct": day["gain_pct"].median(),
        "winning_days": int(
            (day["gain_pct"] > 0).sum()
        ),
        "worst_day_pct": day["gain_pct"].min()
    })

calendar_exp22b_summary = pd.DataFrame(summary_rows)

calendar_exp22b_summary["win_fraction"] = (
    calendar_exp22b_summary["winning_days"]
    / calendar_exp22b_summary["days"]
)

print("\n===== EXPERIMENT 22B SUMMARY =====")

print(
    calendar_exp22b_summary[
        calendar_exp22b_summary["model"] != "baseline"
    ].round(6).to_string(index=False)
)

print("\n===== DAY-LEVEL GAINS =====")

print(
    calendar_exp22b_daily[
        calendar_exp22b_daily["model"] != "baseline"
    ].pivot_table(
        index=["date", "horizon"],
        columns="model",
        values="gain_pct"
    ).round(5).to_string()
)

print("\n===== EXPERIMENT 22B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")



===== COMMON FEATURE AVAILABILITY =====
Total rows: 459122
Complete rows: 450498
Complete fraction: 0.98122
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== EXPERIMENT 22B SUMMARY =====
 horizon     model  days  pooled_gain_pct  median_daily_gain_pct  winning_days  worst_day_pct  win_fraction
      30  plus_300    11         0.047930               0.030089             6      -0.133340      0.545455
      30   plus_60    11         0.031282              -0.018623             5      -0.065274      0.454545
      30 plus_both    11         0.052765               0.054723             6      -0.141688      0.545455
      60  plus_300    11         0.054061               0.028457             7      -0.274507      0.636364
      60   plus_60    11         0.030437               0.027346         

In [49]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22C — CAUSAL LIQUIDITY RECOVERY SPEED
#
# Research notebook: df_1s
#
# When the previous second's book was depleted,
# measure how much normalized depth recovered now.
#
# Aggregate recovery over 30, 60, 300 seconds.
#
# Outcomes:
#   Future absolute mid-return: 30, 60, 300 seconds
#   Future spread change: 30, 60, 300 seconds
#
# This is an initial screening experiment.
# No modification to df_1s.
# ============================================================

WINDOWS_22C = [30, 60, 300]
HORIZONS_22C = [30, 60, 300]
REFERENCE_22C = 300
DEPLETION_THRESHOLD_22C = 0.75
MIN_COVERAGE_22C = 0.90

qty_cols = [
    f"{side}_q{i}"
    for side in ("bid", "ask")
    for i in range(1, 6)
]

required = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
] + qty_cols

if "df_1s" not in globals():
    raise ValueError("df_1s not found")

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing fields: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate master columns")

x = df_1s[required].copy()

for col in required:
    if col != "symbol":
        x[col] = pd.to_numeric(x[col], errors="coerce")

if x["ts_sec"].isna().any():
    raise ValueError("Invalid timestamp values")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp pairs")

ts = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ts.dt.date
x["clock"] = (
    ts.dt.hour * 3600
    + ts.dt.minute * 60
    + ts.dt.second
)

x = x.loc[
    x["clock"].ge(32400)
    & x["clock"].lt(61200)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

# Match the previously established session coverage.
coverage = (
    x.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage.loc[
    coverage["rows"].ge(0.97 * 28800)
    & coverage["date"].astype(str).ne("2026-09-28"),
    ["symbol", "date"]
]

x = x.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

print("Eligible sessions:", len(eligible))
print("Working rows:", len(x))

# ------------------------------------------------------------
# 1. Current depth, mid and spread
# ------------------------------------------------------------

bid = x["bid_p1"]
ask = x["ask_p1"]

valid_quote = (
    bid.gt(0)
    & ask.gt(0)
    & ask.ge(bid)
)

x["mid"] = ((bid + ask) / 2).where(valid_quote)

x["spread_bps"] = (
    1e4 * (ask - bid) / x["mid"]
).where(valid_quote)

valid_depth = (
    x[qty_cols].notna().all(axis=1)
    & np.isfinite(x[qty_cols]).all(axis=1)
    & x[qty_cols].ge(0).all(axis=1)
)

x["depth_l5"] = (
    x[qty_cols].sum(axis=1)
).where(valid_depth)

# ------------------------------------------------------------
# 2. Previous-reference depth and normalized depth
# ------------------------------------------------------------

g = x.groupby(["symbol", "date"], sort=False)

x["reference_depth"] = (
    g["depth_l5"].transform(
        lambda s: s.shift(1).rolling(
            REFERENCE_22C,
            min_periods=270
        ).median()
    )
)

x["relative_depth"] = (
    x["depth_l5"]
    / x["reference_depth"].where(
        x["reference_depth"] > 0
    )
)

# ------------------------------------------------------------
# 3. Causal one-second recovery
#
# Only when t-1 is exactly one second before t.
# Only count recovery following a depleted book.
# ------------------------------------------------------------

g = x.groupby(["symbol", "date"], sort=False)

previous_ts = g["ts_sec"].shift(1)
previous_ratio = g["relative_depth"].shift(1)

consecutive = (
    x["ts_sec"].sub(previous_ts).eq(1)
)

valid_pair = (
    consecutive
    & np.isfinite(previous_ratio)
    & np.isfinite(x["relative_depth"])
)

depleted_before = (
    valid_pair
    & previous_ratio.lt(DEPLETION_THRESHOLD_22C)
)

# Signed normalized change:
# positive -> liquidity recovery
# negative -> further depletion
x["recovery_step"] = (
    x["relative_depth"] - previous_ratio
).where(depleted_before)

# A zero-filled sum would conceal whether there were
# any depleted observations. Keep a separate event count.
x["depleted_transition"] = np.where(
    valid_pair,
    previous_ratio.lt(
        DEPLETION_THRESHOLD_22C
    ).astype(float),
    np.nan
)

# ------------------------------------------------------------
# 4. Trailing recovery features
#
# Recovery rate is the average signed change
# CONDITIONAL on previously depleted observations.
#
# Require >=5 depleted transitions.
# This intentionally allows missing values if
# there were insufficient depletion events.
# ------------------------------------------------------------

CANDIDATES_22C = []

for w in WINDOWS_22C:

    group = x.groupby(
        ["symbol", "date"], sort=False
    )

    minp = int(np.ceil(w * MIN_COVERAGE_22C))

    eligible_transitions = group[
        "depleted_transition"
    ].transform(
        lambda s: s.rolling(
            w, min_periods=minp
        ).sum()
    )

    recovery_sum = group[
        "recovery_step"
    ].transform(
        lambda s: s.fillna(0.0).rolling(
            w, min_periods=minp
        ).sum()
    )

    name = f"liquidity_recovery_speed_{w}"

    x[name] = (
        recovery_sum
        / eligible_transitions.where(
            eligible_transitions >= 5
        )
    )

    CANDIDATES_22C.append(name)

# ------------------------------------------------------------
# 5. Exact future targets: no cross-gap labels
# ------------------------------------------------------------

for h in HORIZONS_22C:

    future = x[
        ["symbol", "date", "ts_sec",
         "mid", "spread_bps"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(columns={
        "mid": f"future_mid_{h}",
        "spread_bps": f"future_spread_{h}"
    })

    x = x.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    x[f"future_absret_{h}"] = (
        1e4 * np.log(
            x[f"future_mid_{h}"] / x["mid"]
        )
    ).abs()

    x[f"future_spread_change_{h}"] = (
        x[f"future_spread_{h}"]
        - x["spread_bps"]
    )

# ------------------------------------------------------------
# 6. Spearman daily IC
# ------------------------------------------------------------

def safe_ic_22c(a, b):
    mask = np.isfinite(a) & np.isfinite(b)

    if mask.sum() < 200:
        return np.nan

    aa = pd.Series(a[mask]).rank().to_numpy()
    bb = pd.Series(b[mask]).rank().to_numpy()

    if aa.std() < 1e-12 or bb.std() < 1e-12:
        return np.nan

    return float(np.corrcoef(aa, bb)[0, 1])

BASELINES_22C = [
    "depth_l5",
    "relative_depth",
    "spread_bps"
]

records = []

for (symbol, date), day in x.groupby(
    ["symbol", "date"], sort=False
):
    for h in HORIZONS_22C:

        targets = {
            "absret": f"future_absret_{h}",
            "spread_change": f"future_spread_change_{h}"
        }

        for task, target in targets.items():

            y = day[target].to_numpy(dtype=float)

            for feature in (
                CANDIDATES_22C + BASELINES_22C
            ):
                z = day[feature].to_numpy(dtype=float)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": safe_ic_22c(z, y)
                })

calendar_exp22c_daily = pd.DataFrame(records)

calendar_exp22c_summary = (
    calendar_exp22c_daily.groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda z: int((z > 0).sum())
        ),
        negative_days=(
            "ic", lambda z: int((z < 0).sum())
        )
    )
    .reset_index()
)

calendar_exp22c_summary["sign_consistency"] = (
    calendar_exp22c_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp22c_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 7. Diagnostics and outputs
# ------------------------------------------------------------

print("\n===== RECOVERY-SPEED IC RESULTS =====")

print(
    calendar_exp22c_summary.loc[
        calendar_exp22c_summary[
            "feature"
        ].isin(CANDIDATES_22C)
    ].round(5).to_string(index=False)
)

print("\n===== COMPONENT BASELINES =====")

print(
    calendar_exp22c_summary.loc[
        calendar_exp22c_summary[
            "feature"
        ].isin(BASELINES_22C)
    ].round(5).to_string(index=False)
)

print("\n===== RECOVERY FEATURE COVERAGE =====")

for feature in CANDIDATES_22C:
    print(
        feature,
        "missing_pct:",
        round(100 * x[feature].isna().mean(), 4),
        "unique:",
        x[feature].nunique()
    )

print("\n===== FEATURE REDUNDANCY =====")

print(
    x[
        CANDIDATES_22C + BASELINES_22C
    ].corr(method="spearman")
    .loc[
        CANDIDATES_22C,
        BASELINES_22C
    ]
    .round(5)
    .to_string()
)

calendar_exp22c_features = x

print("\n===== EXPERIMENT 22C COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


Eligible sessions: 16
Working rows: 459122

===== RECOVERY-SPEED IC RESULTS =====
         task  horizon                      feature  days  mean_ic  median_ic  positive_days  negative_days  sign_consistency
       absret       30  liquidity_recovery_speed_30    16  0.09424    0.09565             15              1            0.9375
       absret       30 liquidity_recovery_speed_300    16  0.06277    0.07266             14              2            0.8750
       absret       30  liquidity_recovery_speed_60    16  0.08997    0.09585             15              1            0.9375
       absret       60  liquidity_recovery_speed_30    16  0.08361    0.08100             13              3            0.8125
       absret       60 liquidity_recovery_speed_300    16  0.06098    0.06252             13              3            0.8125
       absret       60  liquidity_recovery_speed_60    16  0.07883    0.09849             13              3            0.8125
       absret      300  liquidity_re

In [50]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22D
# RECOVERY SPEED VS DEPLETION EVENT FREQUENCY
#
# Run after Experiment 22C.
# Reads calendar_exp22c_features only.
#
# Outcomes: future abs return, 30s and 60s.
# Expanding-window ridge; identical samples across models.
# All computations within symbol and IST trading session.
#
# No changes to df_1s.
# ============================================================

if "calendar_exp22c_features" not in globals():
    raise ValueError("Run Experiment 22C first.")

x = calendar_exp22c_features.copy()

WINS = [30, 60, 300]
HORIZONS = [30, 60]
MIN_DAYS = 5
TRAIN_STRIDE = 10
ALPHA = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date
x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

gkeys = ["symbol", "date"]

# ------------------------------------------------------------
# 1. Audit coverage and build frequency controls
#
# Valid transitions are checked explicitly.
# Recovery speed is defined only when:
#   - >=90% valid consecutive transitions in window
#   - >=5 transitions that start from depleted depth
#
# Missing recovery speed is represented by:
#   value = 0 and availability = 0.
# These must always be supplied together.
# ------------------------------------------------------------

for w in WINS:

    min_valid = int(np.ceil(0.90 * w))

    valid = (
        x["depleted_transition"]
        .notna()
        .astype(float)
    )

    depleted_count = (
        x["depleted_transition"]
        .fillna(0.0)
    )

    recovery_sum = (
        x["recovery_step"]
        .fillna(0.0)
    )

    # Rolling calculations remain within each session.
    work = pd.DataFrame({
        "symbol": x["symbol"],
        "date": x["date"],
        "valid": valid,
        "depleted": depleted_count,
        "recovery": recovery_sum
    })

    grouped = work.groupby(gkeys, sort=False)

    n_valid = grouped["valid"].transform(
        lambda s: s.rolling(
            w, min_periods=1
        ).sum()
    )

    n_events = grouped["depleted"].transform(
        lambda s: s.rolling(
            w, min_periods=1
        ).sum()
    )

    sum_recovery = grouped["recovery"].transform(
        lambda s: s.rolling(
            w, min_periods=1
        ).sum()
    )

    sufficient_coverage = n_valid >= min_valid
    sufficient_events = n_events >= 5

    x[f"recovery_event_rate_{w}"] = (
        n_events / n_valid.where(n_valid > 0)
    ).where(sufficient_coverage)

    speed = (
        sum_recovery /
        n_events.where(n_events > 0)
    )

    available = (
        sufficient_coverage
        & sufficient_events
        & np.isfinite(speed)
    )

    x[f"recovery_available_{w}"] = (
        available.astype(float)
    )

    x[f"recovery_speed_value_{w}"] = (
        speed.where(available, 0.0)
    )

    print(
        f"Window {w}: "
        f"available={available.mean():.2%}; "
        f"median events={n_events.median():.1f}"
    )

# ------------------------------------------------------------
# 2. Causal realized volatility and clock controls
# ------------------------------------------------------------

g = x.groupby(gkeys, sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

r = np.log(x["mid"] / prev_mid)
r = r.where((x["ts_sec"] - prev_ts) == 1)

x["_ret_sq_22d"] = r ** 2

for w in [60, 300]:
    x[f"rv_{w}_22d"] = (
        x.groupby(gkeys, sort=False)["_ret_sq_22d"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(np.ceil(0.9 * w))
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5) * 10000
    )

x["clock_22d"] = (
    (x["ts_sec"] + 19800) % 86400
) - 32400

# ------------------------------------------------------------
# 3. Feature configurations
# ------------------------------------------------------------

BASE = [
    "depth_l5",
    "relative_depth",
    "reference_depth",
    "spread_bps",
    "rv_60_22d",
    "rv_300_22d",
    "clock_22d"
]

MODELS = {"baseline": BASE}

for w in WINS:
    MODELS[f"events_{w}"] = BASE + [
        f"recovery_event_rate_{w}"
    ]

    MODELS[f"events_plus_speed_{w}"] = BASE + [
        f"recovery_event_rate_{w}",
        f"recovery_available_{w}",
        f"recovery_speed_value_{w}"
    ]

# Complete baseline data shared by all comparisons.
for col in BASE:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_base_valid"] = np.isfinite(
    x[BASE].to_numpy(dtype=float)
).all(axis=1)

# ------------------------------------------------------------
# 4. Fixed ridge, training-only standardization
# ------------------------------------------------------------

def ridge_22d(train, test, columns, target):

    A = train[columns].to_numpy(dtype=float, copy=True)
    B = test[columns].to_numpy(dtype=float, copy=True)

    # Event rates can be missing during warm-up.
    # Imputation and an availability indicator for
    # recovery speed must remain distinct.
    med = np.array([
        np.nanmedian(v) if np.isfinite(v).any()
        else 0.0
        for v in A.T
    ])

    A = np.where(np.isfinite(A), A, med)
    B = np.where(np.isfinite(B), B, med)

    mu = A.mean(axis=0)
    sd = A.std(axis=0)
    sd = np.where(sd > 1e-12, sd, 1.0)

    A = (A - mu) / sd
    B = (B - mu) / sd

    y = train[target].to_numpy(dtype=float)
    ym = y.mean()

    beta = np.linalg.solve(
        A.T @ A + ALPHA * len(A) * np.eye(len(columns)),
        A.T @ (y - ym)
    )

    return np.maximum(ym + B @ beta, 0.0)

# ------------------------------------------------------------
# 5. Chronological evaluation
#
# Comparisons are made pairwise on equal rows
# for each window, including the baseline model.
# ------------------------------------------------------------

dates = sorted(x["date"].unique())
records = []

for i in range(MIN_DAYS, len(dates)):

    date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_base_valid"]
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test_pool = x[
        (x["date"] == date)
        & x["_base_valid"]
    ]

    for h in HORIZONS:

        target = f"future_absret_{h}"

        for w in WINS:

            # Require event-rate availability in both
            # models so evaluation rows are identical.
            event_col = f"recovery_event_rate_{w}"

            tr = train_pool[
                np.isfinite(train_pool[target])
                & np.isfinite(train_pool[event_col])
            ]

            te = test_pool[
                np.isfinite(test_pool[target])
                & np.isfinite(test_pool[event_col])
            ]

            if len(tr) < 1000 or len(te) < 200:
                continue

            model_names = [
                "baseline",
                f"events_{w}",
                f"events_plus_speed_{w}"
            ]

            y = te[target].to_numpy(dtype=float)

            errors = {}

            for name in model_names:

                pred = ridge_22d(
                    tr, te, MODELS[name], target
                )

                errors[name] = float(
                    np.abs(y - pred).mean()
                )

            event_mae = errors[f"events_{w}"]
            speed_mae = errors[
                f"events_plus_speed_{w}"
            ]

            records.append({
                "date": date,
                "horizon": h,
                "window": w,
                "rows": len(te),
                "base_mae": errors["baseline"],
                "events_mae": event_mae,
                "speed_mae": speed_mae,
                "events_gain_pct": 100 * (
                    1 - event_mae / errors["baseline"]
                ),
                "speed_increment_pct": 100 * (
                    1 - speed_mae / event_mae
                ),
                "total_gain_pct": 100 * (
                    1 - speed_mae / errors["baseline"]
                )
            })

    print("Completed:", date)

calendar_exp22d_daily = pd.DataFrame(records)

if calendar_exp22d_daily.empty:
    raise ValueError("No evaluations completed.")

# ------------------------------------------------------------
# 6. Aggregate, preserving true error weighting
# ------------------------------------------------------------

rows = []

for (h, w), d in calendar_exp22d_daily.groupby(
    ["horizon", "window"]
):

    weight = d["rows"].to_numpy(dtype=float)

    base = np.sum(weight * d["base_mae"])
    events = np.sum(weight * d["events_mae"])
    speed = np.sum(weight * d["speed_mae"])

    rows.append({
        "horizon": h,
        "window": w,
        "days": d["date"].nunique(),
        "events_vs_base_pct": 100 * (1 - events/base),
        "speed_vs_events_pct": 100 * (1 - speed/events),
        "total_vs_base_pct": 100 * (1 - speed/base),
        "speed_winning_days": int(
            (d["speed_increment_pct"] > 0).sum()
        ),
        "worst_speed_day_pct":
            d["speed_increment_pct"].min()
    })

calendar_exp22d_summary = pd.DataFrame(rows)

print("\n===== RECOVERY SPEED INCREMENTAL TEST =====")
print(
    calendar_exp22d_summary.round(6)
    .to_string(index=False)
)

print("\n===== DAILY RECOVERY-SPEED GAINS =====")
print(
    calendar_exp22d_daily.pivot_table(
        index=["date", "horizon"],
        columns="window",
        values="speed_increment_pct"
    ).round(5).to_string()
)

calendar_exp22d_features = x

print("\n===== EXPERIMENT 22D COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


Window 30: available=27.88%; median events=0.0
Window 60: available=37.44%; median events=0.0
Window 300: available=76.21%; median events=39.0
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== RECOVERY SPEED INCREMENTAL TEST =====
 horizon  window  days  events_vs_base_pct  speed_vs_events_pct  total_vs_base_pct  speed_winning_days  worst_speed_day_pct
      30      30    11            0.036856            -0.006322           0.030536                   8            -0.270101
      30      60    11            0.029356            -0.025904           0.003459                   7            -0.235529
      30     300    11            0.047599             0.033112           0.080696                   7            -0.383589
      60      30    11            0.028359            -0.012183           

In [51]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22E — BID/ASK RECOVERY ASYMMETRY
#
# Causal bid-side vs ask-side depth recovery.
#
# Each side:
#   1. Build L5 displayed depth
#   2. Calculate previous-300s reference median
#   3. Identify depleted prior observations (<75%)
#   4. Calculate signed depth recovery at valid 1s steps
#
# Asymmetry:
#   bid recovery speed - ask recovery speed
#
# Windows: 60s, 300s
# Forecast horizons: 30s, 60s, 300s
#
# Screening via daily Spearman IC.
# No modification to df_1s.
# ============================================================

WINDOWS_22E = [60, 300]
HORIZONS_22E = [30, 60, 300]
REFERENCE_22E = 300
THRESHOLD_22E = 0.75
MIN_EVENTS_22E = 5

qty_cols = [
    f"{side}_q{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
]

required = [
    "symbol", "ts_sec",
    "bid_p1", "ask_p1"
] + qty_cols

missing = sorted(set(required) - set(df_1s.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

x = df_1s[required].copy()

for col in required:
    if col != "symbol":
        x[col] = pd.to_numeric(
            x[col], errors="coerce"
        )

if x["ts_sec"].isna().any():
    raise ValueError("Invalid ts_sec")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows")

# ------------------------------------------------------------
# 1. Session filtering
# ------------------------------------------------------------

ts = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ts.dt.date

x["clock"] = (
    ts.dt.hour * 3600
    + ts.dt.minute * 60
    + ts.dt.second
)

x = x[
    x["clock"].ge(32400)
    & x["clock"].lt(61200)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage[
    (coverage["rows"] >= 0.97 * 28800)
    & (
        coverage["date"].astype(str)
        != "2026-09-28"
    )
][["symbol", "date"]]

x = x.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

print("Eligible sessions:", len(eligible))
print("Rows:", len(x))

GROUP = ["symbol", "date"]

# ------------------------------------------------------------
# 2. Mid-price and spread
# ------------------------------------------------------------

bid = x["bid_p1"]
ask = x["ask_p1"]

valid_quote = (
    bid.gt(0)
    & ask.gt(0)
    & ask.ge(bid)
)

x["mid"] = (
    (bid + ask) / 2
).where(valid_quote)

x["spread_bps"] = (
    1e4 * (ask - bid) / x["mid"]
).where(valid_quote)

# ------------------------------------------------------------
# 3. Side-specific book depths
# ------------------------------------------------------------

for side in ["bid", "ask"]:

    cols = [
        f"{side}_q{i}"
        for i in range(1, 6)
    ]

    q = x[cols].replace(
        [np.inf, -np.inf], np.nan
    )

    valid = (
        q.notna().all(axis=1)
        & q.ge(0).all(axis=1)
    )

    x[f"{side}_depth"] = (
        q.sum(axis=1).where(valid)
    )

# ------------------------------------------------------------
# 4. Causal reference depth
# ------------------------------------------------------------

for side in ["bid", "ask"]:

    col = f"{side}_depth"

    x[f"{side}_reference"] = (
        x.groupby(GROUP, sort=False)[col]
        .transform(
            lambda s: s.shift(1).rolling(
                REFERENCE_22E,
                min_periods=270
            ).median()
        )
    )

    x[f"{side}_relative"] = (
        x[col]
        / x[f"{side}_reference"].where(
            x[f"{side}_reference"] > 0
        )
    )

# ------------------------------------------------------------
# 5. Valid consecutive-second transitions
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

prev_ts = g["ts_sec"].shift(1)

consecutive = (
    x["ts_sec"] - prev_ts == 1
)

for side in ["bid", "ask"]:

    current = x[f"{side}_relative"]
    previous = g[f"{side}_relative"].shift(1)

    valid_pair = (
        consecutive
        & np.isfinite(current)
        & np.isfinite(previous)
    )

    depleted = (
        previous < THRESHOLD_22E
    )

    x[f"{side}_valid_transition"] = (
        valid_pair.astype(float)
    )

    x[f"{side}_depleted_event"] = (
        (valid_pair & depleted).astype(float)
    )

    x[f"{side}_recovery_step"] = (
        (current - previous)
        .where(valid_pair & depleted, 0.0)
    )

# ------------------------------------------------------------
# 6. Window recovery and availability
# ------------------------------------------------------------

CANDIDATES_22E = []

for w in WINDOWS_22E:

    min_valid = int(np.ceil(0.90 * w))

    for side in ["bid", "ask"]:

        g = x.groupby(GROUP, sort=False)

        valid_n = g[
            f"{side}_valid_transition"
        ].transform(
            lambda s: s.rolling(
                w, min_periods=1
            ).sum()
        )

        event_n = g[
            f"{side}_depleted_event"
        ].transform(
            lambda s: s.rolling(
                w, min_periods=1
            ).sum()
        )

        recovery_sum = g[
            f"{side}_recovery_step"
        ].transform(
            lambda s: s.rolling(
                w, min_periods=1
            ).sum()
        )

        x[f"{side}_event_rate_{w}"] = (
            event_n
            / valid_n.where(valid_n >= min_valid)
        )

        sufficient = (
            (valid_n >= min_valid)
            & (event_n >= MIN_EVENTS_22E)
        )

        x[f"{side}_recovery_{w}"] = (
            recovery_sum
            / event_n.where(sufficient)
        )

    name = f"bid_ask_recovery_asymmetry_{w}"

    x[name] = (
        x[f"bid_recovery_{w}"]
        - x[f"ask_recovery_{w}"]
    )

    CANDIDATES_22E.append(name)

    # Additional diagnostic:
    # difference in depleted-event frequency.
    x[f"depletion_event_asymmetry_{w}"] = (
        x[f"bid_event_rate_{w}"]
        - x[f"ask_event_rate_{w}"]
    )

# ------------------------------------------------------------
# 7. Future targets using exact timestamps
# ------------------------------------------------------------

for h in HORIZONS_22E:

    future = x[
        [
            "symbol", "date", "ts_sec",
            "mid", "spread_bps"
        ]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(columns={
        "mid": f"future_mid_{h}",
        "spread_bps": f"future_spread_{h}"
    })

    x = x.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    x[f"future_return_{h}"] = (
        1e4 * np.log(
            x[f"future_mid_{h}"] / x["mid"]
        )
    )

    x[f"future_absret_{h}"] = (
        x[f"future_return_{h}"].abs()
    )

    x[f"future_spread_change_{h}"] = (
        x[f"future_spread_{h}"]
        - x["spread_bps"]
    )

# ------------------------------------------------------------
# 8. Spearman IC
# ------------------------------------------------------------

def safe_ic_22e(a, b):

    mask = (
        np.isfinite(a)
        & np.isfinite(b)
    )

    if mask.sum() < 200:
        return np.nan

    ra = pd.Series(
        a[mask]
    ).rank().to_numpy()

    rb = pd.Series(
        b[mask]
    ).rank().to_numpy()

    if ra.std() == 0 or rb.std() == 0:
        return np.nan

    return float(
        np.corrcoef(ra, rb)[0, 1]
    )

BASELINES_22E = [
    "bid_relative",
    "ask_relative",
    "spread_bps"
]

ASYMMETRY_CONTROLS_22E = [
    f"depletion_event_asymmetry_{w}"
    for w in WINDOWS_22E
]

FEATURES_22E = (
    CANDIDATES_22E
    + ASYMMETRY_CONTROLS_22E
    + BASELINES_22E
)

records = []

for (symbol, date), day in x.groupby(
    GROUP, sort=False
):

    for h in HORIZONS_22E:

        targets = {
            "return": f"future_return_{h}",
            "absret": f"future_absret_{h}",
            "spread_change":
                f"future_spread_change_{h}"
        }

        for task, target in targets.items():

            y = day[target].to_numpy(dtype=float)

            for feature in FEATURES_22E:

                z = day[feature].to_numpy(
                    dtype=float
                )

                valid = (
                    np.isfinite(z)
                    & np.isfinite(y)
                )

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "valid_rows": int(valid.sum()),
                    "ic": safe_ic_22e(z, y)
                })

calendar_exp22e_daily = pd.DataFrame(records)

calendar_exp22e_summary = (
    calendar_exp22e_daily
    .groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic",
            lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

calendar_exp22e_summary["sign_consistency"] = (
    calendar_exp22e_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp22e_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 9. Print results
# ------------------------------------------------------------

print("\n===== RECOVERY ASYMMETRY IC RESULTS =====")

print(
    calendar_exp22e_summary[
        calendar_exp22e_summary[
            "feature"
        ].isin(CANDIDATES_22E)
    ].round(5).to_string(index=False)
)

print("\n===== EVENT-ASYMMETRY CONTROL IC =====")

print(
    calendar_exp22e_summary[
        calendar_exp22e_summary[
            "feature"
        ].isin(ASYMMETRY_CONTROLS_22E)
    ].round(5).to_string(index=False)
)

print("\n===== BASELINE IC RESULTS =====")

print(
    calendar_exp22e_summary[
        calendar_exp22e_summary[
            "feature"
        ].isin(BASELINES_22E)
    ].round(5).to_string(index=False)
)

print("\n===== ASYMMETRY FEATURE COVERAGE =====")

for feature in CANDIDATES_22E:
    print(
        feature,
        "missing_pct:",
        round(
            100 * x[feature].isna().mean(), 4
        ),
        "unique:",
        x[feature].nunique()
    )

print("\n===== ASYMMETRY REDUNDANCY =====")

print(
    x[
        CANDIDATES_22E
        + ASYMMETRY_CONTROLS_22E
        + BASELINES_22E
    ].corr(method="spearman")
    .loc[
        CANDIDATES_22E,
        ASYMMETRY_CONTROLS_22E
        + BASELINES_22E
    ].round(5).to_string()
)

calendar_exp22e_features = x

print("\n===== EXPERIMENT 22E COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


Eligible sessions: 16
Rows: 459122

===== RECOVERY ASYMMETRY IC RESULTS =====
         task  horizon                        feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
       absret       30 bid_ask_recovery_asymmetry_300    16 -0.00292    0.00595              9              7            18603.0            0.5625
       absret       30  bid_ask_recovery_asymmetry_60    16 -0.01424    0.00106              9              7             5610.0            0.5625
       absret       60 bid_ask_recovery_asymmetry_300    16 -0.00472    0.00267              8              8            18573.0            0.5000
       absret       60  bid_ask_recovery_asymmetry_60    16  0.00227    0.00593             10              6             5595.0            0.6250
       absret      300 bid_ask_recovery_asymmetry_300    16  0.02222    0.01876              9              7            18341.5            0.5625
       absret      300  bid_ask_recovery

In [52]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22F — DIRECTIONAL INCREMENTAL ABLATION
#
# Requires calendar_exp22e_features from Experiment 22E
#
# Fixed chronological expanding-window ridge regression
# Targets: signed 30s, 60s, 300s mid-price returns
# Metrics: MSE (primary), MAE (secondary)
#
# No modification to df_1s
# No automatic feature promotion
# ============================================================

if "calendar_exp22e_features" not in globals():
    raise ValueError("Run Experiment 22E first.")

x = calendar_exp22e_features.copy()

GROUP = ["symbol", "date"]
HORIZONS = [30, 60, 300]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10
ALPHA = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date
x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp records.")

# ------------------------------------------------------------
# 1. Causal signed momentum
#
# Exact timestamp offsets, no crossing gaps or sessions.
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

for w in [30, 60]:
    previous_mid = g["mid"].shift(w)
    previous_ts = g["ts_sec"].shift(w)

    x[f"momentum_{w}_22f"] = (
        1e4 * np.log(x["mid"] / previous_mid)
    ).where(
        (x["ts_sec"] - previous_ts) == w
    )

# ------------------------------------------------------------
# 2. Causal realized volatility
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

previous_mid = g["mid"].shift(1)
previous_ts = g["ts_sec"].shift(1)

ret1 = np.log(x["mid"] / previous_mid)
ret1 = ret1.where(
    (x["ts_sec"] - previous_ts) == 1
)

x["_ret_sq_22f"] = ret1 ** 2

for w in [60, 300]:
    x[f"rv_{w}_22f"] = (
        x.groupby(GROUP, sort=False)["_ret_sq_22f"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 1e4
    )

# ------------------------------------------------------------
# 3. Book state and clock
# ------------------------------------------------------------

x["book_imbalance_22f"] = (
    (x["bid_depth"] - x["ask_depth"])
    /
    (x["bid_depth"] + x["ask_depth"]).where(
        (x["bid_depth"] + x["ask_depth"]) > 0
    )
)

x["clock_22f"] = (
    (x["ts_sec"] + 19800) % 86400
) - 32400

x["open_1_5m_22f"] = (
    x["clock_22f"].ge(60)
    & x["clock_22f"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 4. Recovery speed availability and encoding
#
# Crucial: Do not treat missing recovery as genuine zero.
# ------------------------------------------------------------

for w in [60, 300]:

    source = f"bid_ask_recovery_asymmetry_{w}"

    values = pd.to_numeric(
        x[source], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    available = np.isfinite(values)

    x[f"recovery_available_{w}_22f"] = (
        available.astype(float)
    )

    x[f"recovery_value_{w}_22f"] = (
        values.where(available, 0.0)
    )

# ------------------------------------------------------------
# 5. Feature sets
# ------------------------------------------------------------

BASE = [
    "bid_depth",
    "ask_depth",
    "bid_relative",
    "ask_relative",
    "book_imbalance_22f",
    "spread_bps",
    "momentum_30_22f",
    "momentum_60_22f",
    "rv_60_22f",
    "rv_300_22f",
    "clock_22f",
    "open_1_5m_22f"
]

EVENTS = [
    "depletion_event_asymmetry_60",
    "depletion_event_asymmetry_300"
]

RECOVERY_60 = [
    "recovery_available_60_22f",
    "recovery_value_60_22f"
]

RECOVERY_300 = [
    "recovery_available_300_22f",
    "recovery_value_300_22f"
]

MODELS = {
    "A_components": BASE,
    "B_plus_events": BASE + EVENTS,
    "C_plus_recovery60": BASE + EVENTS + RECOVERY_60,
    "D_plus_recovery300": BASE + EVENTS + RECOVERY_300,
    "E_plus_both": (
        BASE + EVENTS + RECOVERY_60 + RECOVERY_300
    )
}

all_features = sorted(set(
    feature
    for features in MODELS.values()
    for feature in features
))

targets = [
    f"future_return_{h}"
    for h in HORIZONS
]

missing = sorted(
    set(all_features + targets) - set(x.columns)
)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in all_features + targets:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Baseline and event information must be available.
# Recovery information is encoded using explicit flags.
COMMON = BASE + EVENTS

x["_common_22f"] = np.isfinite(
    x[COMMON].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 22F SAMPLE =====")
print("Total rows:", len(x))
print("Common valid rows:", int(x["_common_22f"].sum()))

for w in [60, 300]:
    print(
        f"Recovery {w}s availability:",
        round(
            x[f"recovery_available_{w}_22f"].mean() * 100,
            3
        ),
        "%"
    )

# ------------------------------------------------------------
# 6. Fixed ridge with training-only standardization
# ------------------------------------------------------------

def ridge_predict_22f(train, test, features, target):

    A = train[features].to_numpy(dtype=np.float64)
    B = test[features].to_numpy(dtype=np.float64)
    y = train[target].to_numpy(dtype=np.float64)

    if not (
        np.isfinite(A).all()
        and np.isfinite(B).all()
        and np.isfinite(y).all()
    ):
        raise ValueError("Nonfinite values in model data.")

    mean = A.mean(axis=0)
    std = A.std(axis=0)

    std = np.where(std > 1e-12, std, 1.0)

    A = (A - mean) / std
    B = (B - mean) / std

    ymean = y.mean()

    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A + ALPHA * n * np.eye(k),
        A.T @ (y - ymean)
    )

    # Signed returns: DO NOT clip predictions to zero.
    return ymean + B @ beta

# ------------------------------------------------------------
# 7. Expanding-window evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient trading dates.")

records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_22f"]
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test_pool = x[
        (x["date"] == test_date)
        & x["_common_22f"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y = test[target].to_numpy(dtype=float)

        preds = {
            name: ridge_predict_22f(
                train, test, features, target
            )
            for name, features in MODELS.items()
        }

        squared_errors = {
            name: (y - pred) ** 2
            for name, pred in preds.items()
        }

        absolute_errors = {
            name: np.abs(y - pred)
            for name, pred in preds.items()
        }

        # Compare every model against BOTH relevant baselines.
        mse_A = squared_errors["A_components"].mean()
        mse_B = squared_errors["B_plus_events"].mean()

        mae_A = absolute_errors["A_components"].mean()
        mae_B = absolute_errors["B_plus_events"].mean()

        for name in MODELS:

            mse = squared_errors[name].mean()
            mae = absolute_errors[name].mean()

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "mse": mse,
                "mae": mae,
                "mse_A": mse_A,
                "mse_B": mse_B,
                "mae_A": mae_A,
                "mae_B": mae_B,
                "mse_gain_vs_A_pct": (
                    100 * (1 - mse / mse_A)
                    if mse_A > 0 else np.nan
                ),
                "mse_gain_vs_B_pct": (
                    100 * (1 - mse / mse_B)
                    if mse_B > 0 else np.nan
                ),
                "mae_gain_vs_B_pct": (
                    100 * (1 - mae / mae_B)
                    if mae_B > 0 else np.nan
                )
            })

    print("Completed:", test_date)

calendar_exp22f_daily = pd.DataFrame(records)

if calendar_exp22f_daily.empty:
    raise ValueError("No completed evaluations.")

# ------------------------------------------------------------
# 8. Pooled summary
# ------------------------------------------------------------

summary = []

for (h, name), d in calendar_exp22f_daily.groupby(
    ["horizon", "model"]
):

    weights = d["rows"].to_numpy(dtype=float)

    mse = np.sum(d["mse"] * weights)
    mse_A = np.sum(d["mse_A"] * weights)
    mse_B = np.sum(d["mse_B"] * weights)

    mae = np.sum(d["mae"] * weights)
    mae_B = np.sum(d["mae_B"] * weights)

    summary.append({
        "horizon": h,
        "model": name,
        "days": d["date"].nunique(),
        "mse_gain_vs_A_pct": 100 * (1 - mse / mse_A),
        "mse_gain_vs_B_pct": 100 * (1 - mse / mse_B),
        "mae_gain_vs_B_pct": 100 * (1 - mae / mae_B),
        "winning_days_vs_B": int(
            (d["mse_gain_vs_B_pct"] > 0).sum()
        ),
        "worst_day_vs_B_pct": (
            d["mse_gain_vs_B_pct"].min()
        )
    })

calendar_exp22f_summary = pd.DataFrame(summary)

print("\n===== EXPERIMENT 22F POOLED RESULTS =====")

print(
    calendar_exp22f_summary.round(6)
    .to_string(index=False)
)

print("\n===== DAILY MSE GAINS VS EVENT BASELINE =====")

print(
    calendar_exp22f_daily[
        calendar_exp22f_daily["model"].isin([
            "C_plus_recovery60",
            "D_plus_recovery300",
            "E_plus_both"
        ])
    ].pivot_table(
        index=["date", "horizon"],
        columns="model",
        values="mse_gain_vs_B_pct"
    ).round(5).to_string()
)

print("\n===== EXPERIMENT 22F COMPLETE =====")
print("df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 22F SAMPLE =====
Total rows: 459122
Common valid rows: 450482
Recovery 60s availability: 20.565 %
Recovery 300s availability: 66.267 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== EXPERIMENT 22F POOLED RESULTS =====
 horizon              model  days  mse_gain_vs_A_pct  mse_gain_vs_B_pct  mae_gain_vs_B_pct  winning_days_vs_B  worst_day_vs_B_pct
      30       A_components    11           0.000000           0.056188           0.065182                  6           -0.226698
      30      B_plus_events    11          -0.056219           0.000000           0.000000                  0            0.000000
      30  C_plus_recovery60    11          -0.095300          -0.039058          -0.056693                  3           -0.327962
      30 D_plus_recovery300    11          

In [53]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 22G
# RECOVERY MAGNITUDE VS AVAILABILITY
#
# Input: calendar_exp22e_features
# Target: signed return, 300-second horizon
#
# Fixed ridge model; expanding walk-forward
# Same model settings as Experiment 22F
#
# No modification to df_1s
# ============================================================

if "calendar_exp22e_features" not in globals():
    raise ValueError("Run Experiment 22E first.")

calendar_exp22g_work = calendar_exp22e_features.copy()

x = calendar_exp22g_work
x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

GROUP = ["symbol", "date"]
TARGET = "future_return_300"
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10
ALPHA = 0.01

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate timestamps")

# ------------------------------------------------------------
# 1. Reconstruct Experiment 22F baseline
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

for w in [30, 60]:

    old_mid = g["mid"].shift(w)
    old_ts = g["ts_sec"].shift(w)

    x[f"momentum_{w}_22g"] = (
        1e4 * np.log(x["mid"] / old_mid)
    ).where(x["ts_sec"] - old_ts == w)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

ret = np.log(x["mid"] / prev_mid)
ret = ret.where(x["ts_sec"] - prev_ts == 1)

x["_ret_sq_22g"] = ret ** 2

for w in [60, 300]:

    x[f"rv_{w}_22g"] = (
        x.groupby(GROUP, sort=False)["_ret_sq_22g"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5) * 1e4
    )

total_depth = x["bid_depth"] + x["ask_depth"]

x["book_imbalance_22g"] = (
    (x["bid_depth"] - x["ask_depth"])
    / total_depth.where(total_depth > 0)
)

x["clock_22g"] = (
    (x["ts_sec"] + 19800) % 86400
) - 32400

x["open_1_5m_22g"] = (
    x["clock_22g"].ge(60)
    & x["clock_22g"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 2. Recovery availability and magnitude
# ------------------------------------------------------------

source = pd.to_numeric(
    x["bid_ask_recovery_asymmetry_300"],
    errors="coerce"
).replace([np.inf, -np.inf], np.nan)

x["recovery_available_300_22g"] = (
    source.notna().astype(float)
)

x["recovery_value_300_22g"] = (
    source.fillna(0.0)
)

# ------------------------------------------------------------
# 3. Frozen feature configurations
# ------------------------------------------------------------

BASE = [
    "bid_depth",
    "ask_depth",
    "bid_relative",
    "ask_relative",
    "book_imbalance_22g",
    "spread_bps",
    "momentum_30_22g",
    "momentum_60_22g",
    "rv_60_22g",
    "rv_300_22g",
    "clock_22g",
    "open_1_5m_22g"
]

EVENTS = [
    "depletion_event_asymmetry_60",
    "depletion_event_asymmetry_300"
]

AVAIL = ["recovery_available_300_22g"]
VALUE = ["recovery_value_300_22g"]

MODELS = {
    "A_components": BASE,

    "A_plus_availability": (
        BASE + AVAIL
    ),

    "A_plus_recovery": (
        BASE + AVAIL + VALUE
    ),

    "B_plus_events": (
        BASE + EVENTS
    ),

    "B_plus_availability": (
        BASE + EVENTS + AVAIL
    ),

    "B_plus_recovery": (
        BASE + EVENTS + AVAIL + VALUE
    )
}

all_features = sorted(set(
    f for fs in MODELS.values() for f in fs
))

required = all_features + [TARGET]

missing = sorted(set(required) - set(x.columns))
if missing:
    raise ValueError(f"Missing: {missing}")

for c in required:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Same common sample as 22F:
# components + event asymmetry must be valid.
COMMON = BASE + EVENTS

x["_common_22g"] = np.isfinite(
    x[COMMON].to_numpy(dtype=float)
).all(axis=1)

print("===== SAMPLE =====")
print("Total rows:", len(x))
print("Common valid rows:", int(x["_common_22g"].sum()))
print(
    "Recovery available (%):",
    round(
        100 * x["recovery_available_300_22g"].mean(),
        3
    )
)

# ------------------------------------------------------------
# 4. Fixed ridge predictor
# ------------------------------------------------------------

def ridge_predict_22g(train, test, features):

    A = train[features].to_numpy(dtype=float)
    B = test[features].to_numpy(dtype=float)
    y = train[TARGET].to_numpy(dtype=float)

    if (
        not np.isfinite(A).all()
        or not np.isfinite(B).all()
        or not np.isfinite(y).all()
    ):
        raise ValueError("Nonfinite regression input")

    mu = A.mean(axis=0)
    sd = A.std(axis=0)
    sd = np.where(sd > 1e-12, sd, 1.0)

    A = (A - mu) / sd
    B = (B - mu) / sd

    ym = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A + ALPHA * n * np.eye(k),
        A.T @ (y - ym)
    )

    return ym + B @ beta

# ------------------------------------------------------------
# 5. Chronological expanding-window evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())
records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train = x[
        x["date"].isin(dates[:i])
        & x["_common_22g"]
        & np.isfinite(x[TARGET])
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test = x[
        (x["date"] == test_date)
        & x["_common_22g"]
        & np.isfinite(x[TARGET])
    ]

    if len(train) < 1000 or len(test) < 200:
        continue

    y = test[TARGET].to_numpy(dtype=float)

    for name, features in MODELS.items():

        prediction = ridge_predict_22g(
            train, test, features
        )

        error = y - prediction

        records.append({
            "date": test_date,
            "model": name,
            "rows": len(test),
            "sse": float(np.sum(error ** 2)),
            "sae": float(np.sum(np.abs(error))),
            "mse": float(np.mean(error ** 2)),
            "mae": float(np.mean(np.abs(error)))
        })

    print("Completed:", test_date)

calendar_exp22g_daily = pd.DataFrame(records)

if calendar_exp22g_daily.empty:
    raise ValueError("No completed evaluations")

# ------------------------------------------------------------
# 6. Compare against appropriate nested baselines
# ------------------------------------------------------------

daily = calendar_exp22g_daily.pivot(
    index="date",
    columns="model",
    values="mse"
)

comparisons = {
    "events_vs_components": (
        "B_plus_events", "A_components"
    ),
    "availability_vs_A": (
        "A_plus_availability", "A_components"
    ),
    "recovery_vs_A_availability": (
        "A_plus_recovery", "A_plus_availability"
    ),
    "availability_vs_B": (
        "B_plus_availability", "B_plus_events"
    ),
    "recovery_vs_B_availability": (
        "B_plus_recovery", "B_plus_availability"
    ),
    "full_recovery_vs_B": (
        "B_plus_recovery", "B_plus_events"
    ),
    "full_recovery_vs_A": (
        "B_plus_recovery", "A_components"
    )
}

sums = (
    calendar_exp22g_daily
    .groupby("model")[["sse", "sae"]]
    .sum()
)

summary = []
daily_gains = pd.DataFrame(index=daily.index)

for label, (candidate, baseline) in comparisons.items():

    pooled_gain = 100 * (
        1 - sums.loc[candidate, "sse"]
        / sums.loc[baseline, "sse"]
    )

    gain = 100 * (
        1 - daily[candidate] / daily[baseline]
    )

    daily_gains[label] = gain

    summary.append({
        "comparison": label,
        "pooled_mse_gain_pct": pooled_gain,
        "winning_days": int((gain > 0).sum()),
        "days": len(gain),
        "median_daily_gain_pct": gain.median(),
        "worst_day_pct": gain.min()
    })

calendar_exp22g_summary = pd.DataFrame(summary)
calendar_exp22g_daily_gains = daily_gains

print("\n===== 22G NESTED ABLATION RESULTS =====")
print(
    calendar_exp22g_summary.round(6)
    .to_string(index=False)
)

print("\n===== DAILY RECOVERY MAGNITUDE GAINS =====")
print(
    calendar_exp22g_daily_gains[
        [
            "recovery_vs_A_availability",
            "recovery_vs_B_availability"
        ]
    ].round(6).to_string()
)

print("\n===== EXPERIMENT 22G COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== SAMPLE =====
Total rows: 459122
Common valid rows: 450482
Recovery available (%): 66.267
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 22G NESTED ABLATION RESULTS =====
                comparison  pooled_mse_gain_pct  winning_days  days  median_daily_gain_pct  worst_day_pct
      events_vs_components            -0.054493             5    11              -0.022211      -0.461794
         availability_vs_A            -0.055791             5    11              -0.003440      -0.571253
recovery_vs_A_availability             0.313329             7    11               0.025227      -0.155497
         availability_vs_B            -0.053756             5    11              -0.002544      -0.561607
recovery_vs_B_availability             0.274753             6    11               0.046300  

In [54]:

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except ImportError:
    raise ImportError("Install LightGBM with %pip install lightgbm")

# ============================================================
# EXPERIMENT 22H — NONLINEAR RECOVERY ABLATION
#
# Target: future_return_300 (signed bps)
# Expanding chronological validation
# Fixed LightGBM hyperparameters
# Common train/test rows across models
#
# Original df_1s remains unchanged.
# ============================================================

if "calendar_exp22e_features" not in globals():
    raise ValueError("Run Experiment 22E first.")

x = calendar_exp22e_features.copy()

GROUP = ["symbol", "date"]
TARGET = "future_return_300"
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

x["date"] = pd.to_datetime(x["date"]).dt.date
x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows")

# ------------------------------------------------------------
# 1. Rebuild the same causal controls as 22F/22G
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

for w in [30, 60]:

    previous_mid = g["mid"].shift(w)
    previous_ts = g["ts_sec"].shift(w)

    x[f"momentum_{w}_22h"] = (
        1e4 * np.log(x["mid"] / previous_mid)
    ).where(
        (x["ts_sec"] - previous_ts) == w
    )

previous_mid = g["mid"].shift(1)
previous_ts = g["ts_sec"].shift(1)

ret1 = np.log(x["mid"] / previous_mid)
ret1 = ret1.where(
    (x["ts_sec"] - previous_ts) == 1
)

x["_ret_sq_22h"] = ret1 ** 2

for w in [60, 300]:

    x[f"rv_{w}_22h"] = (
        x.groupby(GROUP, sort=False)["_ret_sq_22h"]
        .transform(
            lambda s: s.rolling(
                window=w,
                min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5) * 1e4
    )

total_depth = x["bid_depth"] + x["ask_depth"]

x["book_imbalance_22h"] = (
    (x["bid_depth"] - x["ask_depth"])
    / total_depth.where(total_depth > 0)
)

x["clock_22h"] = (
    (x["ts_sec"] + 19800) % 86400
) - 32400

x["open_1_5m_22h"] = (
    x["clock_22h"].ge(60)
    & x["clock_22h"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 2. Explicit recovery availability
# ------------------------------------------------------------

recovery = pd.to_numeric(
    x["bid_ask_recovery_asymmetry_300"],
    errors="coerce"
).replace([np.inf, -np.inf], np.nan)

x["recovery_available_300_22h"] = (
    recovery.notna().astype(float)
)

x["recovery_value_300_22h"] = (
    recovery.fillna(0.0)
)

# ------------------------------------------------------------
# 3. Fixed model specifications
# ------------------------------------------------------------

BASE = [
    "bid_depth",
    "ask_depth",
    "bid_relative",
    "ask_relative",
    "book_imbalance_22h",
    "spread_bps",
    "momentum_30_22h",
    "momentum_60_22h",
    "rv_60_22h",
    "rv_300_22h",
    "clock_22h",
    "open_1_5m_22h"
]

EVENTS = [
    "depletion_event_asymmetry_60",
    "depletion_event_asymmetry_300"
]

AVAIL = ["recovery_available_300_22h"]
VALUE = ["recovery_value_300_22h"]

MODELS = {
    "A_components": BASE,
    "B_events": BASE + EVENTS,
    "C_availability": BASE + EVENTS + AVAIL,
    "D_recovery": BASE + EVENTS + AVAIL + VALUE
}

all_features = sorted(set(
    f for features in MODELS.values() for f in features
))

required = all_features + [TARGET]

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in required:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Match Experiment 22G's common sample.
COMMON = BASE + EVENTS

x["_common_22h"] = np.isfinite(
    x[COMMON + [TARGET]].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 22H DATA =====")
print("Rows:", len(x))
print("Common valid rows:", int(x["_common_22h"].sum()))
print(
    "Recovery availability:",
    round(
        100 * x["recovery_available_300_22h"].mean(),
        3
    ),
    "%"
)

# ------------------------------------------------------------
# 4. Walk-forward evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient trading sessions")

records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    date = dates[i]

    train = x[
        x["date"].isin(dates[:i])
        & x["_common_22h"]
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test = x[
        (x["date"] == date)
        & x["_common_22h"]
    ]

    if len(train) < 1000 or len(test) < 200:
        continue

    y_train = train[TARGET].to_numpy(dtype=float)
    y_test = test[TARGET].to_numpy(dtype=float)

    for name, features in MODELS.items():

        model = lgb.LGBMRegressor(**PARAMS)

        model.fit(
            train[features],
            y_train
        )

        pred = model.predict(
            test[features]
        )

        # Signed returns: no clipping.
        error = y_test - pred

        records.append({
            "date": date,
            "model": name,
            "rows": len(test),
            "sse": float(np.sum(error ** 2)),
            "sae": float(np.sum(np.abs(error))),
            "mse": float(np.mean(error ** 2)),
            "mae": float(np.mean(np.abs(error)))
        })

    print("Completed:", date)

calendar_exp22h_daily = pd.DataFrame(records)

if calendar_exp22h_daily.empty:
    raise ValueError("No model evaluations completed")

# ------------------------------------------------------------
# 5. Nested pooled MSE / MAE comparison
# ------------------------------------------------------------

comparisons = {
    "events_vs_components": (
        "B_events", "A_components"
    ),
    "availability_vs_events": (
        "C_availability", "B_events"
    ),
    "recovery_vs_availability": (
        "D_recovery", "C_availability"
    ),
    "recovery_vs_components": (
        "D_recovery", "A_components"
    )
}

pooled = calendar_exp22h_daily.groupby(
    "model"
)[["sse", "sae"]].sum()

daily_mse = calendar_exp22h_daily.pivot(
    index="date", columns="model", values="mse"
)

summary = []
daily_gains = pd.DataFrame(index=daily_mse.index)

for label, (candidate, baseline) in comparisons.items():

    gain_by_day = 100 * (
        1 - daily_mse[candidate] / daily_mse[baseline]
    )

    daily_gains[label] = gain_by_day

    summary.append({
        "comparison": label,
        "days": len(gain_by_day),
        "pooled_mse_gain_pct": (
            100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            )
        ),
        "pooled_mae_gain_pct": (
            100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            )
        ),
        "winning_days": int(
            (gain_by_day > 0).sum()
        ),
        "median_daily_gain_pct": gain_by_day.median(),
        "worst_day_pct": gain_by_day.min()
    })

calendar_exp22h_summary = pd.DataFrame(summary)
calendar_exp22h_daily_gains = daily_gains

print("\n===== EXPERIMENT 22H NONLINEAR ABLATION =====")
print(
    calendar_exp22h_summary.round(6)
    .to_string(index=False)
)

print("\n===== DAILY RECOVERY VS AVAILABILITY =====")
print(
    daily_gains[
        ["recovery_vs_availability"]
    ].round(6).to_string()
)

print("\n===== EXPERIMENT 22H COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 22H DATA =====
Rows: 459122
Common valid rows: 445682
Recovery availability: 66.267 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== EXPERIMENT 22H NONLINEAR ABLATION =====
              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
    events_vs_components    11            -0.437271            -0.144944             4              -0.435446      -2.095774
  availability_vs_events    11             0.073896             0.074218             5              -0.018332      -0.708199
recovery_vs_availability    11             0.344506             0.246576             6               0.263480      -1.814633
  recovery_vs_components    11            -0.017295             0.176131             7               0.43851

In [55]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 22I
# DIRECT LIGHTGBM RECOVERY-MAGNITUDE ABLATION
#
# Requires completed Experiment 22H:
#   x contains the prepared 22H feature columns
#
# Target: signed 300-second return
# Model settings unchanged from 22H
# No modification to df_1s
# ============================================================

required_22i = [
    "symbol", "date", "ts_sec",
    "future_return_300",
    "bid_depth", "ask_depth",
    "bid_relative", "ask_relative",
    "book_imbalance_22h",
    "spread_bps",
    "momentum_30_22h",
    "momentum_60_22h",
    "rv_60_22h",
    "rv_300_22h",
    "clock_22h",
    "open_1_5m_22h",
    "recovery_available_300_22h",
    "recovery_value_300_22h"
]

if "x" not in globals():
    raise ValueError("Run Experiment 22H first.")

missing = sorted(set(required_22i) - set(x.columns))
if missing:
    raise ValueError(
        f"Required 22H fields missing: {missing}. "
        "Rerun Experiment 22H first."
    )

work = x[required_22i].copy()

work["date"] = pd.to_datetime(
    work["date"]
).dt.date

work = work.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

BASE = [
    "bid_depth", "ask_depth",
    "bid_relative", "ask_relative",
    "book_imbalance_22h",
    "spread_bps",
    "momentum_30_22h",
    "momentum_60_22h",
    "rv_60_22h",
    "rv_300_22h",
    "clock_22h",
    "open_1_5m_22h"
]

AVAIL = ["recovery_available_300_22h"]
VALUE = ["recovery_value_300_22h"]

MODELS = {
    "A_components": BASE,
    "A_plus_availability": BASE + AVAIL,
    "A_plus_recovery": BASE + AVAIL + VALUE
}

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

TARGET = "future_return_300"

numeric_cols = sorted(set(
    BASE + AVAIL + VALUE + [TARGET]
))

for col in numeric_cols:
    work[col] = pd.to_numeric(
        work[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Use exactly the same common sample as 22H,
# including event-asymmetry validity, so results
# remain directly comparable to the earlier test.
#
# The event controls are used only for sample
# selection, not as predictors.

if "_common_22h" not in x.columns:
    raise ValueError(
        "Experiment 22H common-sample mask missing."
    )

work["_common"] = (
    x["_common_22h"]
    .reset_index(drop=True)
    .astype(bool)
)

# IMPORTANT: x from 22H is already sorted in
# symbol/date/ts order. Re-align explicitly by
# recreating from the unmodified 22H input order.
# Avoid relying on positional matching after sort.

source = x.copy()
source["date"] = pd.to_datetime(
    source["date"]
).dt.date

source = source.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

work["_common"] = (
    source["_common_22h"].astype(bool)
)

work["_common"] &= np.isfinite(
    work[numeric_cols].to_numpy(dtype=float)
).all(axis=1)

dates = sorted(work["date"].unique())

records = []

for i in range(5, len(dates)):

    date = dates[i]

    train = work[
        work["date"].isin(dates[:i])
        & work["_common"]
        & (work["ts_sec"] % 10 == 0)
    ]

    test = work[
        (work["date"] == date)
        & work["_common"]
    ]

    if len(train) < 1000 or len(test) < 200:
        continue

    y_train = train[TARGET].to_numpy(dtype=float)
    y_test = test[TARGET].to_numpy(dtype=float)

    for name, cols in MODELS.items():

        model = lgb.LGBMRegressor(**PARAMS)

        model.fit(train[cols], y_train)

        pred = model.predict(test[cols])

        err = y_test - pred

        records.append({
            "date": date,
            "model": name,
            "rows": len(test),
            "sse": float(np.sum(err ** 2)),
            "sae": float(np.sum(np.abs(err)))
        })

    print("Completed:", date)

calendar_exp22i_daily = pd.DataFrame(records)

if calendar_exp22i_daily.empty:
    raise ValueError("No valid evaluations.")

wide = calendar_exp22i_daily.pivot(
    index="date",
    columns="model",
    values="sse"
)

pooled = calendar_exp22i_daily.groupby(
    "model"
)[["sse", "sae"]].sum()

COMPARISONS = {
    "availability_vs_components": (
        "A_plus_availability",
        "A_components"
    ),
    "recovery_vs_availability": (
        "A_plus_recovery",
        "A_plus_availability"
    ),
    "recovery_vs_components": (
        "A_plus_recovery",
        "A_components"
    )
}

summary = []
daily = pd.DataFrame(index=wide.index)

for name, (candidate, baseline) in COMPARISONS.items():

    gain = 100 * (
        1 - wide[candidate] / wide[baseline]
    )

    daily[name] = gain

    summary.append({
        "comparison": name,
        "pooled_mse_gain_pct": 100 * (
            1 - pooled.loc[candidate, "sse"]
            / pooled.loc[baseline, "sse"]
        ),
        "pooled_mae_gain_pct": 100 * (
            1 - pooled.loc[candidate, "sae"]
            / pooled.loc[baseline, "sae"]
        ),
        "winning_days": int((gain > 0).sum()),
        "days": len(gain),
        "worst_day_pct": gain.min()
    })

calendar_exp22i_summary = pd.DataFrame(summary)
calendar_exp22i_daily_gains = daily

print("\n===== EXPERIMENT 22I SUMMARY =====")
print(
    calendar_exp22i_summary.round(6)
    .to_string(index=False)
)

print("\n===== DAILY RECOVERY VS COMPONENT BASELINE =====")
print(
    daily[
        [
            "recovery_vs_availability",
            "recovery_vs_components"
        ]
    ].round(6).to_string()
)

print("\n===== EXPERIMENT 22I COMPLETE =====")
print("No original dataframe modified.")
print("No feature promoted.")


Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== EXPERIMENT 22I SUMMARY =====
                comparison  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  days  worst_day_pct
availability_vs_components             0.245699             0.158803             8    11      -0.044096
  recovery_vs_availability            -0.340866            -0.159037             6    11      -4.230891
    recovery_vs_components            -0.094330             0.000019             6    11      -3.850151

===== DAILY RECOVERY VS COMPONENT BASELINE =====
            recovery_vs_availability  recovery_vs_components
date                                                        
2026-10-05                  4.357181                4.613291
2026-10-06                 -2.398397               -2.019380
2026-10-07  

In [56]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 23A — OFI ABSORPTION SCREENING
#
# Hypothesis:
# Strong directional Level-1 OFI with weak realized
# price displacement may predict subsequent returns.
#
# Windows: 60, 300 seconds
# Targets: signed and absolute mid-price returns
# Horizons: 30, 60, 300 seconds
#
# Causal within-session calculations
# Exact future timestamp targets
# Daily Spearman IC
# No modification to df_1s
# ============================================================

WINDOWS_23A = [60, 300]
HORIZONS_23A = [30, 60, 300]
MIN_SESSION_COVERAGE_23A = 0.97
VOL_FLOOR_BPS_23A = 0.1

required = [
    "symbol", "ts_sec",
    "bid_p1", "ask_p1",
    "bid_q1", "ask_q1"
]

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing required fields: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate column names")

x = df_1s[required].copy()

for c in required:
    if c != "symbol":
        x[c] = pd.to_numeric(x[c], errors="coerce")

if x["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp records")

# ------------------------------------------------------------
# 1. IST session filtering
# ------------------------------------------------------------

ts = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ts.dt.date

x["clock"] = (
    ts.dt.hour * 3600
    + ts.dt.minute * 60
    + ts.dt.second
)

x = x[
    x["clock"].ge(32400)
    & x["clock"].lt(61200)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage[
    coverage["rows"].ge(
        28800 * MIN_SESSION_COVERAGE_23A
    )
    & coverage["date"].astype(str).ne("2026-09-28")
][["symbol", "date"]]

x = x.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

if x.empty:
    raise ValueError("No eligible sessions")

GROUP = ["symbol", "date"]

print("===== EXPERIMENT 23A DATA =====")
print("Eligible sessions:", len(eligible))
print("Rows:", len(x))

# ------------------------------------------------------------
# 2. Quote validity and mid-price
# ------------------------------------------------------------

bid_p = x["bid_p1"]
ask_p = x["ask_p1"]
bid_q = x["bid_q1"]
ask_q = x["ask_q1"]

valid_quote = (
    bid_p.gt(0)
    & ask_p.gt(0)
    & ask_p.ge(bid_p)
    & bid_q.ge(0)
    & ask_q.ge(0)
    & np.isfinite(
        x[[
            "bid_p1", "ask_p1",
            "bid_q1", "ask_q1"
        ]]
    ).all(axis=1)
)

x["mid"] = (
    (bid_p + ask_p) / 2
).where(valid_quote)

x["spread_bps"] = (
    1e4 * (ask_p - bid_p) / x["mid"]
).where(valid_quote)

# ------------------------------------------------------------
# 3. Causal quote-based OFI increment
#
# Only exact consecutive-second quote pairs.
# At an unchanged quote, e_t = 0.
# A forward-filled quote contributes zero, not
# invented activity.
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

prev_ts = g["ts_sec"].shift(1)
prev_bp = g["bid_p1"].shift(1)
prev_ap = g["ask_p1"].shift(1)
prev_bq = g["bid_q1"].shift(1)
prev_aq = g["ask_q1"].shift(1)

valid_prev_quote = (
    prev_bp.gt(0)
    & prev_ap.gt(0)
    & prev_ap.ge(prev_bp)
    & prev_bq.ge(0)
    & prev_aq.ge(0)
    & np.isfinite(prev_bp)
    & np.isfinite(prev_ap)
    & np.isfinite(prev_bq)
    & np.isfinite(prev_aq)
)

valid_pair = (
    valid_quote
    & valid_prev_quote
    & (x["ts_sec"] - prev_ts).eq(1)
)

bid_contrib = (
    (bid_p >= prev_bp).astype(float) * bid_q
    - (bid_p <= prev_bp).astype(float) * prev_bq
)

ask_contrib = (
    - (ask_p <= prev_ap).astype(float) * ask_q
    + (ask_p >= prev_ap).astype(float) * prev_aq
)

x["ofi_increment_23a"] = (
    bid_contrib + ask_contrib
).where(valid_pair)

x["abs_ofi_increment_23a"] = (
    x["ofi_increment_23a"].abs()
)

x["quote_changed_23a"] = (
    (
        bid_p.ne(prev_bp)
        | ask_p.ne(prev_ap)
        | bid_q.ne(prev_bq)
        | ask_q.ne(prev_aq)
    ).astype(float)
).where(valid_pair)

# ------------------------------------------------------------
# 4. Causal one-second mid returns
# ------------------------------------------------------------

prev_mid = g["mid"].shift(1)

r1 = np.log(x["mid"] / prev_mid)
r1 = r1.where(valid_pair)

x["ret_sq_23a"] = r1 ** 2

# ------------------------------------------------------------
# 5. Candidate families
#
# pressure_W:
#     signed net OFI / gross OFI
#
# absorption_W:
#     pressure_W / (1 + abs momentum / RV)
#
# Includes controls so absorption is not confused
# with OFI or low volatility by themselves.
# ------------------------------------------------------------

CANDIDATES_23A = []
CONTROLS_23A = [
    "spread_bps"
]

for w in WINDOWS_23A:

    minp = int(np.ceil(0.90 * w))

    # Rolling net and gross OFI
    x[f"ofi_sum_{w}_23a"] = (
        x.groupby(GROUP, sort=False)
        ["ofi_increment_23a"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).sum()
        )
    )

    x[f"ofi_gross_{w}_23a"] = (
        x.groupby(GROUP, sort=False)
        ["abs_ofi_increment_23a"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).sum()
        )
    )

    gross = x[f"ofi_gross_{w}_23a"]
    net = x[f"ofi_sum_{w}_23a"]

    x[f"ofi_pressure_{w}_23a"] = (
        net / gross.where(gross > 0)
    )

    # Valid zero-flow window:
    # net=0 and gross=0 => pressure=0.
    x.loc[
        gross.eq(0) & net.notna(),
        f"ofi_pressure_{w}_23a"
    ] = 0.0

    # Exact W-second historical return.
    grouped = x.groupby(GROUP, sort=False)

    old_mid = grouped["mid"].shift(w)
    old_ts = grouped["ts_sec"].shift(w)

    x[f"momentum_{w}_23a"] = (
        1e4 * np.log(x["mid"] / old_mid)
    ).where(
        (x["ts_sec"] - old_ts).eq(w)
    )

    # Trailing realized volatility.
    x[f"rv_{w}_23a"] = (
        x.groupby(GROUP, sort=False)
        ["ret_sq_23a"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 1e4
    )

    # Quote-change frequency diagnostic/control.
    x[f"quote_change_rate_{w}_23a"] = (
        x.groupby(GROUP, sort=False)
        ["quote_changed_23a"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).mean()
        )
    )

    pressure = x[f"ofi_pressure_{w}_23a"]
    momentum = x[f"momentum_{w}_23a"]
    rv = x[f"rv_{w}_23a"]

    displacement_ratio = (
        momentum.abs()
        / rv.clip(lower=VOL_FLOOR_BPS_23A)
    )

    feature_name = f"ofi_absorption_{w}_23a"

    x[feature_name] = (
        pressure / (1.0 + displacement_ratio)
    )

    CANDIDATES_23A.append(feature_name)

    CONTROLS_23A.extend([
        f"ofi_pressure_{w}_23a",
        f"ofi_sum_{w}_23a",
        f"momentum_{w}_23a",
        f"rv_{w}_23a",
        f"quote_change_rate_{w}_23a"
    ])

# ------------------------------------------------------------
# 6. Exact future signed and absolute returns
# ------------------------------------------------------------

for h in HORIZONS_23A:

    future = x[
        ["symbol", "date", "ts_sec", "mid"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    x = x.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    x[f"future_return_{h}"] = (
        1e4 * np.log(
            x[f"future_mid_{h}"] / x["mid"]
        )
    )

    x[f"future_absret_{h}"] = (
        x[f"future_return_{h}"].abs()
    )

# ------------------------------------------------------------
# 7. Daily Spearman IC
# ------------------------------------------------------------

def safe_spearman_23a(a, b):

    mask = (
        np.isfinite(a)
        & np.isfinite(b)
    )

    if mask.sum() < 200:
        return np.nan, int(mask.sum())

    ra = pd.Series(a[mask]).rank().to_numpy()
    rb = pd.Series(b[mask]).rank().to_numpy()

    if ra.std() < 1e-12 or rb.std() < 1e-12:
        return np.nan, int(mask.sum())

    return (
        float(np.corrcoef(ra, rb)[0, 1]),
        int(mask.sum())
    )

features = CANDIDATES_23A + CONTROLS_23A

records = []

for (symbol, date), day in x.groupby(
    GROUP, sort=False
):

    for h in HORIZONS_23A:

        targets = {
            "signed_return":
                f"future_return_{h}",
            "absret":
                f"future_absret_{h}"
        }

        for task, target in targets.items():

            y = day[target].to_numpy(dtype=float)

            for feature in features:

                z = day[feature].to_numpy(
                    dtype=float
                )

                ic, n = safe_spearman_23a(z, y)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "valid_rows": n,
                    "ic": ic
                })

calendar_exp23a_daily = pd.DataFrame(records)

calendar_exp23a_summary = (
    calendar_exp23a_daily
    .groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

calendar_exp23a_summary["sign_consistency"] = (
    calendar_exp23a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp23a_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 8. Redundancy with component signals
# ------------------------------------------------------------

redundancy_cols = (
    CANDIDATES_23A + CONTROLS_23A
)

calendar_exp23a_redundancy = (
    x[redundancy_cols]
    .corr(method="spearman")
    .loc[
        CANDIDATES_23A,
        CONTROLS_23A
    ]
)

# ------------------------------------------------------------
# 9. Diagnostics
# ------------------------------------------------------------

print("\n===== ABSORPTION SIGNED-RETURN IC =====")

print(
    calendar_exp23a_summary[
        calendar_exp23a_summary[
            "feature"
        ].isin(CANDIDATES_23A)
        & calendar_exp23a_summary[
            "task"
        ].eq("signed_return")
    ].round(6).to_string(index=False)
)

print("\n===== ABSORPTION ABSOLUTE-RETURN IC =====")

print(
    calendar_exp23a_summary[
        calendar_exp23a_summary[
            "feature"
        ].isin(CANDIDATES_23A)
        & calendar_exp23a_summary[
            "task"
        ].eq("absret")
    ].round(6).to_string(index=False)
)

print("\n===== DIRECTIONAL COMPONENT BASELINES =====")

print(
    calendar_exp23a_summary[
        calendar_exp23a_summary[
            "task"
        ].eq("signed_return")
        & calendar_exp23a_summary[
            "feature"
        ].isin([
            "ofi_pressure_60_23a",
            "ofi_pressure_300_23a",
            "momentum_60_23a",
            "momentum_300_23a",
            "quote_change_rate_60_23a"
        ])
    ].round(6).to_string(index=False)
)

print("\n===== ABSORPTION REDUNDANCY =====")

print(
    calendar_exp23a_redundancy.round(4)
    .to_string()
)

print("\n===== FEATURE AVAILABILITY =====")

for name in CANDIDATES_23A:

    s = x[name]

    print(
        name,
        "valid_pct:",
        round(100 * s.notna().mean(), 3),
        "mean:",
        round(s.mean(), 6),
        "std:",
        round(s.std(), 6),
        "zero_pct:",
        round(100 * s.eq(0).mean(), 3)
    )

print("\n===== OFI / PRICE DATA DIAGNOSTICS =====")

for w in WINDOWS_23A:

    gross = x[f"ofi_gross_{w}_23a"]

    print(
        f"W={w}: zero_gross_ofi_pct=",
        round(100 * gross.eq(0).mean(), 3),
        "quote_change_rate_mean=",
        round(
            x[f"quote_change_rate_{w}_23a"].mean(),
            5
        )
    )

calendar_exp23a_features = x

print("\n===== EXPERIMENT 23A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


Task was destroyed but it is pending!
task: <Task pending name='Task-326' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-327' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\projects\hedge_fund\venv\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-327' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>


===== EXPERIMENT 23A DATA =====
Eligible sessions: 16
Rows: 459122

===== ABSORPTION SIGNED-RETURN IC =====
         task  horizon                feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
signed_return       30 ofi_absorption_300_23a    16 0.030612   0.025440             14              2            28423.0            0.8750
signed_return       30  ofi_absorption_60_23a    16 0.039572   0.036815             13              3            28663.0            0.8125
signed_return       60 ofi_absorption_300_23a    16 0.034121   0.038132             14              2            28393.0            0.8750
signed_return       60  ofi_absorption_60_23a    16 0.046744   0.052945             12              4            28633.0            0.7500
signed_return      300 ofi_absorption_300_23a    16 0.023434   0.019341             10              6            28153.0            0.6250
signed_return      300  ofi_absorption_60_23a    16 0.0363

In [57]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 23B — OFI/PRICE DIRECTIONAL DISAGREEMENT
#
# Source: calendar_exp23a_features
# Windows: 60, 300 seconds
# Future signed-return horizons: 30, 60, 300 seconds
#
# Hypotheses:
#   1. OFI and historical price return disagree.
#   2. The magnitude of that disagreement matters.
#
# No df_1s modification.
# No feature promotion.
# ============================================================

if "calendar_exp23a_features" not in globals():
    raise ValueError("Run Experiment 23A first.")

x = calendar_exp23a_features.copy()

WINDOWS = [60, 300]
HORIZONS = [30, 60, 300]
RV_FLOOR = 0.1

GROUP = ["symbol", "date"]

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

required = [
    "symbol", "date", "ts_sec", "spread_bps"
]

for w in WINDOWS:
    required += [
        f"ofi_pressure_{w}_23a",
        f"momentum_{w}_23a",
        f"rv_{w}_23a"
    ]

for h in HORIZONS:
    required += [f"future_return_{h}"]

missing = sorted(set(required) - set(x.columns))
if missing:
    raise ValueError(f"Missing fields: {missing}")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp pairs.")

# ------------------------------------------------------------
# 1. Build signed disagreement candidates
# ------------------------------------------------------------

CANDIDATES = []
CONTROLS = []

for w in WINDOWS:

    pressure = pd.to_numeric(
        x[f"ofi_pressure_{w}_23a"],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    momentum = pd.to_numeric(
        x[f"momentum_{w}_23a"],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    rv = pd.to_numeric(
        x[f"rv_{w}_23a"],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    valid = (
        np.isfinite(pressure)
        & np.isfinite(momentum)
        & np.isfinite(rv)
    )

    disagree = (
        (pressure * momentum) < 0
    )

    relative_move = (
        momentum.abs()
        / rv.clip(lower=RV_FLOOR)
    ).clip(upper=3.0)

    simple = f"ofi_disagreement_{w}_23b"
    weighted = f"ofi_disagreement_weighted_{w}_23b"
    event = f"ofi_disagreement_event_{w}_23b"

    x[simple] = (
        pressure.where(disagree, 0.0)
        .where(valid)
    )

    x[weighted] = (
        (pressure * relative_move)
        .where(disagree, 0.0)
        .where(valid)
    )

    x[event] = (
        disagree.astype(float)
        .where(valid)
    )

    CANDIDATES.extend([simple, weighted])
    CONTROLS.extend([
        f"ofi_pressure_{w}_23a",
        f"momentum_{w}_23a",
        f"rv_{w}_23a",
        event
    ])

# ------------------------------------------------------------
# 2. Daily signed-return Spearman IC
# ------------------------------------------------------------

def spearman_ic(a, b):

    mask = np.isfinite(a) & np.isfinite(b)

    if mask.sum() < 200:
        return np.nan, int(mask.sum())

    ra = pd.Series(a[mask]).rank().to_numpy()
    rb = pd.Series(b[mask]).rank().to_numpy()

    if ra.std() < 1e-12 or rb.std() < 1e-12:
        return np.nan, int(mask.sum())

    return (
        float(np.corrcoef(ra, rb)[0, 1]),
        int(mask.sum())
    )

records = []

for (symbol, date), day in x.groupby(
    GROUP, sort=False
):

    for h in HORIZONS:

        y = day[
            f"future_return_{h}"
        ].to_numpy(dtype=float)

        for name in CANDIDATES + CONTROLS:

            z = day[name].to_numpy(dtype=float)

            ic, valid_rows = spearman_ic(z, y)

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "feature": name,
                "ic": ic,
                "valid_rows": valid_rows
            })

calendar_exp23b_daily = pd.DataFrame(records)

calendar_exp23b_summary = (
    calendar_exp23b_daily
    .groupby(["horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Conditional forward-return test
#
# Compare the average future signed return
# after opposing OFI/price directions.
#
# These conditional means are descriptive, not
# evidence of tradable profitability.
# ------------------------------------------------------------

conditional_records = []

for (symbol, date), day in x.groupby(
    GROUP, sort=False
):

    for w in WINDOWS:

        p = day[
            f"ofi_pressure_{w}_23a"
        ].to_numpy(dtype=float)

        m = day[
            f"momentum_{w}_23a"
        ].to_numpy(dtype=float)

        for h in HORIZONS:

            y = day[
                f"future_return_{h}"
            ].to_numpy(dtype=float)

            valid = (
                np.isfinite(p)
                & np.isfinite(m)
                & np.isfinite(y)
            )

            states = {
                "positive_ofi_negative_mom": (
                    (p > 0) & (m < 0)
                ),
                "negative_ofi_positive_mom": (
                    (p < 0) & (m > 0)
                ),
                "aligned_positive": (
                    (p > 0) & (m > 0)
                ),
                "aligned_negative": (
                    (p < 0) & (m < 0)
                )
            }

            for label, condition in states.items():

                mask = valid & condition

                conditional_records.append({
                    "symbol": symbol,
                    "date": date,
                    "window": w,
                    "horizon": h,
                    "state": label,
                    "n": int(mask.sum()),
                    "mean_future_bps": (
                        float(y[mask].mean())
                        if mask.sum() >= 100
                        else np.nan
                    )
                })

calendar_exp23b_conditional = pd.DataFrame(
    conditional_records
)

condition_summary = (
    calendar_exp23b_conditional
    .groupby(["window", "horizon", "state"])
    .agg(
        days=("mean_future_bps", "count"),
        median_rows=("n", "median"),
        mean_daily_future_bps=(
            "mean_future_bps", "mean"
        ),
        median_daily_future_bps=(
            "mean_future_bps", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. Redundancy analysis
# ------------------------------------------------------------

redundancy_columns = sorted(
    set(CANDIDATES + CONTROLS)
)

calendar_exp23b_redundancy = (
    x[redundancy_columns]
    .corr(method="spearman")
    .loc[CANDIDATES, CONTROLS]
)

# ------------------------------------------------------------
# 5. Coverage and event frequency diagnostics
# ------------------------------------------------------------

print("\n===== DISAGREEMENT SIGNED-RETURN IC =====")

print(
    calendar_exp23b_summary[
        calendar_exp23b_summary[
            "feature"
        ].isin(CANDIDATES)
    ].round(6).to_string(index=False)
)

print("\n===== DISAGREEMENT CONTROL IC =====")

print(
    calendar_exp23b_summary[
        calendar_exp23b_summary[
            "feature"
        ].isin(CONTROLS)
    ].round(6).to_string(index=False)
)

print("\n===== CONDITIONAL FUTURE RETURNS =====")

print(
    condition_summary.round(6)
    .to_string(index=False)
)

print("\n===== DISAGREEMENT REDUNDANCY =====")

print(
    calendar_exp23b_redundancy
    .round(5)
    .to_string()
)

print("\n===== CANDIDATE COVERAGE =====")

for name in CANDIDATES:
    s = x[name]
    print(
        name,
        "valid_pct=",
        round(s.notna().mean() * 100, 3),
        "zero_pct=",
        round(s.eq(0).mean() * 100, 3)
    )

print("\n===== DISAGREEMENT EVENT FREQUENCY =====")

for w in WINDOWS:

    s = x[f"ofi_disagreement_event_{w}_23b"]

    print(
        f"W={w}",
        "event_rate_among_valid_pct=",
        round(s.mean() * 100, 3),
        "valid_pct=",
        round(s.notna().mean() * 100, 3)
    )

calendar_exp23b_features = x

print("\n===== EXPERIMENT 23B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")



===== DISAGREEMENT SIGNED-RETURN IC =====
 horizon                           feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows
      30          ofi_disagreement_300_23b    16 0.053448   0.060586             15              1            28423.0
      30           ofi_disagreement_60_23b    16 0.086699   0.084029             16              0            28663.0
      30 ofi_disagreement_weighted_300_23b    16 0.054475   0.061593             15              1            28423.0
      30  ofi_disagreement_weighted_60_23b    16 0.087283   0.085351             16              0            28663.0
      60          ofi_disagreement_300_23b    16 0.056703   0.064794             15              1            28393.0
      60           ofi_disagreement_60_23b    16 0.088881   0.081596             16              0            28633.0
      60 ofi_disagreement_weighted_300_23b    16 0.057962   0.065961             15              1            28393.0
      60  ofi

In [58]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 23C — INCREMENTAL OFI DISAGREEMENT
#
# Requires: calendar_exp23b_features
#
# Fixed ridge regression
# Expanding chronological walk-forward
# Forecast signed returns at 30, 60, 300 seconds
# Identical samples across all models
#
# No changes to df_1s.
# No feature promotion.
# ============================================================

if "calendar_exp23b_features" not in globals():
    raise ValueError("Run Experiment 23B first.")

x = calendar_exp23b_features.copy()

HORIZONS = [30, 60, 300]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10
RIDGE_ALPHA = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows.")

# ------------------------------------------------------------
# 1. Clock controls
# ------------------------------------------------------------

x["clock_23c"] = (
    (x["ts_sec"] + 19800) % 86400
) - 32400

x["open_first30_23c"] = (
    x["clock_23c"] < 30
).astype(float)

x["open_1_5m_23c"] = (
    x["clock_23c"].ge(60)
    & x["clock_23c"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 2. Component baseline
#
# Both 60s and 300s components included, so that
# the candidate cannot win merely because the
# baseline lacks longer-term OFI/momentum context.
# ------------------------------------------------------------

BASE = [
    "ofi_pressure_60_23a",
    "ofi_pressure_300_23a",
    "ofi_sum_60_23a",
    "ofi_sum_300_23a",
    "momentum_60_23a",
    "momentum_300_23a",
    "rv_60_23a",
    "rv_300_23a",
    "quote_change_rate_60_23a",
    "quote_change_rate_300_23a",
    "spread_bps",
    "clock_23c",
    "open_first30_23c",
    "open_1_5m_23c"
]

SIMPLE = "ofi_disagreement_60_23b"
WEIGHTED = "ofi_disagreement_weighted_60_23b"

MODELS = {
    "A_components": BASE,
    "B_simple": BASE + [SIMPLE],
    "C_weighted": BASE + [WEIGHTED],
    "D_both": BASE + [SIMPLE, WEIGHTED]
}

TARGETS = [
    f"future_return_{h}" for h in HORIZONS
]

ALL_FEATURES = sorted(set(
    feature
    for features in MODELS.values()
    for feature in features
))

required = ALL_FEATURES + TARGETS

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in required:

    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Common finite-feature mask; every model uses
# exactly the same training and evaluation rows.

x["_common_23c"] = np.isfinite(
    x[ALL_FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 23C SAMPLE =====")
print("Total rows:", len(x))
print("Complete input rows:", int(x["_common_23c"].sum()))
print(
    "Complete input coverage:",
    round(100 * x["_common_23c"].mean(), 3),
    "%"
)

# ------------------------------------------------------------
# 3. Fixed ridge regression
#
# Preprocessing exclusively on training dates.
# Signed-return predictions are NOT clipped.
# ------------------------------------------------------------

def ridge_predict_23c(train, test, features, target):

    A = train[features].to_numpy(
        dtype=np.float64
    )
    B = test[features].to_numpy(
        dtype=np.float64
    )
    y = train[target].to_numpy(
        dtype=np.float64
    )

    if (
        not np.isfinite(A).all()
        or not np.isfinite(B).all()
        or not np.isfinite(y).all()
    ):
        raise ValueError("Nonfinite model inputs.")

    mean = A.mean(axis=0)
    std = A.std(axis=0)

    std = np.where(std > 1e-12, std, 1.0)

    A = (A - mean) / std
    B = (B - mean) / std

    ymean = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A
        + RIDGE_ALPHA * n * np.eye(k),
        A.T @ (y - ymean)
    )

    return ymean + B @ beta

# ------------------------------------------------------------
# 4. Expanding-window evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Not enough eligible sessions.")

records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_23c"]
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test_pool = x[
        (x["date"] == test_date)
        & x["_common_23c"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y = test[target].to_numpy(dtype=float)

        for name, features in MODELS.items():

            prediction = ridge_predict_23c(
                train, test, features, target
            )

            error = y - prediction

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

    print("Completed:", test_date)

calendar_exp23c_daily = pd.DataFrame(records)

if calendar_exp23c_daily.empty:
    raise ValueError("No completed evaluations.")

# ------------------------------------------------------------
# 5. Pooled comparisons with matched observations
# ------------------------------------------------------------

COMPARISONS = {
    "simple_vs_components": (
        "B_simple", "A_components"
    ),
    "weighted_vs_components": (
        "C_weighted", "A_components"
    ),
    "both_vs_components": (
        "D_both", "A_components"
    ),
    "weighted_vs_simple": (
        "C_weighted", "B_simple"
    ),
    "both_vs_simple": (
        "D_both", "B_simple"
    )
}

summary = []
daily_gain_records = []

for h, block in calendar_exp23c_daily.groupby("horizon"):

    totals = block.groupby("model")[["sse", "sae"]].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gain_by_day = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in gain_by_day.items():

            daily_gain_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "daily_mse_gain_pct": gain
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gain_by_day),
            "pooled_mse_gain_pct": 100 * (
                1 - totals.loc[candidate, "sse"]
                / totals.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - totals.loc[candidate, "sae"]
                / totals.loc[baseline, "sae"]
            ),
            "winning_days": int(
                (gain_by_day > 0).sum()
            ),
            "median_daily_gain_pct": gain_by_day.median(),
            "worst_day_pct": gain_by_day.min()
        })

calendar_exp23c_summary = pd.DataFrame(summary)

calendar_exp23c_daily_gains = pd.DataFrame(
    daily_gain_records
)

print("\n===== EXPERIMENT 23C INCREMENTAL RESULTS =====")

print(
    calendar_exp23c_summary.round(6)
    .to_string(index=False)
)

print("\n===== DAILY SIMPLE-DISAGREEMENT GAINS =====")

print(
    calendar_exp23c_daily_gains[
        calendar_exp23c_daily_gains[
            "comparison"
        ].eq("simple_vs_components")
    ].pivot_table(
        index="date",
        columns="horizon",
        values="daily_mse_gain_pct"
    ).round(6).to_string()
)

print("\n===== EXPERIMENT 23C COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")

===== EXPERIMENT 23C SAMPLE =====
Total rows: 459122
Complete input rows: 454322
Complete input coverage: 98.955 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== EXPERIMENT 23C INCREMENTAL RESULTS =====
 horizon             comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30   simple_vs_components    11             0.040955             0.001950             7               0.046228      -0.060191
      30 weighted_vs_components    11             0.055210             0.025586             9               0.042485      -0.031277
      30     both_vs_components    11             0.049703             0.012188             7               0.049141      -0.052920
      30     weighted_vs_simple    11             0.014261         

In [59]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 23D — NONLINEAR OFI DISAGREEMENT ABLATION
#
# Source: calendar_exp23b_features
# Forecast: signed mid-price returns (30, 60, 300s)
#
# Fixed LightGBM parameters
# Chronological expanding-window evaluation
# Matched samples across all models
# No changes to df_1s
# No feature promotion
# ============================================================

if "calendar_exp23b_features" not in globals():
    raise ValueError("Run Experiment 23B first.")

x = calendar_exp23b_features.copy()

HORIZONS = [30, 60, 300]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

x["date"] = pd.to_datetime(x["date"]).dt.date
x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp observations")

# ------------------------------------------------------------
# 1. Clock controls, identical to 23C
# ------------------------------------------------------------

x["clock_23d"] = (
    (x["ts_sec"] + 19800) % 86400
) - 32400

x["open_first30_23d"] = (
    x["clock_23d"] < 30
).astype(float)

x["open_1_5m_23d"] = (
    x["clock_23d"].between(
        60, 300, inclusive="left"
    )
).astype(float)

# ------------------------------------------------------------
# 2. Frozen feature sets
# ------------------------------------------------------------

BASE = [
    "ofi_pressure_60_23a",
    "ofi_pressure_300_23a",
    "ofi_sum_60_23a",
    "ofi_sum_300_23a",
    "momentum_60_23a",
    "momentum_300_23a",
    "rv_60_23a",
    "rv_300_23a",
    "quote_change_rate_60_23a",
    "quote_change_rate_300_23a",
    "spread_bps",
    "clock_23d",
    "open_first30_23d",
    "open_1_5m_23d"
]

SIMPLE = "ofi_disagreement_60_23b"
WEIGHTED = "ofi_disagreement_weighted_60_23b"

MODELS = {
    "A_components": BASE,
    "B_simple": BASE + [SIMPLE],
    "C_weighted": BASE + [WEIGHTED],
    "D_both": BASE + [SIMPLE, WEIGHTED]
}

TARGETS = [
    f"future_return_{h}" for h in HORIZONS
]

ALL_FEATURES = sorted(set(
    f
    for feature_list in MODELS.values()
    for f in feature_list
))

required = ALL_FEATURES + TARGETS + [
    "date", "symbol", "ts_sec"
]

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in ALL_FEATURES + TARGETS:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common_23d"] = np.isfinite(
    x[ALL_FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 23D SAMPLE =====")
print("Total rows:", len(x))
print("Complete rows:", int(x["_common_23d"].sum()))
print(
    "Coverage (%):",
    round(100 * x["_common_23d"].mean(), 3)
)

# ------------------------------------------------------------
# 3. Expanding chronological validation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient sessions")

records = []
importance_records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_23d"]
        & (x["ts_sec"] % TRAIN_STRIDE == 0)
    ]

    test_pool = x[
        (x["date"] == test_date)
        & x["_common_23d"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for model_name, columns in MODELS.items():

            model = lgb.LGBMRegressor(**PARAMS)

            model.fit(
                train[columns],
                y_train
            )

            pred = model.predict(test[columns])
            error = y_test - pred

            records.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

            importance = (
                model.booster_.feature_importance(
                    importance_type="gain"
                )
            )

            for feature, value in zip(columns, importance):
                if feature in [SIMPLE, WEIGHTED]:
                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": model_name,
                        "feature": feature,
                        "gain_importance": float(value)
                    })

    print("Completed:", test_date)

calendar_exp23d_daily = pd.DataFrame(records)
calendar_exp23d_importance = pd.DataFrame(
    importance_records
)

if calendar_exp23d_daily.empty:
    raise ValueError("No model evaluations")

# ------------------------------------------------------------
# 4. Pooled comparisons
# ------------------------------------------------------------

COMPARISONS = {
    "simple_vs_components": (
        "B_simple", "A_components"
    ),
    "weighted_vs_components": (
        "C_weighted", "A_components"
    ),
    "both_vs_components": (
        "D_both", "A_components"
    ),
    "weighted_vs_simple": (
        "C_weighted", "B_simple"
    ),
    "both_vs_weighted": (
        "D_both", "C_weighted"
    )
}

summary = []
daily_rows = []

for h, block in calendar_exp23d_daily.groupby(
    "horizon"
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    daily = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for name, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gains = 100 * (
            1 - daily[candidate] / daily[baseline]
        )

        for date, value in gains.items():
            daily_rows.append({
                "date": date,
                "horizon": h,
                "comparison": name,
                "gain_pct": value
            })

        summary.append({
            "horizon": h,
            "comparison": name,
            "days": len(gains),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gains > 0).sum()),
            "median_daily_gain_pct": gains.median(),
            "worst_day_pct": gains.min()
        })

calendar_exp23d_summary = pd.DataFrame(summary)
calendar_exp23d_daily_gains = pd.DataFrame(
    daily_rows
)

# ------------------------------------------------------------
# 5. Diagnostics
# ------------------------------------------------------------

print("\n===== 23D LIGHTGBM INCREMENTAL RESULTS =====")

print(
    calendar_exp23d_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 23D DAILY WEIGHTED FEATURE GAINS =====")

print(
    calendar_exp23d_daily_gains[
        calendar_exp23d_daily_gains[
            "comparison"
        ].eq("weighted_vs_components")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== 23D ENGINEERED FEATURE IMPORTANCE =====")

if not calendar_exp23d_importance.empty:

    print(
        calendar_exp23d_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 23D COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 23D SAMPLE =====
Total rows: 459122
Complete rows: 454322
Coverage (%): 98.955
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 23D LIGHTGBM INCREMENTAL RESULTS =====
 horizon             comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30   simple_vs_components    11            -0.061168             0.006280             5              -0.035530      -0.339301
      30 weighted_vs_components    11            -0.083310             0.052854             3              -0.015705      -0.409662
      30     both_vs_components    11            -0.052018             0.055131             4              -0.094819      -0.247485
      30     weighted_vs_simple    11            -0.022129             0.046578         

In [60]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 23E — FROZEN FORWARD TEST
#
# Requires refreshed calendar_exp23b_features.
# Train: sessions through 2026-10-19
# Test: complete, previously unseen sessions after cutoff
#
# Target: signed 60-second mid-price return
# No hyperparameter search
# No modification to df_1s
# ============================================================

if "calendar_exp23b_features" not in globals():
    raise ValueError(
        "Refresh daily data and rerun 23A + 23B."
    )

x = calendar_exp23b_features.copy()

CUTOFF = pd.Timestamp("2026-10-19").date()
TARGET = "future_return_60"
TRAIN_STRIDE = 10

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

x["date"] = pd.to_datetime(x["date"]).dt.date

x["clock_23e"] = (
    (pd.to_numeric(x["ts_sec"]) + 19800) % 86400
) - 32400

x["open_first30_23e"] = (
    x["clock_23e"] < 30
).astype(float)

x["open_1_5m_23e"] = (
    x["clock_23e"].ge(60)
    & x["clock_23e"].lt(300)
).astype(float)

BASE = [
    "ofi_pressure_60_23a",
    "ofi_pressure_300_23a",
    "ofi_sum_60_23a",
    "ofi_sum_300_23a",
    "momentum_60_23a",
    "momentum_300_23a",
    "rv_60_23a",
    "rv_300_23a",
    "quote_change_rate_60_23a",
    "quote_change_rate_300_23a",
    "spread_bps",
    "clock_23e",
    "open_first30_23e",
    "open_1_5m_23e"
]

SIMPLE = "ofi_disagreement_60_23b"
WEIGHTED = "ofi_disagreement_weighted_60_23b"

MODELS = {
    "baseline": BASE,
    "simple": BASE + [SIMPLE],
    "weighted": BASE + [WEIGHTED]
}

ALL = sorted(set(
    col for cols in MODELS.values() for col in cols
))

required = (
    ["symbol", "date", "ts_sec", TARGET]
    + ALL
)

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate timestamp records.")

for col in ALL + [TARGET]:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

valid = np.isfinite(
    x[ALL + [TARGET]].to_numpy(dtype=float)
).all(axis=1)

x = x.loc[valid].copy()

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
)

train = x[
    x["date"].le(CUTOFF)
    & x["ts_sec"].mod(TRAIN_STRIDE).eq(0)
]

test = x[x["date"].gt(CUTOFF)]

test_dates = sorted(test["date"].unique())

print("===== FROZEN TEST SAMPLE =====")
print("Historical training rows:", len(train))
print("Post-cutoff test dates:", test_dates)

if len(train) < 1000:
    raise ValueError("Insufficient training data.")

if not test_dates:
    print(
        "No post-cutoff complete sessions available. "
        "Refresh calendar data first."
    )

records = []

# Fit each model once on the frozen historical sample.
fitted = {}

for name, features in MODELS.items():

    model = lgb.LGBMRegressor(**PARAMS)

    model.fit(
        train[features],
        train[TARGET]
    )

    fitted[name] = model

for date in test_dates:

    day = test[test["date"].eq(date)]

    if len(day) < 200:
        continue

    y = day[TARGET].to_numpy(dtype=float)

    for name, features in MODELS.items():

        prediction = fitted[name].predict(
            day[features]
        )

        error = y - prediction

        records.append({
            "date": date,
            "model": name,
            "rows": len(day),
            "sse": float(np.sum(error ** 2)),
            "sae": float(np.sum(np.abs(error))),
            "mse": float(np.mean(error ** 2))
        })

    print("Completed:", date)

calendar_exp23e_daily = pd.DataFrame(records)

if not calendar_exp23e_daily.empty:

    pooled = calendar_exp23e_daily.groupby(
        "model"
    )[["sse", "sae"]].sum()

    daily = calendar_exp23e_daily.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    summary = []

    for candidate in ["simple", "weighted"]:

        gain_by_day = 100 * (
            1 - daily[candidate] / daily["baseline"]
        )

        summary.append({
            "model": candidate,
            "days": len(gain_by_day),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc["baseline", "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc["baseline", "sae"]
            ),
            "winning_days": int(
                (gain_by_day > 0).sum()
            ),
            "worst_day_pct": gain_by_day.min()
        })

    calendar_exp23e_summary = pd.DataFrame(summary)

    print("\n===== 23E FORWARD SUMMARY =====")
    print(
        calendar_exp23e_summary
        .round(6)
        .to_string(index=False)
    )

    print("\n===== 23E DAILY MSE GAINS =====")
    print(
        (100 * (
            1 - daily[["simple", "weighted"]]
            .div(daily["baseline"], axis=0)
        )).round(6).to_string()
    )

print("\n===== EXPERIMENT 23E COMPLETE =====")
print("Historical training cutoff: 2026-10-19")
print("Model configuration frozen.")
print("No feature promoted.")
print("Original df_1s unchanged.")


Task was destroyed but it is pending!
task: <Task pending name='Task-396' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-397' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\projects\hedge_fund\venv\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-397' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>


===== FROZEN TEST SAMPLE =====
Historical training rows: 45343
Post-cutoff test dates: []
No post-cutoff complete sessions available. Refresh calendar data first.

===== EXPERIMENT 23E COMPLETE =====
Historical training cutoff: 2026-10-19
Model configuration frozen.
No feature promoted.
Original df_1s unchanged.


In [61]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 24A — QUOTE RETREAT INTENSITY
#
# Candidate features:
#   quote_retreat_imbalance_30
#   quote_retreat_imbalance_60
#   quote_retreat_imbalance_300
#
# Controls:
#   quote_advance_imbalance
#   total_retreat_rate
#   quote_price_change_rate
#   historical momentum
#   current spread
#
# Target:
#   Future signed returns at 30, 60, 300 seconds
#
# No modifications to df_1s
# No feature promotion
# ============================================================

WINDOWS_24A = [30, 60, 300]
HORIZONS_24A = [30, 60, 300]
MIN_COVERAGE_24A = 0.90
MIN_SESSION_COVERAGE_24A = 0.97

required = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
]

missing = sorted(
    set(required) - set(df_1s.columns)
)

if missing:
    raise ValueError(f"Missing columns: {missing}")

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate dataframe columns")

x = df_1s[required].copy()

for c in ["ts_sec", "bid_p1", "ask_p1"]:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

if x["ts_sec"].isna().any():
    raise ValueError("Invalid timestamp")

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol-timestamp records")

# ------------------------------------------------------------
# 1. IST session classification
# ------------------------------------------------------------

ts_ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ts_ist.dt.date

x["clock_seconds"] = (
    ts_ist.dt.hour * 3600
    + ts_ist.dt.minute * 60
    + ts_ist.dt.second
    - 32400
)

x = x.loc[
    x["clock_seconds"].ge(0)
    & x["clock_seconds"].lt(28800)
].copy()

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage.loc[
    coverage["rows"].ge(
        MIN_SESSION_COVERAGE_24A * 28800
    )
    & coverage["date"].astype(str).ne(
        "2026-09-28"
    ),
    ["symbol", "date"]
]

x = x.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

if x.empty:
    raise ValueError("No eligible sessions")

GROUP = ["symbol", "date"]

print("===== EXPERIMENT 24A DATA =====")
print("Eligible sessions:", len(eligible))
print("Rows:", len(x))

# ------------------------------------------------------------
# 2. Current and previous valid quotes
# ------------------------------------------------------------

bid = x["bid_p1"]
ask = x["ask_p1"]

valid_quote = (
    bid.gt(0)
    & ask.gt(0)
    & ask.ge(bid)
    & np.isfinite(bid)
    & np.isfinite(ask)
)

x["mid"] = ((bid + ask) / 2).where(
    valid_quote
)

x["spread_bps"] = (
    1e4 * (ask - bid) / x["mid"]
).where(valid_quote)

group = x.groupby(GROUP, sort=False)

prev_ts = group["ts_sec"].shift(1)
prev_bid = group["bid_p1"].shift(1)
prev_ask = group["ask_p1"].shift(1)

valid_prev = (
    prev_bid.gt(0)
    & prev_ask.gt(0)
    & prev_ask.ge(prev_bid)
    & np.isfinite(prev_bid)
    & np.isfinite(prev_ask)
)

valid_pair = (
    valid_quote
    & valid_prev
    & (x["ts_sec"] - prev_ts).eq(1)
)

x["valid_transition_24a"] = (
    valid_pair.astype(float)
)

# ------------------------------------------------------------
# 3. Quote retreat and advance indicators
#
# For an invalid pair, indicators remain NaN.
#
# Bid retreat   : bid price decreases
# Ask retreat   : ask price increases
# Bid advance   : bid price increases
# Ask advance   : ask price decreases
# ------------------------------------------------------------

x["bid_retreat_24a"] = (
    bid.lt(prev_bid).astype(float)
).where(valid_pair)

x["ask_retreat_24a"] = (
    ask.gt(prev_ask).astype(float)
).where(valid_pair)

x["bid_advance_24a"] = (
    bid.gt(prev_bid).astype(float)
).where(valid_pair)

x["ask_advance_24a"] = (
    ask.lt(prev_ask).astype(float)
).where(valid_pair)

# Signed directional withdrawal:
# Positive when ask retreats dominate.
x["retreat_signed_24a"] = (
    x["ask_retreat_24a"]
    - x["bid_retreat_24a"]
)

# Signed directional quote improvement:
# Positive when bid advances dominate.
x["advance_signed_24a"] = (
    x["bid_advance_24a"]
    - x["ask_advance_24a"]
)

# Total retreats on either side:
x["retreat_total_24a"] = (
    x["ask_retreat_24a"]
    + x["bid_retreat_24a"]
)

# Price-level quote change activity.
x["quote_price_changed_24a"] = (
    (
        bid.ne(prev_bid)
        | ask.ne(prev_ask)
    ).astype(float)
).where(valid_pair)

# ------------------------------------------------------------
# 4. Rolling features
#
# Divide by number of VALID transitions.
# Minimum 90% coverage within each W rows.
#
# Missing quote pairs cannot be treated as
# observed no-change events.
# ------------------------------------------------------------

CANDIDATES_24A = []
CONTROLS_24A = ["spread_bps"]

group = x.groupby(GROUP, sort=False)

for w in WINDOWS_24A:

    min_valid = int(np.ceil(w * MIN_COVERAGE_24A))

    n_valid = group[
        "valid_transition_24a"
    ].transform(
        lambda s: s.rolling(
            w, min_periods=1
        ).sum()
    )

    sufficient = n_valid >= min_valid

    def roll_sum(col):
        return (
            x.groupby(GROUP, sort=False)[col]
            .transform(
                lambda s: s.fillna(0.0).rolling(
                    w, min_periods=1
                ).sum()
            )
        )

    signed_retreat = roll_sum(
        "retreat_signed_24a"
    )
    signed_advance = roll_sum(
        "advance_signed_24a"
    )
    total_retreat = roll_sum(
        "retreat_total_24a"
    )
    quote_changed = roll_sum(
        "quote_price_changed_24a"
    )

    denom = n_valid.where(sufficient)

    qri = f"quote_retreat_imbalance_{w}_24a"
    qai = f"quote_advance_imbalance_{w}_24a"
    trr = f"total_retreat_rate_{w}_24a"
    qcr = f"quote_price_change_rate_{w}_24a"

    x[qri] = signed_retreat / denom
    x[qai] = signed_advance / denom
    x[trr] = total_retreat / denom
    x[qcr] = quote_changed / denom

    CANDIDATES_24A.append(qri)
    CONTROLS_24A.extend([qai, trr, qcr])

    # Exact W-second historical mid-price return.
    group = x.groupby(GROUP, sort=False)

    prev_mid_w = group["mid"].shift(w)
    prev_ts_w = group["ts_sec"].shift(w)

    mom = f"momentum_{w}_24a"

    x[mom] = (
        1e4 * np.log(
            x["mid"] / prev_mid_w
        )
    ).where(
        (x["ts_sec"] - prev_ts_w).eq(w)
        & prev_mid_w.gt(0)
        & x["mid"].gt(0)
    )

    CONTROLS_24A.append(mom)

# ------------------------------------------------------------
# 5. Exact future signed returns
#
# Merge by symbol, date, and exact future ts.
# No shift over missing seconds.
# ------------------------------------------------------------

for h in HORIZONS_24A:

    future = x[
        ["symbol", "date", "ts_sec", "mid"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    x = x.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    x[f"future_return_{h}"] = (
        1e4 * np.log(
            x[f"future_mid_{h}"] / x["mid"]
        )
    )

# ------------------------------------------------------------
# 6. Optionally inspect OFI controls from earlier
# experiments, without depending on them.
#
# We avoid recreating existing core features in
# the master frame.
# ------------------------------------------------------------

for optional in [
    "ofi_pressure_60_23a",
    "ofi_pressure_300_23a"
]:
    if optional in df_1s.columns:
        print(
            "Optional original OFI field exists:",
            optional,
            "(not automatically joined)"
        )

# ------------------------------------------------------------
# 7. Per-session Spearman IC
# ------------------------------------------------------------

def safe_ic_24a(a, b):

    mask = np.isfinite(a) & np.isfinite(b)

    if mask.sum() < 200:
        return np.nan, int(mask.sum())

    ra = pd.Series(a[mask]).rank().to_numpy()
    rb = pd.Series(b[mask]).rank().to_numpy()

    if ra.std() < 1e-12 or rb.std() < 1e-12:
        return np.nan, int(mask.sum())

    return (
        float(np.corrcoef(ra, rb)[0, 1]),
        int(mask.sum())
    )

records = []

FEATURES_24A = CANDIDATES_24A + CONTROLS_24A

for (symbol, date), day in x.groupby(
    GROUP, sort=False
):

    for h in HORIZONS_24A:

        y = day[
            f"future_return_{h}"
        ].to_numpy(dtype=float)

        for feature in FEATURES_24A:

            z = day[feature].to_numpy(dtype=float)

            ic, n = safe_ic_24a(z, y)

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "feature": feature,
                "ic": ic,
                "valid_rows": n
            })

calendar_exp24a_daily = pd.DataFrame(records)

calendar_exp24a_summary = (
    calendar_exp24a_daily
    .groupby(["horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic",
            lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

calendar_exp24a_summary["sign_consistency"] = (
    calendar_exp24a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp24a_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 8. Redundancy and feature availability
# ------------------------------------------------------------

calendar_exp24a_redundancy = (
    x[FEATURES_24A]
    .corr(method="spearman")
    .loc[CANDIDATES_24A, CONTROLS_24A]
)

# ------------------------------------------------------------
# 9. Diagnostic tables
# ------------------------------------------------------------

print("\n===== QUOTE RETREAT IC =====")

print(
    calendar_exp24a_summary.loc[
        calendar_exp24a_summary[
            "feature"
        ].isin(CANDIDATES_24A)
    ].round(6).to_string(index=False)
)

print("\n===== QUOTE ADVANCE AND MOMENTUM BASELINES =====")

comparison_controls = [
    f"quote_advance_imbalance_{w}_24a"
    for w in WINDOWS_24A
] + [
    f"momentum_{w}_24a"
    for w in WINDOWS_24A
]

print(
    calendar_exp24a_summary.loc[
        calendar_exp24a_summary[
            "feature"
        ].isin(comparison_controls)
    ].round(6).to_string(index=False)
)

print("\n===== QUOTE RETREAT REDUNDANCY =====")

print(
    calendar_exp24a_redundancy
    .round(5)
    .to_string()
)

print("\n===== FEATURE AVAILABILITY =====")

for name in CANDIDATES_24A:

    feature = x[name]

    print(
        name,
        "valid_pct:",
        round(100 * feature.notna().mean(), 4),
        "zero_pct:",
        round(100 * feature.eq(0).mean(), 4),
        "std:",
        round(feature.std(), 7),
        "unique:",
        feature.nunique()
    )

print("\n===== QUOTE EVENT DIAGNOSTICS =====")

for col in [
    "bid_retreat_24a",
    "ask_retreat_24a",
    "bid_advance_24a",
    "ask_advance_24a"
]:
    print(
        col,
        "event_rate_among_valid:",
        round(x[col].mean(), 6)
    )

print(
    "Valid consecutive quote pair pct:",
    round(100 * valid_pair.mean(), 3)
)

calendar_exp24a_features = x

print("\n===== EXPERIMENT 24A COMPLETE =====")
print("Original df_1s unchanged.")
print("No new feature promoted.")


===== EXPERIMENT 24A DATA =====
Eligible sessions: 16
Rows: 459122

===== QUOTE RETREAT IC =====
 horizon                         feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
      30 quote_retreat_imbalance_300_24a    16 -0.041876  -0.050542              2             14            28453.0            0.8750
      30  quote_retreat_imbalance_30_24a    16 -0.092243  -0.086797              0             16            28696.0            1.0000
      30  quote_retreat_imbalance_60_24a    16 -0.083975  -0.070638              0             16            28669.0            1.0000
      60 quote_retreat_imbalance_300_24a    16 -0.043225  -0.045141              4             12            28423.0            0.7500
      60  quote_retreat_imbalance_30_24a    16 -0.092293  -0.085856              0             16            28666.0            1.0000
      60  quote_retreat_imbalance_60_24a    16 -0.086051  -0.078133              1           

In [62]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 24B — INCREMENTAL QUOTE RETREAT TEST
#
# Input: calendar_exp24a_features
# Models: components, +retreat30, +retreat60, +both
# Targets: future signed returns at 30, 60, 300s
#
# Expanding chronological ridge regression
# Fixed parameters; no tuning
# Same train/test rows for every model
# Original df_1s unchanged
# ============================================================

if "calendar_exp24a_features" not in globals():
    raise ValueError("Run Experiment 24A first.")

x = calendar_exp24a_features.copy()

HORIZONS_24B = [30, 60, 300]
MIN_TRAIN_DAYS_24B = 5
TRAIN_STRIDE_24B = 10
RIDGE_ALPHA_24B = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date
x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp records.")

# ------------------------------------------------------------
# 1. Clock controls
# ------------------------------------------------------------

x["open_first30_24b"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_24b"] = (
    (x["clock_seconds"] >= 60)
    & (x["clock_seconds"] < 300)
).astype(float)

# ------------------------------------------------------------
# 2. Full component baseline
#
# Include matched-window momentum, quote advances,
# retreat frequency and quote-change activity.
# ------------------------------------------------------------

BASE_24B = [
    "spread_bps",
    "clock_seconds",
    "open_first30_24b",
    "open_1_5m_24b",

    "momentum_30_24a",
    "momentum_60_24a",
    "momentum_300_24a",

    "quote_advance_imbalance_30_24a",
    "quote_advance_imbalance_60_24a",
    "quote_advance_imbalance_300_24a",

    "total_retreat_rate_30_24a",
    "total_retreat_rate_60_24a",
    "total_retreat_rate_300_24a",

    "quote_price_change_rate_30_24a",
    "quote_price_change_rate_60_24a",
    "quote_price_change_rate_300_24a"
]

RETREAT_30 = "quote_retreat_imbalance_30_24a"
RETREAT_60 = "quote_retreat_imbalance_60_24a"

MODELS_24B = {
    "A_components": BASE_24B,
    "B_retreat30": BASE_24B + [RETREAT_30],
    "C_retreat60": BASE_24B + [RETREAT_60],
    "D_both": BASE_24B + [
        RETREAT_30, RETREAT_60
    ]
}

TARGETS_24B = [
    f"future_return_{h}"
    for h in HORIZONS_24B
]

ALL_FEATURES_24B = sorted(set(
    f
    for features in MODELS_24B.values()
    for f in features
))

required_24b = (
    ["symbol", "date", "ts_sec"]
    + ALL_FEATURES_24B
    + TARGETS_24B
)

missing = sorted(
    set(required_24b) - set(x.columns)
)

if missing:
    raise ValueError(f"Missing fields: {missing}")

for c in ALL_FEATURES_24B + TARGETS_24B:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Every model uses exactly the same sample.
x["_common_24b"] = np.isfinite(
    x[ALL_FEATURES_24B].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 24B SAMPLE =====")
print("Total rows:", len(x))
print("Complete rows:", int(x["_common_24b"].sum()))
print(
    "Complete fraction:",
    round(x["_common_24b"].mean(), 5)
)

# ------------------------------------------------------------
# 3. Ridge regression
#
# Training-only standardization.
# Signed predictions are not clipped.
# ------------------------------------------------------------

def ridge_predict_24b(train, test, features, target):

    A = train[features].to_numpy(dtype=float)
    B = test[features].to_numpy(dtype=float)
    y = train[target].to_numpy(dtype=float)

    if not (
        np.isfinite(A).all()
        and np.isfinite(B).all()
        and np.isfinite(y).all()
    ):
        raise ValueError("Nonfinite model inputs.")

    mu = A.mean(axis=0)
    sd = A.std(axis=0)

    sd = np.where(sd > 1e-12, sd, 1.0)

    A = (A - mu) / sd
    B = (B - mu) / sd

    ym = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A
        + RIDGE_ALPHA_24B * n * np.eye(k),
        A.T @ (y - ym)
    )

    return ym + B @ beta

# ------------------------------------------------------------
# 4. Expanding walk-forward validation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS_24B:
    raise ValueError("Not enough sessions.")

records = []

for i in range(
    MIN_TRAIN_DAYS_24B,
    len(dates)
):
    date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_24b"]
        & (x["ts_sec"] % TRAIN_STRIDE_24B == 0)
    ]

    test_pool = x[
        (x["date"] == date)
        & x["_common_24b"]
    ]

    for h in HORIZONS_24B:

        target = f"future_return_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        actual = test[target].to_numpy(dtype=float)

        for name, features in MODELS_24B.items():

            pred = ridge_predict_24b(
                train, test, features, target
            )

            error = actual - pred

            records.append({
                "date": date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

    print("Completed:", date)

calendar_exp24b_daily = pd.DataFrame(records)

if calendar_exp24b_daily.empty:
    raise ValueError("No completed model evaluations.")

# ------------------------------------------------------------
# 5. Pooled summary and daily gains
# ------------------------------------------------------------

COMPARISONS_24B = {
    "retreat30_vs_components": (
        "B_retreat30", "A_components"
    ),
    "retreat60_vs_components": (
        "C_retreat60", "A_components"
    ),
    "both_vs_components": (
        "D_both", "A_components"
    ),
    "retreat30_vs_retreat60": (
        "B_retreat30", "C_retreat60"
    ),
    "both_vs_retreat30": (
        "D_both", "B_retreat30"
    )
}

summary = []
daily_records = []

for h, block in calendar_exp24b_daily.groupby(
    "horizon"
):

    totals = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS_24B.items()
    ):

        daily_gains = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in daily_gains.items():
            daily_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(gain)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(daily_gains),
            "pooled_mse_gain_pct": 100 * (
                1 - totals.loc[candidate, "sse"]
                / totals.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - totals.loc[candidate, "sae"]
                / totals.loc[baseline, "sae"]
            ),
            "winning_days": int(
                (daily_gains > 0).sum()
            ),
            "median_daily_gain_pct":
                daily_gains.median(),
            "worst_day_pct":
                daily_gains.min()
        })

calendar_exp24b_summary = pd.DataFrame(summary)
calendar_exp24b_daily_gains = pd.DataFrame(
    daily_records
)

print("\n===== 24B INCREMENTAL RESULTS =====")

print(
    calendar_exp24b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== DAILY RETREAT 30S GAINS =====")

print(
    calendar_exp24b_daily_gains[
        calendar_exp24b_daily_gains[
            "comparison"
        ].eq("retreat30_vs_components")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== EXPERIMENT 24B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 24B SAMPLE =====
Total rows: 459122
Complete rows: 454322
Complete fraction: 0.98955
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 24B INCREMENTAL RESULTS =====
 horizon              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30 retreat30_vs_components    11             0.024399             0.038950             8               0.279513      -1.032923
      30 retreat60_vs_components    11            -0.007135            -0.092279             7               0.308693      -1.001176
      30      both_vs_components    11            -0.012912            -0.042139             8               0.233506      -1.234825
      30  retreat30_vs_retreat60    11             0.031531             0.131108       

In [63]:

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except ImportError:
    raise ImportError(
        "Install LightGBM using %pip install lightgbm"
    )

# ============================================================
# EXPERIMENT 24C — NONLINEAR QUOTE RETREAT ABLATION
#
# Source: calendar_exp24a_features
#
# Horizons: 30s, 60s
# Target: signed future mid-price return (bps)
#
# Fixed LightGBM configuration
# Expanding chronological evaluation
# Identical rows across model configurations
#
# Does not modify df_1s.
# ============================================================

if "calendar_exp24a_features" not in globals():
    raise ValueError("Run Experiment 24A first.")

x = calendar_exp24a_features.copy()

HORIZONS = [30, 60]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

# ------------------------------------------------------------
# 1. Sort chronologically and construct clock controls
# ------------------------------------------------------------

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows")

x["open_first30_24c"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_24c"] = (
    x["clock_seconds"].ge(60)
    & x["clock_seconds"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 2. Same component baseline as 24B
# ------------------------------------------------------------

BASE = [
    "spread_bps",
    "clock_seconds",
    "open_first30_24c",
    "open_1_5m_24c",

    "momentum_30_24a",
    "momentum_60_24a",
    "momentum_300_24a",

    "quote_advance_imbalance_30_24a",
    "quote_advance_imbalance_60_24a",
    "quote_advance_imbalance_300_24a",

    "total_retreat_rate_30_24a",
    "total_retreat_rate_60_24a",
    "total_retreat_rate_300_24a",

    "quote_price_change_rate_30_24a",
    "quote_price_change_rate_60_24a",
    "quote_price_change_rate_300_24a"
]

RETREAT_30 = "quote_retreat_imbalance_30_24a"
RETREAT_60 = "quote_retreat_imbalance_60_24a"

MODELS = {
    "A_components": BASE,
    "B_retreat30": BASE + [RETREAT_30],
    "C_retreat60": BASE + [RETREAT_60],
    "D_both": BASE + [RETREAT_30, RETREAT_60]
}

TARGETS = [
    f"future_return_{h}"
    for h in HORIZONS
]

ALL_FEATURES = sorted(set(
    f
    for columns in MODELS.values()
    for f in columns
))

required = (
    ["symbol", "date", "ts_sec"]
    + ALL_FEATURES
    + TARGETS
)

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

for c in ALL_FEATURES + TARGETS:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common_24c"] = np.isfinite(
    x[ALL_FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 24C SAMPLE =====")
print("Total rows:", len(x))
print("Complete rows:", int(x["_common_24c"].sum()))
print(
    "Complete percentage:",
    round(x["_common_24c"].mean() * 100, 4)
)

# ------------------------------------------------------------
# 3. Chronological expanding-window evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient sessions")

records = []
importance_records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_24c"]
        & x["ts_sec"].mod(TRAIN_STRIDE).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_24c"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for name, columns in MODELS.items():

            model = lgb.LGBMRegressor(**PARAMS)

            model.fit(
                train[columns],
                y_train
            )

            predictions = model.predict(
                test[columns]
            )

            # Signed returns must not be clipped.
            errors = y_test - predictions

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(errors ** 2)),
                "sae": float(np.sum(np.abs(errors))),
                "mse": float(np.mean(errors ** 2)),
                "mae": float(np.mean(np.abs(errors)))
            })

            importance = (
                model.booster_.feature_importance(
                    importance_type="gain"
                )
            )

            for feature, value in zip(
                columns, importance
            ):

                if feature in [RETREAT_30, RETREAT_60]:

                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": name,
                        "feature": feature,
                        "gain_importance": float(value)
                    })

    print("Completed:", test_date)

calendar_exp24c_daily = pd.DataFrame(records)
calendar_exp24c_importance = pd.DataFrame(
    importance_records
)

if calendar_exp24c_daily.empty:
    raise ValueError("No completed evaluations")

# ------------------------------------------------------------
# 4. Pooled and day-level comparisons
# ------------------------------------------------------------

COMPARISONS = {
    "retreat30_vs_components": (
        "B_retreat30", "A_components"
    ),
    "retreat60_vs_components": (
        "C_retreat60", "A_components"
    ),
    "both_vs_components": (
        "D_both", "A_components"
    ),
    "retreat30_vs_retreat60": (
        "B_retreat30", "C_retreat60"
    ),
    "both_vs_retreat30": (
        "D_both", "B_retreat30"
    )
}

summary = []
daily_records = []

for h, block in calendar_exp24c_daily.groupby(
    "horizon"
):

    totals = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    daily = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gains = 100 * (
            1 - daily[candidate] / daily[baseline]
        )

        for date, gain in gains.items():

            daily_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(gain)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gains),
            "pooled_mse_gain_pct": 100 * (
                1 - totals.loc[candidate, "sse"]
                / totals.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - totals.loc[candidate, "sae"]
                / totals.loc[baseline, "sae"]
            ),
            "winning_days": int(
                (gains > 0).sum()
            ),
            "median_daily_gain_pct": gains.median(),
            "worst_day_pct": gains.min()
        })

calendar_exp24c_summary = pd.DataFrame(summary)

calendar_exp24c_daily_gains = pd.DataFrame(
    daily_records
)

# ------------------------------------------------------------
# 5. Print results
# ------------------------------------------------------------

print("\n===== 24C LIGHTGBM INCREMENTAL RESULTS =====")

print(
    calendar_exp24c_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 24C DAILY RETREAT 30S GAINS =====")

print(
    calendar_exp24c_daily_gains[
        calendar_exp24c_daily_gains[
            "comparison"
        ].eq("retreat30_vs_components")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== 24C RETREAT FEATURE IMPORTANCE =====")

if not calendar_exp24c_importance.empty:

    print(
        calendar_exp24c_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 24C COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 24C SAMPLE =====
Total rows: 459122
Complete rows: 454322
Complete percentage: 98.9545
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 24C LIGHTGBM INCREMENTAL RESULTS =====
 horizon              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30 retreat30_vs_components    11             0.042055             0.124896             7               0.006464      -0.249500
      30 retreat60_vs_components    11             0.083096             0.179739             8               0.072540      -0.447630
      30      both_vs_components    11             0.111557             0.242899             8               0.186320      -0.349820
      30  retreat30_vs_retreat60    11            -0.041075            -0.05

In [67]:

import pandas as pd

core_31 = pd.read_parquet("usdinr_31_features.parquet")

CORE = [
    c for c in core_31.columns
    if c not in ["symbol", "ts_sec"]
]

assert len(CORE) == 31
assert core_31[["symbol", "ts_sec"]].notna().all().all()
assert not core_31.duplicated(["symbol", "ts_sec"]).any()

print("Loaded", len(CORE), "features")
print("Rows:", len(core_31))


Loaded 31 features
Rows: 524325


In [68]:
left = core_31[["symbol", "ts_sec"] + CORE].copy()

In [70]:
print("Core features:", len(CORE))
print("Feature data shape:", core_31.shape)
print("Research data shape:", calendar_exp24a_features.shape)

Core features: 31
Feature data shape: (524325, 33)
Research data shape: (459122, 38)


In [72]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 24D — FULL 31-FEATURE ABLATION
#
# Run in: CALENDAR RESEARCH NOTEBOOK
#
# Required:
#   1. core_31
#      Loaded from usdinr_31_features.parquet
#
#   2. calendar_exp24a_features
#      Created in Experiment 24A
#
# Models:
#   A: 31 existing features + common quote controls
#   B: Model A + retreat imbalance 30s
#   C: Model A + retreat imbalance 60s
#   D: Model A + both retreat features
#
# Targets:
#   Signed 30-second future mid-price return
#   Signed 60-second future mid-price return
#
# Validation:
#   Expanding chronological walk-forward
#   First 5 dates used for initial training
#   Train on every 10th second
#   Test on complete eligible future sessions
#
# No changes to df_1s
# No feature promotion
# ============================================================

HORIZONS_24D = [30, 60]
MIN_TRAIN_DAYS_24D = 5
TRAIN_STRIDE_24D = 10

PARAMS_24D = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

# ------------------------------------------------------------
# 1. Verify the two research DataFrames exist
# ------------------------------------------------------------

if "core_31" not in globals():
    raise ValueError(
        "core_31 not found. Load the exported "
        "usdinr_31_features.parquet file first."
    )

if "calendar_exp24a_features" not in globals():
    raise ValueError(
        "calendar_exp24a_features not found. "
        "Run Experiment 24A first."
    )

# ------------------------------------------------------------
# 2. Identify the 31 features directly from Parquet
#    No registry required.
# ------------------------------------------------------------

KEYS_24D = ["symbol", "ts_sec"]

CORE_24D = [
    c for c in core_31.columns
    if c not in KEYS_24D
]

if len(CORE_24D) != 31:
    raise ValueError(
        f"Expected 31 features, found {len(CORE_24D)}"
    )

if len(set(CORE_24D)) != 31:
    raise ValueError("Duplicate core feature names.")

RETREAT_30_24D = "quote_retreat_imbalance_30_24a"
RETREAT_60_24D = "quote_retreat_imbalance_60_24a"

COMMON_CONTROLS_24D = [
    "spread_bps",
    "clock_seconds",

    "momentum_30_24a",
    "momentum_60_24a",
    "momentum_300_24a",

    "quote_advance_imbalance_30_24a",
    "quote_advance_imbalance_60_24a",
    "quote_advance_imbalance_300_24a",

    "total_retreat_rate_30_24a",
    "total_retreat_rate_60_24a",
    "total_retreat_rate_300_24a",

    "quote_price_change_rate_30_24a",
    "quote_price_change_rate_60_24a",
    "quote_price_change_rate_300_24a"
]

TARGETS_24D = [
    f"future_return_{h}"
    for h in HORIZONS_24D
]

# ------------------------------------------------------------
# 3. Validate raw join keys
# ------------------------------------------------------------

left_24d = core_31[
    KEYS_24D + CORE_24D
].copy()

required_research_24d = list(dict.fromkeys(
    KEYS_24D
    + ["date"]
    + COMMON_CONTROLS_24D
    + [RETREAT_30_24D, RETREAT_60_24D]
    + TARGETS_24D
))

missing_research_24d = [
    c for c in required_research_24d
    if c not in calendar_exp24a_features.columns
]

if missing_research_24d:
    raise ValueError(
        "Missing research columns: "
        f"{missing_research_24d}"
    )

right_24d = calendar_exp24a_features[
    required_research_24d
].copy()

# Ensure join keys use compatible data types.
for frame in [left_24d, right_24d]:
    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"], errors="raise"
    ).astype("int64")

    frame["symbol"] = frame["symbol"].astype(str)

if left_24d[KEYS_24D].isna().any().any():
    raise ValueError("Missing keys in core_31.")

if right_24d[KEYS_24D].isna().any().any():
    raise ValueError("Missing keys in research frame.")

if left_24d.duplicated(KEYS_24D).any():
    raise ValueError(
        "Duplicate symbol/timestamp keys in core_31."
    )

if right_24d.duplicated(KEYS_24D).any():
    raise ValueError(
        "Duplicate symbol/timestamp keys in 24A data."
    )

# ------------------------------------------------------------
# 4. Merge on exact symbol + timestamp
#
# Left = eligible experiment observations
# Right = all 31 engineered features
# ------------------------------------------------------------

x_24d = right_24d.merge(
    left_24d,
    on=KEYS_24D,
    how="left",
    validate="one_to_one",
    indicator=True
)

match_rate_24d = (
    x_24d["_merge"].eq("both").mean()
)

print("===== 24D DATA JOIN =====")
print("Research rows:", len(right_24d))
print("Core rows:", len(left_24d))
print(
    "Matched percentage:",
    round(100 * match_rate_24d, 4)
)

if match_rate_24d < 0.99:
    raise ValueError(
        "Less than 99% of research rows matched "
        "the engineered feature data. "
        "Check data versions and timestamps."
    )

x_24d.drop(columns=["_merge"], inplace=True)

x_24d["date"] = pd.to_datetime(
    x_24d["date"]
).dt.date

x_24d = x_24d.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Build model feature sets
#
# All models share the same 31 core features
# and quote component controls.
# ------------------------------------------------------------

BASE_24D = list(dict.fromkeys(
    CORE_24D + COMMON_CONTROLS_24D
))

MODELS_24D = {
    "A_full_baseline": BASE_24D,

    "B_retreat30": (
        BASE_24D + [RETREAT_30_24D]
    ),

    "C_retreat60": (
        BASE_24D + [RETREAT_60_24D]
    ),

    "D_both": (
        BASE_24D
        + [RETREAT_30_24D, RETREAT_60_24D]
    )
}

ALL_FEATURES_24D = list(dict.fromkeys(
    feature
    for columns in MODELS_24D.values()
    for feature in columns
))

# ------------------------------------------------------------
# 6. Convert numerical features
# ------------------------------------------------------------

for c in ALL_FEATURES_24D + TARGETS_24D:

    x_24d[c] = pd.to_numeric(
        x_24d[c],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

# Enforce identical sample eligibility across
# all four models, avoiding sample-selection bias.

common_mask_24d = np.isfinite(
    x_24d[ALL_FEATURES_24D].to_numpy(
        dtype=np.float64
    )
).all(axis=1)

x_24d["_common_24d"] = common_mask_24d

print("\n===== EXPERIMENT 24D SAMPLE =====")
print("Total rows:", len(x_24d))
print(
    "Complete feature rows:",
    int(common_mask_24d.sum())
)
print(
    "Complete feature coverage (%):",
    round(100 * common_mask_24d.mean(), 4)
)
print("Core feature count:", len(CORE_24D))
print("Baseline feature count:", len(BASE_24D))
print(
    "Candidate-inclusive feature count:",
    len(ALL_FEATURES_24D)
)

print(
    "\nEligible dates:",
    sorted(x_24d["date"].unique())
)

# ------------------------------------------------------------
# 7. Fixed LightGBM model
#    Expanding chronological evaluation
# ------------------------------------------------------------

dates_24d = sorted(
    x_24d["date"].unique()
)

if len(dates_24d) <= MIN_TRAIN_DAYS_24D:
    raise ValueError(
        "Not enough sessions for expanding "
        "walk-forward evaluation."
    )

records_24d = []
importance_records_24d = []

for i in range(
    MIN_TRAIN_DAYS_24D,
    len(dates_24d)
):

    test_date_24d = dates_24d[i]

    train_dates_24d = dates_24d[:i]

    train_pool_24d = x_24d[
        x_24d["date"].isin(train_dates_24d)
        & x_24d["_common_24d"]
        & (
            x_24d["ts_sec"] % TRAIN_STRIDE_24D
            == 0
        )
    ]

    test_pool_24d = x_24d[
        x_24d["date"].eq(test_date_24d)
        & x_24d["_common_24d"]
    ]

    for horizon_24d in HORIZONS_24D:

        target_24d = (
            f"future_return_{horizon_24d}"
        )

        train_24d = train_pool_24d[
            np.isfinite(
                train_pool_24d[target_24d]
            )
        ]

        test_24d = test_pool_24d[
            np.isfinite(
                test_pool_24d[target_24d]
            )
        ]

        if (
            len(train_24d) < 1000
            or len(test_24d) < 200
        ):
            continue

        y_train_24d = train_24d[
            target_24d
        ].to_numpy(dtype=float)

        y_test_24d = test_24d[
            target_24d
        ].to_numpy(dtype=float)

        for model_name_24d, cols_24d in (
            MODELS_24D.items()
        ):

            model_24d = lgb.LGBMRegressor(
                **PARAMS_24D
            )

            model_24d.fit(
                train_24d[cols_24d],
                y_train_24d
            )

            pred_24d = model_24d.predict(
                test_24d[cols_24d]
            )

            error_24d = (
                y_test_24d - pred_24d
            )

            records_24d.append({
                "date": test_date_24d,
                "horizon": horizon_24d,
                "model": model_name_24d,
                "rows": len(test_24d),
                "sse": float(
                    np.sum(error_24d ** 2)
                ),
                "sae": float(
                    np.sum(np.abs(error_24d))
                ),
                "mse": float(
                    np.mean(error_24d ** 2)
                ),
                "mae": float(
                    np.mean(np.abs(error_24d))
                )
            })

            # Track candidate split gain importance.
            importances_24d = (
                model_24d.booster_.feature_importance(
                    importance_type="gain"
                )
            )

            for feature_24d, importance_24d in zip(
                cols_24d,
                importances_24d
            ):

                if feature_24d in [
                    RETREAT_30_24D,
                    RETREAT_60_24D
                ]:

                    importance_records_24d.append({
                        "date": test_date_24d,
                        "horizon": horizon_24d,
                        "model": model_name_24d,
                        "feature": feature_24d,
                        "gain_importance": float(
                            importance_24d
                        )
                    })

    print("Completed:", test_date_24d)

calendar_exp24d_daily = pd.DataFrame(
    records_24d
)

calendar_exp24d_importance = pd.DataFrame(
    importance_records_24d
)

if calendar_exp24d_daily.empty:
    raise ValueError(
        "No completed model evaluations. "
        "Check feature availability and dates."
    )

# ------------------------------------------------------------
# 8. Pooled metrics and daily comparisons
# ------------------------------------------------------------

COMPARISONS_24D = {
    "retreat30_vs_full": (
        "B_retreat30",
        "A_full_baseline"
    ),
    "retreat60_vs_full": (
        "C_retreat60",
        "A_full_baseline"
    ),
    "both_vs_full": (
        "D_both",
        "A_full_baseline"
    ),
    "retreat60_vs_retreat30": (
        "C_retreat60",
        "B_retreat30"
    ),
    "both_vs_retreat30": (
        "D_both",
        "B_retreat30"
    ),
    "both_vs_retreat60": (
        "D_both",
        "C_retreat60"
    )
}

summary_rows_24d = []
daily_gain_rows_24d = []

for horizon_24d, block_24d in (
    calendar_exp24d_daily.groupby("horizon")
):

    pooled_24d = block_24d.groupby(
        "model"
    )[["sse", "sae"]].sum()

    wide_24d = block_24d.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison_24d, (
        candidate_24d,
        baseline_24d
    ) in COMPARISONS_24D.items():

        daily_gain_24d = 100 * (
            1
            - (
                wide_24d[candidate_24d]
                / wide_24d[baseline_24d]
            )
        )

        for date_24d, gain_24d in (
            daily_gain_24d.items()
        ):

            daily_gain_rows_24d.append({
                "date": date_24d,
                "horizon": horizon_24d,
                "comparison": comparison_24d,
                "gain_pct": float(gain_24d)
            })

        summary_rows_24d.append({
            "horizon": horizon_24d,
            "comparison": comparison_24d,
            "days": len(daily_gain_24d),

            "pooled_mse_gain_pct": 100 * (
                1
                - (
                    pooled_24d.loc[
                        candidate_24d, "sse"
                    ]
                    / pooled_24d.loc[
                        baseline_24d, "sse"
                    ]
                )
            ),

            "pooled_mae_gain_pct": 100 * (
                1
                - (
                    pooled_24d.loc[
                        candidate_24d, "sae"
                    ]
                    / pooled_24d.loc[
                        baseline_24d, "sae"
                    ]
                )
            ),

            "winning_days": int(
                (daily_gain_24d > 0).sum()
            ),

            "median_daily_gain_pct": float(
                daily_gain_24d.median()
            ),

            "worst_day_pct": float(
                daily_gain_24d.min()
            )
        })

calendar_exp24d_summary = pd.DataFrame(
    summary_rows_24d
)

calendar_exp24d_daily_gains = pd.DataFrame(
    daily_gain_rows_24d
)

# ------------------------------------------------------------
# 9. Print incremental results
# ------------------------------------------------------------

print("\n===== 24D FULL BASELINE RESULTS =====")

print(
    calendar_exp24d_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 24D DAILY RETREAT GAINS =====")

selected_24d = calendar_exp24d_daily_gains[
    calendar_exp24d_daily_gains[
        "comparison"
    ].isin([
        "retreat30_vs_full",
        "retreat60_vs_full"
    ])
]

print(
    selected_24d.pivot_table(
        index="date",
        columns=[
            "horizon",
            "comparison"
        ],
        values="gain_pct"
    )
    .round(6)
    .to_string()
)

print("\n===== 24D FEATURE IMPORTANCE =====")

if not calendar_exp24d_importance.empty:

    importance_summary_24d = (
        calendar_exp24d_importance
        .groupby([
            "horizon",
            "model",
            "feature"
        ])["gain_importance"]
        .median()
        .reset_index()
    )

    print(
        importance_summary_24d
        .round(4)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 10. Completion
# ------------------------------------------------------------

print("\n===== EXPERIMENT 24D COMPLETE =====")
print("Original df_1s unchanged.")
print("31-feature baseline preserved.")
print("No feature promoted.")


===== 24D DATA JOIN =====
Research rows: 459122
Core rows: 524325
Matched percentage: 100.0

===== EXPERIMENT 24D SAMPLE =====
Total rows: 459122
Complete feature rows: 402979
Complete feature coverage (%): 87.7717
Core feature count: 31
Baseline feature count: 45
Candidate-inclusive feature count: 47

Eligible dates: [datetime.date(2026, 9, 23), datetime.date(2026, 9, 24), datetime.date(2026, 9, 25), datetime.date(2026, 9, 30), datetime.date(2026, 10, 1), datetime.date(2026, 10, 5), datetime.date(2026, 10, 6), datetime.date(2026, 10, 7), datetime.date(2026, 10, 8), datetime.date(2026, 10, 9), datetime.date(2026, 10, 12), datetime.date(2026, 10, 13), datetime.date(2026, 10, 14), datetime.date(2026, 10, 15), datetime.date(2026, 10, 16), datetime.date(2026, 10, 19)]
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Compl

In [73]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 25A — QUOTE PRICE AGE ASYMMETRY
#
# Source: raw df_1s in CALENDAR notebook
#
# Candidates:
#   quote_age_asymmetry_25a
#   joint_quote_age_25a
#
# Controls:
#   bid and ask individual quote ages
#   momentum 30s/60s
#   spread
#   quote-price-change rate 60s/300s
#   update rate 300s, if available
#
# Targets:
#   Future signed mid returns: 30/60/300s
#   Future absolute mid returns: 30/60/300s
#
# Exact timestamp targets
# No master dataframe mutation
# No feature promotion
# ============================================================

HORIZONS_25A = [30, 60, 300]
WINDOWS_25A = [30, 60, 300]
MIN_SESSION_COVERAGE_25A = 0.97

required_25a = [
    "symbol", "ts_sec",
    "bid_p1", "ask_p1"
]

missing_25a = [
    c for c in required_25a
    if c not in df_1s.columns
]

if missing_25a:
    raise ValueError(
        f"Missing required columns: {missing_25a}"
    )

# ------------------------------------------------------------
# 1. Independent research frame
# ------------------------------------------------------------

x25 = df_1s[required_25a].copy()

for c in ["ts_sec", "bid_p1", "ask_p1"]:
    x25[c] = pd.to_numeric(
        x25[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

if x25["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps.")

if x25.duplicated(
    ["symbol", "ts_sec"]
).any():
    raise ValueError(
        "Duplicate symbol/timestamp observations."
    )

ts_ist = pd.to_datetime(
    x25["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x25["date"] = ts_ist.dt.date

x25["clock_seconds"] = (
    ts_ist.dt.hour * 3600
    + ts_ist.dt.minute * 60
    + ts_ist.dt.second
    - 32400
)

# Market window 09:00–17:00 IST
x25 = x25.loc[
    x25["clock_seconds"].ge(0)
    & x25["clock_seconds"].lt(28800)
].copy()

x25 = x25.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x25.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage.loc[
    coverage["rows"].ge(
        28800 * MIN_SESSION_COVERAGE_25A
    )
    & coverage["date"].astype(str).ne(
        "2026-09-28"
    ),
    ["symbol", "date"]
]

x25 = x25.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

if x25.empty:
    raise ValueError("No eligible sessions.")

GROUP25 = ["symbol", "date"]

print("===== EXPERIMENT 25A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x25))

# ------------------------------------------------------------
# 2. Valid quote pairs
# ------------------------------------------------------------

bid = x25["bid_p1"]
ask = x25["ask_p1"]

valid_quote = (
    bid.gt(0)
    & ask.gt(0)
    & ask.ge(bid)
    & np.isfinite(bid)
    & np.isfinite(ask)
)

x25["mid"] = (
    (bid + ask) / 2
).where(valid_quote)

x25["spread_bps_25a"] = (
    1e4 * (ask - bid) / x25["mid"]
).where(valid_quote)

g25 = x25.groupby(GROUP25, sort=False)

prev_ts = g25["ts_sec"].shift(1)
prev_bid = g25["bid_p1"].shift(1)
prev_ask = g25["ask_p1"].shift(1)

valid_previous = (
    prev_bid.gt(0)
    & prev_ask.gt(0)
    & prev_ask.ge(prev_bid)
    & np.isfinite(prev_bid)
    & np.isfinite(prev_ask)
)

consecutive = (
    (x25["ts_sec"] - prev_ts).eq(1)
)

valid_pair = (
    valid_quote
    & valid_previous
    & consecutive
)

# ------------------------------------------------------------
# 3. Price-change events and quote age anchors
#
# A valid first observation, or first valid quote
# after a gap, starts a new observed age segment.
#
# Ages also reset when corresponding best quote
# PRICE changes. Quantity-only updates do NOT
# reset quote-price age.
# ------------------------------------------------------------

bid_changed = (
    bid.ne(prev_bid) & valid_pair
)

ask_changed = (
    ask.ne(prev_ask) & valid_pair
)

segment_start = (
    valid_quote & ~valid_pair
)

bid_anchor = (
    valid_quote
    & (segment_start | bid_changed)
)

ask_anchor = (
    valid_quote
    & (segment_start | ask_changed)
)

# Last anchor timestamp is forward-filled ONLY
# within the same symbol/date.
x25["_bid_anchor_ts"] = (
    x25["ts_sec"].where(bid_anchor)
)

x25["_ask_anchor_ts"] = (
    x25["ts_sec"].where(ask_anchor)
)

x25["_bid_anchor_ts"] = (
    x25.groupby(GROUP25, sort=False)[
        "_bid_anchor_ts"
    ].ffill()
)

x25["_ask_anchor_ts"] = (
    x25.groupby(GROUP25, sort=False)[
        "_ask_anchor_ts"
    ].ffill()
)

x25["bid_quote_age_25a"] = (
    x25["ts_sec"] - x25["_bid_anchor_ts"]
).where(valid_quote)

x25["ask_quote_age_25a"] = (
    x25["ts_sec"] - x25["_ask_anchor_ts"]
).where(valid_quote)

# ------------------------------------------------------------
# 4. Candidate quote-age features
# ------------------------------------------------------------

bid_age = x25["bid_quote_age_25a"]
ask_age = x25["ask_quote_age_25a"]

x25["quote_age_asymmetry_25a"] = (
    np.log1p(ask_age)
    - np.log1p(bid_age)
)

x25["joint_quote_age_25a"] = (
    np.log1p(
        np.minimum(bid_age, ask_age)
    )
)

x25["log_bid_quote_age_25a"] = (
    np.log1p(bid_age)
)

x25["log_ask_quote_age_25a"] = (
    np.log1p(ask_age)
)

CANDIDATES25 = [
    "quote_age_asymmetry_25a",
    "joint_quote_age_25a"
]

CONTROLS25 = [
    "log_bid_quote_age_25a",
    "log_ask_quote_age_25a",
    "spread_bps_25a"
]

# ------------------------------------------------------------
# 5. Historical momentum controls
# ------------------------------------------------------------

for w in [30, 60]:

    g25 = x25.groupby(GROUP25, sort=False)

    old_mid = g25["mid"].shift(w)
    old_ts = g25["ts_sec"].shift(w)

    name = f"momentum_{w}_25a"

    x25[name] = (
        1e4 * np.log(
            x25["mid"] / old_mid
        )
    ).where(
        (x25["ts_sec"] - old_ts).eq(w)
        & old_mid.gt(0)
        & x25["mid"].gt(0)
    )

    CONTROLS25.append(name)

# ------------------------------------------------------------
# 6. Quote-price change intensity controls
# ------------------------------------------------------------

x25["_any_price_change"] = (
    (bid_changed | ask_changed)
    .astype(float)
    .where(valid_pair)
)

for w in [60, 300]:

    n_valid = (
        x25.groupby(GROUP25, sort=False)[
            "_any_price_change"
        ].transform(
            lambda s: s.notna().astype(float)
            .rolling(w, min_periods=1).sum()
        )
    )

    n_changed = (
        x25.groupby(GROUP25, sort=False)[
            "_any_price_change"
        ].transform(
            lambda s: s.fillna(0.0)
            .rolling(w, min_periods=1).sum()
        )
    )

    feature = f"quote_price_change_rate_{w}_25a"

    x25[feature] = (
        n_changed / n_valid.where(
            n_valid >= int(np.ceil(0.90 * w))
        )
    )

    CONTROLS25.append(feature)

# ------------------------------------------------------------
# 7. Optional already-engineered update-rate control
#
# Not required for this experiment.
# ------------------------------------------------------------

if "update_rate_300s" in df_1s.columns:

    extra = df_1s[
        ["symbol", "ts_sec", "update_rate_300s"]
    ].copy()

    x25 = x25.merge(
        extra,
        on=["symbol", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    CONTROLS25.append("update_rate_300s")

# ------------------------------------------------------------
# 8. Exact-horizon future return targets
# ------------------------------------------------------------

for h in HORIZONS_25A:

    future = x25[
        ["symbol", "date", "ts_sec", "mid"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={"mid": f"future_mid_{h}"}
    )

    x25 = x25.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    signed_name = f"future_return_{h}_25a"
    abs_name = f"future_absret_{h}_25a"

    x25[signed_name] = (
        1e4 * np.log(
            x25[f"future_mid_{h}"] / x25["mid"]
        )
    )

    x25[abs_name] = x25[signed_name].abs()

# ------------------------------------------------------------
# 9. Daily Spearman IC
# ------------------------------------------------------------

def safe_ic_25a(a, b):

    valid = (
        np.isfinite(a) & np.isfinite(b)
    )

    if valid.sum() < 200:
        return np.nan, int(valid.sum())

    ra = pd.Series(a[valid]).rank().to_numpy()
    rb = pd.Series(b[valid]).rank().to_numpy()

    if (
        ra.std() < 1e-12
        or rb.std() < 1e-12
    ):
        return np.nan, int(valid.sum())

    return (
        float(np.corrcoef(ra, rb)[0, 1]),
        int(valid.sum())
    )

features25 = CANDIDATES25 + CONTROLS25

records = []

for (symbol, date), day in x25.groupby(
    GROUP25, sort=False
):

    for h in HORIZONS_25A:

        targets = {
            "signed_return":
                f"future_return_{h}_25a",
            "absret":
                f"future_absret_{h}_25a"
        }

        for task, target in targets.items():

            y = day[target].to_numpy(dtype=float)

            for feature in features25:

                z = day[feature].to_numpy(
                    dtype=float
                )

                ic, n = safe_ic_25a(z, y)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp25a_daily = pd.DataFrame(records)

calendar_exp25a_summary = (
    calendar_exp25a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

calendar_exp25a_summary["sign_consistency"] = (
    calendar_exp25a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp25a_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 10. Redundancy and distribution diagnostics
# ------------------------------------------------------------

calendar_exp25a_redundancy = (
    x25[features25]
    .corr(method="spearman")
    .loc[CANDIDATES25, CONTROLS25]
)

print("\n===== 25A QUOTE AGE SIGNED RETURN IC =====")

print(
    calendar_exp25a_summary[
        calendar_exp25a_summary["task"].eq(
            "signed_return"
        )
        & calendar_exp25a_summary[
            "feature"
        ].isin(CANDIDATES25)
    ].round(6).to_string(index=False)
)

print("\n===== 25A QUOTE AGE ABSRET IC =====")

print(
    calendar_exp25a_summary[
        calendar_exp25a_summary["task"].eq(
            "absret"
        )
        & calendar_exp25a_summary[
            "feature"
        ].isin(CANDIDATES25)
    ].round(6).to_string(index=False)
)

print("\n===== 25A CONTROL BASELINE IC =====")

print(
    calendar_exp25a_summary[
        calendar_exp25a_summary[
            "feature"
        ].isin(CONTROLS25)
    ].round(6).to_string(index=False)
)

print("\n===== 25A QUOTE AGE REDUNDANCY =====")

print(
    calendar_exp25a_redundancy
    .round(5)
    .to_string()
)

print("\n===== 25A FEATURE DIAGNOSTICS =====")

for c in CANDIDATES25 + [
    "bid_quote_age_25a",
    "ask_quote_age_25a"
]:
    s = x25[c]

    print(
        c,
        "valid_pct:",
        round(100 * s.notna().mean(), 3),
        "median:",
        round(s.median(), 4),
        "p95:",
        round(s.quantile(0.95), 4),
        "zero_pct:",
        round(100 * s.eq(0).mean(), 3)
    )

print("\n===== 25A QUOTE CHANGE DIAGNOSTICS =====")

print(
    "Bid price change rate:",
    round(bid_changed.sum() / valid_pair.sum(), 6)
)

print(
    "Ask price change rate:",
    round(ask_changed.sum() / valid_pair.sum(), 6)
)

print(
    "Valid consecutive quote pairs:",
    round(100 * valid_pair.mean(), 4), "%"
)

calendar_exp25a_features = x25

print("\n===== EXPERIMENT 25A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 25A SAMPLE =====
Sessions: 16
Rows: 459122

===== 25A QUOTE AGE SIGNED RETURN IC =====
         task  horizon                 feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
signed_return       30     joint_quote_age_25a    16 0.017514   0.016242             13              3            28723.0            0.8125
signed_return       30 quote_age_asymmetry_25a    16 0.045842   0.047052             15              1            28723.0            0.9375
signed_return       60     joint_quote_age_25a    16 0.018534   0.021378             11              5            28693.0            0.6875
signed_return       60 quote_age_asymmetry_25a    16 0.054587   0.056719             15              1            28693.0            0.9375
signed_return      300     joint_quote_age_25a    16 0.005905  -0.001679              7              9            28453.0            0.5625
signed_return      300 quote_age_asymmetry_25a    16 0.0

In [74]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 25B — JOINT QUOTE AGE / FUTURE ABSRET
#
# Input:
#   calendar_exp25a_features
#
# Target:
#   Future absolute returns at 30, 60, 300 seconds
#
# Models:
#   A: Quote activity + volatility + price controls
#   B: A + individual bid/ask quote ages
#   C: B + joint quote age
#
# Expanding chronological ridge regression
# Identical observations across models
# No modification to df_1s
# ============================================================

if "calendar_exp25a_features" not in globals():
    raise ValueError("Run Experiment 25A first.")

x = calendar_exp25a_features.copy()

HORIZONS = [30, 60, 300]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10
RIDGE_ALPHA = 0.01

GROUP = ["symbol", "date"]

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows.")

# ------------------------------------------------------------
# 1. Construct causal realized volatility controls
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

r1 = np.log(x["mid"] / prev_mid).where(
    (x["ts_sec"] - prev_ts).eq(1)
    & x["mid"].gt(0)
    & prev_mid.gt(0)
)

x["_ret_sq_25b"] = r1 ** 2

for w in [60, 300]:

    minp = int(np.ceil(0.9 * w))

    x[f"rv_{w}_25b"] = (
        x.groupby(GROUP, sort=False)[
            "_ret_sq_25b"
        ].transform(
            lambda s: s.rolling(
                w, min_periods=minp
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5) * 1e4
    )

# ------------------------------------------------------------
# 2. Intraday timing controls
# ------------------------------------------------------------

x["clock_25b"] = x["clock_seconds"]

x["open_first30_25b"] = (
    x["clock_25b"] < 30
).astype(float)

x["open_1_5m_25b"] = (
    x["clock_25b"].ge(60)
    & x["clock_25b"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 3. Freeze model inputs
# ------------------------------------------------------------

BASE = [
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a",
    "rv_60_25b",
    "rv_300_25b",
    "spread_bps_25a",
    "momentum_30_25a",
    "momentum_60_25a",
    "clock_25b",
    "open_first30_25b",
    "open_1_5m_25b"
]

# Include update activity if actually available.
if "update_rate_300s" in x.columns:
    BASE.append("update_rate_300s")

INDIVIDUAL_AGES = [
    "log_bid_quote_age_25a",
    "log_ask_quote_age_25a"
]

JOINT_AGE = "joint_quote_age_25a"

MODELS = {
    "A_activity": BASE,
    "B_individual_ages": BASE + INDIVIDUAL_AGES,
    "C_joint_age": BASE + INDIVIDUAL_AGES + [JOINT_AGE]
}

ALL_FEATURES = list(dict.fromkeys(
    f for cols in MODELS.values() for f in cols
))

TARGETS = [
    f"future_absret_{h}_25a"
    for h in HORIZONS
]

required = ALL_FEATURES + TARGETS

missing = sorted(set(required) - set(x.columns))

if missing:
    raise ValueError(f"Missing fields: {missing}")

for c in required:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common_25b"] = np.isfinite(
    x[ALL_FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("===== 25B DATA COVERAGE =====")
print("Rows:", len(x))
print("Complete feature rows:", int(x["_common_25b"].sum()))
print("Coverage %:", round(100*x["_common_25b"].mean(), 4))
print("Baseline input count:", len(BASE))

# ------------------------------------------------------------
# 4. Fixed ridge regression
#
# Standardization fitted exclusively on train.
# Absolute-return predictions clipped to zero.
# ------------------------------------------------------------

def predict_ridge_25b(train, test, features, target):

    A = train[features].to_numpy(dtype=float)
    B = test[features].to_numpy(dtype=float)
    y = train[target].to_numpy(dtype=float)

    mu = A.mean(axis=0)
    sd = A.std(axis=0)
    sd = np.where(sd > 1e-12, sd, 1.0)

    A = (A - mu) / sd
    B = (B - mu) / sd

    ym = y.mean()
    n, k = A.shape

    beta = np.linalg.solve(
        A.T @ A + RIDGE_ALPHA * n * np.eye(k),
        A.T @ (y - ym)
    )

    return np.maximum(0.0, ym + B @ beta)

# ------------------------------------------------------------
# 5. Expanding walk-forward validation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient sessions.")

records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_25b"]
        & x["ts_sec"].mod(TRAIN_STRIDE).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_25b"]
    ]

    for h in HORIZONS:

        target = f"future_absret_{h}_25a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        actual = test[target].to_numpy(dtype=float)

        for name, features in MODELS.items():

            pred = predict_ridge_25b(
                train, test, features, target
            )

            error = actual - pred

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

    print("Completed:", test_date)

calendar_exp25b_daily = pd.DataFrame(records)

if calendar_exp25b_daily.empty:
    raise ValueError("No evaluations completed.")

# ------------------------------------------------------------
# 6. Matched pooled and day-level comparisons
# ------------------------------------------------------------

COMPARISONS = {
    "individual_vs_activity": (
        "B_individual_ages",
        "A_activity"
    ),
    "joint_vs_individual": (
        "C_joint_age",
        "B_individual_ages"
    ),
    "joint_vs_activity": (
        "C_joint_age",
        "A_activity"
    )
}

summary = []
daily_rows = []

for h, block in calendar_exp25b_daily.groupby("horizon"):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    daily = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gain = 100 * (
            1 - daily[candidate] / daily[baseline]
        )

        for date, value in gain.items():

            daily_rows.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "mse_gain_pct": float(value)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gain),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gain > 0).sum()),
            "median_daily_gain_pct": float(gain.median()),
            "worst_day_pct": float(gain.min())
        })

calendar_exp25b_summary = pd.DataFrame(summary)

calendar_exp25b_daily_gains = pd.DataFrame(daily_rows)

print("\n===== 25B ABSRET INCREMENTAL RESULTS =====")

print(
    calendar_exp25b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 25B DAILY JOINT-AGE GAINS =====")

print(
    calendar_exp25b_daily_gains[
        calendar_exp25b_daily_gains[
            "comparison"
        ].eq("joint_vs_individual")
    ].pivot(
        index="date",
        columns="horizon",
        values="mse_gain_pct"
    ).round(6).to_string()
)

print("\n===== EXPERIMENT 25B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 25B DATA COVERAGE =====
Rows: 459122
Complete feature rows: 454802
Coverage %: 99.0591
Baseline input count: 10
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 25B ABSRET INCREMENTAL RESULTS =====
 horizon             comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30 individual_vs_activity    11             0.841654             0.652817            10               0.413600      -0.172340
      30    joint_vs_individual    11             0.007170            -0.003693             5              -0.000721      -0.172621
      30      joint_vs_activity    11             0.848764             0.649148            10               0.427803      -0.173062
      60 individual_vs_activity    11             0.441893          

In [75]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 25C — DIRECTIONAL QUOTE-AGE TEST
#
# Models:
# A: Existing components
# B: A + quote-age asymmetry
# C: A + individual bid/ask log ages
#
# Targets: signed future returns, 30/60/300s
# Expanding chronological ridge regression
# Matched samples and fixed parameters
#
# No changes to df_1s
# ============================================================

if "calendar_exp25a_features" not in globals():
    raise ValueError("Run Experiment 25A first.")

x = calendar_exp25a_features.copy()

HORIZONS = [30, 60, 300]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10
RIDGE_ALPHA = 0.01

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows")

# ------------------------------------------------------------
# 1. Causal historical volatility
# ------------------------------------------------------------

grp = x.groupby(["symbol", "date"], sort=False)

old_mid = grp["mid"].shift(1)
old_ts = grp["ts_sec"].shift(1)

r1 = np.log(x["mid"] / old_mid).where(
    (x["ts_sec"] - old_ts).eq(1)
    & x["mid"].gt(0)
    & old_mid.gt(0)
)

x["_ret_sq_25c"] = r1 ** 2

for w in [60, 300]:
    x[f"rv_{w}_25c"] = (
        x.groupby(
            ["symbol", "date"], sort=False
        )["_ret_sq_25c"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5) * 1e4
    )

# ------------------------------------------------------------
# 2. Time-of-day controls
# ------------------------------------------------------------

x["open_first30_25c"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_25c"] = (
    x["clock_seconds"].ge(60)
    & x["clock_seconds"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 3. Fixed feature sets
# ------------------------------------------------------------

BASE = [
    "momentum_30_25a",
    "momentum_60_25a",
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a",
    "rv_60_25c",
    "rv_300_25c",
    "spread_bps_25a",
    "clock_seconds",
    "open_first30_25c",
    "open_1_5m_25c"
]

if "update_rate_300s" in x.columns:
    BASE.append("update_rate_300s")

ASYM = "quote_age_asymmetry_25a"

INDIVIDUAL = [
    "log_bid_quote_age_25a",
    "log_ask_quote_age_25a"
]

MODELS = {
    "A_components": BASE,
    "B_asymmetry": BASE + [ASYM],
    "C_individual_ages": BASE + INDIVIDUAL
}

FEATURES = list(dict.fromkeys(
    feature
    for cols in MODELS.values()
    for feature in cols
))

TARGETS = [
    f"future_return_{h}_25a"
    for h in HORIZONS
]

missing = sorted(
    set(FEATURES + TARGETS) - set(x.columns)
)

if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in FEATURES + TARGETS:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common_25c"] = np.isfinite(
    x[FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 25C SAMPLE =====")
print("Rows:", len(x))
print(
    "Complete rows:",
    int(x["_common_25c"].sum())
)
print(
    "Coverage:",
    round(100 * x["_common_25c"].mean(), 4),
    "%"
)

# ------------------------------------------------------------
# 4. Fixed standardized ridge regression
# ------------------------------------------------------------

def ridge_predict_25c(train, test, columns, target):

    A = train[columns].to_numpy(
        dtype=np.float64
    )
    B = test[columns].to_numpy(
        dtype=np.float64
    )
    y = train[target].to_numpy(
        dtype=np.float64
    )

    mu = A.mean(axis=0)
    sd = A.std(axis=0)
    sd = np.where(sd > 1e-12, sd, 1.0)

    A = (A - mu) / sd
    B = (B - mu) / sd

    ym = y.mean()

    beta = np.linalg.solve(
        A.T @ A
        + RIDGE_ALPHA * len(A) * np.eye(
            A.shape[1]
        ),
        A.T @ (y - ym)
    )

    # Signed returns: do not clip.
    return ym + B @ beta

# ------------------------------------------------------------
# 5. Expanding chronological evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient sessions")

records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_25c"]
        & x["ts_sec"].mod(TRAIN_STRIDE).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_25c"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}_25a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y = test[target].to_numpy(dtype=float)

        for name, columns in MODELS.items():

            prediction = ridge_predict_25c(
                train, test, columns, target
            )

            error = y - prediction

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

    print("Completed:", test_date)

calendar_exp25c_daily = pd.DataFrame(records)

if calendar_exp25c_daily.empty:
    raise ValueError("No completed evaluations")

# ------------------------------------------------------------
# 6. Compare pooled errors and daily wins
# ------------------------------------------------------------

COMPARISONS = {
    "asymmetry_vs_components": (
        "B_asymmetry", "A_components"
    ),
    "individual_vs_components": (
        "C_individual_ages", "A_components"
    ),
    "asymmetry_vs_individual": (
        "B_asymmetry", "C_individual_ages"
    )
}

summary = []
daily_records = []

for h, block in calendar_exp25c_daily.groupby(
    "horizon"
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gain = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, value in gain.items():
            daily_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(value)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gain),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int(
                (gain > 0).sum()
            ),
            "median_daily_gain_pct": float(
                gain.median()
            ),
            "worst_day_pct": float(
                gain.min()
            )
        })

calendar_exp25c_summary = pd.DataFrame(summary)
calendar_exp25c_daily_gains = pd.DataFrame(
    daily_records
)

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n===== 25C DIRECTIONAL INCREMENTAL RESULTS =====")

print(
    calendar_exp25c_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 25C DAILY ASYMMETRY GAINS =====")

print(
    calendar_exp25c_daily_gains[
        calendar_exp25c_daily_gains[
            "comparison"
        ].eq("asymmetry_vs_components")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== EXPERIMENT 25C COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 25C SAMPLE =====
Rows: 459122
Complete rows: 454802
Coverage: 99.0591 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 25C DIRECTIONAL INCREMENTAL RESULTS =====
 horizon               comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30  asymmetry_vs_components    11             0.079179            -0.379430             8               0.040298      -0.228033
      30 individual_vs_components    11             0.047200            -0.448393             8               0.076770      -0.183178
      30  asymmetry_vs_individual    11             0.031993             0.068655             5              -0.000618      -0.235177
      60  asymmetry_vs_components    11             0.135555            -0.194706   

In [76]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 25D — NONLINEAR QUOTE-AGE DIRECTIONAL TEST
#
# Input: calendar_exp25a_features
# Targets: signed returns at 30, 60, 300 seconds
#
# A: components
# B: components + quote-age asymmetry
# C: components + bid and ask log ages
#
# Expanding chronological LightGBM
# Fixed hyperparameters
# Matched samples
# No modification to df_1s
# ============================================================

if "calendar_exp25a_features" not in globals():
    raise ValueError("Run Experiment 25A first.")

x = calendar_exp25a_features.copy()

HORIZONS = [30, 60, 300]
MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

# ------------------------------------------------------------
# 1. Chronological ordering
# ------------------------------------------------------------

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp rows.")

GROUP = ["symbol", "date"]

# ------------------------------------------------------------
# 2. Historical realized volatility
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

previous_mid = g["mid"].shift(1)
previous_ts = g["ts_sec"].shift(1)

r1 = np.log(
    x["mid"] / previous_mid
).where(
    (x["ts_sec"] - previous_ts).eq(1)
    & x["mid"].gt(0)
    & previous_mid.gt(0)
)

x["_ret_sq_25d"] = r1 ** 2

for w in [60, 300]:

    x[f"rv_{w}_25d"] = (
        x.groupby(GROUP, sort=False)[
            "_ret_sq_25d"
        ].transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 1e4
    )

# ------------------------------------------------------------
# 3. Intraday controls
# ------------------------------------------------------------

x["open_first30_25d"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_25d"] = (
    x["clock_seconds"].ge(60)
    & x["clock_seconds"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 4. Frozen model inputs
# ------------------------------------------------------------

BASE = [
    "momentum_30_25a",
    "momentum_60_25a",
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a",
    "rv_60_25d",
    "rv_300_25d",
    "spread_bps_25a",
    "clock_seconds",
    "open_first30_25d",
    "open_1_5m_25d"
]

if "update_rate_300s" in x.columns:
    BASE.append("update_rate_300s")

ASYM = "quote_age_asymmetry_25a"

INDIVIDUAL = [
    "log_bid_quote_age_25a",
    "log_ask_quote_age_25a"
]

MODELS = {
    "A_components": BASE,
    "B_asymmetry": BASE + [ASYM],
    "C_individual_ages": BASE + INDIVIDUAL
}

ALL_FEATURES = list(dict.fromkeys(
    f
    for cols in MODELS.values()
    for f in cols
))

TARGETS = [
    f"future_return_{h}_25a"
    for h in HORIZONS
]

missing = sorted(
    set(ALL_FEATURES + TARGETS) - set(x.columns)
)

if missing:
    raise ValueError(f"Missing columns: {missing}")

for c in ALL_FEATURES + TARGETS:

    x[c] = pd.to_numeric(
        x[c],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

x["_common_25d"] = np.isfinite(
    x[ALL_FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 25D SAMPLE =====")

print("Total rows:", len(x))

print(
    "Complete feature rows:",
    int(x["_common_25d"].sum())
)

print(
    "Feature coverage (%):",
    round(
        100 * x["_common_25d"].mean(),
        4
    )
)

print("Baseline inputs:", len(BASE))

# ------------------------------------------------------------
# 5. Expanding walk-forward evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Insufficient trading sessions.")

records = []
importance_records = []

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_25d"]
        & x["ts_sec"].mod(TRAIN_STRIDE).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_25d"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}_25a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for name, cols in MODELS.items():

            model = lgb.LGBMRegressor(**PARAMS)

            model.fit(
                train[cols],
                y_train
            )

            pred = model.predict(test[cols])

            # Signed returns are not clipped.
            error = y_test - pred

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

            gains = model.booster_.feature_importance(
                importance_type="gain"
            )

            for feature, value in zip(cols, gains):

                if feature in [ASYM] + INDIVIDUAL:

                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": name,
                        "feature": feature,
                        "gain_importance": float(value)
                    })

    print("Completed:", test_date)

calendar_exp25d_daily = pd.DataFrame(records)

calendar_exp25d_importance = pd.DataFrame(
    importance_records
)

if calendar_exp25d_daily.empty:
    raise ValueError("No evaluations completed.")

# ------------------------------------------------------------
# 6. Model comparisons
# ------------------------------------------------------------

COMPARISONS = {
    "asymmetry_vs_components": (
        "B_asymmetry",
        "A_components"
    ),
    "individual_vs_components": (
        "C_individual_ages",
        "A_components"
    ),
    "asymmetry_vs_individual": (
        "B_asymmetry",
        "C_individual_ages"
    )
}

summary = []
daily_gains = []

for h, block in calendar_exp25d_daily.groupby(
    "horizon"
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gain = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, value in gain.items():

            daily_gains.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(value)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gain),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int(
                (gain > 0).sum()
            ),
            "median_daily_gain_pct": float(
                gain.median()
            ),
            "worst_day_pct": float(
                gain.min()
            )
        })

calendar_exp25d_summary = pd.DataFrame(summary)

calendar_exp25d_daily_gains = pd.DataFrame(
    daily_gains
)

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n===== 25D LIGHTGBM DIRECTIONAL RESULTS =====")

print(
    calendar_exp25d_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 25D DAILY ASYMMETRY GAINS =====")

print(
    calendar_exp25d_daily_gains[
        calendar_exp25d_daily_gains[
            "comparison"
        ].eq("asymmetry_vs_components")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== 25D QUOTE AGE IMPORTANCE =====")

if not calendar_exp25d_importance.empty:

    print(
        calendar_exp25d_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 25D COMPLETE =====")

print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 25D SAMPLE =====
Total rows: 459122
Complete feature rows: 454802
Feature coverage (%): 99.0591
Baseline inputs: 10
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 25D LIGHTGBM DIRECTIONAL RESULTS =====
 horizon               comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30  asymmetry_vs_components    11             0.240229            -0.319950             8               0.178424      -0.227632
      30 individual_vs_components    11             0.288984            -0.078088             7               0.415308      -0.320869
      30  asymmetry_vs_individual    11            -0.048897            -0.241674             6               0.075696      -0.783579
      60  asymmetry_vs_components    11    

In [77]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 25E — NONLINEAR QUOTE AGE / ABSOLUTE RETURNS
#
# Research notebook only
#
# Targets:
#   Future absolute mid returns at 30, 60, 300s
#
# A: Activity, RV, momentum, spread, clock
# B: A + individual bid/ask log ages
# C: B + joint quote age
#
# Fixed expanding LightGBM
# Identical observations across models
# Does not modify df_1s
# ============================================================

if "calendar_exp25a_features" not in globals():
    raise ValueError("Run Experiment 25A first.")

x = calendar_exp25a_features.copy()

HORIZONS_25E = [30, 60, 300]
MIN_TRAIN_DAYS_25E = 5
TRAIN_STRIDE_25E = 10

PARAMS_25E = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

GROUP = ["symbol", "date"]

if x.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp observations")

# ------------------------------------------------------------
# 1. Construct causal historical realized volatility
# ------------------------------------------------------------

g = x.groupby(GROUP, sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

ret1 = np.log(x["mid"] / prev_mid).where(
    (x["ts_sec"] - prev_ts).eq(1)
    & x["mid"].gt(0)
    & prev_mid.gt(0)
)

x["_ret_sq_25e"] = ret1 ** 2

for w in [60, 300]:

    x[f"rv_{w}_25e"] = (
        x.groupby(GROUP, sort=False)["_ret_sq_25e"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.90 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 1e4
    )

# ------------------------------------------------------------
# 2. Intraday controls
# ------------------------------------------------------------

x["open_first30_25e"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_25e"] = (
    x["clock_seconds"].ge(60)
    & x["clock_seconds"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 3. Frozen feature sets
# ------------------------------------------------------------

BASE_25E = [
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a",
    "rv_60_25e",
    "rv_300_25e",
    "spread_bps_25a",
    "momentum_30_25a",
    "momentum_60_25a",
    "clock_seconds",
    "open_first30_25e",
    "open_1_5m_25e"
]

if "update_rate_300s" in x.columns:
    BASE_25E.append("update_rate_300s")

AGES_25E = [
    "log_bid_quote_age_25a",
    "log_ask_quote_age_25a"
]

JOINT_25E = "joint_quote_age_25a"

MODELS_25E = {
    "A_activity": BASE_25E,
    "B_individual_ages": BASE_25E + AGES_25E,
    "C_joint_age": BASE_25E + AGES_25E + [JOINT_25E]
}

ALL_FEATURES_25E = list(dict.fromkeys(
    f for cols in MODELS_25E.values() for f in cols
))

TARGETS_25E = [
    f"future_absret_{h}_25a"
    for h in HORIZONS_25E
]

missing = sorted(
    set(ALL_FEATURES_25E + TARGETS_25E)
    - set(x.columns)
)

if missing:
    raise ValueError(f"Missing columns: {missing}")

for c in ALL_FEATURES_25E + TARGETS_25E:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common_25e"] = np.isfinite(
    x[ALL_FEATURES_25E].to_numpy(dtype=float)
).all(axis=1)

print("===== EXPERIMENT 25E SAMPLE =====")
print("Rows:", len(x))
print("Complete feature rows:", int(x["_common_25e"].sum()))
print(
    "Feature coverage (%):",
    round(100 * x["_common_25e"].mean(), 4)
)

# ------------------------------------------------------------
# 4. Chronological expanding walk-forward
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS_25E:
    raise ValueError("Insufficient sessions.")

records = []
importance_records = []

for i in range(MIN_TRAIN_DAYS_25E, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_25e"]
        & x["ts_sec"].mod(TRAIN_STRIDE_25E).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_25e"]
    ]

    for h in HORIZONS_25E:

        target = f"future_absret_{h}_25a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for name, columns in MODELS_25E.items():

            model = lgb.LGBMRegressor(**PARAMS_25E)

            model.fit(train[columns], y_train)

            prediction = np.maximum(
                model.predict(test[columns]),
                0.0
            )

            error = y_test - prediction

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

            gain = model.booster_.feature_importance(
                importance_type="gain"
            )

            for feature, value in zip(columns, gain):
                if feature in AGES_25E + [JOINT_25E]:
                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": name,
                        "feature": feature,
                        "gain_importance": float(value)
                    })

    print("Completed:", test_date)

calendar_exp25e_daily = pd.DataFrame(records)
calendar_exp25e_importance = pd.DataFrame(importance_records)

if calendar_exp25e_daily.empty:
    raise ValueError("No evaluations completed.")

# ------------------------------------------------------------
# 5. Pooled and day-level comparisons
# ------------------------------------------------------------

COMPARISONS_25E = {
    "individual_vs_activity": (
        "B_individual_ages",
        "A_activity"
    ),
    "joint_vs_individual": (
        "C_joint_age",
        "B_individual_ages"
    ),
    "joint_vs_activity": (
        "C_joint_age",
        "A_activity"
    )
}

summary = []
daily_rows = []

for h, block in calendar_exp25e_daily.groupby("horizon"):

    pooled = block.groupby("model")[["sse", "sae"]].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS_25E.items()
    ):

        gains = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, value in gains.items():
            daily_rows.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(value)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gains),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gains > 0).sum()),
            "median_daily_gain_pct": float(gains.median()),
            "worst_day_pct": float(gains.min())
        })

calendar_exp25e_summary = pd.DataFrame(summary)
calendar_exp25e_daily_gains = pd.DataFrame(daily_rows)

# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

print("\n===== 25E LIGHTGBM ABSRET RESULTS =====")

print(
    calendar_exp25e_summary.round(6)
    .to_string(index=False)
)

print("\n===== 25E DAILY INDIVIDUAL AGE GAINS =====")

print(
    calendar_exp25e_daily_gains[
        calendar_exp25e_daily_gains[
            "comparison"
        ].eq("individual_vs_activity")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== 25E QUOTE AGE IMPORTANCE =====")

if not calendar_exp25e_importance.empty:
    print(
        calendar_exp25e_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 25E COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 25E SAMPLE =====
Rows: 459122
Complete feature rows: 454802
Feature coverage (%): 99.0591
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 25E LIGHTGBM ABSRET RESULTS =====
 horizon             comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30 individual_vs_activity    11             0.164141             0.236774             8               0.161857      -0.882332
      30    joint_vs_individual    11            -0.123108            -0.096077             5              -0.112214      -0.541582
      30      joint_vs_activity    11             0.041236             0.140924             5              -0.002902      -1.428693
      60 individual_vs_activity    11             0.283648             0.174391   

In [78]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 25F — FULL 32-FEATURE QUOTE-AGE ABLATION
#
# Calendar notebook only.
#
# Sources:
#   core_31
#   calendar_exp24a_features
#   calendar_exp25a_features
#
# Full baseline:
#   Existing 31 features
#   + quote_retreat_imbalance_30 (feature #32)
#   + common quote activity / RV / clock controls
#
# Candidates:
#   log_bid_quote_age_25a
#   log_ask_quote_age_25a
#
# Target:
#   Future absolute returns: 30/60/300 seconds
#
# Fixed expanding LightGBM.
# Identical train/test samples.
# No modifications to df_1s.
# ============================================================

HORIZONS_25F = [30, 60, 300]
MIN_TRAIN_DAYS_25F = 5
TRAIN_STRIDE_25F = 10

PARAMS_25F = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

# ------------------------------------------------------------
# 1. Check required research DataFrames
# ------------------------------------------------------------

for name in [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp25a_features"
]:
    if name not in globals():
        raise ValueError(f"{name} is not available.")

KEYS_25F = ["symbol", "ts_sec"]

CORE_25F = [
    c for c in core_31.columns
    if c not in KEYS_25F
]

if len(CORE_25F) != 31:
    raise ValueError(
        f"Expected 31 core features, got {len(CORE_25F)}."
    )

if len(set(CORE_25F)) != 31:
    raise ValueError("Duplicate core feature names.")

RETREAT_25F = "quote_retreat_imbalance_30_24a"

BID_AGE_25F = "log_bid_quote_age_25a"
ASK_AGE_25F = "log_ask_quote_age_25a"

# ------------------------------------------------------------
# 2. Construct separate source tables
# ------------------------------------------------------------

source_core = core_31[
    KEYS_25F + CORE_25F
].copy()

source_retreat = calendar_exp24a_features[
    KEYS_25F + [RETREAT_25F]
].copy()

age_source_cols = [
    "date",
    "clock_seconds",
    "spread_bps_25a",
    "momentum_30_25a",
    "momentum_60_25a",
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a",
    "mid",
    BID_AGE_25F,
    ASK_AGE_25F
]

TARGETS_25F = [
    f"future_absret_{h}_25a"
    for h in HORIZONS_25F
]

source_ages = calendar_exp25a_features[
    KEYS_25F + age_source_cols + TARGETS_25F
].copy()

# Normalize join key dtypes and verify uniqueness.
for frame in [
    source_core,
    source_retreat,
    source_ages
]:
    frame["symbol"] = frame["symbol"].astype(str)

    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"], errors="raise"
    ).astype("int64")

    if frame.duplicated(KEYS_25F).any():
        raise ValueError(
            "Duplicate symbol/timestamp join keys."
        )

# ------------------------------------------------------------
# 3. Join exact timestamps
#
# Keep all research observations, then attach
# feature #32 and the existing 31.
# ------------------------------------------------------------

x = source_ages.merge(
    source_retreat,
    on=KEYS_25F,
    how="left",
    validate="one_to_one",
    indicator="_retreat_match"
)

x = x.merge(
    source_core,
    on=KEYS_25F,
    how="left",
    validate="one_to_one",
    indicator="_core_match"
)

retreat_match_pct = (
    100 * x["_retreat_match"].eq("both").mean()
)

core_match_pct = (
    100 * x["_core_match"].eq("both").mean()
)

print("===== 25F JOIN DIAGNOSTICS =====")
print("Research rows:", len(x))
print("Core match %:", round(core_match_pct, 4))
print("Retreat match %:", round(retreat_match_pct, 4))

if (
    core_match_pct < 99
    or retreat_match_pct < 99
):
    raise ValueError(
        "Source datasets do not match sufficiently. "
        "Check symbol, timestamps and data versions."
    )

x = x.drop(
    columns=["_retreat_match", "_core_match"]
)

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Causal realized volatility controls
# ------------------------------------------------------------

GROUP_25F = ["symbol", "date"]

g = x.groupby(GROUP_25F, sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

r1 = np.log(
    x["mid"] / prev_mid
).where(
    (x["ts_sec"] - prev_ts).eq(1)
    & x["mid"].gt(0)
    & prev_mid.gt(0)
)

x["_ret_sq_25f"] = r1 ** 2

for w in [60, 300]:

    x[f"rv_{w}_25f"] = (
        x.groupby(GROUP_25F, sort=False)[
            "_ret_sq_25f"
        ]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=int(0.9 * w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5)
        * 1e4
    )

# ------------------------------------------------------------
# 5. Time-of-day controls
# ------------------------------------------------------------

x["open_first30_25f"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_25f"] = (
    x["clock_seconds"].ge(60)
    & x["clock_seconds"].lt(300)
).astype(float)

# ------------------------------------------------------------
# 6. Freeze baseline and candidate models
# ------------------------------------------------------------

COMMON_CONTROLS_25F = [
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a",
    "rv_60_25f",
    "rv_300_25f",
    "spread_bps_25a",
    "momentum_30_25a",
    "momentum_60_25a",
    "clock_seconds",
    "open_first30_25f",
    "open_1_5m_25f"
]

BASE_25F = list(dict.fromkeys(
    CORE_25F
    + [RETREAT_25F]
    + COMMON_CONTROLS_25F
))

MODELS_25F = {
    "A_full32": BASE_25F,

    "B_bid_age": (
        BASE_25F + [BID_AGE_25F]
    ),

    "C_ask_age": (
        BASE_25F + [ASK_AGE_25F]
    ),

    "D_both_ages": (
        BASE_25F + [
            BID_AGE_25F,
            ASK_AGE_25F
        ]
    )
}

ALL_FEATURES_25F = list(dict.fromkeys(
    f for cols in MODELS_25F.values()
    for f in cols
))

for c in ALL_FEATURES_25F + TARGETS_25F:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

# Same finite feature rows across ALL models.
x["_common_25f"] = np.isfinite(
    x[ALL_FEATURES_25F].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 25F MODEL SAMPLE =====")
print("Total rows:", len(x))
print(
    "Complete rows:",
    int(x["_common_25f"].sum())
)
print(
    "Coverage %:",
    round(100 * x["_common_25f"].mean(), 4)
)
print("Core baseline features:", 32)
print("Baseline including controls:", len(BASE_25F))

# Per-session coverage is important because a
# late-available core feature can remove early data.
session_coverage = (
    x.groupby(GROUP_25F)["_common_25f"]
    .agg(["sum", "mean"])
)

print("\n===== 25F SESSION COVERAGE =====")
print(
    session_coverage.round(4).to_string()
)

# ------------------------------------------------------------
# 7. Expanding chronological validation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS_25F:
    raise ValueError("Insufficient eligible sessions.")

records = []
importance_records = []

for i in range(
    MIN_TRAIN_DAYS_25F,
    len(dates)
):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_25f"]
        & x["ts_sec"].mod(
            TRAIN_STRIDE_25F
        ).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_25f"]
    ]

    for h in HORIZONS_25F:

        target = f"future_absret_{h}_25a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for name, columns in MODELS_25F.items():

            model = lgb.LGBMRegressor(
                **PARAMS_25F
            )

            model.fit(
                train[columns],
                y_train
            )

            prediction = np.maximum(
                model.predict(test[columns]),
                0.0
            )

            error = y_test - prediction

            records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

            importance = (
                model.booster_.feature_importance(
                    importance_type="gain"
                )
            )

            for feature, gain in zip(
                columns,
                importance
            ):

                if feature in [
                    BID_AGE_25F,
                    ASK_AGE_25F
                ]:

                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": name,
                        "feature": feature,
                        "gain_importance": float(gain)
                    })

    print("Completed:", test_date)

calendar_exp25f_daily = pd.DataFrame(records)

calendar_exp25f_importance = pd.DataFrame(
    importance_records
)

if calendar_exp25f_daily.empty:
    raise ValueError(
        "No evaluations completed."
    )

# ------------------------------------------------------------
# 8. Paired model comparisons
# ------------------------------------------------------------

COMPARISONS_25F = {
    "bid_vs_full32": (
        "B_bid_age",
        "A_full32"
    ),
    "ask_vs_full32": (
        "C_ask_age",
        "A_full32"
    ),
    "both_vs_full32": (
        "D_both_ages",
        "A_full32"
    ),
    "both_vs_bid": (
        "D_both_ages",
        "B_bid_age"
    ),
    "both_vs_ask": (
        "D_both_ages",
        "C_ask_age"
    )
}

summary = []
daily_records = []

for h, block in calendar_exp25f_daily.groupby(
    "horizon"
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (
        candidate, baseline
    ) in COMPARISONS_25F.items():

        gains = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in gains.items():

            daily_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(gain)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gains),

            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),

            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),

            "winning_days": int(
                (gains > 0).sum()
            ),

            "median_daily_gain_pct": float(
                gains.median()
            ),

            "worst_day_pct": float(
                gains.min()
            )
        })

calendar_exp25f_summary = pd.DataFrame(summary)

calendar_exp25f_daily_gains = pd.DataFrame(
    daily_records
)

# ------------------------------------------------------------
# 9. Results
# ------------------------------------------------------------

print("\n===== 25F FULL32 ABSRET RESULTS =====")

print(
    calendar_exp25f_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 25F DAILY BOTH-AGE GAINS =====")

print(
    calendar_exp25f_daily_gains[
        calendar_exp25f_daily_gains[
            "comparison"
        ].eq("both_vs_full32")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== 25F INDIVIDUAL AGE IMPORTANCE =====")

if not calendar_exp25f_importance.empty:

    print(
        calendar_exp25f_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 25F COMPLETE =====")
print("No changes to original df_1s.")
print("Full 32-feature baseline preserved.")
print("No new feature promoted.")


===== 25F JOIN DIAGNOSTICS =====
Research rows: 459122
Core match %: 100.0
Retreat match %: 100.0

===== 25F MODEL SAMPLE =====
Total rows: 459122
Complete rows: 402979
Coverage %: 87.7717
Core baseline features: 32
Baseline including controls: 42

===== 25F SESSION COVERAGE =====
                             sum    mean
symbol         date                     
USDINR26OCTFUT 2026-09-30  24790  0.8623
               2026-10-01  25406  0.8945
               2026-10-05  24006  0.8345
               2026-10-06  24892  0.8657
               2026-10-07  27113  0.9430
               2026-10-08  24678  0.8584
               2026-10-09  23427  0.8149
               2026-10-12  27524  0.9572
               2026-10-13  26231  0.9110
               2026-10-14  27070  0.9402
               2026-10-15  27429  0.9525
               2026-10-16  24102  0.8370
               2026-10-19  24339  0.8456
USDINR26SEPFUT 2026-09-23  19915  0.7003
               2026-09-24  25021  0.8746
               2026-0

In [79]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 25G — FULL 32-FEATURE DIRECTIONAL ABLATION
#
# Target: signed future mid-price returns, 30/60/300s
# A: 32 existing features + common activity controls
# B: A + quote-age asymmetry
# C: A + individual bid/ask quote ages
#
# Historical expanding walk-forward
# Frozen LightGBM parameters
# No changes to df_1s
# ============================================================

for name in [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp25a_features"
]:
    if name not in globals():
        raise ValueError(f"Missing {name}")

HORIZONS_25G = [30, 60, 300]
KEYS_25G = ["symbol", "ts_sec"]

core_features_25g = [
    c for c in core_31.columns
    if c not in KEYS_25G
]

assert len(core_features_25g) == 31
assert len(set(core_features_25g)) == 31

RETREAT_25G = "quote_retreat_imbalance_30_24a"
ASYM_25G = "quote_age_asymmetry_25a"

INDIVIDUAL_25G = [
    "log_bid_quote_age_25a",
    "log_ask_quote_age_25a"
]

TARGETS_25G = [
    f"future_return_{h}_25a"
    for h in HORIZONS_25G
]

CONTROLS_25G = [
    "clock_seconds",
    "spread_bps_25a",
    "momentum_30_25a",
    "momentum_60_25a",
    "quote_price_change_rate_60_25a",
    "quote_price_change_rate_300_25a"
]

# ------------------------------------------------------------
# 1. Prepare existing research tables
# ------------------------------------------------------------

left = core_31[
    KEYS_25G + core_features_25g
].copy()

retreat = calendar_exp24a_features[
    KEYS_25G + [RETREAT_25G]
].copy()

age_cols = list(dict.fromkeys(
    KEYS_25G
    + ["date", "mid"]
    + CONTROLS_25G
    + [ASYM_25G]
    + INDIVIDUAL_25G
    + TARGETS_25G
))

ages = calendar_exp25a_features[
    age_cols
].copy()

if "update_rate_300s" in ages.columns:
    CONTROLS_25G.append("update_rate_300s")

for frame in [left, retreat, ages]:
    frame["symbol"] = frame["symbol"].astype(str)
    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"], errors="raise"
    ).astype("int64")

    if frame.duplicated(KEYS_25G).any():
        raise ValueError("Duplicate source join keys")

x = ages.merge(
    retreat,
    on=KEYS_25G,
    how="left",
    validate="one_to_one",
    indicator="_retreat_match"
).merge(
    left,
    on=KEYS_25G,
    how="left",
    validate="one_to_one",
    indicator="_core_match"
)

print("===== EXPERIMENT 25G JOIN =====")
print("Rows:", len(x))
print(
    "Retreat match:",
    round(100*x["_retreat_match"].eq("both").mean(), 3),
    "%"
)
print(
    "Core match:",
    round(100*x["_core_match"].eq("both").mean(), 3),
    "%"
)

if (
    x["_retreat_match"].eq("both").mean() < 0.99
    or x["_core_match"].eq("both").mean() < 0.99
):
    raise ValueError("Insufficient source data overlap")

x = x.drop(
    columns=["_retreat_match", "_core_match"]
)

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Add causal volatility and clock controls
# ------------------------------------------------------------

g = x.groupby(["symbol", "date"], sort=False)

prev_mid = g["mid"].shift(1)
prev_ts = g["ts_sec"].shift(1)

r1 = np.log(x["mid"] / prev_mid).where(
    (x["ts_sec"] - prev_ts).eq(1)
    & x["mid"].gt(0)
    & prev_mid.gt(0)
)

x["_ret_sq_25g"] = r1 ** 2

for w in [60, 300]:
    name = f"rv_{w}_25g"

    x[name] = (
        x.groupby(
            ["symbol", "date"], sort=False
        )["_ret_sq_25g"]
        .transform(
            lambda s: s.rolling(
                w, min_periods=int(0.9*w)
            ).sum()
        )
        .clip(lower=0)
        .pow(0.5) * 1e4
    )

    CONTROLS_25G.append(name)

x["open_first30_25g"] = (
    x["clock_seconds"] < 30
).astype(float)

x["open_1_5m_25g"] = (
    x["clock_seconds"].ge(60)
    & x["clock_seconds"].lt(300)
).astype(float)

CONTROLS_25G += [
    "open_first30_25g",
    "open_1_5m_25g"
]

# ------------------------------------------------------------
# 3. Model sets
# ------------------------------------------------------------

BASE_25G = list(dict.fromkeys(
    core_features_25g
    + [RETREAT_25G]
    + CONTROLS_25G
))

MODELS_25G = {
    "A_full32": BASE_25G,
    "B_asymmetry": BASE_25G + [ASYM_25G],
    "C_individual": BASE_25G + INDIVIDUAL_25G
}

ALL_25G = list(dict.fromkeys(
    c for cols in MODELS_25G.values()
    for c in cols
))

missing = [
    c for c in ALL_25G + TARGETS_25G
    if c not in x.columns
]

if missing:
    raise ValueError(f"Missing features: {missing}")

for col in ALL_25G + TARGETS_25G:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common_25g"] = np.isfinite(
    x[ALL_25G].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 25G SAMPLE =====")
print("Core feature count:", 32)
print("Baseline incl. controls:", len(BASE_25G))
print(
    "Finite-input coverage:",
    round(100*x["_common_25g"].mean(), 3),
    "%"
)

# ------------------------------------------------------------
# 4. Frozen nonlinear walk-forward evaluation
# ------------------------------------------------------------

PARAMS_25G = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

dates = sorted(x["date"].unique())

if len(dates) <= 5:
    raise ValueError("Insufficient historical sessions")

records = []

for i in range(5, len(dates)):

    date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_25g"]
        & x["ts_sec"].mod(10).eq(0)
    ]

    test_pool = x[
        x["date"].eq(date)
        & x["_common_25g"]
    ]

    for h in HORIZONS_25G:

        target = f"future_return_{h}_25a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]
        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for name, columns in MODELS_25G.items():

            model = lgb.LGBMRegressor(
                **PARAMS_25G
            )

            model.fit(
                train[columns],
                y_train
            )

            prediction = model.predict(
                test[columns]
            )

            error = y_test - prediction

            records.append({
                "date": date,
                "horizon": h,
                "model": name,
                "rows": len(test),
                "sse": float(np.sum(error**2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error**2)),
                "mae": float(np.mean(np.abs(error)))
            })

    print("Completed:", date)

calendar_exp25g_daily = pd.DataFrame(records)

if calendar_exp25g_daily.empty:
    raise ValueError("No evaluations completed")

# ------------------------------------------------------------
# 5. Pooled and day-level comparisons
# ------------------------------------------------------------

COMPARISONS_25G = {
    "asymmetry_vs_full32": (
        "B_asymmetry", "A_full32"
    ),
    "individual_vs_full32": (
        "C_individual", "A_full32"
    ),
    "asymmetry_vs_individual": (
        "B_asymmetry", "C_individual"
    )
}

summary = []
daily_rows = []

for h, block in calendar_exp25g_daily.groupby("horizon"):

    pooled = block.groupby("model")[["sse", "sae"]].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for label, (candidate, baseline) in (
        COMPARISONS_25G.items()
    ):

        gains = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in gains.items():
            daily_rows.append({
                "date": date,
                "horizon": h,
                "comparison": label,
                "gain_pct": float(gain)
            })

        summary.append({
            "horizon": h,
            "comparison": label,
            "days": len(gains),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gains > 0).sum()),
            "median_daily_gain_pct": float(gains.median()),
            "worst_day_pct": float(gains.min())
        })

calendar_exp25g_summary = pd.DataFrame(summary)
calendar_exp25g_daily_gains = pd.DataFrame(daily_rows)

print("\n===== 25G FULL32 DIRECTIONAL RESULTS =====")

print(
    calendar_exp25g_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 25G DAILY ASYMMETRY GAINS =====")

print(
    calendar_exp25g_daily_gains[
        calendar_exp25g_daily_gains["comparison"]
        .eq("asymmetry_vs_full32")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== EXPERIMENT 25G COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== EXPERIMENT 25G JOIN =====
Rows: 459122
Retreat match: 100.0 %
Core match: 100.0 %

===== 25G SAMPLE =====
Core feature count: 32
Baseline incl. controls: 42
Finite-input coverage: 87.772 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 25G FULL32 DIRECTIONAL RESULTS =====
 horizon              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30     asymmetry_vs_full32    11             0.137328             0.104409             7               0.061670      -0.240997
      30    individual_vs_full32    11             0.033547             0.061208             5              -0.027329      -0.577372
      30 asymmetry_vs_individual    11             0.103815             0.043227             7               0.015688   

In [80]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 26A — DEPTH CONCENTRATION
#
# Standalone research notebook experiment.
#
# Candidates:
#   depth_concentration_asymmetry_26a
#   touch_concentration_26a
#
# Controls:
#   bid_l1_concentration_26a
#   ask_l1_concentration_26a
#   queue_imbalance_l1_26a
#   depth_imbalance_l5_26a
#   log_total_depth_l5_26a
#   spread_bps_26a
#   momentum_30_26a
#   momentum_60_26a
#
# Future targets:
#   signed return: 30/60/300 seconds
#   absolute return: 30/60/300 seconds
#
# 16 historical eligible sessions expected.
# No lookahead; exact timestamp forward returns.
# No modifications to df_1s.
# No feature promotion.
# ============================================================

HORIZONS_26A = [30, 60, 300]
MIN_SESSION_COVERAGE_26A = 0.97

BID_Q_COLS_26A = [
    f"bid_q{i}" for i in range(1, 6)
]

ASK_Q_COLS_26A = [
    f"ask_q{i}" for i in range(1, 6)
]

REQUIRED_26A = [
    "symbol",
    "ts_sec",
    "bid_p1",
    "ask_p1"
] + BID_Q_COLS_26A + ASK_Q_COLS_26A

missing_26a = sorted(
    set(REQUIRED_26A) - set(df_1s.columns)
)

if missing_26a:
    raise ValueError(
        f"Missing required columns: {missing_26a}"
    )

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate DataFrame columns.")

# ------------------------------------------------------------
# 1. Create independent research DataFrame
# ------------------------------------------------------------

x26 = df_1s[REQUIRED_26A].copy()

numeric_26a = [
    c for c in REQUIRED_26A
    if c != "symbol"
]

for c in numeric_26a:

    x26[c] = pd.to_numeric(
        x26[c],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

if x26["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps.")

if x26.duplicated(
    ["symbol", "ts_sec"]
).any():
    raise ValueError(
        "Duplicate symbol/timestamp observations."
    )

# ------------------------------------------------------------
# 2. IST trading dates and session filtering
# ------------------------------------------------------------

ts_ist_26a = pd.to_datetime(
    x26["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata")

x26["date"] = ts_ist_26a.dt.date

x26["clock_seconds"] = (
    ts_ist_26a.dt.hour * 3600
    + ts_ist_26a.dt.minute * 60
    + ts_ist_26a.dt.second
    - 32400
)

# Indian market research window:
# 09:00:00 through 16:59:59 IST

x26 = x26.loc[
    x26["clock_seconds"].ge(0)
    & x26["clock_seconds"].lt(28800)
].copy()

x26 = x26.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage_26a = (
    x26.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "nunique")
    )
    .reset_index()
)

eligible_26a = coverage_26a.loc[
    coverage_26a["rows"].ge(
        MIN_SESSION_COVERAGE_26A * 28800
    )
    & coverage_26a["date"].astype(str).ne(
        "2026-09-28"
    ),
    ["symbol", "date"]
]

x26 = x26.merge(
    eligible_26a,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

if x26.empty:
    raise ValueError("No eligible sessions.")

GROUP_26A = ["symbol", "date"]

print("===== 26A DATA SUMMARY =====")
print("Eligible sessions:", len(eligible_26a))
print("Rows:", len(x26))

# ------------------------------------------------------------
# 3. Quote and depth validation
#
# Require:
#   valid crossed-book check
#   all 10 quantity levels finite and >= 0
#   positive total depth on BOTH sides
#
# Undefined concentration is NaN, not zero.
# ------------------------------------------------------------

bid_p_26a = x26["bid_p1"]
ask_p_26a = x26["ask_p1"]

valid_prices_26a = (
    bid_p_26a.gt(0)
    & ask_p_26a.gt(0)
    & ask_p_26a.ge(bid_p_26a)
    & np.isfinite(bid_p_26a)
    & np.isfinite(ask_p_26a)
)

depth_cols_26a = (
    BID_Q_COLS_26A + ASK_Q_COLS_26A
)

valid_depth_26a = (
    x26[depth_cols_26a].notna().all(axis=1)
    & x26[depth_cols_26a].ge(0).all(axis=1)
)

x26["_bid_depth"] = (
    x26[BID_Q_COLS_26A]
    .sum(axis=1, min_count=5)
)

x26["_ask_depth"] = (
    x26[ASK_Q_COLS_26A]
    .sum(axis=1, min_count=5)
)

valid_book_26a = (
    valid_prices_26a
    & valid_depth_26a
    & x26["_bid_depth"].gt(0)
    & x26["_ask_depth"].gt(0)
)

x26["mid_26a"] = (
    (bid_p_26a + ask_p_26a) / 2
).where(valid_prices_26a)

x26["spread_bps_26a"] = (
    1e4 * (
        ask_p_26a - bid_p_26a
    ) / x26["mid_26a"]
).where(valid_prices_26a)

# ------------------------------------------------------------
# 4. Side-specific depth concentration
# ------------------------------------------------------------

x26["bid_l1_concentration_26a"] = (
    x26["bid_q1"] / x26["_bid_depth"]
).where(valid_book_26a)

x26["ask_l1_concentration_26a"] = (
    x26["ask_q1"] / x26["_ask_depth"]
).where(valid_book_26a)

# ------------------------------------------------------------
# 5. Main depth concentration candidates
# ------------------------------------------------------------

ASYM_26A = "depth_concentration_asymmetry_26a"
TOUCH_26A = "touch_concentration_26a"

x26[ASYM_26A] = (
    x26["bid_l1_concentration_26a"]
    - x26["ask_l1_concentration_26a"]
)

total_depth = (
    x26["_bid_depth"] + x26["_ask_depth"]
)

x26[TOUCH_26A] = (
    (x26["bid_q1"] + x26["ask_q1"])
    / total_depth
).where(valid_book_26a)

CANDIDATES_26A = [
    ASYM_26A,
    TOUCH_26A
]

# ------------------------------------------------------------
# 6. Existing-information controls
# ------------------------------------------------------------

l1_total = x26["bid_q1"] + x26["ask_q1"]

x26["queue_imbalance_l1_26a"] = (
    (x26["bid_q1"] - x26["ask_q1"])
    / l1_total.where(l1_total > 0)
).where(valid_book_26a)

x26["depth_imbalance_l5_26a"] = (
    (x26["_bid_depth"] - x26["_ask_depth"])
    / total_depth
).where(valid_book_26a)

x26["log_total_depth_l5_26a"] = (
    np.log1p(total_depth)
).where(valid_book_26a)

CONTROLS_26A = [
    "bid_l1_concentration_26a",
    "ask_l1_concentration_26a",
    "queue_imbalance_l1_26a",
    "depth_imbalance_l5_26a",
    "log_total_depth_l5_26a",
    "spread_bps_26a"
]

# ------------------------------------------------------------
# 7. Historical 30s and 60s momentum controls
#
# Require exact W-second endpoint difference.
# ------------------------------------------------------------

for w in [30, 60]:

    group = x26.groupby(
        GROUP_26A,
        sort=False
    )

    old_mid = group["mid_26a"].shift(w)
    old_ts = group["ts_sec"].shift(w)

    name = f"momentum_{w}_26a"

    valid_endpoint = (
        (x26["ts_sec"] - old_ts).eq(w)
        & old_mid.gt(0)
        & x26["mid_26a"].gt(0)
    )

    x26[name] = (
        1e4 * np.log(
            x26["mid_26a"] / old_mid
        )
    ).where(valid_endpoint)

    CONTROLS_26A.append(name)

# ------------------------------------------------------------
# 8. Exact future return targets
#
# No row shifting into gaps or another session.
# ------------------------------------------------------------

for h in HORIZONS_26A:

    future_26a = x26[
        ["symbol", "date", "ts_sec", "mid_26a"]
    ].copy()

    future_26a["ts_sec"] -= h

    future_26a = future_26a.rename(
        columns={
            "mid_26a": f"future_mid_{h}_26a"
        }
    )

    x26 = x26.merge(
        future_26a,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    signed = f"future_return_{h}_26a"
    absolute = f"future_absret_{h}_26a"

    x26[signed] = (
        1e4 * np.log(
            x26[f"future_mid_{h}_26a"]
            / x26["mid_26a"]
        )
    )

    x26[absolute] = x26[signed].abs()

# ------------------------------------------------------------
# 9. Daily Spearman information coefficient
# ------------------------------------------------------------

def safe_spearman_26a(a, b):

    mask = (
        np.isfinite(a)
        & np.isfinite(b)
    )

    n = int(mask.sum())

    if n < 200:
        return np.nan, n

    rank_a = (
        pd.Series(a[mask])
        .rank()
        .to_numpy()
    )

    rank_b = (
        pd.Series(b[mask])
        .rank()
        .to_numpy()
    )

    if (
        rank_a.std() < 1e-12
        or rank_b.std() < 1e-12
    ):
        return np.nan, n

    ic = np.corrcoef(
        rank_a, rank_b
    )[0, 1]

    return float(ic), n

FEATURES_26A = (
    CANDIDATES_26A + CONTROLS_26A
)

records_26a = []

for (symbol, date), day in x26.groupby(
    GROUP_26A,
    sort=False
):

    for h in HORIZONS_26A:

        target_names = {
            "signed_return":
                f"future_return_{h}_26a",
            "absret":
                f"future_absret_{h}_26a"
        }

        for task, target in target_names.items():

            y = day[target].to_numpy(
                dtype=float
            )

            for feature in FEATURES_26A:

                values = day[feature].to_numpy(
                    dtype=float
                )

                ic, n = safe_spearman_26a(
                    values, y
                )

                records_26a.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp26a_daily = pd.DataFrame(
    records_26a
)

calendar_exp26a_summary = (
    calendar_exp26a_daily
    .groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic",
            lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

calendar_exp26a_summary["sign_consistency"] = (
    calendar_exp26a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp26a_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 10. Spearman redundancy
# ------------------------------------------------------------

calendar_exp26a_redundancy = (
    x26[FEATURES_26A]
    .corr(method="spearman")
    .loc[
        CANDIDATES_26A,
        CONTROLS_26A
    ]
)

# ------------------------------------------------------------
# 11. Diagnostics
# ------------------------------------------------------------

print("\n===== 26A DEPTH CONCENTRATION SIGNED IC =====")

print(
    calendar_exp26a_summary.loc[
        calendar_exp26a_summary[
            "task"
        ].eq("signed_return")
        & calendar_exp26a_summary[
            "feature"
        ].isin(CANDIDATES_26A)
    ].round(6).to_string(index=False)
)

print("\n===== 26A DEPTH CONCENTRATION ABSRET IC =====")

print(
    calendar_exp26a_summary.loc[
        calendar_exp26a_summary[
            "task"
        ].eq("absret")
        & calendar_exp26a_summary[
            "feature"
        ].isin(CANDIDATES_26A)
    ].round(6).to_string(index=False)
)

print("\n===== 26A DEPTH / MOMENTUM CONTROL IC =====")

print(
    calendar_exp26a_summary.loc[
        calendar_exp26a_summary[
            "feature"
        ].isin([
            "queue_imbalance_l1_26a",
            "depth_imbalance_l5_26a",
            "log_total_depth_l5_26a",
            "momentum_30_26a",
            "momentum_60_26a"
        ])
    ].round(6).to_string(index=False)
)

print("\n===== 26A DEPTH CONCENTRATION REDUNDANCY =====")

print(
    calendar_exp26a_redundancy
    .round(5)
    .to_string()
)

print("\n===== 26A FEATURE DIAGNOSTICS =====")

diagnostic_cols = (
    CANDIDATES_26A
    + [
        "bid_l1_concentration_26a",
        "ask_l1_concentration_26a"
    ]
)

for name in diagnostic_cols:

    s = x26[name]

    print(
        name,
        "valid_pct:",
        round(100 * s.notna().mean(), 4),
        "mean:",
        round(s.mean(), 6),
        "std:",
        round(s.std(), 6),
        "p05:",
        round(s.quantile(0.05), 6),
        "median:",
        round(s.median(), 6),
        "p95:",
        round(s.quantile(0.95), 6)
    )

print("\n===== 26A BOOK VALIDITY =====")

print(
    "Valid full five-level books (%):",
    round(
        100 * valid_book_26a.mean(), 4
    )
)

print(
    "Zero total bid depth:",
    int(x26["_bid_depth"].eq(0).sum())
)

print(
    "Zero total ask depth:",
    int(x26["_ask_depth"].eq(0).sum())
)

# ------------------------------------------------------------
# 12. Optional conditional comparison
#
# Compare mean future signed returns across
# concentration-asymmetry quintiles.
#
# These are descriptive associations only.
# ------------------------------------------------------------

cond_26a = x26[
    [
        "date",
        ASYM_26A,
        "future_return_30_26a",
        "future_return_60_26a"
    ]
].dropna().copy()

if cond_26a[ASYM_26A].nunique() >= 5:

    # Rank first to handle many duplicate values.
    cond_26a["asym_quintile"] = (
        pd.qcut(
            cond_26a[ASYM_26A].rank(
                method="first"
            ),
            q=5,
            labels=[
                "Q1_low", "Q2",
                "Q3", "Q4", "Q5_high"
            ]
        )
    )

    calendar_exp26a_quintiles = (
        cond_26a.groupby(
            "asym_quintile",
            observed=True
        )
        .agg(
            rows=(ASYM_26A, "size"),
            mean_future_30_bps=(
                "future_return_30_26a",
                "mean"
            ),
            mean_future_60_bps=(
                "future_return_60_26a",
                "mean"
            )
        )
        .reset_index()
    )

    print(
        "\n===== 26A ASYMMETRY QUINTILE RETURNS ====="
    )

    print(
        calendar_exp26a_quintiles
        .round(6)
        .to_string(index=False)
    )

calendar_exp26a_features = x26

print("\n===== EXPERIMENT 26A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 26A DATA SUMMARY =====
Eligible sessions: 16
Rows: 459122

===== 26A DEPTH CONCENTRATION SIGNED IC =====
         task  horizon                           feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
signed_return       30 depth_concentration_asymmetry_26a    16 0.100194   0.099022             16              0            28723.0            1.0000
signed_return       30           touch_concentration_26a    16 0.007313   0.001983              8              8            28723.0            0.5000
signed_return       60 depth_concentration_asymmetry_26a    16 0.099264   0.092847             16              0            28693.0            1.0000
signed_return       60           touch_concentration_26a    16 0.001707   0.003233              9              7            28693.0            0.5625
signed_return      300 depth_concentration_asymmetry_26a    16 0.066947   0.054765             15              1            28453.0        

In [81]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 26B — FULL BASELINE DEPTH CONCENTRATION
#
# Target: future signed returns at 30/60/300s
#
# A: Existing 32 features + common controls
# B: A + concentration asymmetry
# C: A + separate bid and ask L1 concentration
#
# Expanding chronological LightGBM
# Fixed parameters / matched samples
# Research notebook only
# ============================================================

for name in [
    "calendar_exp26a_features",
    "calendar_exp24a_features",
    "core_31"
]:
    if name not in globals():
        raise ValueError(f"Missing {name}")

HORIZONS_26B = [30, 60, 300]
KEYS_26B = ["symbol", "ts_sec"]

CORE_26B = [
    c for c in core_31.columns
    if c not in KEYS_26B
]

if len(CORE_26B) != 31 or len(set(CORE_26B)) != 31:
    raise ValueError("Expected exactly 31 distinct core features.")

RETREAT_26B = "quote_retreat_imbalance_30_24a"
ASYM_26B = "depth_concentration_asymmetry_26a"

SIDE_26B = [
    "bid_l1_concentration_26a",
    "ask_l1_concentration_26a"
]

TARGETS_26B = [
    f"future_return_{h}_26a"
    for h in HORIZONS_26B
]

CONTROL_26B = [
    "queue_imbalance_l1_26a",
    "depth_imbalance_l5_26a",
    "log_total_depth_l5_26a",
    "spread_bps_26a",
    "momentum_30_26a",
    "momentum_60_26a",
    "clock_seconds"
]

# ------------------------------------------------------------
# 1. Build research frame from exact timestamp joins
# ------------------------------------------------------------

cols_26b = list(dict.fromkeys(
    KEYS_26B
    + ["date"]
    + CONTROL_26B
    + [ASYM_26B]
    + SIDE_26B
    + TARGETS_26B
))

x = calendar_exp26a_features[cols_26b].copy()

retreat = calendar_exp24a_features[
    KEYS_26B + [RETREAT_26B]
].copy()

core = core_31[
    KEYS_26B + CORE_26B
].copy()

for frame in [x, retreat, core]:
    frame["symbol"] = frame["symbol"].astype(str)
    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"],
        errors="raise"
    ).astype("int64")

    if frame.duplicated(KEYS_26B).any():
        raise ValueError("Duplicate symbol/timestamp keys.")

x = x.merge(
    retreat,
    on=KEYS_26B,
    how="left",
    validate="one_to_one",
    indicator="_retreat_match"
)

x = x.merge(
    core,
    on=KEYS_26B,
    how="left",
    validate="one_to_one",
    indicator="_core_match"
)

print("===== 26B JOIN DIAGNOSTICS =====")
print("Research rows:", len(x))
print(
    "Core match (%):",
    round(
        100 * x["_core_match"].eq("both").mean(),
        4
    )
)
print(
    "Retreat match (%):",
    round(
        100 * x["_retreat_match"].eq("both").mean(),
        4
    )
)

if (
    x["_core_match"].eq("both").mean() < 0.99
    or x["_retreat_match"].eq("both").mean() < 0.99
):
    raise ValueError("Insufficient source alignment.")

x = x.drop(
    columns=["_retreat_match", "_core_match"]
)

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Freeze model features
#
# Both existing imbalance features are already in
# the core 31. The reconstructed controls are also
# included to keep comparisons explicit.
# ------------------------------------------------------------

BASE_26B = list(dict.fromkeys(
    CORE_26B
    + [RETREAT_26B]
    + CONTROL_26B
))

MODELS_26B = {
    "A_full32": BASE_26B,
    "B_asymmetry": BASE_26B + [ASYM_26B],
    "C_side_concentrations": BASE_26B + SIDE_26B
}

ALL_FEATURES_26B = list(dict.fromkeys(
    c for cols in MODELS_26B.values()
    for c in cols
))

missing = sorted(
    set(ALL_FEATURES_26B + TARGETS_26B)
    - set(x.columns)
)

if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in ALL_FEATURES_26B + TARGETS_26B:
    x[col] = pd.to_numeric(
        x[col],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Strict matched sample across all models.
x["_common_26b"] = np.isfinite(
    x[ALL_FEATURES_26B].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 26B MODEL COVERAGE =====")
print("Full baseline columns:", len(BASE_26B))
print("Complete rows:", int(x["_common_26b"].sum()))
print(
    "Coverage (%):",
    round(100*x["_common_26b"].mean(), 4)
)

# ------------------------------------------------------------
# 3. Fixed LightGBM configuration
# ------------------------------------------------------------

PARAMS_26B = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

# ------------------------------------------------------------
# 4. Expanding walk-forward evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= 5:
    raise ValueError("Insufficient sessions.")

records = []
importance_records = []

for i in range(5, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common_26b"]
        & x["ts_sec"].mod(10).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common_26b"]
    ]

    for h in HORIZONS_26B:

        target = f"future_return_{h}_26a"

        train = train_pool[
            np.isfinite(train_pool[target])
        ]

        test = test_pool[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for model_name, features in MODELS_26B.items():

            model = lgb.LGBMRegressor(
                **PARAMS_26B
            )

            model.fit(
                train[features],
                y_train
            )

            pred = model.predict(
                test[features]
            )

            error = y_test - pred

            records.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

            importance = (
                model.booster_.feature_importance(
                    importance_type="gain"
                )
            )

            for feature, value in zip(
                features, importance
            ):
                if feature in [ASYM_26B] + SIDE_26B:
                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": model_name,
                        "feature": feature,
                        "gain_importance": float(value)
                    })

    print("Completed:", test_date)

calendar_exp26b_daily = pd.DataFrame(records)
calendar_exp26b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp26b_daily.empty:
    raise ValueError("No evaluations completed.")

# ------------------------------------------------------------
# 5. Pooled and daily comparisons
# ------------------------------------------------------------

COMPARISONS_26B = {
    "asymmetry_vs_full32": (
        "B_asymmetry", "A_full32"
    ),
    "sides_vs_full32": (
        "C_side_concentrations", "A_full32"
    ),
    "asymmetry_vs_sides": (
        "B_asymmetry", "C_side_concentrations"
    )
}

summary = []
daily_records = []

for h, block in calendar_exp26b_daily.groupby(
    "horizon"
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS_26B.items()
    ):

        gains = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, value in gains.items():
            daily_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "gain_pct": float(value)
            })

        summary.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(gains),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gains > 0).sum()),
            "median_daily_gain_pct": float(
                gains.median()
            ),
            "worst_day_pct": float(gains.min())
        })

calendar_exp26b_summary = pd.DataFrame(summary)

calendar_exp26b_daily_gains = pd.DataFrame(
    daily_records
)

# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

print("\n===== 26B FULL32 DIRECTIONAL RESULTS =====")

print(
    calendar_exp26b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 26B DAILY ASYMMETRY GAINS =====")

print(
    calendar_exp26b_daily_gains[
        calendar_exp26b_daily_gains[
            "comparison"
        ].eq("asymmetry_vs_full32")
    ].pivot(
        index="date",
        columns="horizon",
        values="gain_pct"
    ).round(6).to_string()
)

print("\n===== 26B DEPTH CONCENTRATION IMPORTANCE =====")

if not calendar_exp26b_importance.empty:
    print(
        calendar_exp26b_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 26B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 26B JOIN DIAGNOSTICS =====
Research rows: 459122
Core match (%): 100.0
Retreat match (%): 100.0

===== 26B MODEL COVERAGE =====
Full baseline columns: 39
Complete rows: 402979
Coverage (%): 87.7717
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 26B FULL32 DIRECTIONAL RESULTS =====
 horizon          comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30 asymmetry_vs_full32    11            -0.066329            -0.047783             4              -0.118336      -0.816839
      30     sides_vs_full32    11            -0.076631            -0.087899             5              -0.053442      -0.369085
      30  asymmetry_vs_sides    11             0.010294             0.040081             5              -0.067618      -0.4

In [82]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 27A — LIQUIDITY RESILIENCE
#
# Standalone research experiment.
# Source: df_1s in the CALENDAR notebook.
#
# Windows: 60s, 300s
# Targets: future signed and absolute returns, 30/60/300s
#
# Candidate families:
#   recovery_pressure
#   magnitude_reduction
#   directional_imbalance_change
#
# Diagnostics:
#   Daily Spearman IC
#   Sign consistency
#   Spearman redundancy
#   Feature distributions
#   Overshoot rate
#   Recovery conditional on significant initial imbalance
#
# No future information in features.
# No modifications to df_1s.
# No feature promotion.
# ============================================================

WINDOWS_27A = [60, 300]
HORIZONS_27A = [30, 60, 300]
MIN_COVERAGE_27A = 0.97

BID_Q_27A = [f"bid_q{i}" for i in range(1, 6)]
ASK_Q_27A = [f"ask_q{i}" for i in range(1, 6)]

REQUIRED_27A = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
] + BID_Q_27A + ASK_Q_27A

missing = sorted(set(REQUIRED_27A) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing raw columns: {missing}")

# ------------------------------------------------------------
# 1. Copy raw data and optional existing controls
# ------------------------------------------------------------

OPTIONAL_27A = [
    "queue_replenishment_pressure_60s",
    "recovery_available_300",
    "depth_imbalance_delta",
    "queue_imbalance_delta",
    "depth_imbalance_l5",
    "queue_imbalance_l1",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "update_rate_300s",
    "ofi_l1_sum_60s_signed_log"
]

available_optional = [
    c for c in OPTIONAL_27A if c in df_1s.columns
]

source_cols = list(dict.fromkeys(
    REQUIRED_27A + available_optional
))

x27 = df_1s[source_cols].copy()

numeric_cols = [
    c for c in source_cols if c != "symbol"
]

for c in numeric_cols:
    x27[c] = pd.to_numeric(
        x27[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

if x27["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

if x27.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp keys")

# ------------------------------------------------------------
# 2. IST sessions: 09:00:00–16:59:59
# ------------------------------------------------------------

ist = pd.to_datetime(
    x27["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x27["date"] = ist.dt.date

x27["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x27 = x27.loc[
    x27["clock_seconds"].between(0, 28799)
].copy()

x27 = x27.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x27.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage.loc[
    coverage["rows"].ge(28800 * MIN_COVERAGE_27A)
    & coverage["date"].astype(str).ne("2026-09-28"),
    ["symbol", "date"]
]

x27 = x27.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

if x27.empty:
    raise ValueError("No eligible sessions")

GROUP_27A = ["symbol", "date"]

print("===== 27A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x27))
print("Existing controls found:", available_optional)

# ------------------------------------------------------------
# 3. Reconstruct five-level bid/ask depth
# ------------------------------------------------------------

bid = x27["bid_p1"]
ask = x27["ask_p1"]

valid_prices = (
    bid.gt(0)
    & ask.gt(0)
    & ask.ge(bid)
    & np.isfinite(bid)
    & np.isfinite(ask)
)

quantity_cols = BID_Q_27A + ASK_Q_27A

valid_quantities = (
    x27[quantity_cols].notna().all(axis=1)
    & x27[quantity_cols].ge(0).all(axis=1)
)

x27["bid_depth_27a"] = (
    x27[BID_Q_27A].sum(axis=1, min_count=5)
)

x27["ask_depth_27a"] = (
    x27[ASK_Q_27A].sum(axis=1, min_count=5)
)

db = x27["bid_depth_27a"]
da = x27["ask_depth_27a"]

valid_book = (
    valid_prices
    & valid_quantities
    & db.gt(0)
    & da.gt(0)
)

total = db + da

x27["imbalance_27a"] = (
    (db - da) / total
).where(valid_book)

x27["log_bid_depth_27a"] = (
    np.log1p(db)
).where(valid_book)

x27["log_ask_depth_27a"] = (
    np.log1p(da)
).where(valid_book)

x27["mid_27a"] = (
    (bid + ask) / 2
).where(valid_prices)

x27["spread_bps_27a"] = (
    1e4 * (ask - bid) / x27["mid_27a"]
).where(valid_prices)

# ------------------------------------------------------------
# 4. Lagged depth imbalance and depth changes
#
# Valid only if lag endpoints are exactly W seconds
# apart in the same symbol and IST trading date.
# ------------------------------------------------------------

CANDIDATES_27A = []
CONTROLS_27A = [
    "imbalance_27a",
    "log_bid_depth_27a",
    "log_ask_depth_27a",
    "spread_bps_27a"
]

g = x27.groupby(GROUP_27A, sort=False)

for w in WINDOWS_27A:

    old_ts = g["ts_sec"].shift(w)
    exact = (
        (x27["ts_sec"] - old_ts).eq(w)
    )

    old_imb = g["imbalance_27a"].shift(w)
    old_bid = g["log_bid_depth_27a"].shift(w)
    old_ask = g["log_ask_depth_27a"].shift(w)

    now_imb = x27["imbalance_27a"]

    valid_pair = (
        exact
        & now_imb.notna()
        & old_imb.notna()
        & old_bid.notna()
        & old_ask.notna()
    )

    prior = f"prior_imbalance_{w}_27a"
    delta = f"imbalance_change_{w}_27a"
    recovery = f"recovery_pressure_{w}_27a"
    reduction = f"magnitude_reduction_{w}_27a"
    bid_delta = f"log_bid_depth_change_{w}_27a"
    ask_delta = f"log_ask_depth_change_{w}_27a"
    overshoot = f"overshoot_{w}_27a"

    x27[prior] = old_imb.where(valid_pair)

    x27[delta] = (
        now_imb - old_imb
    ).where(valid_pair)

    x27[recovery] = (
        -old_imb * (now_imb - old_imb)
    ).where(valid_pair)

    x27[reduction] = (
        old_imb.abs() - now_imb.abs()
    ).where(valid_pair)

    x27[bid_delta] = (
        x27["log_bid_depth_27a"] - old_bid
    ).where(valid_pair)

    x27[ask_delta] = (
        x27["log_ask_depth_27a"] - old_ask
    ).where(valid_pair)

    # Detect crossing from one imbalance sign
    # to the other, rather than calling all such
    # observations ordinary recovery.
    x27[overshoot] = (
        (old_imb * now_imb < 0).astype(float)
    ).where(valid_pair)

    CANDIDATES_27A.extend([
        recovery, reduction, delta
    ])

    CONTROLS_27A.extend([
        prior, bid_delta, ask_delta, overshoot
    ])

# ------------------------------------------------------------
# 5. Recent price momentum controls
# ------------------------------------------------------------

for w in [30, 60]:

    g = x27.groupby(GROUP_27A, sort=False)

    previous_mid = g["mid_27a"].shift(w)
    previous_ts = g["ts_sec"].shift(w)

    name = f"momentum_{w}_27a"

    x27[name] = (
        1e4 * np.log(
            x27["mid_27a"] / previous_mid
        )
    ).where(
        (x27["ts_sec"] - previous_ts).eq(w)
        & x27["mid_27a"].gt(0)
        & previous_mid.gt(0)
    )

    CONTROLS_27A.append(name)

CONTROLS_27A += available_optional
CONTROLS_27A = list(dict.fromkeys(CONTROLS_27A))

# ------------------------------------------------------------
# 6. Exact future signed and absolute returns
# ------------------------------------------------------------

for h in HORIZONS_27A:

    future = x27[
        ["symbol", "date", "ts_sec", "mid_27a"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={
            "mid_27a": f"future_mid_{h}_27a"
        }
    )

    x27 = x27.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    signed = f"future_return_{h}_27a"
    absolute = f"future_absret_{h}_27a"

    x27[signed] = (
        1e4 * np.log(
            x27[f"future_mid_{h}_27a"]
            / x27["mid_27a"]
        )
    )

    x27[absolute] = x27[signed].abs()

# ------------------------------------------------------------
# 7. Daily Spearman IC
# ------------------------------------------------------------

def safe_spearman_27a(a, b):

    mask = np.isfinite(a) & np.isfinite(b)
    n = int(mask.sum())

    if n < 200:
        return np.nan, n

    ra = pd.Series(a[mask]).rank().to_numpy()
    rb = pd.Series(b[mask]).rank().to_numpy()

    if ra.std() < 1e-12 or rb.std() < 1e-12:
        return np.nan, n

    return float(np.corrcoef(ra, rb)[0, 1]), n


all_features = list(dict.fromkeys(
    CANDIDATES_27A + CONTROLS_27A
))

records = []

for (symbol, date), day in x27.groupby(
    GROUP_27A, sort=False
):

    for h in HORIZONS_27A:

        targets = {
            "signed_return": f"future_return_{h}_27a",
            "absret": f"future_absret_{h}_27a"
        }

        for task, target in targets.items():

            y = day[target].to_numpy(dtype=float)

            for feature in all_features:

                z = day[feature].to_numpy(dtype=float)

                ic, n = safe_spearman_27a(z, y)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp27a_daily = pd.DataFrame(records)

calendar_exp27a_summary = (
    calendar_exp27a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=("valid_rows", "median")
    )
    .reset_index()
)

calendar_exp27a_summary["sign_consistency"] = (
    calendar_exp27a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp27a_summary["days"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 8. Redundancy: rank correlations
# ------------------------------------------------------------

calendar_exp27a_redundancy = (
    x27[all_features]
    .corr(method="spearman")
    .loc[CANDIDATES_27A, CONTROLS_27A]
)

# ------------------------------------------------------------
# 9. Conditional recovery diagnostics
#
# Avoid interpreting tiny initial imbalance
# as a meaningful liquidity shock.
# Use a fixed, descriptive threshold.
# ------------------------------------------------------------

SHOCK_THRESHOLD_27A = 0.30

shock_records = []

for w in WINDOWS_27A:

    prior = f"prior_imbalance_{w}_27a"
    reduction = f"magnitude_reduction_{w}_27a"
    recovery = f"recovery_pressure_{w}_27a"
    overshoot = f"overshoot_{w}_27a"

    sample = x27.loc[
        x27[prior].abs().ge(SHOCK_THRESHOLD_27A)
    ].copy()

    shock_records.append({
        "window": w,
        "shock_rows": len(sample),
        "shock_fraction": (
            len(sample) / len(x27)
        ),
        "magnitude_reduced_pct": (
            100 * sample[reduction].gt(0).mean()
        ),
        "recovery_pressure_positive_pct": (
            100 * sample[recovery].gt(0).mean()
        ),
        "overshoot_pct": (
            100 * sample[overshoot].eq(1).mean()
        )
    })

calendar_exp27a_shocks = pd.DataFrame(
    shock_records
)

# ------------------------------------------------------------
# 10. Outputs
# ------------------------------------------------------------

print("\n===== 27A RECOVERY SIGNED IC =====")

print(
    calendar_exp27a_summary.loc[
        calendar_exp27a_summary["task"].eq(
            "signed_return"
        )
        & calendar_exp27a_summary[
            "feature"
        ].isin(CANDIDATES_27A)
    ].round(6).to_string(index=False)
)

print("\n===== 27A RECOVERY ABSRET IC =====")

print(
    calendar_exp27a_summary.loc[
        calendar_exp27a_summary["task"].eq("absret")
        & calendar_exp27a_summary[
            "feature"
        ].isin(CANDIDATES_27A)
    ].round(6).to_string(index=False)
)

print("\n===== 27A REDUNDANCY =====")

print(
    calendar_exp27a_redundancy
    .round(5)
    .to_string()
)

print("\n===== 27A FEATURE DIAGNOSTICS =====")

for feature in CANDIDATES_27A:

    s = x27[feature]

    print(
        feature,
        "valid_pct:", round(100*s.notna().mean(), 3),
        "mean:", round(s.mean(), 6),
        "std:", round(s.std(), 6),
        "zero_pct:", round(100*s.eq(0).mean(), 3),
        "p05:", round(s.quantile(0.05), 6),
        "p95:", round(s.quantile(0.95), 6)
    )

print("\n===== 27A SHOCK RECOVERY DIAGNOSTICS =====")

print(
    calendar_exp27a_shocks
    .round(6)
    .to_string(index=False)
)

print("\n===== 27A EXISTING CONTROL IC =====")

existing_control_ic = calendar_exp27a_summary.loc[
    calendar_exp27a_summary["feature"].isin(
        available_optional
    )
]

print(
    existing_control_ic
    .round(6)
    .to_string(index=False)
)

calendar_exp27a_features = x27

print("\n===== EXPERIMENT 27A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 27A SAMPLE =====
Sessions: 16
Rows: 459122
Existing controls found: []

===== 27A RECOVERY SIGNED IC =====
         task  horizon                     feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
signed_return       30    imbalance_change_300_27a    16  0.086321   0.088776             16              0            28423.0            1.0000
signed_return       30     imbalance_change_60_27a    16  0.088982   0.089574             16              0            28663.0            1.0000
signed_return       30 magnitude_reduction_300_27a    16 -0.034089  -0.039472              2             14            28423.0            0.8750
signed_return       30  magnitude_reduction_60_27a    16 -0.023844  -0.023718              4             12            28663.0            0.7500
signed_return       30   recovery_pressure_300_27a    16 -0.036355  -0.044487              3             13            28423.0            0.8125
signed_return    

In [83]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 27B — FULL BASELINE IMBALANCE CHANGE
#
# Source tables:
#   calendar_exp27a_features
#   calendar_exp24a_features
#   core_31
#
# Targets: signed returns at 30, 60, 300 seconds
#
# A: full32 + component controls
# B: A + 60s imbalance change
# C: A + 300s imbalance change
# D: A + both imbalance changes
#
# Strict matched samples.
# Fixed expanding LightGBM.
# No modification to df_1s.
# ============================================================

for name in [
    "calendar_exp27a_features",
    "calendar_exp24a_features",
    "core_31"
]:
    if name not in globals():
        raise ValueError(f"Missing {name}")

KEYS = ["symbol", "ts_sec"]
HORIZONS = [30, 60, 300]

CORE = [
    c for c in core_31.columns
    if c not in KEYS
]

if len(CORE) != 31 or len(set(CORE)) != 31:
    raise ValueError("Expected 31 unique core features.")

RETREAT = "quote_retreat_imbalance_30_24a"

DELTA60 = "imbalance_change_60_27a"
DELTA300 = "imbalance_change_300_27a"

TARGETS = [
    f"future_return_{h}_27a"
    for h in HORIZONS
]

# ------------------------------------------------------------
# 1. Explicit component and liquidity controls
# ------------------------------------------------------------

COMPONENTS = [
    "imbalance_27a",
    "prior_imbalance_60_27a",
    "prior_imbalance_300_27a",
    "log_bid_depth_27a",
    "log_ask_depth_27a",
    "log_bid_depth_change_60_27a",
    "log_ask_depth_change_60_27a",
    "log_bid_depth_change_300_27a",
    "log_ask_depth_change_300_27a",
    "spread_bps_27a",
    "momentum_30_27a",
    "momentum_60_27a",
    "clock_seconds"
]

RESEARCH_COLS = list(dict.fromkeys(
    KEYS
    + ["date"]
    + COMPONENTS
    + [DELTA60, DELTA300]
    + TARGETS
))

research = calendar_exp27a_features[
    RESEARCH_COLS
].copy()

retreat = calendar_exp24a_features[
    KEYS + [RETREAT]
].copy()

core = core_31[
    KEYS + CORE
].copy()

# ------------------------------------------------------------
# 2. Align sources on exact symbol and timestamp
# ------------------------------------------------------------

for frame in [research, retreat, core]:

    frame["symbol"] = frame["symbol"].astype(str)

    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"], errors="raise"
    ).astype("int64")

    if frame.duplicated(KEYS).any():
        raise ValueError("Duplicate join keys.")

x = research.merge(
    retreat,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator="_retreat_match"
)

x = x.merge(
    core,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator="_core_match"
)

print("===== 27B SOURCE ALIGNMENT =====")

for c in ["_retreat_match", "_core_match"]:
    match = x[c].eq("both").mean()
    print(c, "matched %:", round(100 * match, 4))

    if match < 0.99:
        raise ValueError(
            f"Insufficient source alignment: {c}"
        )

x = x.drop(
    columns=["_retreat_match", "_core_match"]
)

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 3. Freeze feature sets
# ------------------------------------------------------------

BASE = list(dict.fromkeys(
    CORE + [RETREAT] + COMPONENTS
))

MODELS = {
    "A_full32_components": BASE,
    "B_delta60": BASE + [DELTA60],
    "C_delta300": BASE + [DELTA300],
    "D_both": BASE + [DELTA60, DELTA300]
}

ALL_FEATURES = list(dict.fromkeys(
    f for cols in MODELS.values() for f in cols
))

for c in ALL_FEATURES + TARGETS:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Matched finite sample across all four variants.
x["_common"] = np.isfinite(
    x[ALL_FEATURES].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 27B MODEL COVERAGE =====")
print("Total rows:", len(x))
print("Complete rows:", int(x["_common"].sum()))
print("Coverage %:", round(100*x["_common"].mean(), 4))
print("Baseline columns:", len(BASE))
print("Existing engineered features:", 32)

# ------------------------------------------------------------
# 4. Fixed LightGBM configuration
# ------------------------------------------------------------

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

# ------------------------------------------------------------
# 5. Expanding chronological walk-forward evaluation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= 5:
    raise ValueError("Insufficient trading dates.")

records = []
importance_records = []

for i in range(5, len(dates)):

    test_date = dates[i]

    train_pool = x[
        x["date"].isin(dates[:i])
        & x["_common"]
        & x["ts_sec"].mod(10).eq(0)
    ]

    test_pool = x[
        x["date"].eq(test_date)
        & x["_common"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}_27a"

        train = train_pool.loc[
            np.isfinite(train_pool[target])
        ]

        test = test_pool.loc[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for model_name, features in MODELS.items():

            model = lgb.LGBMRegressor(**PARAMS)

            model.fit(
                train[features],
                y_train
            )

            prediction = model.predict(
                test[features]
            )

            error = y_test - prediction

            records.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test),
                "sse": float(np.sum(error ** 2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error)))
            })

            gains = model.booster_.feature_importance(
                importance_type="gain"
            )

            for feature, gain in zip(features, gains):

                if feature in [DELTA60, DELTA300]:

                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": model_name,
                        "feature": feature,
                        "gain_importance": float(gain)
                    })

    print("Completed:", test_date)

calendar_exp27b_daily = pd.DataFrame(records)

calendar_exp27b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp27b_daily.empty:
    raise ValueError("No evaluations completed.")

# ------------------------------------------------------------
# 6. Paired pooled and daily error comparisons
# ------------------------------------------------------------

COMPARISONS = {
    "delta60_vs_full32": (
        "B_delta60",
        "A_full32_components"
    ),
    "delta300_vs_full32": (
        "C_delta300",
        "A_full32_components"
    ),
    "both_vs_full32": (
        "D_both",
        "A_full32_components"
    ),
    "both_vs_delta60": (
        "D_both",
        "B_delta60"
    ),
    "both_vs_delta300": (
        "D_both",
        "C_delta300"
    )
}

summary = []
daily_records = []

for h, block in calendar_exp27b_daily.groupby("horizon"):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for label, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        gain = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, value in gain.items():

            daily_records.append({
                "date": date,
                "horizon": h,
                "comparison": label,
                "gain_pct": float(value)
            })

        summary.append({
            "horizon": h,
            "comparison": label,
            "days": len(gain),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gain > 0).sum()),
            "median_daily_gain_pct": float(
                gain.median()
            ),
            "worst_day_pct": float(gain.min())
        })

calendar_exp27b_summary = pd.DataFrame(summary)

calendar_exp27b_daily_gains = pd.DataFrame(
    daily_records
)

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n===== 27B FULL32 DIRECTIONAL RESULTS =====")

print(
    calendar_exp27b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 27B DAILY IMBALANCE CHANGE GAINS =====")

for comparison in [
    "delta60_vs_full32",
    "delta300_vs_full32",
    "both_vs_full32"
]:

    print("\n", comparison)

    subset = calendar_exp27b_daily_gains.loc[
        calendar_exp27b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="gain_pct"
        ).round(6).to_string()
    )

print("\n===== 27B CANDIDATE FEATURE IMPORTANCE =====")

if not calendar_exp27b_importance.empty:

    print(
        calendar_exp27b_importance
        .groupby(
            ["horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 27B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 27B SOURCE ALIGNMENT =====
_retreat_match matched %: 100.0
_core_match matched %: 100.0

===== 27B MODEL COVERAGE =====
Total rows: 459122
Complete rows: 402979
Coverage %: 87.7717
Baseline columns: 45
Existing engineered features: 32
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 27B FULL32 DIRECTIONAL RESULTS =====
 horizon         comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30  delta60_vs_full32    11             0.013378            -0.016063             6               0.011211      -0.524213
      30 delta300_vs_full32    11             0.012820            -0.031919             5              -0.111671      -0.304637
      30     both_vs_full32    11            -0.002810             0.042312             4

In [84]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 28A — ORDER-BOOK SLOPE AND CURVATURE
#
# Input: raw df_1s, 5 levels of prices and quantities.
#
# Candidates:
#   slope_asymmetry_28a
#   curvature_asymmetry_28a
#   mean_slope_28a
#   mean_curvature_28a
#
# Targets: 30s / 60s / 300s signed & absolute returns
# Daily Spearman IC, redundancy and validity.
#
# Research only. No mutation of df_1s.
# ============================================================

HORIZONS_28A = [30, 60, 300]
MIN_SESSION_COVERAGE_28A = 0.97

bp = [f"bid_p{i}" for i in range(1, 6)]
ap = [f"ask_p{i}" for i in range(1, 6)]
bq = [f"bid_q{i}" for i in range(1, 6)]
aq = [f"ask_q{i}" for i in range(1, 6)]

required = ["symbol", "ts_sec"] + bp + ap + bq + aq

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

x28 = df_1s[required].copy()

for c in required:
    if c != "symbol":
        x28[c] = pd.to_numeric(
            x28[c], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

if x28["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps.")

if x28.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp keys.")

# ------------------------------------------------------------
# 1. IST session filtering
# ------------------------------------------------------------

ist = pd.to_datetime(
    x28["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x28["date"] = ist.dt.date

x28["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x28 = x28.loc[
    x28["clock_seconds"].between(0, 28799)
].copy()

x28 = x28.sort_values(
    ["symbol", "date", "ts_sec"]
).reset_index(drop=True)

coverage = (
    x28.groupby(["symbol", "date"])
    .agg(rows=("ts_sec", "nunique"))
    .reset_index()
)

eligible = coverage.loc[
    coverage["rows"].ge(
        MIN_SESSION_COVERAGE_28A * 28800
    )
    & coverage["date"].astype(str).ne("2026-09-28"),
    ["symbol", "date"]
]

x28 = x28.merge(
    eligible,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

if x28.empty:
    raise ValueError("No eligible sessions.")

GROUP = ["symbol", "date"]

print("===== 28A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x28))

# ------------------------------------------------------------
# 2. Extract complete book snapshots
# ------------------------------------------------------------

BP = x28[bp].to_numpy(dtype=np.float64)
AP = x28[ap].to_numpy(dtype=np.float64)
BQ = x28[bq].to_numpy(dtype=np.float64)
AQ = x28[aq].to_numpy(dtype=np.float64)

mid = (BP[:, 0] + AP[:, 0]) / 2.0

valid_price = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (AP[:, 0] >= BP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
)

valid_qty = (
    np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
    & (AQ.sum(axis=1) > 0)
)

valid_book = valid_price & valid_qty

print("\n===== 28A BOOK VALIDITY =====")
print(
    "Strict valid 5-level books (%):",
    round(100 * valid_book.mean(), 4)
)
print(
    "Invalid price ladders:",
    int((~valid_price).sum())
)
print(
    "Invalid quantities:",
    int((~valid_qty).sum())
)

# ------------------------------------------------------------
# 3. Price distance in basis points
# ------------------------------------------------------------

with np.errstate(divide="ignore", invalid="ignore"):

    DB = 1e4 * (
        BP[:, [0]] - BP
    ) / mid[:, None]

    DA = 1e4 * (
        AP - AP[:, [0]]
    ) / mid[:, None]

# ------------------------------------------------------------
# 4. Vectorized OLS slope for cumulative depth
# ------------------------------------------------------------

def rowwise_slope(y, d):
    """
    OLS slope of y on d with intercept,
    computed independently per book snapshot.
    """
    dx = d - d.mean(axis=1, keepdims=True)
    dy = y - y.mean(axis=1, keepdims=True)

    numerator = np.sum(dx * dy, axis=1)
    denominator = np.sum(dx * dx, axis=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        return numerator / np.where(
            denominator > 1e-16,
            denominator,
            np.nan
        )

CB = np.cumsum(BQ, axis=1)
CA = np.cumsum(AQ, axis=1)

slope_b = rowwise_slope(CB, DB)
slope_a = rowwise_slope(CA, DA)

with np.errstate(divide="ignore", invalid="ignore"):
    slope_b = slope_b / CB[:, -1]
    slope_a = slope_a / CA[:, -1]

# ------------------------------------------------------------
# 5. Liquidity-density gradient ("curvature")
#
# Density = quantity / local price gap
# Only levels 2..5 are included.
# ------------------------------------------------------------

gap_b = np.diff(DB, axis=1)
gap_a = np.diff(DA, axis=1)

with np.errstate(divide="ignore", invalid="ignore"):
    density_b = BQ[:, 1:] / np.where(
        gap_b > 0, gap_b, np.nan
    )
    density_a = AQ[:, 1:] / np.where(
        gap_a > 0, gap_a, np.nan
    )

curvature_b = rowwise_slope(
    np.log1p(density_b),
    DB[:, 1:]
)

curvature_a = rowwise_slope(
    np.log1p(density_a),
    DA[:, 1:]
)

# Enforce strict book validity for every output.
for arr in [
    slope_b, slope_a,
    curvature_b, curvature_a
]:
    arr[~valid_book] = np.nan
    arr[~np.isfinite(arr)] = np.nan

x28["bid_slope_28a"] = slope_b
x28["ask_slope_28a"] = slope_a

x28["bid_curvature_28a"] = curvature_b
x28["ask_curvature_28a"] = curvature_a

# ------------------------------------------------------------
# 6. Main experimental candidates
# ------------------------------------------------------------

x28["slope_asymmetry_28a"] = (
    x28["bid_slope_28a"]
    - x28["ask_slope_28a"]
)

x28["curvature_asymmetry_28a"] = (
    x28["bid_curvature_28a"]
    - x28["ask_curvature_28a"]
)

x28["mean_slope_28a"] = (
    x28["bid_slope_28a"]
    + x28["ask_slope_28a"]
) / 2

x28["mean_curvature_28a"] = (
    x28["bid_curvature_28a"]
    + x28["ask_curvature_28a"]
) / 2

CANDIDATES = [
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "mean_slope_28a",
    "mean_curvature_28a"
]

# ------------------------------------------------------------
# 7. Existing-information controls
# ------------------------------------------------------------

bid_depth = BQ.sum(axis=1)
ask_depth = AQ.sum(axis=1)
total_depth = bid_depth + ask_depth

with np.errstate(divide="ignore", invalid="ignore"):

    qi = (
        (BQ[:, 0] - AQ[:, 0])
        / (BQ[:, 0] + AQ[:, 0])
    )

    di = (
        (bid_depth - ask_depth)
        / total_depth
    )

controls = {
    "queue_imbalance_28a": qi,
    "depth_imbalance_28a": di,
    "log_total_depth_28a": np.log1p(total_depth),
    "spread_bps_28a": (
        1e4 * (AP[:, 0] - BP[:, 0]) / mid
    ),
    "bid_l1_concentration_28a": (
        BQ[:, 0] / bid_depth
    ),
    "ask_l1_concentration_28a": (
        AQ[:, 0] / ask_depth
    )
}

for name, values in controls.items():

    values = np.asarray(values, dtype=float)

    values[~valid_book] = np.nan
    values[~np.isfinite(values)] = np.nan

    x28[name] = values

CONTROL_NAMES = list(controls) + [
    "bid_slope_28a",
    "ask_slope_28a",
    "bid_curvature_28a",
    "ask_curvature_28a"
]

# ------------------------------------------------------------
# 8. Midprice and historical momentum
# ------------------------------------------------------------

x28["mid_28a"] = pd.Series(
    mid
).where(
    np.isfinite(mid)
    & (mid > 0)
    & (AP[:, 0] >= BP[:, 0])
)

for w in [30, 60]:

    g = x28.groupby(GROUP, sort=False)

    prev_mid = g["mid_28a"].shift(w)
    prev_ts = g["ts_sec"].shift(w)

    col = f"momentum_{w}_28a"

    x28[col] = (
        1e4 * np.log(
            x28["mid_28a"] / prev_mid
        )
    ).where(
        (x28["ts_sec"] - prev_ts).eq(w)
        & prev_mid.gt(0)
        & x28["mid_28a"].gt(0)
    )

    CONTROL_NAMES.append(col)

# ------------------------------------------------------------
# 9. Exact future signed/absolute returns
# ------------------------------------------------------------

for h in HORIZONS_28A:

    future = x28[
        ["symbol", "date", "ts_sec", "mid_28a"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={
            "mid_28a": f"future_mid_{h}_28a"
        }
    )

    x28 = x28.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    signed = f"future_return_{h}_28a"
    absolute = f"future_absret_{h}_28a"

    with np.errstate(divide="ignore", invalid="ignore"):
        x28[signed] = (
            1e4 * np.log(
                x28[f"future_mid_{h}_28a"]
                / x28["mid_28a"]
            )
        )

    x28[absolute] = x28[signed].abs()

# ------------------------------------------------------------
# 10. Daily Spearman IC
# ------------------------------------------------------------

def daily_ic_28a(a, b):

    mask = np.isfinite(a) & np.isfinite(b)
    n = int(mask.sum())

    if n < 200:
        return np.nan, n

    ra = pd.Series(a[mask]).rank().to_numpy()
    rb = pd.Series(b[mask]).rank().to_numpy()

    if ra.std() < 1e-12 or rb.std() < 1e-12:
        return np.nan, n

    return float(np.corrcoef(ra, rb)[0, 1]), n

FEATURES = list(dict.fromkeys(
    CANDIDATES + CONTROL_NAMES
))

records = []

for (symbol, date), day in x28.groupby(
    GROUP, sort=False
):

    for h in HORIZONS_28A:

        for task, target in {
            "signed_return": f"future_return_{h}_28a",
            "absret": f"future_absret_{h}_28a"
        }.items():

            y = day[target].to_numpy(dtype=float)

            for feature in FEATURES:

                z = day[feature].to_numpy(
                    dtype=float
                )

                ic, n = daily_ic_28a(z, y)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp28a_daily = pd.DataFrame(records)

calendar_exp28a_summary = (
    calendar_exp28a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=("valid_rows", "median")
    )
    .reset_index()
)

calendar_exp28a_summary["sign_consistency"] = (
    calendar_exp28a_summary[
        ["positive_days", "negative_days"]
    ].max(axis=1)
    / calendar_exp28a_summary[
        "days"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 11. Redundancy and distribution diagnostics
# ------------------------------------------------------------

calendar_exp28a_redundancy = (
    x28[FEATURES]
    .corr(method="spearman")
    .loc[CANDIDATES, CONTROL_NAMES]
)

print("\n===== 28A SLOPE / CURVATURE SIGNED IC =====")

print(
    calendar_exp28a_summary.loc[
        calendar_exp28a_summary["task"].eq(
            "signed_return"
        )
        & calendar_exp28a_summary[
            "feature"
        ].isin(CANDIDATES)
    ].round(6).to_string(index=False)
)

print("\n===== 28A SLOPE / CURVATURE ABSRET IC =====")

print(
    calendar_exp28a_summary.loc[
        calendar_exp28a_summary["task"].eq(
            "absret"
        )
        & calendar_exp28a_summary[
            "feature"
        ].isin(CANDIDATES)
    ].round(6).to_string(index=False)
)

print("\n===== 28A REDUNDANCY =====")

print(
    calendar_exp28a_redundancy
    .round(5)
    .to_string()
)

print("\n===== 28A FEATURE DIAGNOSTICS =====")

for name in CANDIDATES:

    s = x28[name]

    print(
        name,
        "valid_pct:", round(100*s.notna().mean(), 3),
        "mean:", round(s.mean(), 6),
        "std:", round(s.std(), 6),
        "median:", round(s.median(), 6),
        "p05:", round(s.quantile(0.05), 6),
        "p95:", round(s.quantile(0.95), 6)
    )

calendar_exp28a_features = x28

print("\n===== EXPERIMENT 28A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 28A SAMPLE =====
Sessions: 16
Rows: 459122

===== 28A BOOK VALIDITY =====
Strict valid 5-level books (%): 100.0
Invalid price ladders: 0
Invalid quantities: 0

===== 28A SLOPE / CURVATURE SIGNED IC =====
         task  horizon                 feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows  sign_consistency
signed_return       30 curvature_asymmetry_28a    16 -0.072315  -0.069952              0             16            28723.0            1.0000
signed_return       30      mean_curvature_28a    16  0.004208   0.004359             11              5            28723.0            0.6875
signed_return       30          mean_slope_28a    16  0.010081   0.004963              9              7            28723.0            0.5625
signed_return       30     slope_asymmetry_28a    16  0.035909   0.027604             12              4            28723.0            0.7500
signed_return       60 curvature_asymmetry_28a    16 -0.079384  -0.076383            

In [85]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 28B — FULL32 BOOK-GEOMETRY ABLATION
#
# Tasks:
#   signed_return:
#       curvature_asymmetry
#       slope_asymmetry
#
#   absolute_return:
#       mean_slope
#       mean_curvature
#
# Horizons: 30, 60, 300 seconds
# Models: baseline, primary, secondary, both
#
# Fixed LightGBM + chronological expanding evaluation
# Identical train/test rows within each comparison
#
# No changes to df_1s
# No feature promotion
# ============================================================

KEYS_28B = ["symbol", "ts_sec"]
HORIZONS_28B = [30, 60, 300]
MIN_TRAIN_DAYS_28B = 5
TRAIN_STRIDE_28B = 10

PARAMS_28B = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

for name in [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp28a_features"
]:
    if name not in globals():
        raise ValueError(f"Missing {name}")

CORE_28B = [
    c for c in core_31.columns
    if c not in KEYS_28B
]

if len(CORE_28B) != 31 or len(set(CORE_28B)) != 31:
    raise ValueError("Expected exactly 31 core features.")

RETREAT_28B = "quote_retreat_imbalance_30_24a"

CANDIDATES_28B = [
    "curvature_asymmetry_28a",
    "slope_asymmetry_28a",
    "mean_slope_28a",
    "mean_curvature_28a"
]

CONTROLS_28B = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

TARGETS_28B = [
    f"future_return_{h}_28a"
    for h in HORIZONS_28B
] + [
    f"future_absret_{h}_28a"
    for h in HORIZONS_28B
]

# ------------------------------------------------------------
# 1. Exact timestamp joins
# ------------------------------------------------------------

source_cols = list(dict.fromkeys(
    KEYS_28B
    + ["date"]
    + CONTROLS_28B
    + CANDIDATES_28B
    + TARGETS_28B
))

research = calendar_exp28a_features[
    source_cols
].copy()

retreat = calendar_exp24a_features[
    KEYS_28B + [RETREAT_28B]
].copy()

core = core_31[
    KEYS_28B + CORE_28B
].copy()

for frame in [research, retreat, core]:
    frame["symbol"] = frame["symbol"].astype(str)
    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"], errors="raise"
    ).astype("int64")

    if frame.duplicated(KEYS_28B).any():
        raise ValueError("Duplicate symbol/timestamp keys")

x = research.merge(
    retreat,
    on=KEYS_28B,
    how="left",
    validate="one_to_one",
    indicator="_retreat_match"
)

x = x.merge(
    core,
    on=KEYS_28B,
    how="left",
    validate="one_to_one",
    indicator="_core_match"
)

print("===== 28B SOURCE ALIGNMENT =====")
print("Rows:", len(x))

for name in ["_retreat_match", "_core_match"]:
    matched = x[name].eq("both").mean()
    print(name, "match %:", round(100 * matched, 4))

    if matched < 0.99:
        raise ValueError(
            f"Insufficient source alignment: {name}"
        )

x = x.drop(
    columns=["_retreat_match", "_core_match"]
)

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Feature sets
#
# Extra controls deliberately make baseline stricter.
# This baseline has the 32 engineered features,
# plus common reconstructed market information.
# ------------------------------------------------------------

BASE_28B = list(dict.fromkeys(
    CORE_28B
    + [RETREAT_28B]
    + CONTROLS_28B
))

TASKS_28B = {
    "signed_return": {
        "primary": "curvature_asymmetry_28a",
        "secondary": "slope_asymmetry_28a",
        "target_prefix": "future_return"
    },
    "absolute_return": {
        "primary": "mean_slope_28a",
        "secondary": "mean_curvature_28a",
        "target_prefix": "future_absret"
    }
}

ALL_FEATURES_28B = list(dict.fromkeys(
    BASE_28B + CANDIDATES_28B
))

missing_cols = sorted(
    set(ALL_FEATURES_28B + TARGETS_28B)
    - set(x.columns)
)

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

for col in ALL_FEATURES_28B + TARGETS_28B:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Strict common sample across both forecasting tasks.
x["_common_28b"] = np.isfinite(
    x[ALL_FEATURES_28B].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 28B DATA COVERAGE =====")
print("Full baseline columns:", len(BASE_28B))
print("Complete feature rows:", int(x["_common_28b"].sum()))
print(
    "Coverage %:",
    round(100 * x["_common_28b"].mean(), 4)
)

# ------------------------------------------------------------
# 3. Expanding chronological LightGBM validation
# ------------------------------------------------------------

dates = sorted(x["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS_28B:
    raise ValueError("Insufficient sessions")

records = []
importance_records = []

for i in range(MIN_TRAIN_DAYS_28B, len(dates)):

    test_date = dates[i]

    train_pool = x.loc[
        x["date"].isin(dates[:i])
        & x["_common_28b"]
        & x["ts_sec"].mod(TRAIN_STRIDE_28B).eq(0)
    ]

    test_pool = x.loc[
        x["date"].eq(test_date)
        & x["_common_28b"]
    ]

    for task, spec in TASKS_28B.items():

        p = spec["primary"]
        s = spec["secondary"]

        models = {
            "A_full32": BASE_28B,
            "B_primary": BASE_28B + [p],
            "C_secondary": BASE_28B + [s],
            "D_both": BASE_28B + [p, s]
        }

        for h in HORIZONS_28B:

            target = f"{spec['target_prefix']}_{h}_28a"

            train = train_pool.loc[
                np.isfinite(train_pool[target])
            ]

            test = test_pool.loc[
                np.isfinite(test_pool[target])
            ]

            if len(train) < 1000 or len(test) < 200:
                continue

            y_train = train[target].to_numpy(
                dtype=float
            )

            y_test = test[target].to_numpy(
                dtype=float
            )

            for model_name, features in models.items():

                model = lgb.LGBMRegressor(
                    **PARAMS_28B
                )

                model.fit(train[features], y_train)

                pred = model.predict(test[features])

                if task == "absolute_return":
                    pred = np.maximum(pred, 0.0)

                error = y_test - pred

                records.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test),
                    "sse": float(np.sum(error ** 2)),
                    "sae": float(np.sum(np.abs(error))),
                    "mse": float(np.mean(error ** 2)),
                    "mae": float(np.mean(np.abs(error)))
                })

                importance = (
                    model.booster_.feature_importance(
                        importance_type="gain"
                    )
                )

                for feature, gain in zip(features, importance):
                    if feature in [p, s]:
                        importance_records.append({
                            "date": test_date,
                            "task": task,
                            "horizon": h,
                            "model": model_name,
                            "feature": feature,
                            "gain_importance": float(gain)
                        })

    print("Completed:", test_date)

calendar_exp28b_daily = pd.DataFrame(records)

calendar_exp28b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp28b_daily.empty:
    raise ValueError("No evaluations completed")

# ------------------------------------------------------------
# 4. Pooled MSE/MAE and day-level comparisons
# ------------------------------------------------------------

COMPARISONS_28B = {
    "primary_vs_full32": (
        "B_primary", "A_full32"
    ),
    "secondary_vs_full32": (
        "C_secondary", "A_full32"
    ),
    "both_vs_full32": (
        "D_both", "A_full32"
    ),
    "both_vs_primary": (
        "D_both", "B_primary"
    ),
    "both_vs_secondary": (
        "D_both", "C_secondary"
    )
}

summary = []
daily_gains = []

for (task, h), block in (
    calendar_exp28b_daily.groupby(["task", "horizon"])
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for label, (candidate, baseline) in (
        COMPARISONS_28B.items()
    ):

        gains = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, value in gains.items():
            daily_gains.append({
                "date": date,
                "task": task,
                "horizon": h,
                "comparison": label,
                "gain_pct": float(value)
            })

        summary.append({
            "task": task,
            "horizon": h,
            "comparison": label,
            "days": len(gains),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((gains > 0).sum()),
            "median_daily_gain_pct": float(
                gains.median()
            ),
            "worst_day_pct": float(gains.min())
        })

calendar_exp28b_summary = pd.DataFrame(summary)

calendar_exp28b_daily_gains = pd.DataFrame(
    daily_gains
)

# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print("\n===== 28B SIGNED RETURN RESULTS =====")

print(
    calendar_exp28b_summary.loc[
        calendar_exp28b_summary[
            "task"
        ].eq("signed_return")
    ].round(6).to_string(index=False)
)

print("\n===== 28B ABSRET RESULTS =====")

print(
    calendar_exp28b_summary.loc[
        calendar_exp28b_summary[
            "task"
        ].eq("absolute_return")
    ].round(6).to_string(index=False)
)

print("\n===== 28B DAILY PRIMARY FEATURE GAINS =====")

for task in TASKS_28B:

    print("\nTask:", task)

    subset = calendar_exp28b_daily_gains.loc[
        calendar_exp28b_daily_gains[
            "task"
        ].eq(task)
        & calendar_exp28b_daily_gains[
            "comparison"
        ].eq("primary_vs_full32")
    ]

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="gain_pct"
        ).round(6).to_string()
    )

print("\n===== 28B CANDIDATE IMPORTANCE =====")

if not calendar_exp28b_importance.empty:
    print(
        calendar_exp28b_importance.groupby(
            ["task", "horizon", "model", "feature"]
        )["gain_importance"]
        .median()
        .reset_index()
        .round(4)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 28B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


===== 28B SOURCE ALIGNMENT =====
Rows: 459122
_retreat_match match %: 100.0
_core_match match %: 100.0

===== 28B DATA COVERAGE =====
Full baseline columns: 41
Complete feature rows: 402979
Coverage %: 87.7717
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 28B SIGNED RETURN RESULTS =====
         task  horizon          comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
signed_return       30   primary_vs_full32    11             0.463620             0.114507             8               0.399014      -0.051979
signed_return       30 secondary_vs_full32    11             1.081533            -0.150804             9               1.248737      -1.118345
signed_return       30      both_vs_full32    11             1.172242           

In [86]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 28C — ROBUSTNESS OF BOOK GEOMETRY
#
# Tests:
# 1. Leave-one-evaluation-day-out pooled MSE/MAE
# 2. Alternative, more regularized LightGBM
# 3. Performance conditional on price-ladder spacing
#
# Primary target: signed future 60-second return.
#
# Input DataFrames:
#   calendar_exp28b_daily
#   calendar_exp28a_features
#   calendar_exp24a_features
#   core_31
#
# No edits to df_1s or feature registry.
# ============================================================

REQUIRED_FRAMES_28C = [
    "calendar_exp28b_daily",
    "calendar_exp28a_features",
    "calendar_exp24a_features",
    "core_31"
]

for name in REQUIRED_FRAMES_28C:
    if name not in globals():
        raise ValueError(f"Missing {name}")

H_28C = 60
KEYS_28C = ["symbol", "ts_sec"]

SLOPE_28C = "slope_asymmetry_28a"
CURV_28C = "curvature_asymmetry_28a"
RETREAT_28C = "quote_retreat_imbalance_30_24a"
TARGET_28C = "future_return_60_28a"

# ============================================================
# PART A — LEAVE-ONE-DAY-OUT STABILITY
#
# Uses the ORIGINAL Experiment 28B predictions'
# aggregated errors, without retraining.
# ============================================================

old = calendar_exp28b_daily.copy()

old = old.loc[
    old["task"].eq("signed_return")
    & old["horizon"].eq(H_28C)
].copy()

MODEL_MAP_28C = {
    "B_primary": "curvature",
    "C_secondary": "slope",
    "D_both": "both"
}

required_models = (
    {"A_full32"} | set(MODEL_MAP_28C)
)

if not required_models.issubset(set(old["model"])):
    raise ValueError("Missing Experiment 28B models.")

loo_records = []

for model, label in MODEL_MAP_28C.items():

    daily = old.loc[
        old["model"].isin(["A_full32", model])
    ]

    # Align the two model records by evaluation date.
    a = daily.loc[
        daily["model"].eq("A_full32")
    ].set_index("date")

    b = daily.loc[
        daily["model"].eq(model)
    ].set_index("date")

    common_dates = a.index.intersection(b.index)

    for omitted_date in common_dates:

        keep = common_dates.drop(omitted_date)

        base_sse = a.loc[keep, "sse"].sum()
        cand_sse = b.loc[keep, "sse"].sum()

        base_sae = a.loc[keep, "sae"].sum()
        cand_sae = b.loc[keep, "sae"].sum()

        loo_records.append({
            "candidate": label,
            "omitted_date": omitted_date,
            "pooled_mse_gain_pct": 100 * (
                1 - cand_sse / base_sse
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - cand_sae / base_sae
            )
        })

calendar_exp28c_loo = pd.DataFrame(loo_records)

print("\n===== 28C LEAVE-ONE-DAY-OUT SUMMARY =====")

print(
    calendar_exp28c_loo.groupby("candidate")
    .agg(
        min_mse_gain=("pooled_mse_gain_pct", "min"),
        median_mse_gain=("pooled_mse_gain_pct", "median"),
        max_mse_gain=("pooled_mse_gain_pct", "max"),
        min_mae_gain=("pooled_mae_gain_pct", "min"),
        positive_mse_cases=(
            "pooled_mse_gain_pct",
            lambda s: int((s > 0).sum())
        )
    )
    .round(6)
    .to_string()
)

print("\n===== 28C LEAVE-ONE-DAY-OUT DETAILS =====")

print(
    calendar_exp28c_loo.pivot(
        index="omitted_date",
        columns="candidate",
        values="pooled_mse_gain_pct"
    )
    .round(6)
    .to_string()
)

# ============================================================
# PART B — ALTERNATIVE LIGHTGBM CONFIGURATION
#
# Reconstruct EXACT same full feature baseline
# used in Experiment 28B.
#
# Important: Refit all models on matched rows.
# ============================================================

CORE_28C = [
    c for c in core_31.columns
    if c not in KEYS_28C
]

if len(CORE_28C) != 31 or len(set(CORE_28C)) != 31:
    raise ValueError("Expected exactly 31 core features.")

CONTROLS_28C = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

ALL_CANDIDATES_28C = [
    "curvature_asymmetry_28a",
    "slope_asymmetry_28a",
    "mean_slope_28a",
    "mean_curvature_28a"
]

source_cols = list(dict.fromkeys(
    KEYS_28C
    + ["date"]
    + CONTROLS_28C
    + ALL_CANDIDATES_28C
    + [TARGET_28C]
    + [f"bid_p{i}" for i in range(1, 6)]
    + [f"ask_p{i}" for i in range(1, 6)]
))

research = calendar_exp28a_features[
    source_cols
].copy()

retreat = calendar_exp24a_features[
    KEYS_28C + [RETREAT_28C]
].copy()

core = core_31[
    KEYS_28C + CORE_28C
].copy()

for frame in [research, retreat, core]:

    frame["symbol"] = frame["symbol"].astype(str)

    frame["ts_sec"] = pd.to_numeric(
        frame["ts_sec"], errors="raise"
    ).astype("int64")

    if frame.duplicated(KEYS_28C).any():
        raise ValueError("Duplicate timestamp join keys.")

x = research.merge(
    retreat,
    on=KEYS_28C,
    how="left",
    validate="one_to_one",
    indicator="_retreat_match"
)

x = x.merge(
    core,
    on=KEYS_28C,
    how="left",
    validate="one_to_one",
    indicator="_core_match"
)

print("\n===== 28C SOURCE ALIGNMENT =====")

for col in ["_retreat_match", "_core_match"]:

    match = x[col].eq("both").mean()

    print(col, round(100 * match, 4), "%")

    if match < 0.99:
        raise ValueError("Source alignment below 99%.")

x = x.drop(
    columns=["_retreat_match", "_core_match"]
)

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

BASE_28C = list(dict.fromkeys(
    CORE_28C
    + [RETREAT_28C]
    + CONTROLS_28C
))

MODELS_28C = {
    "A_full32": BASE_28C,
    "B_curvature": BASE_28C + [CURV_28C],
    "C_slope": BASE_28C + [SLOPE_28C],
    "D_both": BASE_28C + [CURV_28C, SLOPE_28C]
}

# Same common feature mask used by Experiment 28B,
# including all 4 geometric candidates.
COMMON_FEATURES_28C = list(dict.fromkeys(
    BASE_28C + ALL_CANDIDATES_28C
))

for col in COMMON_FEATURES_28C + [TARGET_28C]:

    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_common"] = np.isfinite(
    x[COMMON_FEATURES_28C].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 28C MODEL COVERAGE =====")
print("Rows:", len(x))
print("Complete rows:", int(x["_common"].sum()))
print("Baseline inputs:", len(BASE_28C))

# ------------------------------------------------------------
# Price ladder geometry
#
# Median adjacent price gap across both book sides,
# measured in ticks after determining the smallest
# positive observed gap.
#
# Also compute a normalized gap variability measure.
# ------------------------------------------------------------

bid_prices = x[
    [f"bid_p{i}" for i in range(1, 6)]
].to_numpy(dtype=float)

ask_prices = x[
    [f"ask_p{i}" for i in range(1, 6)]
].to_numpy(dtype=float)

bid_gaps = -np.diff(bid_prices, axis=1)
ask_gaps = np.diff(ask_prices, axis=1)

all_gaps = np.concatenate(
    [bid_gaps, ask_gaps], axis=1
)

valid_gaps = (
    np.isfinite(all_gaps).all(axis=1)
    & (all_gaps > 0).all(axis=1)
)

# Relative spacing variability is dimensionless.
gap_mean = np.mean(all_gaps, axis=1)

gap_sd = np.std(all_gaps, axis=1)

x["gap_cv_28c"] = np.where(
    valid_gaps & (gap_mean > 0),
    gap_sd / np.maximum(gap_mean, 1e-12),
    np.nan
)

# Estimate a descriptive minimum positive gap.
positive_gaps = all_gaps[
    np.isfinite(all_gaps) & (all_gaps > 0)
]

if len(positive_gaps) == 0:
    raise ValueError("No positive price gaps.")

reference_gap = float(np.min(positive_gaps))

x["median_gap_units_28c"] = np.where(
    valid_gaps,
    np.median(all_gaps, axis=1)
    / reference_gap,
    np.nan
)

print("\n===== 28C GAP DIAGNOSTICS =====")
print("Minimum observed positive gap:", reference_gap)
print(
    x[
        ["gap_cv_28c", "median_gap_units_28c"]
    ].describe(
        percentiles=[0.1, 0.5, 0.9]
    ).round(6).to_string()
)

# Fixed descriptive geometry bins, NOT learned
# using targets or test-period returns.
x["gap_regime_28c"] = pd.cut(
    x["gap_cv_28c"],
    bins=[-np.inf, 0.10, 0.50, np.inf],
    labels=[
        "uniform_spacing",
        "moderate_spacing",
        "irregular_spacing"
    ]
)

# ------------------------------------------------------------
# Alternative fixed model configuration.
#
# Smaller trees, stronger regularization.
# ------------------------------------------------------------

PARAMS_28C = {
    "objective": "regression",
    "n_estimators": 140,
    "learning_rate": 0.035,
    "num_leaves": 7,
    "max_depth": 3,
    "min_child_samples": 400,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 8.0,
    "reg_alpha": 0.5,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

dates = sorted(x["date"].unique())

if len(dates) <= 5:
    raise ValueError("Not enough sessions.")

records = []
prediction_records = []

for i in range(5, len(dates)):

    test_date = dates[i]

    train = x.loc[
        x["date"].isin(dates[:i])
        & x["_common"]
        & x["ts_sec"].mod(10).eq(0)
        & np.isfinite(x[TARGET_28C])
    ]

    test = x.loc[
        x["date"].eq(test_date)
        & x["_common"]
        & np.isfinite(x[TARGET_28C])
    ].copy()

    if len(train) < 1000 or len(test) < 200:
        continue

    y_train = train[TARGET_28C].to_numpy(dtype=float)
    y_test = test[TARGET_28C].to_numpy(dtype=float)

    result = test[
        ["date", "symbol", "ts_sec", "gap_regime_28c"]
    ].copy()

    result["actual"] = y_test

    for name, cols in MODELS_28C.items():

        model = lgb.LGBMRegressor(**PARAMS_28C)

        model.fit(train[cols], y_train)

        pred = model.predict(test[cols])

        error = y_test - pred

        result[f"sse_{name}"] = error ** 2
        result[f"sae_{name}"] = np.abs(error)

        records.append({
            "date": test_date,
            "model": name,
            "rows": len(test),
            "sse": float(np.sum(error ** 2)),
            "sae": float(np.sum(np.abs(error)))
        })

    prediction_records.append(result)

    print("Completed:", test_date)

calendar_exp28c_alt_daily = pd.DataFrame(records)

if not prediction_records:
    raise ValueError("No evaluation predictions.")

calendar_exp28c_predictions = pd.concat(
    prediction_records,
    ignore_index=True
)

# ------------------------------------------------------------
# Alternative model pooled/daily comparisons
# ------------------------------------------------------------

COMPARISONS_28C = {
    "curvature_vs_full32": (
        "B_curvature", "A_full32"
    ),
    "slope_vs_full32": (
        "C_slope", "A_full32"
    ),
    "both_vs_full32": (
        "D_both", "A_full32"
    ),
    "both_vs_slope": (
        "D_both", "C_slope"
    )
}

summary = []

for label, (candidate, baseline) in (
    COMPARISONS_28C.items()
):

    a = calendar_exp28c_alt_daily.loc[
        calendar_exp28c_alt_daily["model"].eq(baseline)
    ].set_index("date")

    b = calendar_exp28c_alt_daily.loc[
        calendar_exp28c_alt_daily["model"].eq(candidate)
    ].set_index("date")

    dates_common = a.index.intersection(b.index)

    a = a.loc[dates_common]
    b = b.loc[dates_common]

    daily_gain = 100 * (
        1 - b["sse"] / a["sse"]
    )

    summary.append({
        "comparison": label,
        "days": len(dates_common),
        "pooled_mse_gain_pct": 100 * (
            1 - b["sse"].sum() / a["sse"].sum()
        ),
        "pooled_mae_gain_pct": 100 * (
            1 - b["sae"].sum() / a["sae"].sum()
        ),
        "winning_days": int((daily_gain > 0).sum()),
        "median_daily_gain_pct": float(
            daily_gain.median()
        ),
        "worst_day_pct": float(daily_gain.min())
    })

calendar_exp28c_alt_summary = pd.DataFrame(summary)

print("\n===== 28C ALTERNATIVE LIGHTGBM RESULTS =====")

print(
    calendar_exp28c_alt_summary
    .round(6)
    .to_string(index=False)
)

# ============================================================
# PART C — PERFORMANCE BY BOOK SPACING
#
# All models were fitted without using the regime
# as a model feature. The regime is only a
# post-hoc diagnostic.
# ============================================================

geometry_records = []

for regime, day in calendar_exp28c_predictions.groupby(
    "gap_regime_28c",
    observed=True
):

    if len(day) < 200:
        continue

    base_sse = day["sse_A_full32"].sum()
    base_sae = day["sae_A_full32"].sum()

    for label, candidate in [
        ("curvature", "B_curvature"),
        ("slope", "C_slope"),
        ("both", "D_both")
    ]:

        geometry_records.append({
            "gap_regime": str(regime),
            "candidate": label,
            "rows": len(day),
            "sessions": day["date"].nunique(),
            "mse_gain_pct": 100 * (
                1 - day[f"sse_{candidate}"].sum()
                / base_sse
            ),
            "mae_gain_pct": 100 * (
                1 - day[f"sae_{candidate}"].sum()
                / base_sae
            )
        })

calendar_exp28c_geometry = pd.DataFrame(
    geometry_records
)

print("\n===== 28C BOOK SPACING ROBUSTNESS =====")

print(
    calendar_exp28c_geometry
    .round(6)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 28C COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")



===== 28C LEAVE-ONE-DAY-OUT SUMMARY =====
           min_mse_gain  median_mse_gain  max_mse_gain  min_mae_gain  positive_mse_cases
candidate                                                                               
both           1.266815         1.437862      1.614683      0.323244                  11
curvature      0.229866         0.307870      0.382242      0.127739                  11
slope          1.059553         1.329451      1.415769      0.058345                  11

===== 28C LEAVE-ONE-DAY-OUT DETAILS =====
candidate         both  curvature     slope
omitted_date                               
2026-10-05    1.430226   0.307870  1.302404
2026-10-06    1.491459   0.293930  1.367688
2026-10-07    1.437862   0.291257  1.284641
2026-10-08    1.371321   0.287552  1.252436
2026-10-09    1.557715   0.349886  1.415769
2026-10-12    1.614683   0.375995  1.393741
2026-10-13    1.512155   0.382242  1.402208
2026-10-14    1.425273   0.229866  1.344746
2026-10-15    1.266815   0.31

In [87]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 29A — LIQUIDITY GAPS
#
# Input:
#   Raw df_1s containing L1-L5 bid/ask prices,
#   L1-L5 bid/ask quantities, symbol and ts_sec.
#
# Outputs:
#   calendar_exp29a_features
#   calendar_exp29a_daily
#   calendar_exp29a_summary
#   calendar_exp29a_redundancy
#
# Exact future horizons: 30, 60, 300 seconds.
# No changes to original df_1s.
# No features automatically selected.
# ============================================================

HORIZONS_29A = [30, 60, 300]
MIN_COVERAGE_29A = 0.97
DECAY_29A = 0.7

KEYS_29A = ["symbol", "ts_sec"]
BP_COLS = [f"bid_p{i}" for i in range(1, 6)]
AP_COLS = [f"ask_p{i}" for i in range(1, 6)]
BQ_COLS = [f"bid_q{i}" for i in range(1, 6)]
AQ_COLS = [f"ask_q{i}" for i in range(1, 6)]

REQUIRED_29A = (
    KEYS_29A + BP_COLS + AP_COLS + BQ_COLS + AQ_COLS
)

missing = sorted(set(REQUIRED_29A) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing raw columns: {missing}")

x29 = df_1s[REQUIRED_29A].copy()

for c in REQUIRED_29A:
    if c != "symbol":
        x29[c] = pd.to_numeric(
            x29[c], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

if x29["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps.")

x29["ts_sec"] = x29["ts_sec"].astype("int64")
x29["symbol"] = x29["symbol"].astype(str)

if x29.duplicated(KEYS_29A).any():
    raise ValueError("Duplicate symbol/timestamp keys.")

# ------------------------------------------------------------
# 1. IST regular-session eligibility
# ------------------------------------------------------------

ist = pd.to_datetime(
    x29["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x29["date"] = ist.dt.date

x29["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x29 = x29.loc[
    x29["clock_seconds"].between(0, 28799)
].copy()

coverage = (
    x29.groupby(["symbol", "date"])["ts_sec"]
    .nunique()
    .reset_index(name="n_seconds")
)

eligible = coverage.loc[
    coverage["n_seconds"].ge(
        MIN_COVERAGE_29A * 28800
    )
    & coverage["date"].astype(str).ne("2026-09-28"),
    ["symbol", "date"]
]

x29 = (
    x29.merge(
        eligible,
        on=["symbol", "date"],
        how="inner",
        validate="many_to_one"
    )
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

if x29.empty:
    raise ValueError("No eligible sessions.")

print("===== 29A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x29))

# ------------------------------------------------------------
# 2. Validate five-level price ladders
# ------------------------------------------------------------

BP = x29[BP_COLS].to_numpy(dtype=np.float64)
AP = x29[AP_COLS].to_numpy(dtype=np.float64)
BQ = x29[BQ_COLS].to_numpy(dtype=np.float64)
AQ = x29[AQ_COLS].to_numpy(dtype=np.float64)

mid = (BP[:, 0] + AP[:, 0]) / 2.0

valid_book = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (BP[:, 0] <= AP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
    & (AQ.sum(axis=1) > 0)
)

print("\n===== 29A BOOK VALIDITY =====")
print(
    "Valid books (%):",
    round(100 * valid_book.mean(), 5)
)

# ------------------------------------------------------------
# 3. Adjacent price gaps, measured in basis points
# ------------------------------------------------------------

with np.errstate(divide="ignore", invalid="ignore"):

    gap_bid = (
        -np.diff(BP, axis=1)
        / mid[:, None]
        * 1e4
    )

    gap_ask = (
        np.diff(AP, axis=1)
        / mid[:, None]
        * 1e4
    )

gap_bid[~valid_book] = np.nan
gap_ask[~valid_book] = np.nan

# ------------------------------------------------------------
# 4. Fixed near-touch weights
# ------------------------------------------------------------

levels = np.arange(4, dtype=float)

weights = np.exp(-DECAY_29A * levels)
weights /= weights.sum()

print("\n===== 29A LEVEL WEIGHTS =====")
print(dict(zip(
    ["1-2", "2-3", "3-4", "4-5"],
    np.round(weights, 6)
)))

# ------------------------------------------------------------
# 5. Create candidate features
# ------------------------------------------------------------

x29["near_gap_asymmetry_29a"] = (
    gap_bid[:, 0] - gap_ask[:, 0]
)

x29["max_gap_asymmetry_29a"] = (
    np.max(gap_bid, axis=1)
    - np.max(gap_ask, axis=1)
)

weighted_bid = gap_bid @ weights
weighted_ask = gap_ask @ weights

x29["weighted_gap_asymmetry_29a"] = (
    weighted_bid - weighted_ask
)

# Queue-size-adjusted gap pressure.
# Queue at level i is immediately before gap i -> i+1.

with np.errstate(divide="ignore", invalid="ignore"):

    bid_fragility = (
        gap_bid / np.sqrt(1.0 + BQ[:, :4])
    ) @ weights

    ask_fragility = (
        gap_ask / np.sqrt(1.0 + AQ[:, :4])
    ) @ weights

x29["fragility_asymmetry_29a"] = (
    bid_fragility - ask_fragility
)

x29["gap_severity_29a"] = 0.5 * (
    np.max(gap_bid, axis=1)
    + np.max(gap_ask, axis=1)
)

CANDIDATES_29A = [
    "near_gap_asymmetry_29a",
    "max_gap_asymmetry_29a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "gap_severity_29a"
]

# ------------------------------------------------------------
# 6. Basic existing-information controls
# ------------------------------------------------------------

bid_depth = BQ.sum(axis=1)
ask_depth = AQ.sum(axis=1)

with np.errstate(divide="ignore", invalid="ignore"):

    x29["queue_imbalance_29a"] = (
        (BQ[:, 0] - AQ[:, 0])
        / (BQ[:, 0] + AQ[:, 0])
    )

    x29["depth_imbalance_29a"] = (
        (bid_depth - ask_depth)
        / (bid_depth + ask_depth)
    )

x29["spread_bps_29a"] = (
    (AP[:, 0] - BP[:, 0])
    / mid * 1e4
)

x29["log_total_depth_29a"] = (
    np.log1p(bid_depth + ask_depth)
)

CONTROLS_29A = [
    "queue_imbalance_29a",
    "depth_imbalance_29a",
    "spread_bps_29a",
    "log_total_depth_29a"
]

for c in CONTROLS_29A:
    x29.loc[~valid_book, c] = np.nan
    x29[c] = x29[c].replace(
        [np.inf, -np.inf], np.nan
    )

# ------------------------------------------------------------
# 7. Attach exact Experiment 28 geometry features
#    if present (preferred for redundancy).
# ------------------------------------------------------------

if "calendar_exp28a_features" in globals():

    geometry_cols = [
        "slope_asymmetry_28a",
        "curvature_asymmetry_28a"
    ]

    geometry = calendar_exp28a_features[
        KEYS_29A + geometry_cols
    ].copy()

    geometry["symbol"] = geometry["symbol"].astype(str)
    geometry["ts_sec"] = geometry["ts_sec"].astype("int64")

    x29 = x29.merge(
        geometry,
        on=KEYS_29A,
        how="left",
        validate="one_to_one"
    )

    CONTROLS_29A.extend(geometry_cols)

# ------------------------------------------------------------
# 8. Midprice and exact future-return labels
# ------------------------------------------------------------

x29["mid_29a"] = np.where(
    valid_book, mid, np.nan
)

for h in HORIZONS_29A:

    future = x29[
        KEYS_29A + ["date", "mid_29a"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={"mid_29a": f"future_mid_{h}_29a"}
    )

    x29 = x29.merge(
        future,
        on=KEYS_29A + ["date"],
        how="left",
        validate="one_to_one"
    )

    target = f"future_return_{h}_29a"

    with np.errstate(divide="ignore", invalid="ignore"):

        x29[target] = 1e4 * np.log(
            x29[f"future_mid_{h}_29a"]
            / x29["mid_29a"]
        )

    x29[f"future_absret_{h}_29a"] = (
        x29[target].abs()
    )

# ------------------------------------------------------------
# 9. Daily Spearman IC
# ------------------------------------------------------------

def spearman_29a(a, b):

    valid = np.isfinite(a) & np.isfinite(b)

    n = int(valid.sum())

    if n < 200:
        return np.nan, n

    aa = pd.Series(a[valid]).rank().to_numpy()
    bb = pd.Series(b[valid]).rank().to_numpy()

    if aa.std() < 1e-12 or bb.std() < 1e-12:
        return np.nan, n

    return float(np.corrcoef(aa, bb)[0, 1]), n

records = []

FEATURES_TO_TEST = list(dict.fromkeys(
    CANDIDATES_29A + CONTROLS_29A
))

for (symbol, date), day in x29.groupby(
    ["symbol", "date"], sort=False
):

    for h in HORIZONS_29A:

        targets = {
            "signed_return": f"future_return_{h}_29a",
            "absret": f"future_absret_{h}_29a"
        }

        for task, target in targets.items():

            y = day[target].to_numpy(dtype=float)

            for feature in FEATURES_TO_TEST:

                z = day[feature].to_numpy(dtype=float)

                ic, valid_rows = spearman_29a(z, y)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": valid_rows
                })

calendar_exp29a_daily = pd.DataFrame(records)

calendar_exp29a_summary = (
    calendar_exp29a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

calendar_exp29a_redundancy = (
    x29[
        list(dict.fromkeys(
            CANDIDATES_29A + CONTROLS_29A
        ))
    ]
    .corr(method="spearman")
    .loc[CANDIDATES_29A, CONTROLS_29A]
)

# ------------------------------------------------------------
# 10. Diagnostics
# ------------------------------------------------------------

print("\n===== 29A SIGNED RETURN IC =====")

print(
    calendar_exp29a_summary.loc[
        calendar_exp29a_summary["task"].eq(
            "signed_return"
        )
        & calendar_exp29a_summary[
            "feature"
        ].isin(CANDIDATES_29A)
    ].round(6).to_string(index=False)
)

print("\n===== 29A ABSRET IC =====")

print(
    calendar_exp29a_summary.loc[
        calendar_exp29a_summary["task"].eq(
            "absret"
        )
        & calendar_exp29a_summary[
            "feature"
        ].isin(CANDIDATES_29A)
    ].round(6).to_string(index=False)
)

print("\n===== 29A REDUNDANCY =====")

print(
    calendar_exp29a_redundancy
    .round(5).to_string()
)

print("\n===== 29A DISTRIBUTION DIAGNOSTICS =====")

for c in CANDIDATES_29A:

    s = x29[c]

    print(
        c,
        "valid_pct=", round(100*s.notna().mean(), 4),
        "zero_pct=", round(100*s.eq(0).mean(), 4),
        "p01=", round(s.quantile(0.01), 6),
        "median=", round(s.median(), 6),
        "p99=", round(s.quantile(0.99), 6)
    )

# Useful check: one-tick gap prevalence.
positive_raw_gaps = np.concatenate([
    -np.diff(BP, axis=1),
    np.diff(AP, axis=1)
], axis=1)

minimum_gap = np.nanmin(
    positive_raw_gaps[
        np.isfinite(positive_raw_gaps)
        & (positive_raw_gaps > 0)
    ]
)

print("\n===== 29A PRICE GAP DIAGNOSTICS =====")
print("Minimum positive observed gap:", minimum_gap)
print(
    "Median raw adjacent gap:",
    float(np.nanmedian(positive_raw_gaps))
)
print(
    "99th percentile raw adjacent gap:",
    float(np.nanquantile(
        positive_raw_gaps, 0.99
    ))
)

calendar_exp29a_features = x29

print("\n===== EXPERIMENT 29A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")


Task was destroyed but it is pending!
task: <Task pending name='Task-726' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-727' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\projects\hedge_fund\venv\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-727' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-762' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:76> wait_fo

===== 29A SAMPLE =====
Sessions: 16
Rows: 459122

===== 29A BOOK VALIDITY =====
Valid books (%): 100.0

===== 29A LEVEL WEIGHTS =====
{'1-2': np.float64(0.536009), '2-3': np.float64(0.266174), '3-4': np.float64(0.132178), '4-5': np.float64(0.065638)}

===== 29A SIGNED RETURN IC =====
         task  horizon                    feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed_return       30    fragility_asymmetry_29a    16 -0.198117  -0.199260              0             16            28723.0
signed_return       30           gap_severity_29a    16  0.003653   0.000279              8              8            28723.0
signed_return       30      max_gap_asymmetry_29a    16 -0.105179  -0.099979              0             16            28723.0
signed_return       30     near_gap_asymmetry_29a    16 -0.115333  -0.117357              0             16            28723.0
signed_return       30 weighted_gap_asymmetry_29a    16 -0.139039  -0.144079         

In [88]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 29B — FULL34 LIQUIDITY-GAP ABLATION
#
# Calendar research notebook
#
# Original 31 features + quote retreat + slope + curvature
# = 34 engineered baseline features.
#
# Plus the same 9 activity/price/depth controls as 28B.
#
# Evaluate:
#   Signed: fragility, weighted gap, both
#   Absret: gap severity
#
# Exact 30/60/300-second targets.
# Chronological expanding evaluation.
# Matched train/test rows within comparisons.
# Original df_1s is never modified.
# ============================================================

KEYS = ["symbol", "ts_sec"]
HORIZONS = [30, 60, 300]

FRAG = "fragility_asymmetry_29a"
WGAP = "weighted_gap_asymmetry_29a"
SEVERITY = "gap_severity_29a"

RETREAT = "quote_retreat_imbalance_30_24a"
SLOPE = "slope_asymmetry_28a"
CURV = "curvature_asymmetry_28a"

REQUIRED_FRAMES = [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp28a_features",
    "calendar_exp29a_features"
]

for name in REQUIRED_FRAMES:
    if name not in globals():
        raise ValueError(f"Missing {name}")

core_features = [
    c for c in core_31.columns if c not in KEYS
]

if len(core_features) != 31:
    raise ValueError(
        f"Expected 31 core features, got {len(core_features)}"
    )

controls = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

geometry_cols = [SLOPE, CURV] + controls

targets = (
    [f"future_return_{h}_29a" for h in HORIZONS]
    + [f"future_absret_{h}_29a" for h in HORIZONS]
)

# ------------------------------------------------------------
# 1. Reconstruct the full34 baseline
# ------------------------------------------------------------

x = calendar_exp29a_features[
    KEYS + ["date", FRAG, WGAP, SEVERITY] + targets
].copy()

sources = [
    (
        core_31[KEYS + core_features].copy(),
        "core"
    ),
    (
        calendar_exp24a_features[
            KEYS + [RETREAT]
        ].copy(),
        "retreat"
    ),
    (
        calendar_exp28a_features[
            KEYS + geometry_cols
        ].copy(),
        "geometry"
    )
]

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = x["ts_sec"].astype("int64")

if x.duplicated(KEYS).any():
    raise ValueError("Duplicate research keys")

for frame, label in sources:

    frame["symbol"] = frame["symbol"].astype(str)
    frame["ts_sec"] = frame["ts_sec"].astype("int64")

    if frame.duplicated(KEYS).any():
        raise ValueError(f"Duplicate keys in {label}")

    x = x.merge(
        frame,
        on=KEYS,
        how="left",
        validate="one_to_one",
        indicator=f"_{label}_match"
    )

    match = x[f"_{label}_match"].eq("both").mean()

    print(
        f"{label} match: {100 * match:.4f}%"
    )

    if match < 0.99:
        raise ValueError(f"Poor alignment: {label}")

    x.drop(
        columns=[f"_{label}_match"],
        inplace=True
    )

x["date"] = pd.to_datetime(x["date"]).dt.date

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

BASE_34 = list(dict.fromkeys(
    core_features
    + [RETREAT, SLOPE, CURV]
))

assert len(BASE_34) == 34

BASE = list(dict.fromkeys(BASE_34 + controls))

print("\n===== 29B BASELINE =====")
print("Engineered features:", len(BASE_34))
print("Total baseline inputs:", len(BASE))
print("Candidate inputs:", [FRAG, WGAP, SEVERITY])

# ------------------------------------------------------------
# 2. Same complete-input rows for every variant
# ------------------------------------------------------------

all_inputs = list(dict.fromkeys(
    BASE + [FRAG, WGAP, SEVERITY]
))

for col in all_inputs + targets:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

x["_complete"] = np.isfinite(
    x[all_inputs].to_numpy(dtype=np.float64)
).all(axis=1)

print("\n===== 29B DATA COVERAGE =====")
print("Research rows:", len(x))
print("Complete feature rows:", int(x["_complete"].sum()))
print(
    "Complete coverage:",
    round(100 * x["_complete"].mean(), 4), "%"
)

# ------------------------------------------------------------
# 3. Fixed LightGBM configuration
#    Matched to the original Experiment 28B.
# ------------------------------------------------------------

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

TASK_MODELS = {
    "signed": {
        "A_full34": BASE,
        "B_fragility": BASE + [FRAG],
        "C_weighted_gap": BASE + [WGAP],
        "D_both": BASE + [FRAG, WGAP]
    },
    "absret": {
        "A_full34": BASE,
        "B_gap_severity": BASE + [SEVERITY]
    }
}

dates = sorted(x["date"].unique())

if len(dates) <= 5:
    raise ValueError("Need more than five sessions")

records = []
importance_records = []

# ------------------------------------------------------------
# 4. Expanding train/test windows
# ------------------------------------------------------------

for i in range(5, len(dates)):

    test_date = dates[i]

    train_pool = x.loc[
        x["date"].isin(dates[:i])
        & x["_complete"]
        & x["ts_sec"].mod(10).eq(0)
    ]

    test_pool = x.loc[
        x["date"].eq(test_date)
        & x["_complete"]
    ]

    for task, models in TASK_MODELS.items():

        for h in HORIZONS:

            if task == "signed":
                target = f"future_return_{h}_29a"
            else:
                target = f"future_absret_{h}_29a"

            train = train_pool.loc[
                np.isfinite(train_pool[target])
            ]

            test = test_pool.loc[
                np.isfinite(test_pool[target])
            ]

            if len(train) < 1000 or len(test) < 200:
                continue

            y_train = train[target].to_numpy(
                dtype=np.float64
            )

            y_test = test[target].to_numpy(
                dtype=np.float64
            )

            for model_name, columns in models.items():

                model = lgb.LGBMRegressor(**PARAMS)

                model.fit(
                    train[columns],
                    y_train
                )

                pred = model.predict(
                    test[columns]
                )

                if task == "absret":
                    pred = np.maximum(pred, 0.0)

                error = y_test - pred

                records.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test),
                    "sse": float(np.sum(error**2)),
                    "sae": float(np.sum(np.abs(error))),
                    "mse": float(np.mean(error**2)),
                    "mae": float(np.mean(np.abs(error)))
                })

                gains = model.booster_.feature_importance(
                    importance_type="gain"
                )

                total_gain = float(gains.sum())

                for col, gain in zip(columns, gains):

                    if col not in [FRAG, WGAP, SEVERITY]:
                        continue

                    importance_records.append({
                        "date": test_date,
                        "task": task,
                        "horizon": h,
                        "model": model_name,
                        "feature": col,
                        "gain_share_pct": (
                            100 * gain / total_gain
                            if total_gain > 0 else 0.0
                        )
                    })

    print("Completed:", test_date)

calendar_exp29b_daily = pd.DataFrame(records)

calendar_exp29b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp29b_daily.empty:
    raise ValueError("No evaluations completed")

# ------------------------------------------------------------
# 5. Pooled MSE / MAE and day-level performance
# ------------------------------------------------------------

comparisons = {
    "signed": {
        "fragility_vs_full34": (
            "B_fragility", "A_full34"
        ),
        "weighted_vs_full34": (
            "C_weighted_gap", "A_full34"
        ),
        "both_vs_full34": (
            "D_both", "A_full34"
        ),
        "both_vs_fragility": (
            "D_both", "B_fragility"
        ),
        "both_vs_weighted": (
            "D_both", "C_weighted_gap"
        )
    },
    "absret": {
        "severity_vs_full34": (
            "B_gap_severity", "A_full34"
        )
    }
}

summary = []
daily_gains = []

for (task, h), block in (
    calendar_exp29b_daily.groupby(["task", "horizon"])
):

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    for label, (candidate, baseline) in (
        comparisons[task].items()
    ):

        if candidate not in wide or baseline not in wide:
            continue

        daily = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in daily.items():

            daily_gains.append({
                "date": date,
                "task": task,
                "horizon": h,
                "comparison": label,
                "gain_pct": float(gain)
            })

        summary.append({
            "task": task,
            "horizon": h,
            "comparison": label,
            "days": len(daily),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((daily > 0).sum()),
            "median_daily_gain_pct": float(
                daily.median()
            ),
            "worst_day_pct": float(daily.min())
        })

calendar_exp29b_summary = pd.DataFrame(summary)

calendar_exp29b_daily_gains = pd.DataFrame(
    daily_gains
)

# ------------------------------------------------------------
# 6. Output
# ------------------------------------------------------------

print("\n===== 29B SIGNED RETURN RESULTS =====")

print(
    calendar_exp29b_summary.loc[
        calendar_exp29b_summary.task.eq("signed")
    ].round(6).to_string(index=False)
)

print("\n===== 29B ABSRET RESULTS =====")

print(
    calendar_exp29b_summary.loc[
        calendar_exp29b_summary.task.eq("absret")
    ].round(6).to_string(index=False)
)

print("\n===== 29B DAILY GAINS: MAIN CANDIDATES =====")

for task, comparison in [
    ("signed", "fragility_vs_full34"),
    ("signed", "both_vs_full34"),
    ("absret", "severity_vs_full34")
]:

    subset = calendar_exp29b_daily_gains.loc[
        calendar_exp29b_daily_gains.task.eq(task)
        & calendar_exp29b_daily_gains.comparison.eq(
            comparison
        )
    ]

    print("\n", task, comparison)

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="gain_pct"
        ).round(6).to_string()
    )

print("\n===== 29B CANDIDATE IMPORTANCE =====")

if not calendar_exp29b_importance.empty:

    print(
        calendar_exp29b_importance.groupby(
            ["task", "horizon", "model", "feature"]
        )["gain_share_pct"]
        .median()
        .reset_index()
        .round(5)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 29B COMPLETE =====")
print("Original df_1s unchanged.")
print("Feature count remains 34.")


core match: 100.0000%
retreat match: 100.0000%
geometry match: 100.0000%

===== 29B BASELINE =====
Engineered features: 34
Total baseline inputs: 43
Candidate inputs: ['fragility_asymmetry_29a', 'weighted_gap_asymmetry_29a', 'gap_severity_29a']

===== 29B DATA COVERAGE =====
Research rows: 459122
Complete feature rows: 402979
Complete coverage: 87.7717 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 29B SIGNED RETURN RESULTS =====
  task  horizon          comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
signed       30 fragility_vs_full34    11             0.695944            -0.451379            10               0.776373      -0.112882
signed       30  weighted_vs_full34    11             0.993962            -0.622780       

In [89]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 29C — ERROR DISTRIBUTION AND MODEL ROBUSTNESS
# 30-second signed returns only.
#
# Requires Experiment 29B variables:
#   x, BASE, FRAG, WGAP
#
# No feature promotion. No mutation of df_1s.
# ============================================================

TARGET = "future_return_30_29a"

MODELS = {
    "baseline": BASE,
    "fragility": BASE + [FRAG],
    "weighted": BASE + [WGAP],
    "both": BASE + [FRAG, WGAP],
}

CONFIGS = {
    "original": {
        "n_estimators": 180,
        "learning_rate": 0.035,
        "num_leaves": 15,
        "max_depth": 5,
        "min_child_samples": 200,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
    },
    "regularized": {
        "n_estimators": 140,
        "learning_rate": 0.035,
        "num_leaves": 7,
        "max_depth": 3,
        "min_child_samples": 400,
        "reg_lambda": 8.0,
        "reg_alpha": 0.5,
    },
}

dates = sorted(x["date"].unique())
predictions = []

for config_name, params in CONFIGS.items():

    for i in range(5, len(dates)):

        train = x.loc[
            x["date"].isin(dates[:i])
            & x["_complete"]
            & x["ts_sec"].mod(10).eq(0)
            & np.isfinite(x[TARGET])
        ]

        test = x.loc[
            x["date"].eq(dates[i])
            & x["_complete"]
            & np.isfinite(x[TARGET])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        actual = test[TARGET].to_numpy(dtype=float)

        output = test[
            ["date", "symbol", "ts_sec"]
        ].copy()

        output["config"] = config_name
        output["actual"] = actual

        for name, columns in MODELS.items():

            model = lgb.LGBMRegressor(
                objective="regression",
                colsample_bytree=1.0,
                subsample=1.0,
                random_state=42,
                n_jobs=1,
                verbosity=-1,
                **params
            )

            model.fit(
                train[columns],
                train[TARGET]
            )

            output[f"pred_{name}"] = (
                model.predict(test[columns])
            )

        predictions.append(output)

        print(config_name, "completed", dates[i])

calendar_exp29c_predictions = pd.concat(
    predictions, ignore_index=True
)

# ------------------------------------------------------------
# 1. Overall pooled and daily performance
# ------------------------------------------------------------

results = []

for config, group in (
    calendar_exp29c_predictions.groupby("config")
):

    base_err = (
        group["actual"] - group["pred_baseline"]
    ).to_numpy()

    daily_base = (
        group.assign(error2=base_err**2)
        .groupby("date")["error2"].sum()
    )

    for name in ["fragility", "weighted", "both"]:

        err = (
            group["actual"] - group[f"pred_{name}"]
        ).to_numpy()

        daily_cand = (
            group.assign(error2=err**2)
            .groupby("date")["error2"].sum()
        )

        daily_gain = 100 * (
            1 - daily_cand / daily_base
        )

        results.append({
            "config": config,
            "candidate": name,
            "mse_gain_pct": 100 * (
                1 - np.sum(err**2)
                / np.sum(base_err**2)
            ),
            "mae_gain_pct": 100 * (
                1 - np.sum(np.abs(err))
                / np.sum(np.abs(base_err))
            ),
            "winning_days": int(
                (daily_gain > 0).sum()
            ),
            "worst_day_pct": daily_gain.min(),
        })

calendar_exp29c_summary = pd.DataFrame(results)

print("\n===== 29C MODEL ROBUSTNESS =====")
print(
    calendar_exp29c_summary
    .round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 2. Error decomposition by realized return magnitude
#
# Bins defined within each training fold using only
# historically available training targets.
#
# We must NOT define these thresholds using future
# test-session targets.
# ------------------------------------------------------------

# Reconstruct fold-specific magnitude thresholds.
thresholds = {}

for i in range(5, len(dates)):

    train = x.loc[
        x["date"].isin(dates[:i])
        & x["_complete"]
        & x["ts_sec"].mod(10).eq(0)
        & np.isfinite(x[TARGET])
    ]

    if len(train) < 1000:
        continue

    magnitude = train[TARGET].abs()

    thresholds[dates[i]] = (
        float(magnitude.quantile(0.50)),
        float(magnitude.quantile(0.90)),
    )

def magnitude_bin(row):
    p50, p90 = thresholds[row["date"]]
    v = abs(row["actual"])

    if v <= p50:
        return "small_0_50"
    elif v <= p90:
        return "medium_50_90"
    return "large_90_100"

calendar_exp29c_predictions["magnitude_bin"] = (
    calendar_exp29c_predictions.apply(
        magnitude_bin, axis=1
    )
)

segment_results = []

for (config, segment), group in (
    calendar_exp29c_predictions.groupby(
        ["config", "magnitude_bin"]
    )
):

    actual = group["actual"].to_numpy()
    base_err = (
        actual - group["pred_baseline"].to_numpy()
    )

    for candidate in ["fragility", "weighted", "both"]:

        err = (
            actual - group[f"pred_{candidate}"].to_numpy()
        )

        segment_results.append({
            "config": config,
            "segment": segment,
            "candidate": candidate,
            "rows": len(group),
            "mse_gain_pct": 100 * (
                1 - np.sum(err**2)
                / np.sum(base_err**2)
            ),
            "mae_gain_pct": 100 * (
                1 - np.sum(np.abs(err))
                / np.sum(np.abs(base_err))
            ),
        })

calendar_exp29c_segments = pd.DataFrame(
    segment_results
)

print("\n===== 29C ERROR BY RETURN MAGNITUDE =====")
print(
    calendar_exp29c_segments
    .round(6).to_string(index=False)
)

print("\n===== EXPERIMENT 29C COMPLETE =====")
print("No new features selected.")


original completed 2026-10-05
original completed 2026-10-06
original completed 2026-10-07
original completed 2026-10-08
original completed 2026-10-09
original completed 2026-10-12
original completed 2026-10-13
original completed 2026-10-14
original completed 2026-10-15
original completed 2026-10-16
original completed 2026-10-19
regularized completed 2026-10-05
regularized completed 2026-10-06
regularized completed 2026-10-07
regularized completed 2026-10-08
regularized completed 2026-10-09
regularized completed 2026-10-12
regularized completed 2026-10-13
regularized completed 2026-10-14
regularized completed 2026-10-15
regularized completed 2026-10-16
regularized completed 2026-10-19

===== 29C MODEL ROBUSTNESS =====
     config candidate  mse_gain_pct  mae_gain_pct  winning_days  worst_day_pct
   original fragility      0.695944     -0.451379            10      -0.112882
   original  weighted      0.993962     -0.622780            11       0.228196
   original      both      1.129942 

In [90]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 29D — ACTIVITY-CONDITIONED GAP PERFORMANCE
#
# Uses:
#   calendar_exp29c_predictions
#   x from Experiment 29B
#
# Activity states computed using TRAIN-DATE quantiles.
# No future-return magnitude used for regime selection.
# No retraining or changes to df_1s.
# ============================================================

STATE_COLS = [
    "ofi_activity_60s",
    "update_rate_300s",
    "log_rv_300s_new",
]

KEYS = ["symbol", "ts_sec"]
DATE_COL = "date"

required_x = KEYS + [DATE_COL] + STATE_COLS

missing = [c for c in required_x if c not in x.columns]
if missing:
    raise ValueError(f"Missing state columns: {missing}")

states = x[required_x].copy()

if states.duplicated(KEYS).any():
    raise ValueError("Duplicate keys in x")

pred = calendar_exp29c_predictions.copy()

pred = pred.merge(
    states[KEYS + STATE_COLS],
    on=KEYS,
    how="left",
    validate="many_to_one"
)

pred["date"] = pd.to_datetime(pred["date"]).dt.date

dates = sorted(x[DATE_COL].unique())

# ------------------------------------------------------------
# Historical state boundaries
# ------------------------------------------------------------

thresholds = {}

for i in range(5, len(dates)):

    test_date = dates[i]

    train = x.loc[
        x[DATE_COL].isin(dates[:i])
        & x["_complete"]
        & x["ts_sec"].mod(10).eq(0)
        & np.isfinite(x["future_return_30_29a"])
    ]

    thresholds[test_date] = {}

    for col in STATE_COLS:

        values = pd.to_numeric(
            train[col], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan).dropna()

        if len(values) == 0:
            raise ValueError(
                f"No training values for {col} on {test_date}"
            )

        thresholds[test_date][col] = (
            float(values.quantile(0.33)),
            float(values.quantile(0.67))
        )

def assign_state(value, date, column):

    if not np.isfinite(value):
        return "missing"

    q33, q67 = thresholds[date][column]

    if value <= q33:
        return "low"
    if value <= q67:
        return "medium"
    return "high"

records = []

for column in STATE_COLS:

    pred["_state"] = [
        assign_state(v, d, column)
        for v, d in zip(pred[column], pred["date"])
    ]

    for (config, state), group in pred.groupby(
        ["config", "_state"], sort=False
    ):

        if state == "missing" or len(group) < 200:
            continue

        y = group["actual"].to_numpy(dtype=float)

        base = group[
            "pred_baseline"
        ].to_numpy(dtype=float)

        base_error = y - base
        base_sse = np.sum(base_error**2)
        base_sae = np.sum(np.abs(base_error))

        if base_sse <= 0 or base_sae <= 0:
            continue

        for candidate in ["fragility", "weighted", "both"]:

            p = group[
                f"pred_{candidate}"
            ].to_numpy(dtype=float)

            error = y - p

            records.append({
                "state_feature": column,
                "state": state,
                "config": config,
                "candidate": candidate,
                "rows": len(group),
                "sessions": group["date"].nunique(),
                "mse_gain_pct": 100 * (
                    1 - np.sum(error**2) / base_sse
                ),
                "mae_gain_pct": 100 * (
                    1 - np.sum(np.abs(error)) / base_sae
                )
            })

calendar_exp29d_summary = pd.DataFrame(records)

print("\n===== 29D ACTIVITY-CONDITIONED RESULTS =====")

for config in ["original", "regularized"]:

    print("\nMODEL:", config)

    result = calendar_exp29d_summary.loc[
        calendar_exp29d_summary["config"].eq(config)
    ]

    print(
        result.sort_values(
            ["state_feature", "candidate", "state"]
        ).round(6).to_string(index=False)
    )

print("\n===== EXPERIMENT 29D COMPLETE =====")
print("Feature count unchanged.")



===== 29D ACTIVITY-CONDITIONED RESULTS =====

MODEL: original
   state_feature  state   config candidate   rows  sessions  mse_gain_pct  mae_gain_pct
 log_rv_300s_new   high original      both 127159        11      1.305531     -0.618447
 log_rv_300s_new    low original      both  70869        11      2.176941      0.036717
 log_rv_300s_new medium original      both  82453        11      0.077392     -1.623985
 log_rv_300s_new   high original fragility 127159        11      0.780251     -0.299786
 log_rv_300s_new    low original fragility  70869        11      0.938888     -0.211648
 log_rv_300s_new medium original fragility  82453        11      0.301894     -0.943080
 log_rv_300s_new   high original  weighted 127159        11      1.130154     -0.476727
 log_rv_300s_new    low original  weighted  70869        11      1.798503     -0.603063
 log_rv_300s_new medium original  weighted  82453        11      0.180805     -0.970880
ofi_activity_60s   high original      both 105193        

In [91]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 29E — ACTIVITY-GATED GAP PREDICTIONS
#
# 30-second signed-return forecasting.
#
# Compare:
# 1. Baseline model
# 2. Always use both gap features
# 3. Use gap model only in high OFI activity
# 4. Use gap model only in high quote-update activity
#
# Uses already fitted 29C model predictions.
# Thresholds determined from preceding training dates.
#
# IMPORTANT: Retrospective diagnostic on previously
# explored evaluation sessions, not untouched OOS.
# ============================================================

required = [
    "ofi_activity_60s",
    "update_rate_300s",
    "future_return_30_29a",
    "_complete",
    "date",
    "ts_sec",
    "symbol"
]

missing = [c for c in required if c not in x.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

state_data = x[
    ["symbol", "ts_sec",
     "ofi_activity_60s",
     "update_rate_300s"]
].copy()

if state_data.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate state keys")

pred = calendar_exp29c_predictions.merge(
    state_data,
    on=["symbol", "ts_sec"],
    how="left",
    validate="many_to_one"
)

pred["date"] = pd.to_datetime(pred["date"]).dt.date

dates = sorted(pd.to_datetime(x["date"]).dt.date.unique())

# ------------------------------------------------------------
# Training-only activity thresholds
# ------------------------------------------------------------

thresholds = {}

for i in range(5, len(dates)):

    test_date = dates[i]

    train = x.loc[
        pd.to_datetime(x["date"]).dt.date.isin(dates[:i])
        & x["_complete"]
        & x["ts_sec"].mod(10).eq(0)
        & np.isfinite(x["future_return_30_29a"])
    ]

    thresholds[test_date] = {}

    for feature in [
        "ofi_activity_60s",
        "update_rate_300s"
    ]:

        valid = pd.to_numeric(
            train[feature], errors="coerce"
        ).replace(
            [np.inf, -np.inf], np.nan
        ).dropna()

        if valid.empty:
            raise ValueError(
                f"No historical values for {feature}"
            )

        thresholds[test_date][feature] = (
            float(valid.quantile(0.67))
        )

# ------------------------------------------------------------
# Create predictions
# ------------------------------------------------------------

pred["pred_always_gap"] = pred["pred_both"]

for feature, label in [
    ("ofi_activity_60s", "ofi"),
    ("update_rate_300s", "updates")
]:

    cutoff = pred["date"].map(
        {
            date: values[feature]
            for date, values in thresholds.items()
        }
    ).to_numpy(dtype=float)

    state = pred[feature].to_numpy(dtype=float)

    active = np.isfinite(state) & (state > cutoff)

    pred[f"active_{label}"] = active

    pred[f"pred_gate_{label}"] = np.where(
        active,
        pred["pred_both"],
        pred["pred_baseline"]
    )

# ------------------------------------------------------------
# Error metrics
# ------------------------------------------------------------

MODEL_COLUMNS = {
    "baseline": "pred_baseline",
    "always_gap": "pred_always_gap",
    "high_ofi_gate": "pred_gate_ofi",
    "high_update_gate": "pred_gate_updates"
}

records = []
daily_records = []

for config, data in pred.groupby("config"):

    y = data["actual"].to_numpy(dtype=float)

    base_err = y - data["pred_baseline"].to_numpy()
    base_sse = np.sum(base_err**2)
    base_sae = np.sum(np.abs(base_err))

    for model_name, column in MODEL_COLUMNS.items():

        err = y - data[column].to_numpy(dtype=float)

        records.append({
            "config": config,
            "model": model_name,
            "rows": len(data),
            "mse_gain_pct": 100 * (
                1 - np.sum(err**2) / base_sse
            ),
            "mae_gain_pct": 100 * (
                1 - np.sum(np.abs(err)) / base_sae
            )
        })

        for day, part in data.groupby("date"):

            actual = part["actual"].to_numpy()

            reference = (
                actual - part["pred_baseline"].to_numpy()
            )

            candidate = (
                actual - part[column].to_numpy()
            )

            daily_records.append({
                "config": config,
                "date": day,
                "model": model_name,
                "mse_gain_pct": 100 * (
                    1 - np.sum(candidate**2)
                    / np.sum(reference**2)
                ),
                "mae_gain_pct": 100 * (
                    1 - np.sum(np.abs(candidate))
                    / np.sum(np.abs(reference))
                )
            })

calendar_exp29e_summary = pd.DataFrame(records)
calendar_exp29e_daily = pd.DataFrame(daily_records)
calendar_exp29e_predictions = pred

stability = (
    calendar_exp29e_daily.groupby(
        ["config", "model"]
    )
    .agg(
        winning_days=(
            "mse_gain_pct",
            lambda s: int((s > 0).sum())
        ),
        worst_day_pct=("mse_gain_pct", "min"),
        median_daily_gain_pct=("mse_gain_pct", "median")
    )
    .reset_index()
)

print("\n===== 29E GATED MODEL COMPARISON =====")

print(
    calendar_exp29e_summary.merge(
        stability,
        on=["config", "model"],
        how="left"
    ).round(6).to_string(index=False)
)

print("\n===== 29E DAILY GATED MSE GAINS =====")

for config in ["original", "regularized"]:

    print("\n", config)

    subset = calendar_exp29e_daily.loc[
        calendar_exp29e_daily["config"].eq(config)
    ]

    print(
        subset.pivot(
            index="date",
            columns="model",
            values="mse_gain_pct"
        ).round(5).to_string()
    )

print("\n===== 29E GATE ACTIVATION =====")

for config, part in pred.groupby("config"):

    print(
        config,
        "High OFI active:",
        round(100*part["active_ofi"].mean(), 3), "%",
        "High updates active:",
        round(100*part["active_updates"].mean(), 3), "%"
    )

print("\n===== EXPERIMENT 29E COMPLETE =====")
print("No feature promotion.")
print("Research feature count remains 34.")



===== 29E GATED MODEL COMPARISON =====
     config            model   rows  mse_gain_pct  mae_gain_pct  winning_days  worst_day_pct  median_daily_gain_pct
   original         baseline 280481      0.000000      0.000000             0       0.000000               0.000000
   original       always_gap 280481      1.129942     -0.776465            11       0.552825               1.120462
   original    high_ofi_gate 280481      0.943778      0.170917             9      -0.076297               0.893035
   original high_update_gate 280481      0.997912      0.036073            11       0.406027               0.984722
regularized         baseline 280481      0.000000      0.000000             0       0.000000               0.000000
regularized       always_gap 280481      1.272862     -1.097413            11       0.367853               1.166412
regularized    high_ofi_gate 280481      0.912169      0.038731            11       0.005349               0.755674
regularized high_update_gate 280

In [92]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 30A — L1 QUEUE DEPLETION / REPLENISHMENT
#
# Input:
#   raw df_1s with:
#       symbol, ts_sec
#       bid_p1, ask_p1, bid_q1, ask_q1
#
# Output:
#   calendar_exp30a_features
#   calendar_exp30a_summary
#   calendar_exp30a_daily
#   calendar_exp30a_redundancy
#
# Strict causality and session isolation.
# No changes to original df_1s.
# ============================================================

W_30A = 60
HORIZONS_30A = [30, 60, 300]
MIN_COVERAGE_30A = 0.97
MIN_WINDOW_OBS_30A = 54
MIN_VALID_TRANS_30A = 12
EPS_30A = 1e-12

KEYS_30A = ["symbol", "ts_sec"]

RAW_30A = [
    "symbol", "ts_sec",
    "bid_p1", "ask_p1",
    "bid_q1", "ask_q1"
]

missing = sorted(
    set(RAW_30A) - set(df_1s.columns)
)

if missing:
    raise ValueError(f"Missing columns: {missing}")

x30 = df_1s[RAW_30A].copy()

x30["symbol"] = x30["symbol"].astype(str)

for c in RAW_30A:
    if c != "symbol":
        x30[c] = pd.to_numeric(
            x30[c], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

if x30["ts_sec"].isna().any():
    raise ValueError("Invalid ts_sec values.")

x30["ts_sec"] = x30["ts_sec"].astype("int64")

if x30.duplicated(KEYS_30A).any():
    raise ValueError("Duplicate symbol/timestamp keys.")

# ------------------------------------------------------------
# 1. IST session eligibility
# ------------------------------------------------------------

ist = pd.to_datetime(
    x30["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x30["date"] = ist.dt.date

x30["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x30 = x30.loc[
    x30["clock_seconds"].between(0, 28799)
].copy()

coverage = (
    x30.groupby(["symbol", "date"])["ts_sec"]
    .nunique()
    .reset_index(name="n_seconds")
)

eligible = coverage.loc[
    coverage["n_seconds"].ge(
        MIN_COVERAGE_30A * 28800
    )
    & coverage["date"].astype(str).ne("2026-09-28"),
    ["symbol", "date"]
]

x30 = (
    x30.merge(
        eligible,
        on=["symbol", "date"],
        how="inner",
        validate="many_to_one"
    )
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

print("===== 30A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x30))

if x30.empty:
    raise ValueError("No eligible trading sessions.")

# ------------------------------------------------------------
# 2. Current and previous L1 quotes
# ------------------------------------------------------------

G = ["symbol", "date"]
grouped = x30.groupby(G, sort=False)

previous_ts = grouped["ts_sec"].shift(1)

consecutive = (
    x30["ts_sec"] - previous_ts
).eq(1)

valid_bid = (
    consecutive
    & x30["bid_p1"].gt(0)
    & x30["bid_q1"].ge(0)
    & grouped["bid_p1"].shift(1).gt(0)
    & grouped["bid_q1"].shift(1).ge(0)
    & x30["bid_p1"].eq(
        grouped["bid_p1"].shift(1)
    )
    & x30["ask_p1"].ge(x30["bid_p1"])
)

valid_ask = (
    consecutive
    & x30["ask_p1"].gt(0)
    & x30["ask_q1"].ge(0)
    & grouped["ask_p1"].shift(1).gt(0)
    & grouped["ask_q1"].shift(1).ge(0)
    & x30["ask_p1"].eq(
        grouped["ask_p1"].shift(1)
    )
    & x30["ask_p1"].ge(x30["bid_p1"])
)

bid_dq = (
    x30["bid_q1"]
    - grouped["bid_q1"].shift(1)
).where(valid_bid)

ask_dq = (
    x30["ask_q1"]
    - grouped["ask_q1"].shift(1)
).where(valid_ask)

# Missing remains missing for invalid transitions.
x30["_bid_depletion"] = (-bid_dq).clip(lower=0)
x30["_ask_depletion"] = (-ask_dq).clip(lower=0)

x30["_bid_refill"] = bid_dq.clip(lower=0)
x30["_ask_refill"] = ask_dq.clip(lower=0)

x30["_bid_down"] = (
    bid_dq.lt(0).astype(float).where(valid_bid)
)

x30["_ask_down"] = (
    ask_dq.lt(0).astype(float).where(valid_ask)
)

x30["_bid_valid"] = valid_bid.astype(float)
x30["_ask_valid"] = valid_ask.astype(float)

print("\n===== 30A TRANSITION VALIDITY =====")
print("Consecutive %:", round(100*consecutive.mean(), 3))
print("Bid same-price %:", round(100*valid_bid.mean(), 3))
print("Ask same-price %:", round(100*valid_ask.mean(), 3))

# ------------------------------------------------------------
# 3. Clock-time rolling sums within each session
#
# Rolling window (t-60s, t], so current + preceding
# 59 calendar seconds are eligible for inclusion.
#
# No forward look, no cross-day contamination.
# ------------------------------------------------------------

ROLL_COLS = [
    "_bid_depletion", "_ask_depletion",
    "_bid_refill", "_ask_refill",
    "_bid_down", "_ask_down",
    "_bid_valid", "_ask_valid"
]

rolled_parts = []

for (_, _), part in x30.groupby(G, sort=False):

    ts_index = pd.to_datetime(
        part["ts_sec"].to_numpy(),
        unit="s",
        utc=True
    )

    work = part[ROLL_COLS].copy()
    work.index = ts_index

    roll = work.rolling(
        f"{W_30A}s",
        min_periods=1,
        closed="right"
    ).sum()

    roll.columns = [
        f"{c}_sum" for c in roll.columns
    ]

    observed = pd.Series(
        1.0,
        index=ts_index
    ).rolling(
        f"{W_30A}s",
        min_periods=1,
        closed="right"
    ).sum()

    roll["_observed_seconds"] = observed.to_numpy()

    # Restore original row identifiers.
    roll.index = part.index

    rolled_parts.append(roll)

rolling30 = pd.concat(
    rolled_parts
).sort_index()

for col in rolling30.columns:
    x30[col] = rolling30[col].to_numpy()

# ------------------------------------------------------------
# 4. Window validity requirements
# ------------------------------------------------------------

Db = x30["_bid_depletion_sum"].to_numpy()
Da = x30["_ask_depletion_sum"].to_numpy()

Rb = x30["_bid_refill_sum"].to_numpy()
Ra = x30["_ask_refill_sum"].to_numpy()

Vb = x30["_bid_valid_sum"].to_numpy()
Va = x30["_ask_valid_sum"].to_numpy()

valid_window = (
    x30["_observed_seconds"].ge(
        MIN_WINDOW_OBS_30A
    ).to_numpy()
    & (Vb >= MIN_VALID_TRANS_30A)
    & (Va >= MIN_VALID_TRANS_30A)
)

# ------------------------------------------------------------
# 5. Candidate features
# ------------------------------------------------------------

# 1. Ask depletion - bid depletion.
depletion_pressure = (
    (Da - Db) / (Da + Db + EPS_30A)
)

# 2. Bid refill - ask refill.
refill_pressure = (
    (Rb - Ra) / (Rb + Ra + EPS_30A)
)

# 3. Net directional quote change.
net_queue_pressure = (
    ((Rb - Db) - (Ra - Da))
    / (Rb + Db + Ra + Da + EPS_30A)
)

# 4. Frequency of depletion on each side.
with np.errstate(divide="ignore", invalid="ignore"):

    persistence = (
        x30["_ask_down_sum"].to_numpy() / Va
        - x30["_bid_down_sum"].to_numpy() / Vb
    )

# 5. Each side's refill/depletion balance.
refill_balance = (
    (Rb - Db) / (Rb + Db + EPS_30A)
    - (Ra - Da) / (Ra + Da + EPS_30A)
)

CANDIDATES_30A = {
    "depletion_pressure_60_30a": depletion_pressure,
    "refill_pressure_60_30a": refill_pressure,
    "net_queue_pressure_60_30a": net_queue_pressure,
    "depletion_persistence_60_30a": persistence,
    "refill_balance_asymmetry_60_30a": refill_balance
}

for name, values in CANDIDATES_30A.items():

    values = np.asarray(
        values, dtype=np.float64
    ).copy()

    values[~valid_window] = np.nan
    values[~np.isfinite(values)] = np.nan

    x30[name] = values

FEATURES_30A = list(CANDIDATES_30A)

# ------------------------------------------------------------
# 6. Exact future midprice labels
# ------------------------------------------------------------

mid = (
    x30["bid_p1"] + x30["ask_p1"]
) / 2.0

valid_mid = (
    x30["bid_p1"].gt(0)
    & x30["ask_p1"].gt(0)
    & x30["ask_p1"].ge(x30["bid_p1"])
)

x30["_mid"] = mid.where(valid_mid)

for h in HORIZONS_30A:

    future = x30[
        ["symbol", "date", "ts_sec", "_mid"]
    ].copy()

    future["ts_sec"] -= h

    future.rename(
        columns={"_mid": f"_future_mid_{h}"},
        inplace=True
    )

    x30 = x30.merge(
        future,
        on=["symbol", "date", "ts_sec"],
        how="left",
        validate="one_to_one"
    )

    target = f"future_return_{h}_30a"

    with np.errstate(divide="ignore", invalid="ignore"):
        x30[target] = (
            1e4 * np.log(
                x30[f"_future_mid_{h}"]
                / x30["_mid"]
            )
        )

# ------------------------------------------------------------
# 7. Attach selected existing features for redundancy
# ------------------------------------------------------------

CONTROL_FEATURES_30A = []

source_frames = [
    (
        "core_31",
        [
            "ofi_l1_sum_60s_signed_log",
            "queue_replenishment_pressure_60s",
            "queue_imbalance_delta",
            "depth_imbalance_delta",
            "ofi_activity_60s"
        ]
    ),
    (
        "calendar_exp29a_features",
        [
            "weighted_gap_asymmetry_29a",
            "fragility_asymmetry_29a"
        ]
    )
]

for frame_name, requested in source_frames:

    if frame_name not in globals():
        continue

    source = globals()[frame_name]

    available = [
        c for c in requested if c in source.columns
    ]

    if not available:
        continue

    add = source[
        KEYS_30A + available
    ].copy()

    add["symbol"] = add["symbol"].astype(str)
    add["ts_sec"] = add["ts_sec"].astype("int64")

    if add.duplicated(KEYS_30A).any():
        raise ValueError(
            f"Duplicate keys in {frame_name}"
        )

    x30 = x30.merge(
        add,
        on=KEYS_30A,
        how="left",
        validate="one_to_one"
    )

    CONTROL_FEATURES_30A.extend(available)

# ------------------------------------------------------------
# 8. Daily Spearman IC
# ------------------------------------------------------------

def daily_spearman_30a(feature, target):

    good = np.isfinite(feature) & np.isfinite(target)

    n = int(good.sum())

    if n < 200:
        return np.nan, n

    a = pd.Series(feature[good]).rank().to_numpy()
    b = pd.Series(target[good]).rank().to_numpy()

    if a.std() < 1e-12 or b.std() < 1e-12:
        return np.nan, n

    return float(np.corrcoef(a, b)[0, 1]), n

records = []

for (symbol, date), day in x30.groupby(
    G, sort=False
):

    for h in HORIZONS_30A:

        y = day[
            f"future_return_{h}_30a"
        ].to_numpy(dtype=float)

        for feature in FEATURES_30A:

            v = day[feature].to_numpy(dtype=float)

            ic, n = daily_spearman_30a(v, y)

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "feature": feature,
                "ic": ic,
                "valid_rows": n
            })

calendar_exp30a_daily = pd.DataFrame(records)

calendar_exp30a_summary = (
    calendar_exp30a_daily
    .groupby(["horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 9. Redundancy
# ------------------------------------------------------------

calendar_exp30a_redundancy = (
    x30[
        FEATURES_30A + CONTROL_FEATURES_30A
    ]
    .corr(method="spearman")
    .loc[
        FEATURES_30A,
        CONTROL_FEATURES_30A
    ]
    if CONTROL_FEATURES_30A
    else pd.DataFrame()
)

# ------------------------------------------------------------
# 10. Diagnostics
# ------------------------------------------------------------

print("\n===== 30A FEATURE VALIDITY =====")

print("Valid rolling windows:", int(valid_window.sum()))
print(
    "Valid window coverage:",
    round(100 * valid_window.mean(), 4), "%"
)

for c in FEATURES_30A:

    s = x30[c]

    print(
        c,
        "valid_pct=", round(100*s.notna().mean(), 3),
        "zero_pct=", round(100*s.eq(0).mean(), 3),
        "std=", round(s.std(), 6)
    )

print("\n===== 30A DAILY SIGNED IC =====")

print(
    calendar_exp30a_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 30A REDUNDANCY =====")

if not calendar_exp30a_redundancy.empty:
    print(
        calendar_exp30a_redundancy
        .round(5)
        .to_string()
    )
else:
    print("No previous feature controls attached.")

calendar_exp30a_features = x30

print("\n===== EXPERIMENT 30A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")
print("Core feature count: 36.")


===== 30A SAMPLE =====
Sessions: 16
Rows: 459122

===== 30A TRANSITION VALIDITY =====
Consecutive %: 99.997
Bid same-price %: 96.015
Ask same-price %: 94.783

===== 30A FEATURE VALIDITY =====
Valid rolling windows: 458158
Valid window coverage: 99.79 %
depletion_pressure_60_30a valid_pct= 99.79 zero_pct= 22.918 std= 0.751871
refill_pressure_60_30a valid_pct= 99.79 zero_pct= 32.233 std= 0.744133
net_queue_pressure_60_30a valid_pct= 99.79 zero_pct= 16.361 std= 0.645323
depletion_persistence_60_30a valid_pct= 99.79 zero_pct= 24.952 std= 0.079422
refill_balance_asymmetry_60_30a valid_pct= 99.79 zero_pct= 16.266 std= 0.925808

===== 30A DAILY SIGNED IC =====
 horizon                         feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
      30    depletion_persistence_60_30a    16 -0.026966  -0.029631              3             13            28667.0
      30       depletion_pressure_60_30a    16 -0.015473  -0.009770              5             11       

In [93]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 30B — FULL 36-FEATURE ABLATION
#
# Uses existing research DataFrames:
#   core_31
#   calendar_exp24a_features
#   calendar_exp28a_features
#   calendar_exp29a_features
#   calendar_exp30a_features
#
# Tests:
#   Refill pressure
#   Refill balance asymmetry
#   Depletion persistence
#   Net queue pressure
#   Refill balance + depletion persistence
#
# Targets: exact 30s, 60s, 300s signed returns.
# Baseline: 36 engineered features + 9 controls.
#
# No mutation of df_1s.
# ============================================================

KEYS = ["symbol", "ts_sec"]
HORIZONS = [30, 60, 300]

RETREAT = "quote_retreat_imbalance_30_24a"

GEOMETRY = [
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a"
]

GAPS = [
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a"
]

CANDIDATES = [
    "refill_pressure_60_30a",
    "refill_balance_asymmetry_60_30a",
    "depletion_persistence_60_30a",
    "net_queue_pressure_60_30a"
]

CONTROLS = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

TARGETS = [
    f"future_return_{h}_30a"
    for h in HORIZONS
]

required_frames = [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp28a_features",
    "calendar_exp29a_features",
    "calendar_exp30a_features"
]

for name in required_frames:
    if name not in globals():
        raise ValueError(f"Missing {name}")

CORE = [
    c for c in core_31.columns
    if c not in KEYS
]

if len(CORE) != 31 or len(set(CORE)) != 31:
    raise ValueError("Expected 31 unique core features")

# ------------------------------------------------------------
# 1. Source alignment
# ------------------------------------------------------------

def keyed_source(frame, columns, label):

    missing = set(KEYS + columns) - set(frame.columns)

    if missing:
        raise ValueError(
            f"{label} missing columns: {sorted(missing)}"
        )

    result = frame[KEYS + columns].copy()

    result["symbol"] = result["symbol"].astype(str)
    result["ts_sec"] = pd.to_numeric(
        result["ts_sec"], errors="raise"
    ).astype("int64")

    if result.duplicated(KEYS).any():
        raise ValueError(
            f"Duplicate symbol/timestamp keys in {label}"
        )

    return result

x30b = keyed_source(
    calendar_exp30a_features,
    ["date"] + CANDIDATES + TARGETS,
    "30A"
)

sources = [
    ("core", core_31, CORE),
    ("retreat", calendar_exp24a_features, [RETREAT]),
    ("geometry", calendar_exp28a_features,
     GEOMETRY + CONTROLS),
    ("gaps", calendar_exp29a_features, GAPS)
]

for label, frame, columns in sources:

    source = keyed_source(frame, columns, label)

    x30b = x30b.merge(
        source,
        on=KEYS,
        how="left",
        validate="one_to_one",
        indicator=f"_{label}_match"
    )

    matched = x30b[f"_{label}_match"].eq("both").mean()

    print(
        f"{label} source match: {100*matched:.4f}%"
    )

    if matched < 0.99:
        raise ValueError(
            f"Source alignment below 99%: {label}"
        )

    x30b.drop(
        columns=[f"_{label}_match"],
        inplace=True
    )

x30b["date"] = pd.to_datetime(
    x30b["date"]
).dt.date

x30b = x30b.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Model feature sets
# ------------------------------------------------------------

CORE_36 = list(dict.fromkeys(
    CORE + [RETREAT] + GEOMETRY + GAPS
))

if len(CORE_36) != 36:
    raise ValueError(
        f"Expected 36 core features, got {len(CORE_36)}"
    )

BASE = list(dict.fromkeys(
    CORE_36 + CONTROLS
))

REFILL = CANDIDATES[0]
BALANCE = CANDIDATES[1]
PERSIST = CANDIDATES[2]
NET = CANDIDATES[3]

MODELS = {
    "A_full36": BASE,
    "B_refill_pressure": BASE + [REFILL],
    "C_refill_balance": BASE + [BALANCE],
    "D_depletion_persistence": BASE + [PERSIST],
    "E_net_queue": BASE + [NET],
    "F_balance_persistence": BASE + [BALANCE, PERSIST]
}

ALL_INPUTS = list(dict.fromkeys(
    BASE + CANDIDATES
))

for col in ALL_INPUTS + TARGETS:

    x30b[col] = pd.to_numeric(
        x30b[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Common feature-complete observations
# for every model and horizon.
x30b["_complete"] = np.isfinite(
    x30b[ALL_INPUTS].to_numpy(dtype=float)
).all(axis=1)

print("\n===== 30B BASELINE =====")
print("Engineered features:", len(CORE_36))
print("Total baseline inputs:", len(BASE))
print("Research rows:", len(x30b))
print(
    "Complete feature rows:",
    int(x30b["_complete"].sum())
)
print(
    "Complete coverage:",
    round(100*x30b["_complete"].mean(), 4), "%"
)

# ------------------------------------------------------------
# 3. Fixed LightGBM parameters
# ------------------------------------------------------------

PARAMS = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10

dates = sorted(x30b["date"].unique())

if len(dates) <= MIN_TRAIN_DAYS:
    raise ValueError("Not enough trading sessions")

records = []
importance_records = []

# ------------------------------------------------------------
# 4. Expanding chronological validation
# ------------------------------------------------------------

for i in range(MIN_TRAIN_DAYS, len(dates)):

    test_date = dates[i]

    train_pool = x30b.loc[
        x30b["date"].isin(dates[:i])
        & x30b["_complete"]
        & x30b["ts_sec"].mod(TRAIN_STRIDE).eq(0)
    ]

    test_pool = x30b.loc[
        x30b["date"].eq(test_date)
        & x30b["_complete"]
    ]

    for h in HORIZONS:

        target = f"future_return_{h}_30a"

        train = train_pool.loc[
            np.isfinite(train_pool[target])
        ]

        test = test_pool.loc[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(dtype=float)
        y_test = test[target].to_numpy(dtype=float)

        for model_name, columns in MODELS.items():

            model = lgb.LGBMRegressor(**PARAMS)
            model.fit(train[columns], y_train)

            pred = model.predict(test[columns])
            error = y_test - pred

            records.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test),
                "sse": float(np.sum(error**2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error**2)),
                "mae": float(np.mean(np.abs(error)))
            })

            feature_gains = (
                model.booster_.feature_importance(
                    importance_type="gain"
                )
            )

            total_gain = float(feature_gains.sum())

            for feature, gain in zip(
                columns, feature_gains
            ):

                if feature not in CANDIDATES:
                    continue

                importance_records.append({
                    "date": test_date,
                    "horizon": h,
                    "model": model_name,
                    "feature": feature,
                    "gain_share_pct": (
                        100 * gain / total_gain
                        if total_gain > 0 else 0.0
                    )
                })

    print("Completed:", test_date)

calendar_exp30b_daily = pd.DataFrame(records)
calendar_exp30b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp30b_daily.empty:
    raise ValueError("No evaluation results produced")

# ------------------------------------------------------------
# 5. Paired pooled and daily results
# ------------------------------------------------------------

COMPARISONS = {
    "refill_vs_full36": (
        "B_refill_pressure", "A_full36"
    ),
    "balance_vs_full36": (
        "C_refill_balance", "A_full36"
    ),
    "persistence_vs_full36": (
        "D_depletion_persistence", "A_full36"
    ),
    "net_vs_full36": (
        "E_net_queue", "A_full36"
    ),
    "combined_vs_full36": (
        "F_balance_persistence", "A_full36"
    ),
    "combined_vs_balance": (
        "F_balance_persistence", "C_refill_balance"
    ),
    "combined_vs_persistence": (
        "F_balance_persistence", "D_depletion_persistence"
    )
}

summaries = []
daily_gains = []

for h, block in calendar_exp30b_daily.groupby("horizon"):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for label, (candidate, baseline) in (
        COMPARISONS.items()
    ):

        if candidate not in wide or baseline not in wide:
            continue

        daily = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in daily.items():

            daily_gains.append({
                "date": date,
                "horizon": h,
                "comparison": label,
                "mse_gain_pct": float(gain)
            })

        summaries.append({
            "horizon": h,
            "comparison": label,
            "days": len(daily),
            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),
            "winning_days": int((daily > 0).sum()),
            "median_daily_gain_pct": float(
                daily.median()
            ),
            "worst_day_pct": float(daily.min())
        })

calendar_exp30b_summary = pd.DataFrame(summaries)
calendar_exp30b_daily_gains = pd.DataFrame(daily_gains)

# ------------------------------------------------------------
# 6. Print results
# ------------------------------------------------------------

print("\n===== 30B FULL36 ABLATION RESULTS =====")

print(
    calendar_exp30b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 30B DAILY PRIMARY GAINS =====")

for comparison in [
    "refill_vs_full36",
    "balance_vs_full36",
    "persistence_vs_full36",
    "combined_vs_full36"
]:

    subset = calendar_exp30b_daily_gains.loc[
        calendar_exp30b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        ).round(6).to_string()
    )

print("\n===== 30B CANDIDATE IMPORTANCE =====")

if not calendar_exp30b_importance.empty:

    print(
        calendar_exp30b_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_share_pct"]
        .median()
        .reset_index()
        .round(5)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 30B COMPLETE =====")
print("Original df_1s unchanged.")
print("Core feature count remains 36.")


core source match: 100.0000%
retreat source match: 100.0000%
geometry source match: 100.0000%
gaps source match: 100.0000%

===== 30B BASELINE =====
Engineered features: 36
Total baseline inputs: 45
Research rows: 459122
Complete feature rows: 402863
Complete coverage: 87.7464 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 30B FULL36 ABLATION RESULTS =====
 horizon              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30        refill_vs_full36    11             0.027550             0.074732             6               0.053127      -0.368215
      30       balance_vs_full36    11             0.136534             0.102322             7               0.057122      -0.216846
      30   persistence_vs_full36    11

In [94]:

import numpy as np
import pandas as pd

# ============================================================
# EXPERIMENT 31A — MULTI-LEVEL LIQUIDITY MIGRATION
#
# Input: raw df_1s
#   symbol, ts_sec
#   bid_p1..bid_p5, ask_p1..ask_p5
#   bid_q1..bid_q5, ask_q1..ask_q5
#
# Outputs:
#   calendar_exp31a_features
#   calendar_exp31a_daily
#   calendar_exp31a_summary
#   calendar_exp31a_redundancy
#
# Strict session isolation and exact timestamp lags.
# No mutation of the raw df_1s.
# ============================================================

HORIZONS_31A = [30, 60, 300]
LOOKBACKS_31A = [30, 60]
MIN_COVERAGE_31A = 0.97
EPS_31A = 1e-12

KEYS_31A = ["symbol", "ts_sec"]
BP_COLS = [f"bid_p{i}" for i in range(1, 6)]
AP_COLS = [f"ask_p{i}" for i in range(1, 6)]
BQ_COLS = [f"bid_q{i}" for i in range(1, 6)]
AQ_COLS = [f"ask_q{i}" for i in range(1, 6)]

required = (
    KEYS_31A + BP_COLS + AP_COLS
    + BQ_COLS + AQ_COLS
)

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

x31 = df_1s[required].copy()
x31["symbol"] = x31["symbol"].astype(str)

for c in required:
    if c != "symbol":
        x31[c] = pd.to_numeric(
            x31[c], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

if x31["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

if not np.all(
    x31["ts_sec"].to_numpy()
    == np.floor(x31["ts_sec"].to_numpy())
):
    raise ValueError("Timestamps must be whole seconds")

x31["ts_sec"] = x31["ts_sec"].astype("int64")

if x31.duplicated(KEYS_31A).any():
    raise ValueError("Duplicate symbol/timestamp keys")

# ------------------------------------------------------------
# 1. IST trading-session eligibility
# ------------------------------------------------------------

ist = pd.to_datetime(
    x31["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x31["date"] = ist.dt.date

x31["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x31 = x31.loc[
    x31["clock_seconds"].between(0, 28799)
].copy()

coverage = (
    x31.groupby(["symbol", "date"])["ts_sec"]
    .nunique()
    .reset_index(name="n_seconds")
)

eligible = coverage.loc[
    coverage["n_seconds"].ge(
        MIN_COVERAGE_31A * 28800
    )
    & coverage["date"].astype(str).ne("2026-09-28"),
    ["symbol", "date"]
]

x31 = (
    x31.merge(
        eligible,
        on=["symbol", "date"],
        how="inner",
        validate="many_to_one"
    )
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

if x31.empty:
    raise ValueError("No eligible trading sessions")

print("===== 31A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x31))

# ------------------------------------------------------------
# 2. Five-level book validation
# ------------------------------------------------------------

BP = x31[BP_COLS].to_numpy(dtype=np.float64)
AP = x31[AP_COLS].to_numpy(dtype=np.float64)
BQ = x31[BQ_COLS].to_numpy(dtype=np.float64)
AQ = x31[AQ_COLS].to_numpy(dtype=np.float64)

mid = (BP[:, 0] + AP[:, 0]) / 2.0

valid_price = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (AP[:, 0] >= BP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
)

valid_qty = (
    np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
    & (AQ.sum(axis=1) > 0)
)

valid_book = valid_price & valid_qty

print("\n===== 31A BOOK VALIDITY =====")
print("Valid %:", round(100 * valid_book.mean(), 5))

# ------------------------------------------------------------
# 3. Normalize depth distribution at each snapshot
# ------------------------------------------------------------

bid_total = BQ.sum(axis=1)
ask_total = AQ.sum(axis=1)

with np.errstate(divide="ignore", invalid="ignore"):
    pi_b = BQ / bid_total[:, None]
    pi_a = AQ / ask_total[:, None]

    dist_b = 1e4 * (
        BP[:, [0]] - BP
    ) / mid[:, None]

    dist_a = 1e4 * (
        AP - AP[:, [0]]
    ) / mid[:, None]

# ------------------------------------------------------------
# 4. Liquidity center of mass
# ------------------------------------------------------------

mu_b = np.sum(pi_b * dist_b, axis=1)
mu_a = np.sum(pi_a * dist_a, axis=1)

# Near-touch concentration, levels 1 and 2
conc_b = np.sum(pi_b[:, :2], axis=1)
conc_a = np.sum(pi_a[:, :2], axis=1)

# Book concentration entropy
def entropy_rows_31a(p):
    with np.errstate(divide="ignore", invalid="ignore"):
        return -np.sum(
            np.where(p > 0, p * np.log(p), 0.0),
            axis=1
        )

entropy_b = entropy_rows_31a(pi_b)
entropy_a = entropy_rows_31a(pi_a)

snapshot_values = {
    "_mu_bid": mu_b,
    "_mu_ask": mu_a,
    "_conc_bid": conc_b,
    "_conc_ask": conc_a,
    "_entropy_bid": entropy_b,
    "_entropy_ask": entropy_a
}

for name, values in snapshot_values.items():
    values = np.asarray(values, dtype=np.float64).copy()
    values[~valid_book] = np.nan
    values[~np.isfinite(values)] = np.nan
    x31[name] = values

# ------------------------------------------------------------
# 5. Exact clock-time lookbacks
#
# Join t with t-W using symbol/date/ts_sec.
# Avoids treating missing seconds as ordinary rows.
# ------------------------------------------------------------

SNAPSHOT_COLS = list(snapshot_values)

for w in LOOKBACKS_31A:

    previous = x31[
        KEYS_31A + ["date"] + SNAPSHOT_COLS
    ].copy()

    previous["ts_sec"] += w

    previous = previous.rename(
        columns={
            c: f"{c}_lag{w}"
            for c in SNAPSHOT_COLS
        }
    )

    x31 = x31.merge(
        previous,
        on=KEYS_31A + ["date"],
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# 6. Liquidity migration features
# ------------------------------------------------------------

def diff_31a(column, window):
    return (
        x31[column]
        - x31[f"{column}_lag{window}"]
    )

dmu_b_30 = diff_31a("_mu_bid", 30)
dmu_a_30 = diff_31a("_mu_ask", 30)

dmu_b_60 = diff_31a("_mu_bid", 60)
dmu_a_60 = diff_31a("_mu_ask", 60)

x31["liquidity_migration_30_31a"] = (
    dmu_b_30 - dmu_a_30
)

x31["liquidity_migration_60_31a"] = (
    dmu_b_60 - dmu_a_60
)

x31["concentration_migration_60_31a"] = (
    diff_31a("_conc_bid", 60)
    - diff_31a("_conc_ask", 60)
)

with np.errstate(divide="ignore", invalid="ignore"):
    x31["relative_migration_60_31a"] = (
        (dmu_b_60 - dmu_a_60)
        / (
            dmu_b_60.abs()
            + dmu_a_60.abs()
            + EPS_31A
        )
    )

x31["entropy_migration_60_31a"] = (
    diff_31a("_entropy_bid", 60)
    - diff_31a("_entropy_ask", 60)
)

CANDIDATES_31A = [
    "liquidity_migration_30_31a",
    "liquidity_migration_60_31a",
    "concentration_migration_60_31a",
    "relative_migration_60_31a",
    "entropy_migration_60_31a"
]

for col in CANDIDATES_31A:
    x31[col] = x31[col].replace(
        [np.inf, -np.inf], np.nan
    )

# ------------------------------------------------------------
# 7. Exact future return labels
# ------------------------------------------------------------

x31["_mid"] = np.where(
    valid_book, mid, np.nan
)

for h in HORIZONS_31A:

    future = x31[
        KEYS_31A + ["date", "_mid"]
    ].copy()

    future["ts_sec"] -= h

    future = future.rename(
        columns={"_mid": f"_future_mid_{h}"}
    )

    x31 = x31.merge(
        future,
        on=KEYS_31A + ["date"],
        how="left",
        validate="one_to_one"
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        x31[f"future_return_{h}_31a"] = (
            1e4 * np.log(
                x31[f"_future_mid_{h}"]
                / x31["_mid"]
            )
        )

# ------------------------------------------------------------
# 8. Attach existing features for redundancy
# ------------------------------------------------------------

CONTROL_FEATURES_31A = []

source_specs = [
    (
        "core_31",
        [
            "queue_imbalance_l1",
            "depth_imbalance_l5",
            "queue_replenishment_pressure_60s",
            "depth_imbalance_delta",
            "ofi_l1_sum_60s_signed_log",
            "price_mom_30s_new"
        ]
    ),
    (
        "calendar_exp28a_features",
        [
            "slope_asymmetry_28a",
            "curvature_asymmetry_28a"
        ]
    ),
    (
        "calendar_exp29a_features",
        [
            "weighted_gap_asymmetry_29a",
            "fragility_asymmetry_29a"
        ]
    )
]

for source_name, wanted in source_specs:

    if source_name not in globals():
        continue

    source = globals()[source_name]

    cols = [
        c for c in wanted
        if c in source.columns and c not in x31.columns
    ]

    if not cols:
        continue

    extra = source[KEYS_31A + cols].copy()

    extra["symbol"] = extra["symbol"].astype(str)
    extra["ts_sec"] = pd.to_numeric(
        extra["ts_sec"], errors="raise"
    ).astype("int64")

    if extra.duplicated(KEYS_31A).any():
        raise ValueError(
            f"Duplicate keys in {source_name}"
        )

    x31 = x31.merge(
        extra,
        on=KEYS_31A,
        how="left",
        validate="one_to_one"
    )

    CONTROL_FEATURES_31A.extend(cols)

# ------------------------------------------------------------
# 9. Daily Spearman IC
# ------------------------------------------------------------

def daily_spearman_31a(a, b):

    mask = np.isfinite(a) & np.isfinite(b)
    n = int(mask.sum())

    if n < 200:
        return np.nan, n

    ar = pd.Series(a[mask]).rank().to_numpy()
    br = pd.Series(b[mask]).rank().to_numpy()

    if ar.std() < 1e-12 or br.std() < 1e-12:
        return np.nan, n

    return float(np.corrcoef(ar, br)[0, 1]), n

records = []

for (symbol, date), day in x31.groupby(
    ["symbol", "date"], sort=False
):

    for h in HORIZONS_31A:

        y = day[
            f"future_return_{h}_31a"
        ].to_numpy(dtype=float)

        for feature in CANDIDATES_31A:

            values = day[feature].to_numpy(
                dtype=float
            )

            ic, n = daily_spearman_31a(
                values, y
            )

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "feature": feature,
                "ic": ic,
                "valid_rows": n
            })

calendar_exp31a_daily = pd.DataFrame(records)

calendar_exp31a_summary = (
    calendar_exp31a_daily
    .groupby(["horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 10. Redundancy against existing features
# ------------------------------------------------------------

if CONTROL_FEATURES_31A:

    calendar_exp31a_redundancy = (
        x31[
            CANDIDATES_31A + CONTROL_FEATURES_31A
        ]
        .corr(method="spearman")
        .loc[
            CANDIDATES_31A,
            CONTROL_FEATURES_31A
        ]
    )
else:
    calendar_exp31a_redundancy = pd.DataFrame()

# ------------------------------------------------------------
# 11. Diagnostics
# ------------------------------------------------------------

print("\n===== 31A SNAPSHOT DISTRIBUTIONS =====")

for col in [
    "_mu_bid", "_mu_ask",
    "_conc_bid", "_conc_ask",
    "_entropy_bid", "_entropy_ask"
]:
    s = x31[col]
    print(
        col,
        "mean=", round(s.mean(), 6),
        "std=", round(s.std(), 6),
        "median=", round(s.median(), 6)
    )

print("\n===== 31A FEATURE VALIDITY =====")

for col in CANDIDATES_31A:

    s = x31[col]

    print(
        col,
        "valid_pct=", round(100*s.notna().mean(), 4),
        "zero_pct=", round(100*s.eq(0).mean(), 4),
        "std=", round(s.std(), 6),
        "p01=", round(s.quantile(0.01), 6),
        "median=", round(s.median(), 6),
        "p99=", round(s.quantile(0.99), 6)
    )

print("\n===== 31A DAILY SIGNED IC =====")

print(
    calendar_exp31a_summary
    .round(6).to_string(index=False)
)

print("\n===== 31A REDUNDANCY =====")

if not calendar_exp31a_redundancy.empty:
    print(
        calendar_exp31a_redundancy
        .round(5).to_string()
    )
else:
    print("No reference features attached.")

calendar_exp31a_features = x31

print("\n===== EXPERIMENT 31A COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")
print("Core feature count remains 36.")


===== 31A SAMPLE =====
Sessions: 16
Rows: 459122

===== 31A BOOK VALIDITY =====
Valid %: 100.0

===== 31A SNAPSHOT DISTRIBUTIONS =====
_mu_bid mean= 1.107562 std= 0.619493 median= 0.99275
_mu_ask mean= 1.051496 std= 0.638763 median= 0.919413
_conc_bid mean= 0.371873 std= 0.248781 median= 0.339623
_conc_ask mean= 0.400377 std= 0.246471 median= 0.373764
_entropy_bid mean= 1.118228 std= 0.26513 median= 1.15422
_entropy_ask mean= 1.137815 std= 0.256711 median= 1.17695

===== 31A FEATURE VALIDITY =====
liquidity_migration_30_31a valid_pct= 99.8955 zero_pct= 8.0793 std= 0.806497 p01= -2.262629 median= 0.0 p99= 2.325784
liquidity_migration_60_31a valid_pct= 99.7909 zero_pct= 3.3462 std= 0.920185 p01= -2.473072 median= 0.0 p99= 2.520497
concentration_migration_60_31a valid_pct= 99.7909 zero_pct= 3.5309 std= 0.377872 p01= -0.956518 median= 0.0 p99= 0.957919
relative_migration_60_31a valid_pct= 99.7909 zero_pct= 3.3462 std= 0.834048 p01= -1.0 median= 0.0 p99= 1.0
entropy_migration_60_31a valid_p

In [95]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# EXPERIMENT 31B — FULL36 LIQUIDITY MIGRATION ABLATION
#
# Baseline:
#   31 core features
#   + quote retreat
#   + slope and curvature
#   + weighted gap and fragility
#   = 36 engineered features
#   + 9 established controls
#
# Candidates:
#   migration30, migration60, both,
#   concentration60, all three, relative60
#
# Horizons: 30, 60, 300 seconds.
#
# Fixed LightGBM and chronological expanding evaluation.
# Same training/test rows for all model variants.
# No mutation of df_1s.
# ============================================================

KEYS_31B = ["symbol", "ts_sec"]
HORIZONS_31B = [30, 60, 300]

RETREAT_31B = "quote_retreat_imbalance_30_24a"

GEOMETRY_31B = [
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a"
]

GAPS_31B = [
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a"
]

M30 = "liquidity_migration_30_31a"
M60 = "liquidity_migration_60_31a"
CONC = "concentration_migration_60_31a"
REL = "relative_migration_60_31a"

CANDIDATES_31B = [M30, M60, CONC, REL]

CONTROLS_31B = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

TARGETS_31B = [
    f"future_return_{h}_31a"
    for h in HORIZONS_31B
]

REQUIRED_31B = [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp28a_features",
    "calendar_exp29a_features",
    "calendar_exp31a_features"
]

for name in REQUIRED_31B:
    if name not in globals():
        raise ValueError(f"Missing {name}")

CORE_31B = [
    c for c in core_31.columns
    if c not in KEYS_31B
]

if len(CORE_31B) != 31:
    raise ValueError(
        "Expected exactly 31 original core features."
    )

# ------------------------------------------------------------
# 1. Safe exact timestamp joins
# ------------------------------------------------------------

def source_31b(frame, columns, name):

    missing = set(KEYS_31B + columns) - set(
        frame.columns
    )

    if missing:
        raise ValueError(
            f"{name} missing {sorted(missing)}"
        )

    result = frame[
        KEYS_31B + columns
    ].copy()

    result["symbol"] = (
        result["symbol"].astype(str)
    )

    result["ts_sec"] = pd.to_numeric(
        result["ts_sec"],
        errors="raise"
    ).astype("int64")

    if result.duplicated(KEYS_31B).any():
        raise ValueError(
            f"Duplicate keys in {name}"
        )

    return result


x31b = source_31b(
    calendar_exp31a_features,
    ["date"] + CANDIDATES_31B + TARGETS_31B,
    "experiment31"
)

sources = [
    ("core", core_31, CORE_31B),
    (
        "retreat",
        calendar_exp24a_features,
        [RETREAT_31B]
    ),
    (
        "geometry",
        calendar_exp28a_features,
        GEOMETRY_31B + CONTROLS_31B
    ),
    (
        "gaps",
        calendar_exp29a_features,
        GAPS_31B
    )
]

for label, frame, cols in sources:

    data = source_31b(frame, cols, label)

    x31b = x31b.merge(
        data,
        on=KEYS_31B,
        how="left",
        validate="one_to_one",
        indicator=f"_{label}_match"
    )

    match = x31b[
        f"_{label}_match"
    ].eq("both").mean()

    print(
        f"{label} match: {100*match:.4f}%"
    )

    if match < 0.99:
        raise ValueError(
            f"Poor source match: {label}"
        )

    x31b.drop(
        columns=[f"_{label}_match"],
        inplace=True
    )

x31b["date"] = pd.to_datetime(
    x31b["date"]
).dt.date

x31b = x31b.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Baseline and candidate feature sets
# ------------------------------------------------------------

CORE_36_31B = list(dict.fromkeys(
    CORE_31B
    + [RETREAT_31B]
    + GEOMETRY_31B
    + GAPS_31B
))

if len(CORE_36_31B) != 36:
    raise ValueError(
        f"Expected 36 features, got {len(CORE_36_31B)}"
    )

BASE_31B = list(dict.fromkeys(
    CORE_36_31B + CONTROLS_31B
))

MODELS_31B = {
    "A_full36": BASE_31B,
    "B_migration30": BASE_31B + [M30],
    "C_migration60": BASE_31B + [M60],
    "D_both_migrations": BASE_31B + [M30, M60],
    "E_concentration": BASE_31B + [CONC],
    "F_all_three": BASE_31B + [M30, M60, CONC],
    "G_relative60": BASE_31B + [REL]
}

ALL_INPUTS_31B = list(dict.fromkeys(
    BASE_31B + CANDIDATES_31B
))

for col in ALL_INPUTS_31B + TARGETS_31B:
    x31b[col] = pd.to_numeric(
        x31b[col],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

x31b["_complete"] = np.isfinite(
    x31b[ALL_INPUTS_31B].to_numpy(
        dtype=np.float64
    )
).all(axis=1)

print("\n===== 31B BASELINE =====")
print("Engineered features:", len(CORE_36_31B))
print("Total baseline inputs:", len(BASE_31B))
print("Research rows:", len(x31b))
print(
    "Complete feature rows:",
    int(x31b["_complete"].sum())
)
print(
    "Feature coverage:",
    round(100*x31b["_complete"].mean(), 4), "%"
)

# ------------------------------------------------------------
# 3. Fixed LightGBM configuration
# ------------------------------------------------------------

PARAMS_31B = {
    "objective": "regression",
    "n_estimators": 180,
    "learning_rate": 0.035,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 200,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": 1,
    "verbosity": -1
}

MIN_TRAIN_DAYS_31B = 5
TRAIN_STRIDE_31B = 10

dates_31b = sorted(
    x31b["date"].unique()
)

if len(dates_31b) <= MIN_TRAIN_DAYS_31B:
    raise ValueError("Not enough sessions")

records_31b = []
importance_31b = []

# ------------------------------------------------------------
# 4. Expanding walk-forward evaluation
# ------------------------------------------------------------

for i in range(
    MIN_TRAIN_DAYS_31B,
    len(dates_31b)
):

    test_date = dates_31b[i]

    train_pool = x31b.loc[
        x31b["date"].isin(dates_31b[:i])
        & x31b["_complete"]
        & x31b["ts_sec"].mod(
            TRAIN_STRIDE_31B
        ).eq(0)
    ]

    test_pool = x31b.loc[
        x31b["date"].eq(test_date)
        & x31b["_complete"]
    ]

    for h in HORIZONS_31B:

        target = f"future_return_{h}_31a"

        train = train_pool.loc[
            np.isfinite(train_pool[target])
        ]

        test = test_pool.loc[
            np.isfinite(test_pool[target])
        ]

        if len(train) < 1000 or len(test) < 200:
            continue

        y_train = train[target].to_numpy(
            dtype=np.float64
        )

        y_test = test[target].to_numpy(
            dtype=np.float64
        )

        for model_name, columns in MODELS_31B.items():

            model = lgb.LGBMRegressor(
                **PARAMS_31B
            )

            model.fit(
                train[columns],
                y_train
            )

            pred = model.predict(
                test[columns]
            )

            error = y_test - pred

            records_31b.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test),
                "sse": float(np.sum(error**2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error**2)),
                "mae": float(np.mean(np.abs(error)))
            })

            gains = model.booster_.feature_importance(
                importance_type="gain"
            )

            total = float(gains.sum())

            for col, gain in zip(columns, gains):

                if col not in CANDIDATES_31B:
                    continue

                importance_31b.append({
                    "date": test_date,
                    "horizon": h,
                    "model": model_name,
                    "feature": col,
                    "gain_share_pct": (
                        100*gain/total
                        if total > 0 else 0.0
                    )
                })

    print("Completed:", test_date)

calendar_exp31b_daily = pd.DataFrame(
    records_31b
)

calendar_exp31b_importance = pd.DataFrame(
    importance_31b
)

if calendar_exp31b_daily.empty:
    raise ValueError(
        "No evaluation results produced"
    )

# ------------------------------------------------------------
# 5. Model comparisons
# ------------------------------------------------------------

COMPARISONS_31B = {
    "migration30_vs_full36": (
        "B_migration30", "A_full36"
    ),
    "migration60_vs_full36": (
        "C_migration60", "A_full36"
    ),
    "both_vs_full36": (
        "D_both_migrations", "A_full36"
    ),
    "concentration_vs_full36": (
        "E_concentration", "A_full36"
    ),
    "all_three_vs_full36": (
        "F_all_three", "A_full36"
    ),
    "relative_vs_full36": (
        "G_relative60", "A_full36"
    ),
    "both_vs_migration30": (
        "D_both_migrations", "B_migration30"
    ),
    "both_vs_migration60": (
        "D_both_migrations", "C_migration60"
    ),
    "all_three_vs_both": (
        "F_all_three", "D_both_migrations"
    )
}

summaries_31b = []
daily_gains_31b = []

for h, block in (
    calendar_exp31b_daily.groupby("horizon")
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    for label, (candidate, baseline) in (
        COMPARISONS_31B.items()
    ):

        daily_gain = 100 * (
            1 - wide[candidate] / wide[baseline]
        )

        for date, gain in daily_gain.items():

            daily_gains_31b.append({
                "date": date,
                "horizon": h,
                "comparison": label,
                "mse_gain_pct": float(gain)
            })

        summaries_31b.append({
            "horizon": h,
            "comparison": label,
            "days": len(daily_gain),

            "pooled_mse_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sse"]
                / pooled.loc[baseline, "sse"]
            ),

            "pooled_mae_gain_pct": 100 * (
                1 - pooled.loc[candidate, "sae"]
                / pooled.loc[baseline, "sae"]
            ),

            "winning_days": int(
                (daily_gain > 0).sum()
            ),

            "median_daily_gain_pct": float(
                daily_gain.median()
            ),

            "worst_day_pct": float(
                daily_gain.min()
            )
        })

calendar_exp31b_summary = pd.DataFrame(
    summaries_31b
)

calendar_exp31b_daily_gains = pd.DataFrame(
    daily_gains_31b
)

# ------------------------------------------------------------
# 6. Print results
# ------------------------------------------------------------

print("\n===== 31B FULL36 ABLATION RESULTS =====")

print(
    calendar_exp31b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 31B DAILY PRIMARY GAINS =====")

for comparison in [
    "migration30_vs_full36",
    "migration60_vs_full36",
    "both_vs_full36",
    "concentration_vs_full36",
    "all_three_vs_full36"
]:

    subset = calendar_exp31b_daily_gains.loc[
        calendar_exp31b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        ).round(6).to_string()
    )

print("\n===== 31B CANDIDATE IMPORTANCE =====")

if not calendar_exp31b_importance.empty:

    print(
        calendar_exp31b_importance.groupby(
            ["horizon", "model", "feature"]
        )["gain_share_pct"]
        .median()
        .reset_index()
        .round(5)
        .to_string(index=False)
    )

print("\n===== EXPERIMENT 31B COMPLETE =====")
print("Original df_1s unchanged.")
print("No feature promoted.")
print("Core feature count remains 36.")


core match: 100.0000%
retreat match: 100.0000%
geometry match: 100.0000%
gaps match: 100.0000%

===== 31B BASELINE =====
Engineered features: 36
Total baseline inputs: 45
Research rows: 459122
Complete feature rows: 402979
Feature coverage: 87.7717 %
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 31B FULL36 ABLATION RESULTS =====
 horizon              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
      30   migration30_vs_full36    11             0.130725             0.205389             8               0.199245      -1.043081
      30   migration60_vs_full36    11            -0.047096             0.008525             4              -0.027477      -0.525956
      30          both_vs_full36    11             0.030031        

In [98]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 32A — VIRTUAL MARKET-ORDER IMPACT
# MEMORY-OPTIMIZED COMPLETE VERSION
#
# INPUT:
#   df_1s
#   symbol, ts_sec
#   bid_p1..bid_p5, ask_p1..ask_p5
#   bid_q1..bid_q5, ask_q1..ask_q5
#
# OPTIONAL FOR REDUNDANCY:
#   core_31
#   calendar_exp28a_features
#   calendar_exp29a_features
#   calendar_exp31a_features
#
# OUTPUTS:
#   calendar_exp32a_features
#   calendar_exp32a_daily
#   calendar_exp32a_summary
#   calendar_exp32a_redundancy
#
# Does not modify df_1s.
# Core feature count remains 37.
# ============================================================

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

HORIZONS_32A = [30, 60, 300]
FRACTIONS_32A = [0.10, 0.25]

KEYS_32A = ["symbol", "ts_sec"]

BP_COLS_32A = [f"bid_p{i}" for i in range(1, 6)]
AP_COLS_32A = [f"ask_p{i}" for i in range(1, 6)]
BQ_COLS_32A = [f"bid_q{i}" for i in range(1, 6)]
AQ_COLS_32A = [f"ask_q{i}" for i in range(1, 6)]

CANDIDATES_32A = [
    "vimpact_asym_10_32a",
    "vimpact_asym_25_32a",
    "vimpact_mean_25_32a",
    "vimpact_depth_sensitivity_32a"
]

MIN_COVERAGE_32A = 0.97
SESSION_SECONDS_32A = 28800

# ------------------------------------------------------------
# 1. Validate source columns
# ------------------------------------------------------------

required_32a = (
    KEYS_32A
    + BP_COLS_32A
    + AP_COLS_32A
    + BQ_COLS_32A
    + AQ_COLS_32A
)

missing_32a = [
    c for c in required_32a
    if c not in df_1s.columns
]

if missing_32a:
    raise ValueError(
        f"Missing required columns: {missing_32a}"
    )

# Work on only the required raw columns.
x32 = df_1s[required_32a].copy()

x32["symbol"] = x32["symbol"].astype(str)

for col in required_32a:
    if col != "symbol":
        x32[col] = pd.to_numeric(
            x32[col],
            errors="coerce"
        ).replace(
            [np.inf, -np.inf],
            np.nan
        )

if x32["ts_sec"].isna().any():
    raise ValueError("Invalid ts_sec values")

ts_values = x32["ts_sec"].to_numpy(dtype=np.float64)

if not np.all(ts_values == np.floor(ts_values)):
    raise ValueError("ts_sec must contain whole seconds")

x32["ts_sec"] = x32["ts_sec"].astype(np.int64)

if x32.duplicated(KEYS_32A).any():
    raise ValueError(
        "Duplicate symbol/timestamp combinations"
    )

# ------------------------------------------------------------
# 2. Select eligible IST trading sessions
# ------------------------------------------------------------

ist_32a = pd.to_datetime(
    x32["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata")

x32["date"] = ist_32a.dt.date

x32["clock_seconds"] = (
    ist_32a.dt.hour * 3600
    + ist_32a.dt.minute * 60
    + ist_32a.dt.second
    - 9 * 3600
)

# NSE CDS session window used in our previous research.
x32 = x32.loc[
    x32["clock_seconds"].between(
        0, SESSION_SECONDS_32A - 1
    )
].copy()

coverage_32a = (
    x32.groupby(["symbol", "date"])["ts_sec"]
    .nunique()
    .reset_index(name="n_seconds")
)

eligible_32a = coverage_32a.loc[
    coverage_32a["n_seconds"].ge(
        MIN_COVERAGE_32A * SESSION_SECONDS_32A
    )
    & coverage_32a["date"].astype(str).ne(
        "2026-09-28"
    ),
    ["symbol", "date"]
]

eligible_index_32a = pd.MultiIndex.from_frame(
    eligible_32a
)

row_session_index_32a = pd.MultiIndex.from_frame(
    x32[["symbol", "date"]]
)

keep_32a = row_session_index_32a.isin(
    eligible_index_32a
)

x32 = (
    x32.loc[keep_32a]
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

if x32.empty:
    raise ValueError("No eligible research sessions")

print("===== 32A SAMPLE =====")
print("Sessions:", len(eligible_32a))
print("Rows:", len(x32))

# ------------------------------------------------------------
# 3. Extract book arrays
# ------------------------------------------------------------

BP = x32[BP_COLS_32A].to_numpy(
    dtype=np.float64, copy=True
)

AP = x32[AP_COLS_32A].to_numpy(
    dtype=np.float64, copy=True
)

BQ = x32[BQ_COLS_32A].to_numpy(
    dtype=np.float64, copy=True
)

AQ = x32[AQ_COLS_32A].to_numpy(
    dtype=np.float64, copy=True
)

mid_32a = (BP[:, 0] + AP[:, 0]) / 2.0

bid_total_32a = BQ.sum(axis=1)
ask_total_32a = AQ.sum(axis=1)

# ------------------------------------------------------------
# 4. Validate five-level book
# ------------------------------------------------------------

valid_book_32a = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (AP[:, 0] >= BP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (bid_total_32a > 0)
    & (ask_total_32a > 0)
)

shared_depth_32a = np.minimum(
    bid_total_32a,
    ask_total_32a
)

print("\n===== 32A BOOK VALIDITY =====")

print(
    "Valid books (%):",
    round(100 * valid_book_32a.mean(), 5)
)

print(
    "Median shared 5-level depth:",
    round(
        float(
            np.median(
                shared_depth_32a[valid_book_32a]
            )
        ),
        4
    )
)

# ------------------------------------------------------------
# 5. Virtual execution slippage
#
# Fill quantity at each level:
#
# fill_i = min(
#     Q_i,
#     max(V - sum_{j<i} Q_j, 0)
# )
#
# All orders must be completely fillable within L5.
#
# The impact excludes the bid-ask spread:
#
# buy:
#   1e4 * (VWAP_ask - best_ask) / mid
#
# sell:
#   1e4 * (best_bid - VWAP_bid) / mid
# ------------------------------------------------------------

def virtual_slippage_32a(
    prices,
    quantities,
    order_size,
    midprice,
    side
):

    n = len(order_size)

    # Quantity available before each price level.
    cumulative = np.cumsum(
        quantities,
        axis=1
    )

    cumulative_before = (
        cumulative - quantities
    )

    remaining = np.maximum(
        order_size[:, None] - cumulative_before,
        0.0
    )

    fills = np.minimum(
        quantities,
        remaining
    )

    filled_quantity = fills.sum(axis=1)

    if side == "buy":

        price_distance = (
            prices - prices[:, [0]]
        )

    elif side == "sell":

        price_distance = (
            prices[:, [0]] - prices
        )

    else:
        raise ValueError(
            "side must be 'buy' or 'sell'"
        )

    # Cost relative to best quote.
    numerator = np.sum(
        fills * price_distance,
        axis=1
    )

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):

        impact = (
            1e4 * numerator
            / (order_size * midprice)
        )

    # Check full execution, with floating-point tolerance.
    tolerance = (
        1e-8 * np.maximum(order_size, 1.0)
    )

    fully_filled = (
        np.isfinite(filled_quantity)
        & (order_size > 0)
        & (
            np.abs(
                filled_quantity - order_size
            ) <= tolerance
        )
    )

    impact[~fully_filled] = np.nan

    impact[~np.isfinite(impact)] = np.nan

    return impact, fully_filled


# ------------------------------------------------------------
# 6. Calculate virtual impact for 10% and 25% orders
# ------------------------------------------------------------

impact_results_32a = {}

for fraction in FRACTIONS_32A:

    suffix = int(round(fraction * 100))

    # Equal hypothetical buy/sell quantity.
    order_size = (
        fraction * shared_depth_32a
    )

    buy_impact, buy_filled = (
        virtual_slippage_32a(
            prices=AP,
            quantities=AQ,
            order_size=order_size,
            midprice=mid_32a,
            side="buy"
        )
    )

    sell_impact, sell_filled = (
        virtual_slippage_32a(
            prices=BP,
            quantities=BQ,
            order_size=order_size,
            midprice=mid_32a,
            side="sell"
        )
    )

    valid_execution = (
        valid_book_32a
        & buy_filled
        & sell_filled
    )

    buy_impact[~valid_execution] = np.nan
    sell_impact[~valid_execution] = np.nan

    impact_results_32a[f"buy_{suffix}"] = (
        buy_impact
    )

    impact_results_32a[f"sell_{suffix}"] = (
        sell_impact
    )

    print(
        f"\n===== VIRTUAL SIZE {suffix}% ====="
    )

    print(
        "Valid execution (%):",
        round(
            100 * valid_execution.mean(), 5
        )
    )

    print(
        "Median virtual order size:",
        round(
            float(
                np.median(
                    order_size[valid_execution]
                )
            ),
            4
        )
    )

    print(
        "Buy impact median (bps):",
        round(
            float(
                np.nanmedian(buy_impact)
            ),
            6
        )
    )

    print(
        "Sell impact median (bps):",
        round(
            float(
                np.nanmedian(sell_impact)
            ),
            6
        )
    )

    del (
        order_size,
        buy_impact,
        sell_impact,
        buy_filled,
        sell_filled,
        valid_execution
    )

gc.collect()

# ------------------------------------------------------------
# 7. Construct candidate features
# ------------------------------------------------------------

asym_10_32a = (
    impact_results_32a["sell_10"]
    - impact_results_32a["buy_10"]
)

asym_25_32a = (
    impact_results_32a["sell_25"]
    - impact_results_32a["buy_25"]
)

mean_25_32a = (
    impact_results_32a["sell_25"]
    + impact_results_32a["buy_25"]
) / 2.0

sensitivity_32a = (
    asym_25_32a - asym_10_32a
)

# ------------------------------------------------------------
# 8. Build a compact research DataFrame
#
# Avoid retaining the entire raw five-level book.
# ------------------------------------------------------------

calendar_exp32a_features = x32[
    ["symbol", "ts_sec", "date"]
].copy()

calendar_exp32a_features[
    "vimpact_asym_10_32a"
] = asym_10_32a.astype(np.float32)

calendar_exp32a_features[
    "vimpact_asym_25_32a"
] = asym_25_32a.astype(np.float32)

calendar_exp32a_features[
    "vimpact_mean_25_32a"
] = mean_25_32a.astype(np.float32)

calendar_exp32a_features[
    "vimpact_depth_sensitivity_32a"
] = sensitivity_32a.astype(np.float32)

# Keep midprice only temporarily for label creation.
calendar_exp32a_features["_mid_32a"] = (
    np.where(
        valid_book_32a,
        mid_32a,
        np.nan
    )
)

# Release large arrays before further operations.
del (
    x32,
    BP, AP, BQ, AQ,
    impact_results_32a,
    asym_10_32a,
    asym_25_32a,
    mean_25_32a,
    sensitivity_32a,
    mid_32a,
    bid_total_32a,
    ask_total_32a,
    shared_depth_32a,
    valid_book_32a
)

gc.collect()

# ------------------------------------------------------------
# 9. Exact future midprice labels
#
# Rather than repeatedly merging large DataFrames,
# use index lookups.
#
# Match:
#   symbol, date, ts_sec + h
#
# Session key prevents next-day leakage.
# ------------------------------------------------------------

research_index_32a = pd.MultiIndex.from_frame(
    calendar_exp32a_features[
        ["symbol", "date", "ts_sec"]
    ]
)

if research_index_32a.has_duplicates:
    raise ValueError("Duplicate research keys")

current_mid_32a = (
    calendar_exp32a_features["_mid_32a"]
    .to_numpy(dtype=np.float64)
)

symbols_32a = (
    calendar_exp32a_features["symbol"]
)

dates_32a = (
    calendar_exp32a_features["date"]
)

timestamps_32a = (
    calendar_exp32a_features["ts_sec"]
    .to_numpy(dtype=np.int64)
)

for h in HORIZONS_32A:

    lookup_index = pd.MultiIndex.from_arrays(
        [
            symbols_32a,
            dates_32a,
            timestamps_32a + h
        ],
        names=["symbol", "date", "ts_sec"]
    )

    future_positions = (
        research_index_32a.get_indexer(
            lookup_index
        )
    )

    future_mid = np.full(
        len(calendar_exp32a_features),
        np.nan,
        dtype=np.float64
    )

    found = future_positions >= 0

    future_mid[found] = (
        current_mid_32a[
            future_positions[found]
        ]
    )

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):

        future_return = (
            1e4 * np.log(
                future_mid / current_mid_32a
            )
        )

    future_return[
        ~np.isfinite(future_return)
    ] = np.nan

    calendar_exp32a_features[
        f"future_return_{h}_32a"
    ] = future_return.astype(np.float32)

    calendar_exp32a_features[
        f"future_absret_{h}_32a"
    ] = np.abs(future_return).astype(
        np.float32
    )

    print(
        f"\nExact {h}s target valid (%):",
        round(
            100 * np.isfinite(
                future_return
            ).mean(),
            5
        )
    )

    del (
        lookup_index,
        future_positions,
        future_mid,
        future_return,
        found
    )

calendar_exp32a_features.drop(
    columns=["_mid_32a"],
    inplace=True
)

del (
    research_index_32a,
    current_mid_32a,
    symbols_32a,
    dates_32a,
    timestamps_32a
)

gc.collect()

# ------------------------------------------------------------
# 10. Daily Spearman IC
# ------------------------------------------------------------

def daily_spearman_32a(feature, target):

    valid = (
        np.isfinite(feature)
        & np.isfinite(target)
    )

    n = int(valid.sum())

    if n < 200:
        return np.nan, n

    a = pd.Series(
        feature[valid]
    ).rank().to_numpy()

    b = pd.Series(
        target[valid]
    ).rank().to_numpy()

    if (
        np.std(a) < 1e-12
        or np.std(b) < 1e-12
    ):
        return np.nan, n

    return float(
        np.corrcoef(a, b)[0, 1]
    ), n


daily_records_32a = []

for (symbol, date), day in (
    calendar_exp32a_features.groupby(
        ["symbol", "date"],
        sort=False
    )
):

    for h in HORIZONS_32A:

        for task, target_name in [
            (
                "signed",
                f"future_return_{h}_32a"
            ),
            (
                "absret",
                f"future_absret_{h}_32a"
            )
        ]:

            y = day[target_name].to_numpy(
                dtype=np.float64
            )

            for feature in CANDIDATES_32A:

                values = day[feature].to_numpy(
                    dtype=np.float64
                )

                ic, n = daily_spearman_32a(
                    values,
                    y
                )

                daily_records_32a.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp32a_daily = pd.DataFrame(
    daily_records_32a
)

calendar_exp32a_summary = (
    calendar_exp32a_daily
    .groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic",
            lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 11. Redundancy against existing signals
#
# IMPORTANT:
# No merges.
#
# Map timestamps only once per source, then align
# one existing feature column at a time.
# ------------------------------------------------------------

reference_specs_32a = [
    (
        "core_31",
        [
            "queue_imbalance_l1",
            "depth_imbalance_l5",
            "impact_lambda_1800s",
            "queue_replenishment_pressure_60s",
            "spread_rank_1800s"
        ]
    ),
    (
        "calendar_exp28a_features",
        [
            "slope_asymmetry_28a",
            "curvature_asymmetry_28a",
            "log_total_depth_28a",
            "spread_bps_28a"
        ]
    ),
    (
        "calendar_exp29a_features",
        [
            "weighted_gap_asymmetry_29a",
            "fragility_asymmetry_29a",
            "gap_severity_29a"
        ]
    ),
    (
        "calendar_exp31a_features",
        [
            "relative_migration_60_31a"
        ]
    )
]

target_index_32a = pd.MultiIndex.from_frame(
    calendar_exp32a_features[KEYS_32A]
)

candidate_arrays_32a = {
    c: calendar_exp32a_features[c].to_numpy(
        dtype=np.float64
    )
    for c in CANDIDATES_32A
}

redundancy_records_32a = []

for frame_name, requested_cols in (
    reference_specs_32a
):

    if frame_name not in globals():
        print(
            f"Skipping unavailable: {frame_name}"
        )
        continue

    source = globals()[frame_name]

    available_cols = [
        c for c in requested_cols
        if c in source.columns
    ]

    if not available_cols:
        continue

    source_index = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS_32A
    )

    if source_index.has_duplicates:
        raise ValueError(
            f"Duplicate keys in {frame_name}"
        )

    positions = source_index.get_indexer(
        target_index_32a
    )

    matched = positions >= 0

    match_pct = 100 * matched.mean()

    print(
        f"\n{frame_name} match: "
        f"{match_pct:.4f}%"
    )

    if match_pct < 99:
        raise ValueError(
            f"Poor timestamp match for {frame_name}"
        )

    for reference_col in available_cols:

        source_values = pd.to_numeric(
            source[reference_col],
            errors="coerce"
        ).to_numpy(dtype=np.float64)

        aligned = np.full(
            len(target_index_32a),
            np.nan,
            dtype=np.float64
        )

        aligned[matched] = source_values[
            positions[matched]
        ]

        for candidate_name, values in (
            candidate_arrays_32a.items()
        ):

            valid_pair = (
                np.isfinite(values)
                & np.isfinite(aligned)
            )

            if valid_pair.sum() < 200:
                correlation = np.nan
            else:
                correlation = pd.Series(
                    values[valid_pair]
                ).corr(
                    pd.Series(
                        aligned[valid_pair]
                    ),
                    method="spearman"
                )

            redundancy_records_32a.append({
                "candidate": candidate_name,
                "reference": reference_col,
                "spearman": correlation
            })

        del aligned, source_values

    del source_index, positions, matched
    gc.collect()

if redundancy_records_32a:

    calendar_exp32a_redundancy = (
        pd.DataFrame(
            redundancy_records_32a
        )
        .pivot(
            index="candidate",
            columns="reference",
            values="spearman"
        )
        .reindex(CANDIDATES_32A)
    )

else:
    calendar_exp32a_redundancy = (
        pd.DataFrame()
    )

del (
    target_index_32a,
    candidate_arrays_32a
)

gc.collect()

# ------------------------------------------------------------
# 12. Final results
# ------------------------------------------------------------

print("\n===== 32A SIGNED RETURN IC =====")

print(
    calendar_exp32a_summary.loc[
        calendar_exp32a_summary[
            "task"
        ].eq("signed")
    ]
    .round(6)
    .to_string(index=False)
)

print("\n===== 32A ABSRET IC =====")

print(
    calendar_exp32a_summary.loc[
        calendar_exp32a_summary[
            "task"
        ].eq("absret")
    ]
    .round(6)
    .to_string(index=False)
)

print("\n===== 32A REDUNDANCY =====")

if not calendar_exp32a_redundancy.empty:

    print(
        calendar_exp32a_redundancy
        .round(5)
        .to_string()
    )

else:
    print("No reference controls available")

print("\n===== 32A DISTRIBUTIONS =====")

for col in CANDIDATES_32A:

    s = calendar_exp32a_features[col]

    print(
        col,
        "valid_pct=",
        round(100 * s.notna().mean(), 4),
        "zero_pct=",
        round(100 * s.eq(0).mean(), 4),
        "std=",
        round(s.std(), 6),
        "p01=",
        round(s.quantile(0.01), 6),
        "median=",
        round(s.median(), 6),
        "p99=",
        round(s.quantile(0.99), 6)
    )

print("\n===== 32A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp32a_features[
        CANDIDATES_32A
    ]
    .corr(method="spearman")
    .round(5)
    .to_string()
)

print("\n===== EXPERIMENT 32A COMPLETE =====")
print(
    "Research rows:",
    len(calendar_exp32a_features)
)
print("Original df_1s unchanged.")
print("No new features promoted.")
print("Core feature count remains 37.")


Task was destroyed but it is pending!
task: <Task pending name='Task-942' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-943' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\projects\hedge_fund\venv\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-943' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>


===== 32A SAMPLE =====
Sessions: 16
Rows: 459122

===== 32A BOOK VALIDITY =====
Valid books (%): 100.0
Median shared 5-level depth: 1051.0

===== VIRTUAL SIZE 10% =====
Valid execution (%): 100.0
Median virtual order size: 105.1
Buy impact median (bps): 0.0
Sell impact median (bps): 0.0

===== VIRTUAL SIZE 25% =====
Valid execution (%): 100.0
Median virtual order size: 262.75
Buy impact median (bps): 0.173441
Sell impact median (bps): 0.113946

Exact 30s target valid (%): 99.89545

Exact 60s target valid (%): 99.79091

Exact 300s target valid (%): 98.95453

core_31 match: 100.0000%

calendar_exp28a_features match: 100.0000%

calendar_exp29a_features match: 100.0000%

calendar_exp31a_features match: 100.0000%

===== 32A SIGNED RETURN IC =====
  task  horizon                       feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed       30           vimpact_asym_10_32a    16 -0.139532  -0.144960              0             16            28723.0
sign

In [99]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 32B — VIRTUAL IMPACT ABLATION
#
# Directional task:
#   Full37 vs Full37 + vimpact_asym_25_32a
#
# Absolute-return task:
#   Full37 vs Full37 + vimpact_mean_25_32a
#
# Horizons: 30, 60, 300 seconds
# Expanding walk-forward, identical rows.
#
# Memory efficient:
# - No repeated large DataFrame merges
# - Source alignment using timestamp indexers
# - float32 feature matrix
# - Training sampled every 10 seconds
#
# Outputs:
#   calendar_exp32b_summary
#   calendar_exp32b_daily
#   calendar_exp32b_daily_gains
#   calendar_exp32b_importance
# ============================================================

KEYS = ["symbol", "ts_sec"]
HORIZONS = [30, 60, 300]

ASYM = "vimpact_asym_25_32a"
MEAN = "vimpact_mean_25_32a"

CORE = [
    c for c in core_31.columns
    if c not in KEYS
]

if len(CORE) != 31:
    raise ValueError(
        f"Expected 31 original core features, got {len(CORE)}"
    )

RETREAT = ["quote_retreat_imbalance_30_24a"]

GEOMETRY = [
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a"
]

GAPS = [
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a"
]

MIGRATION = ["relative_migration_60_31a"]

CONTROLS = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

ENGINEERED = list(dict.fromkeys(
    CORE + RETREAT + GEOMETRY + GAPS + MIGRATION
))

if len(ENGINEERED) != 37:
    raise ValueError(
        f"Expected 37 engineered features, got {len(ENGINEERED)}"
    )

BASE = list(dict.fromkeys(
    ENGINEERED + CONTROLS
))

# ------------------------------------------------------------
# 1. Source availability
# ------------------------------------------------------------

SOURCES = [
    ("core_31", CORE),
    ("calendar_exp24a_features", RETREAT),
    ("calendar_exp28a_features", GEOMETRY + CONTROLS),
    ("calendar_exp29a_features", GAPS),
    ("calendar_exp31a_features", MIGRATION)
]

for name, cols in SOURCES:

    if name not in globals():
        raise ValueError(f"Missing {name}")

    missing = set(KEYS + cols) - set(
        globals()[name].columns
    )

    if missing:
        raise ValueError(
            f"{name} missing: {sorted(missing)}"
        )

if "calendar_exp32a_features" not in globals():
    raise ValueError(
        "Run Experiment 32A first."
    )

# ------------------------------------------------------------
# 2. Create compact common evaluation table
# ------------------------------------------------------------

TARGETS = [
    f"future_{kind}_{h}_32a"
    for kind in ["return", "absret"]
    for h in HORIZONS
]

needed = (
    KEYS + ["date", ASYM, MEAN] + TARGETS
)

x = calendar_exp32a_features[needed].copy()

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = x["ts_sec"].astype("int64")
x["date"] = pd.to_datetime(x["date"]).dt.date

if x.duplicated(KEYS).any():
    raise ValueError("Duplicate 32A keys.")

# Align once, then assign only needed columns.
target_index = pd.MultiIndex.from_frame(x[KEYS])

for name, cols in SOURCES:

    source = globals()[name]

    source_index = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"],
                errors="raise"
            ).astype("int64")
        ],
        names=KEYS
    )

    if source_index.has_duplicates:
        raise ValueError(
            f"Duplicate keys in {name}"
        )

    indexer = source_index.get_indexer(target_index)
    matched = indexer >= 0

    print(
        f"{name} matched: {100*matched.mean():.4f}%"
    )

    if matched.mean() < 0.99:
        raise ValueError(
            f"Poor match from {name}"
        )

    for col in cols:

        values = pd.to_numeric(
            source[col],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

        aligned = np.full(
            len(x),
            np.nan,
            dtype=np.float32
        )

        aligned[matched] = values[
            indexer[matched]
        ]

        x[col] = aligned

    del source_index, indexer, matched
    gc.collect()

del target_index

# ------------------------------------------------------------
# 3. Common validity across all models
#
# Feature completeness is shared between the tasks.
# Target availability is handled per horizon/task.
# ------------------------------------------------------------

ALL_FEATURES = BASE + [ASYM, MEAN]

for col in [ASYM, MEAN] + TARGETS:
    x[col] = pd.to_numeric(
        x[col],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

feature_matrix = x[ALL_FEATURES].to_numpy(
    dtype=np.float32
)

x["_complete"] = np.isfinite(
    feature_matrix
).all(axis=1)

del feature_matrix

x = x.sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

print("\n===== 32B SAMPLE =====")
print("Research rows:", len(x))
print("Engineered features:", len(ENGINEERED))
print("Baseline inputs:", len(BASE))
print(
    "Feature-complete rows:",
    int(x["_complete"].sum())
)
print(
    "Feature coverage (%):",
    round(100*x["_complete"].mean(), 4)
)

# ------------------------------------------------------------
# 4. Fixed model configuration
# ------------------------------------------------------------

PARAMS = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    colsample_bytree=1.0,
    subsample=1.0,
    reg_lambda=2.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

TASKS = {
    "signed": {
        "candidate": ASYM,
        "target_prefix": "future_return"
    },
    "absret": {
        "candidate": MEAN,
        "target_prefix": "future_absret"
    }
}

dates = sorted(x["date"].unique())

if len(dates) <= 5:
    raise ValueError("Insufficient training sessions")

results = []
importance = []

# ------------------------------------------------------------
# 5. Expanding chronological validation
# ------------------------------------------------------------

for i in range(5, len(dates)):

    test_date = dates[i]

    train_mask = (
        x["date"].isin(dates[:i])
        & x["_complete"]
        & x["ts_sec"].mod(10).eq(0)
    )

    test_mask = (
        x["date"].eq(test_date)
        & x["_complete"]
    )

    for task, config in TASKS.items():

        candidate = config["candidate"]

        model_columns = {
            "baseline": BASE,
            "with_virtual_impact": BASE + [candidate]
        }

        for h in HORIZONS:

            target = (
                f"{config['target_prefix']}_{h}_32a"
            )

            train = x.loc[
                train_mask & x[target].notna()
            ]

            test = x.loc[
                test_mask & x[target].notna()
            ]

            if len(train) < 1000 or len(test) < 200:
                continue

            ytrain = train[target].to_numpy(
                dtype=np.float64
            )

            ytest = test[target].to_numpy(
                dtype=np.float64
            )

            for model_name, columns in (
                model_columns.items()
            ):

                model = lgb.LGBMRegressor(**PARAMS)

                model.fit(
                    train[columns],
                    ytrain
                )

                pred = model.predict(
                    test[columns]
                )

                errors = ytest - pred

                results.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test),
                    "sse": float(np.sum(errors**2)),
                    "sae": float(np.sum(np.abs(errors))),
                    "mse": float(np.mean(errors**2)),
                    "mae": float(np.mean(np.abs(errors)))
                })

                if model_name == "with_virtual_impact":

                    gains = (
                        model.booster_
                        .feature_importance(
                            importance_type="gain"
                        )
                    )

                    total_gain = float(gains.sum())

                    candidate_index = columns.index(
                        candidate
                    )

                    importance.append({
                        "date": test_date,
                        "task": task,
                        "horizon": h,
                        "feature": candidate,
                        "gain_share_pct": (
                            100 * gains[candidate_index]
                            / total_gain
                            if total_gain > 0 else 0.0
                        )
                    })

                del model, pred, errors

    print("Completed:", test_date)

calendar_exp32b_daily = pd.DataFrame(results)

calendar_exp32b_importance = pd.DataFrame(
    importance
)

if calendar_exp32b_daily.empty:
    raise ValueError("No evaluation results")

# ------------------------------------------------------------
# 6. Paired pooled evaluation
# ------------------------------------------------------------

summary = []
daily_gains = []

for (task, h), block in (
    calendar_exp32b_daily.groupby(
        ["task", "horizon"]
    )
):

    wide_mse = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    wide_mae = block.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    # Use only paired dates.
    common_dates = (
        wide_mse.dropna().index
        .intersection(wide_mae.dropna().index)
    )

    paired = block.loc[
        block["date"].isin(common_dates)
    ]

    pooled = paired.groupby("model")[
        ["sse", "sae"]
    ].sum()

    daily_mse_gain = 100 * (
        1
        - wide_mse.loc[
            common_dates, "with_virtual_impact"
        ]
        / wide_mse.loc[
            common_dates, "baseline"
        ]
    )

    daily_mae_gain = 100 * (
        1
        - wide_mae.loc[
            common_dates, "with_virtual_impact"
        ]
        / wide_mae.loc[
            common_dates, "baseline"
        ]
    )

    for date in common_dates:

        daily_gains.append({
            "date": date,
            "task": task,
            "horizon": h,
            "mse_gain_pct": float(
                daily_mse_gain.loc[date]
            ),
            "mae_gain_pct": float(
                daily_mae_gain.loc[date]
            )
        })

    summary.append({
        "task": task,
        "horizon": h,
        "days": len(common_dates),
        "rows": int(
            paired.loc[
                paired.model.eq("baseline"),
                "rows"
            ].sum()
        ),
        "pooled_mse_gain_pct": 100 * (
            1
            - pooled.loc[
                "with_virtual_impact", "sse"
            ]
            / pooled.loc["baseline", "sse"]
        ),
        "pooled_mae_gain_pct": 100 * (
            1
            - pooled.loc[
                "with_virtual_impact", "sae"
            ]
            / pooled.loc["baseline", "sae"]
        ),
        "winning_days": int(
            (daily_mse_gain > 0).sum()
        ),
        "median_daily_gain_pct": float(
            daily_mse_gain.median()
        ),
        "worst_day_pct": float(
            daily_mse_gain.min()
        )
    })

calendar_exp32b_summary = pd.DataFrame(summary)
calendar_exp32b_daily_gains = pd.DataFrame(
    daily_gains
)

# ------------------------------------------------------------
# 7. Diagnostics
# ------------------------------------------------------------

print("\n===== 32B FULL37 ABLATION =====")

print(
    calendar_exp32b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 32B DAILY MSE GAINS =====")

for task in ["signed", "absret"]:

    print("\nTASK:", task)

    subset = calendar_exp32b_daily_gains.loc[
        calendar_exp32b_daily_gains.task.eq(task)
    ]

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 32B DAILY MAE GAINS =====")

for task in ["signed", "absret"]:

    print("\nTASK:", task)

    subset = calendar_exp32b_daily_gains.loc[
        calendar_exp32b_daily_gains.task.eq(task)
    ]

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mae_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 32B FEATURE IMPORTANCE =====")

print(
    calendar_exp32b_importance
    .groupby(["task", "horizon", "feature"])[
        "gain_share_pct"
    ]
    .agg(["median", "mean"])
    .round(5)
    .to_string()
)

print("\n===== EXPERIMENT 32B COMPLETE =====")
print("Original df_1s unchanged.")
print("Core feature count remains 37.")

gc.collect()


core_31 matched: 100.0000%
calendar_exp24a_features matched: 100.0000%
calendar_exp28a_features matched: 100.0000%
calendar_exp29a_features matched: 100.0000%
calendar_exp31a_features matched: 100.0000%

===== 32B SAMPLE =====
Research rows: 459122
Engineered features: 37
Baseline inputs: 46
Feature-complete rows: 402979
Feature coverage (%): 87.7717
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 32B FULL37 ABLATION =====
  task  horizon  days   rows  pooled_mse_gain_pct  pooled_mae_gain_pct  winning_days  median_daily_gain_pct  worst_day_pct
absret       30    11 280481             0.046275             0.038021             6               0.017377      -0.190678
absret       60    11 280151            -0.017507             0.044446             4              -0.055172      -0.736818
abs

0

In [100]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 33A — OFI ABSORPTION AND PRICE RESPONSE
#
# Inputs:
#   df_1s:
#       symbol, ts_sec, bid_p1, ask_p1
#   core_31:
#       symbol, ts_sec,
#       ofi_l1_sum_60s_signed_log
#
# Optional core_31 controls:
#       impact_lambda_1800s
#       impact_r2_1800s
#       ofi_response_gap_z
#       price_mom_30s_new
#       price_mom_60s_new
#       queue_imbalance_l1
#       queue_replenishment_pressure_60s
#
# Outputs:
#   calendar_exp33a_features
#   calendar_exp33a_daily
#   calendar_exp33a_summary
#   calendar_exp33a_redundancy
#
# No mutation of raw data.
# No feature promotion.
# ============================================================

HORIZONS_33A = [30, 60, 300]
SCALE_WINDOW_33A = 300
MIN_SCALE_OBS_33A = 60
EPS_33A = 1e-8

KEYS_33A = ["symbol", "ts_sec"]

CANDIDATES_33A = [
    "signed_absorption_60_33a",
    "ofi_price_mismatch_60_33a",
    "ofi_price_alignment_60_33a",
    "absorption_intensity_60_33a"
]

OFI_33A = "ofi_l1_sum_60s_signed_log"

if "core_31" not in globals():
    raise ValueError("core_31 is required.")

raw_cols = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
]

if any(c not in df_1s.columns for c in raw_cols):
    raise ValueError("Missing required raw columns.")

if any(
    c not in core_31.columns
    for c in KEYS_33A + [OFI_33A]
):
    raise ValueError("Missing OFI columns in core_31.")

# ------------------------------------------------------------
# 1. Base research table
# ------------------------------------------------------------

x33 = df_1s[raw_cols].copy()

x33["symbol"] = x33["symbol"].astype(str)

for c in ["ts_sec", "bid_p1", "ask_p1"]:
    x33[c] = pd.to_numeric(
        x33[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

if x33["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

ts = x33["ts_sec"].to_numpy(dtype=float)

if not np.all(ts == np.floor(ts)):
    raise ValueError("ts_sec must be whole seconds")

x33["ts_sec"] = x33["ts_sec"].astype("int64")

if x33.duplicated(KEYS_33A).any():
    raise ValueError("Duplicate raw timestamp keys")

ist = pd.to_datetime(
    x33["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x33["date"] = ist.dt.date

x33["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x33 = x33.loc[
    x33["clock_seconds"].between(0, 28799)
].copy()

coverage = (
    x33.groupby(["symbol", "date"])["ts_sec"]
    .nunique()
    .reset_index(name="n")
)

eligible = coverage.loc[
    (coverage["n"] >= .97 * 28800)
    & (
        coverage["date"].astype(str)
        != "2026-09-28"
    ),
    ["symbol", "date"]
]

eligible_keys = pd.MultiIndex.from_frame(eligible)
row_keys = pd.MultiIndex.from_frame(
    x33[["symbol", "date"]]
)

x33 = (
    x33.loc[row_keys.isin(eligible_keys)]
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

if x33.empty:
    raise ValueError("No eligible sessions")

print("===== 33A SAMPLE =====")
print("Sessions:", len(eligible))
print("Rows:", len(x33))

# ------------------------------------------------------------
# 2. Timestamp index and historical OFI alignment
# ------------------------------------------------------------

target_index = pd.MultiIndex.from_frame(
    x33[KEYS_33A]
)

def align_source_33a(source, col):
    source_index = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"], errors="raise"
            ).astype("int64")
        ],
        names=KEYS_33A
    )

    if source_index.has_duplicates:
        raise ValueError("Duplicate source keys")

    positions = source_index.get_indexer(
        target_index
    )

    found = positions >= 0

    src_values = pd.to_numeric(
        source[col], errors="coerce"
    ).to_numpy(dtype=np.float64)

    out = np.full(len(x33), np.nan)
    out[found] = src_values[positions[found]]

    return out, found.mean()

ofi, match_rate = align_source_33a(
    core_31, OFI_33A
)

if match_rate < .99:
    raise ValueError(
        f"OFI match too low: {match_rate:.4%}"
    )

x33["_ofi"] = ofi

print(
    "OFI source match:",
    round(100 * match_rate, 4), "%"
)

# ------------------------------------------------------------
# 3. Valid midprice
# ------------------------------------------------------------

bid = x33["bid_p1"].to_numpy(dtype=float)
ask = x33["ask_p1"].to_numpy(dtype=float)

valid_quote = (
    np.isfinite(bid)
    & np.isfinite(ask)
    & (bid > 0)
    & (ask >= bid)
)

mid = np.where(
    valid_quote, (bid + ask) / 2, np.nan
)

x33["_mid"] = mid

# ------------------------------------------------------------
# 4. Exact t-60s previous midprice
#
# Match symbol + session date + exact second.
# No shift(60), so missing seconds cannot change
# the effective lookback duration.
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x33[["symbol", "date", "ts_sec"]]
)

symbols = x33["symbol"]
dates = x33["date"]
timestamps = x33["ts_sec"].to_numpy(dtype="int64")

def lookup_mid_33a(offset):
    lookup_index = pd.MultiIndex.from_arrays(
        [
            symbols,
            dates,
            timestamps + offset
        ],
        names=["symbol", "date", "ts_sec"]
    )

    idx = session_index.get_indexer(lookup_index)

    result = np.full(len(x33), np.nan)

    found = idx >= 0
    result[found] = mid[idx[found]]

    return result

mid_lag60 = lookup_mid_33a(-60)

with np.errstate(divide="ignore", invalid="ignore"):
    past_return = 1e4 * np.log(mid / mid_lag60)

past_return[~np.isfinite(past_return)] = np.nan

x33["_past_return_60"] = past_return

# ------------------------------------------------------------
# 5. Historical scale estimation
#
# Rolling medians use PREVIOUS observations only.
#
# Separate scaling per symbol/session.
# The first 60 observed rows do not have valid scales.
# ------------------------------------------------------------

g = x33.groupby(["symbol", "date"], sort=False)

ofi_scale = g["_ofi"].transform(
    lambda s: (
        s.abs()
        .shift(1)
        .rolling(
            SCALE_WINDOW_33A,
            min_periods=MIN_SCALE_OBS_33A
        )
        .median()
    )
)

return_scale = g["_past_return_60"].transform(
    lambda s: (
        s.abs()
        .shift(1)
        .rolling(
            SCALE_WINDOW_33A,
            min_periods=MIN_SCALE_OBS_33A
        )
        .median()
    )
)

# The zero-median case is common in tick-based FX markets.
# Keep these rows rather than dividing by zero.
# EPS is a numerical guard, not a calibrated market scale.
ofi_scale = ofi_scale.to_numpy(dtype=float)
return_scale = return_scale.to_numpy(dtype=float)

scale_valid = (
    np.isfinite(ofi_scale)
    & np.isfinite(return_scale)
)

print("\n===== 33A SCALE DIAGNOSTICS =====")

print(
    "Scale coverage (%):",
    round(100 * scale_valid.mean(), 4)
)

print(
    "Zero OFI scale (%):",
    round(
        100 * np.mean(
            scale_valid & (ofi_scale == 0)
        ), 4
    )
)

print(
    "Zero return scale (%):",
    round(
        100 * np.mean(
            scale_valid & (return_scale == 0)
        ), 4
    )
)

print(
    "Median OFI scale:",
    float(np.nanmedian(ofi_scale))
)

print(
    "Median return scale (bps):",
    float(np.nanmedian(return_scale))
)

# ------------------------------------------------------------
# 6. Normalize OFI and return
# ------------------------------------------------------------

with np.errstate(divide="ignore", invalid="ignore"):
    z_ofi = np.tanh(
        ofi / (ofi_scale + EPS_33A)
    )

    z_return = np.tanh(
        past_return / (return_scale + EPS_33A)
    )

    price_response_ratio = (
        np.abs(past_return)
        / (return_scale + EPS_33A)
    )

    suppression = np.exp(-price_response_ratio)

valid_inputs = (
    scale_valid
    & np.isfinite(ofi)
    & np.isfinite(past_return)
)

# ------------------------------------------------------------
# 7. Candidate features
# ------------------------------------------------------------

features = {
    "signed_absorption_60_33a":
        z_ofi * suppression,

    "ofi_price_mismatch_60_33a":
        z_ofi - z_return,

    "ofi_price_alignment_60_33a":
        z_ofi * z_return,

    "absorption_intensity_60_33a":
        np.abs(z_ofi) * suppression
}

for name, values in features.items():

    values = np.asarray(values, dtype=float).copy()

    values[~valid_inputs] = np.nan
    values[~np.isfinite(values)] = np.nan

    x33[name] = values.astype(np.float32)

# ------------------------------------------------------------
# 8. Exact future targets
# ------------------------------------------------------------

for h in HORIZONS_33A:

    future_mid = lookup_mid_33a(h)

    with np.errstate(divide="ignore", invalid="ignore"):
        target = (
            1e4 * np.log(future_mid / mid)
        )

    target[~np.isfinite(target)] = np.nan

    x33[f"future_return_{h}_33a"] = (
        target.astype(np.float32)
    )

    x33[f"future_absret_{h}_33a"] = (
        np.abs(target).astype(np.float32)
    )

# ------------------------------------------------------------
# 9. Compact research table
# ------------------------------------------------------------

targets = [
    f"future_{kind}_{h}_33a"
    for kind in ["return", "absret"]
    for h in HORIZONS_33A
]

calendar_exp33a_features = x33[
    KEYS_33A
    + ["date", "clock_seconds"]
    + CANDIDATES_33A
    + targets
].copy()

# ------------------------------------------------------------
# 10. Daily Spearman IC
# ------------------------------------------------------------

def spearman_ic_33a(a, b):

    mask = np.isfinite(a) & np.isfinite(b)
    n = int(mask.sum())

    if n < 200:
        return np.nan, n

    ar = pd.Series(a[mask]).rank().to_numpy()
    br = pd.Series(b[mask]).rank().to_numpy()

    if ar.std() < 1e-12 or br.std() < 1e-12:
        return np.nan, n

    return float(np.corrcoef(ar, br)[0, 1]), n

records = []

for (sym, date), day in (
    calendar_exp33a_features.groupby(
        ["symbol", "date"], sort=False
    )
):

    for h in HORIZONS_33A:

        for task in ["signed", "absret"]:

            target_name = (
                f"future_return_{h}_33a"
                if task == "signed"
                else f"future_absret_{h}_33a"
            )

            y = day[target_name].to_numpy(dtype=float)

            for feature in CANDIDATES_33A:

                v = day[feature].to_numpy(dtype=float)

                ic, n = spearman_ic_33a(v, y)

                records.append({
                    "symbol": sym,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp33a_daily = pd.DataFrame(records)

calendar_exp33a_summary = (
    calendar_exp33a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=("valid_rows", "median")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 11. Redundancy against previous features
#
# Reuse timestamp index mapping.
# No large DataFrame merges.
# ------------------------------------------------------------

controls = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_response_gap_z",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "queue_imbalance_l1",
    "queue_replenishment_pressure_60s"
]

controls = [
    c for c in controls if c in core_31.columns
]

redundancy_records = []

for c in controls:

    if c == OFI_33A:
        aligned = ofi
    else:
        aligned, _ = align_source_33a(core_31, c)

    for feature in CANDIDATES_33A:

        a = calendar_exp33a_features[
            feature
        ].to_numpy(dtype=float)

        mask = np.isfinite(a) & np.isfinite(aligned)

        if mask.sum() >= 200:

            corr = pd.Series(a[mask]).corr(
                pd.Series(aligned[mask]),
                method="spearman"
            )

        else:
            corr = np.nan

        redundancy_records.append({
            "candidate": feature,
            "reference": c,
            "spearman": corr
        })

calendar_exp33a_redundancy = (
    pd.DataFrame(redundancy_records)
    .pivot(
        index="candidate",
        columns="reference",
        values="spearman"
    )
    .reindex(CANDIDATES_33A)
)

# ------------------------------------------------------------
# 12. Diagnostics
# ------------------------------------------------------------

print("\n===== 33A FEATURE DISTRIBUTIONS =====")

for col in CANDIDATES_33A:

    s = calendar_exp33a_features[col]

    print(
        col,
        "valid_pct=", round(100*s.notna().mean(), 4),
        "zero_pct=", round(100*s.eq(0).mean(), 4),
        "std=", round(s.std(), 6),
        "p01=", round(s.quantile(.01), 6),
        "median=", round(s.median(), 6),
        "p99=", round(s.quantile(.99), 6)
    )

print("\n===== 33A SIGNED RETURN IC =====")

print(
    calendar_exp33a_summary.loc[
        calendar_exp33a_summary.task.eq("signed")
    ].round(6).to_string(index=False)
)

print("\n===== 33A ABSRET IC =====")

print(
    calendar_exp33a_summary.loc[
        calendar_exp33a_summary.task.eq("absret")
    ].round(6).to_string(index=False)
)

print("\n===== 33A REDUNDANCY =====")

print(
    calendar_exp33a_redundancy
    .round(5).to_string()
)

print("\n===== 33A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp33a_features[CANDIDATES_33A]
    .corr(method="spearman")
    .round(5).to_string()
)

del x33
gc.collect()

print("\n===== EXPERIMENT 33A COMPLETE =====")
print("Original df_1s unchanged.")
print("Research feature count remains 37.")


===== 33A SAMPLE =====
Sessions: 16
Rows: 459122
OFI source match: 100.0 %

===== 33A SCALE DIAGNOSTICS =====
Scale coverage (%): 99.5818
Zero OFI scale (%): 1.1768
Zero return scale (%): 12.3629
Median OFI scale: 6.1779441140506
Median return scale (bps): 0.2623432499219735

===== 33A FEATURE DISTRIBUTIONS =====
signed_absorption_60_33a valid_pct= 99.5818 zero_pct= 13.8227 std= 0.382241 p01= -0.848382 median= 0.0 p99= 0.862541
ofi_price_mismatch_60_33a valid_pct= 99.5818 zero_pct= 7.9604 std= 0.654478 p01= -1.760949 median= 0.0 p99= 1.77929
ofi_price_alignment_60_33a valid_pct= 99.5818 zero_pct= 26.3268 std= 0.464556 p01= -0.840032 median= 0.349516 p99= 0.983925
absorption_intensity_60_33a valid_pct= 99.5818 zero_pct= 13.8227 std= 0.26307 p01= 0.0 median= 0.218377 p99= 0.915298

===== 33A SIGNED RETURN IC =====
  task  horizon                     feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed       30 absorption_intensity_60_33a    16  0.013

In [101]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 33B
# OFI-PRICE MISMATCH SCALING ROBUSTNESS
# + FULL 37-FEATURE ABLATION
#
# REQUIRED:
#   df_1s
#   core_31
#   calendar_exp24a_features
#   calendar_exp28a_features
#   calendar_exp29a_features
#   calendar_exp31a_features
#   calendar_exp33a_features
#
# TASKS:
#
# Signed:
#   A_full37
#   B_original_mismatch
#   C_tick_floor_mismatch
#
# Absolute return:
#   A_full37
#   D_ofi_price_alignment
#
# HORIZONS:
#   30, 60, 300 seconds
#
# OUTPUT:
#   calendar_exp33b_summary
#   calendar_exp33b_daily
#   calendar_exp33b_daily_gains
#   calendar_exp33b_importance
#   calendar_exp33b_features
#
# No mutation of df_1s.
# No feature promotion.
# ============================================================

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

HORIZONS_33B = [30, 60, 300]

TICK_SIZE_33B = 0.0025
SCALE_WINDOW_33B = 300
MIN_SCALE_OBS_33B = 60
EPS_33B = 1e-8

MIN_TRAIN_DAYS_33B = 5
TRAIN_STRIDE_33B = 10

KEYS_33B = ["symbol", "ts_sec"]

ORIGINAL_33B = "ofi_price_mismatch_60_33a"
FLOORED_33B = "ofi_price_mismatch_tickfloor_60_33b"
ALIGNMENT_33B = "ofi_price_alignment_60_33a"

OFI_33B = "ofi_l1_sum_60s_signed_log"

CORE_33B = [
    c for c in core_31.columns
    if c not in KEYS_33B
]

RETREAT_33B = [
    "quote_retreat_imbalance_30_24a"
]

GEOMETRY_33B = [
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a"
]

GAPS_33B = [
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a"
]

MIGRATION_33B = [
    "relative_migration_60_31a"
]

CONTROLS_33B = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

ENGINEERED_33B = list(dict.fromkeys(
    CORE_33B
    + RETREAT_33B
    + GEOMETRY_33B
    + GAPS_33B
    + MIGRATION_33B
))

if len(ENGINEERED_33B) != 37:
    raise ValueError(
        f"Expected 37 engineered features, "
        f"got {len(ENGINEERED_33B)}"
    )

BASE_33B = list(dict.fromkeys(
    ENGINEERED_33B + CONTROLS_33B
))

print("===== 33B FEATURE REGISTRY =====")
print("Core engineered features:", len(ENGINEERED_33B))
print("Additional controls:", len(CONTROLS_33B))
print("Total baseline inputs:", len(BASE_33B))

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

SOURCE_SPECS_33B = [
    ("core_31", CORE_33B),
    ("calendar_exp24a_features", RETREAT_33B),
    (
        "calendar_exp28a_features",
        GEOMETRY_33B + CONTROLS_33B
    ),
    ("calendar_exp29a_features", GAPS_33B),
    ("calendar_exp31a_features", MIGRATION_33B)
]

required_frames = [
    "df_1s",
    "calendar_exp33a_features"
] + [
    name for name, _ in SOURCE_SPECS_33B
]

for name in required_frames:
    if name not in globals():
        raise ValueError(
            f"Missing required object: {name}"
        )

for name, columns in SOURCE_SPECS_33B:
    source = globals()[name]

    missing = [
        c for c in KEYS_33B + columns
        if c not in source.columns
    ]

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )

# ------------------------------------------------------------
# 2. Compact base table from Experiment 33A
# ------------------------------------------------------------

TARGETS_33B = [
    f"future_{kind}_{h}_33a"
    for kind in ["return", "absret"]
    for h in HORIZONS_33B
]

base_columns = (
    KEYS_33B
    + ["date", ORIGINAL_33B, ALIGNMENT_33B]
    + TARGETS_33B
)

x = calendar_exp33a_features[
    base_columns
].copy()

x["symbol"] = x["symbol"].astype(str)

x["ts_sec"] = pd.to_numeric(
    x["ts_sec"],
    errors="raise"
).astype(np.int64)

x["date"] = pd.to_datetime(
    x["date"]
).dt.date

x = (
    x.sort_values(
        ["date", "symbol", "ts_sec"]
    )
    .reset_index(drop=True)
)

if x.duplicated(KEYS_33B).any():
    raise ValueError(
        "Duplicate timestamps in Experiment 33A"
    )

N = len(x)

print("\n===== 33B SAMPLE =====")
print("Rows:", N)
print("Sessions:", x.groupby(
    ["symbol", "date"]
).ngroups)

# ------------------------------------------------------------
# 3. Align source arrays once
#
# No DataFrame merges.
# Store only numeric feature arrays.
# ------------------------------------------------------------

target_index = pd.MultiIndex.from_frame(
    x[KEYS_33B]
)

def make_indexer_33b(source, name):

    source_index = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS_33B
    )

    if source_index.has_duplicates:
        raise ValueError(
            f"Duplicate keys in {name}"
        )

    positions = source_index.get_indexer(
        target_index
    )

    found = positions >= 0

    print(
        f"{name} match: "
        f"{100 * found.mean():.4f}%"
    )

    if found.mean() < 0.99:
        raise ValueError(
            f"Poor alignment: {name}"
        )

    return positions, found


feature_arrays = {}

for source_name, columns in SOURCE_SPECS_33B:

    source = globals()[source_name]

    positions, found = make_indexer_33b(
        source, source_name
    )

    for col in columns:

        values = pd.to_numeric(
            source[col],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

        aligned = np.full(
            N,
            np.nan,
            dtype=np.float32
        )

        aligned[found] = values[
            positions[found]
        ]

        feature_arrays[col] = aligned

    del positions, found
    gc.collect()

# ------------------------------------------------------------
# 4. Align raw bid/ask Level-1 prices
# ------------------------------------------------------------

raw_columns = [
    "symbol", "ts_sec", "bid_p1", "ask_p1"
]

missing_raw = [
    c for c in raw_columns
    if c not in df_1s.columns
]

if missing_raw:
    raise ValueError(
        f"Missing raw columns: {missing_raw}"
    )

raw_source = df_1s[raw_columns]

raw_positions, raw_found = make_indexer_33b(
    raw_source, "raw df_1s"
)

def raw_numeric_33b(col):

    values = pd.to_numeric(
        raw_source[col],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    aligned = np.full(
        N,
        np.nan,
        dtype=np.float64
    )

    aligned[raw_found] = values[
        raw_positions[raw_found]
    ]

    return aligned


bid = raw_numeric_33b("bid_p1")
ask = raw_numeric_33b("ask_p1")

valid_quote = (
    np.isfinite(bid)
    & np.isfinite(ask)
    & (bid > 0)
    & (ask >= bid)
)

mid = np.where(
    valid_quote,
    (bid + ask) / 2.0,
    np.nan
)

del (
    raw_source,
    raw_positions,
    raw_found,
    bid,
    ask
)

gc.collect()

# ------------------------------------------------------------
# 5. Exact 60-second historical midprice
#
# Match by symbol + IST date + exact timestamp.
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x[["symbol", "date", "ts_sec"]]
)

lag_index = pd.MultiIndex.from_arrays(
    [
        x["symbol"],
        x["date"],
        x["ts_sec"].to_numpy(dtype=np.int64) - 60
    ],
    names=["symbol", "date", "ts_sec"]
)

lag_positions = session_index.get_indexer(
    lag_index
)

mid_lag60 = np.full(N, np.nan)

lag_found = lag_positions >= 0

mid_lag60[lag_found] = mid[
    lag_positions[lag_found]
]

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    past_return = (
        1e4 * np.log(
            mid / mid_lag60
        )
    )

past_return[
    ~np.isfinite(past_return)
] = np.nan

del (
    session_index,
    lag_index,
    lag_positions,
    mid_lag60,
    lag_found
)

gc.collect()

# ------------------------------------------------------------
# 6. Reproduce Experiment 33A historical scales
#
# Rolling median of PREVIOUS 300 observations.
# Minimum 60 historical valid observations.
#
# This reproduces the original scaling methodology.
# ------------------------------------------------------------

scaling_frame = pd.DataFrame({
    "symbol": x["symbol"],
    "date": x["date"],
    "_ofi": feature_arrays[OFI_33B],
    "_return": past_return
})

groups = scaling_frame.groupby(
    ["symbol", "date"],
    sort=False
)

ofi_scale = groups["_ofi"].transform(
    lambda s: (
        s.abs()
        .shift(1)
        .rolling(
            SCALE_WINDOW_33B,
            min_periods=MIN_SCALE_OBS_33B
        )
        .median()
    )
).to_numpy(dtype=np.float64)

return_scale = groups["_return"].transform(
    lambda s: (
        s.abs()
        .shift(1)
        .rolling(
            SCALE_WINDOW_33B,
            min_periods=MIN_SCALE_OBS_33B
        )
        .median()
    )
).to_numpy(dtype=np.float64)

del scaling_frame, groups
gc.collect()

# ------------------------------------------------------------
# 7. Observed tick compatibility diagnostic
#
# Tick size is a fixed research assumption here.
# We report whether observed L1 quotes are consistent
# with it, rather than silently assuming correctness.
#
# Float tolerance is in price units.
# ------------------------------------------------------------

tick_units = np.concatenate([
    mid[np.isfinite(mid)]
])

# Midprices can be half-ticks because they average
# bid and ask. Therefore check raw source prices
# via source snapshots separately if desired.
# Do not use midprice multiples as a tick check.

tick_floor_bps = (
    1e4 * TICK_SIZE_33B / mid
)

print("\n===== 33B SCALE DIAGNOSTICS =====")

scale_valid = (
    np.isfinite(ofi_scale)
    & np.isfinite(return_scale)
)

print(
    "Historical scale coverage (%):",
    round(100 * scale_valid.mean(), 4)
)

print(
    "Zero OFI scale (% of all rows):",
    round(
        100 * np.mean(
            scale_valid & (ofi_scale == 0)
        ),
        4
    )
)

print(
    "Zero return scale (% of all rows):",
    round(
        100 * np.mean(
            scale_valid & (return_scale == 0)
        ),
        4
    )
)

print(
    "Median tick floor (bps):",
    round(
        float(np.nanmedian(
            tick_floor_bps
        )),
        6
    )
)

print(
    "Rows where tick floor exceeds return scale (%):",
    round(
        100 * np.mean(
            scale_valid
            & (tick_floor_bps > return_scale)
        ),
        4
    )
)

# ------------------------------------------------------------
# 8. Construct tick-floor mismatch
# ------------------------------------------------------------

ofi = feature_arrays[OFI_33B].astype(
    np.float64
)

floored_return_scale = np.maximum(
    return_scale,
    tick_floor_bps
)

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    z_ofi = np.tanh(
        ofi / (ofi_scale + EPS_33B)
    )

    z_return_floored = np.tanh(
        past_return
        / (floored_return_scale + EPS_33B)
    )

    mismatch_floored = (
        z_ofi - z_return_floored
    )

valid_floored = (
    scale_valid
    & np.isfinite(ofi)
    & np.isfinite(past_return)
    & np.isfinite(tick_floor_bps)
    & (tick_floor_bps > 0)
)

mismatch_floored[
    ~valid_floored
] = np.nan

feature_arrays[FLOORED_33B] = (
    mismatch_floored.astype(np.float32)
)

del (
    ofi,
    mid,
    past_return,
    ofi_scale,
    return_scale,
    tick_floor_bps,
    floored_return_scale,
    z_ofi,
    z_return_floored,
    mismatch_floored,
    valid_floored,
    tick_units
)

gc.collect()

# ------------------------------------------------------------
# 9. Assemble a compact float32 feature matrix
#
# Avoid repeatedly adding columns to a large DataFrame.
# ------------------------------------------------------------

ALL_FEATURES = (
    BASE_33B
    + [ORIGINAL_33B, FLOORED_33B, ALIGNMENT_33B]
)

for col in [
    ORIGINAL_33B,
    ALIGNMENT_33B
]:

    feature_arrays[col] = pd.to_numeric(
        x[col],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

X = np.column_stack([
    feature_arrays[col]
    for col in ALL_FEATURES
]).astype(
    np.float32,
    copy=False
)

X[~np.isfinite(X)] = np.nan

column_position = {
    col: i
    for i, col in enumerate(ALL_FEATURES)
}

base_indices = [
    column_position[c]
    for c in BASE_33B
]

candidate_indices = {
    ORIGINAL_33B: column_position[ORIGINAL_33B],
    FLOORED_33B: column_position[FLOORED_33B],
    ALIGNMENT_33B: column_position[ALIGNMENT_33B]
}

# Shared finite feature rows across all comparisons.
complete = np.isfinite(X).all(axis=1)

print("\n===== 33B MATCHED SAMPLE =====")

print("Total rows:", N)

print(
    "Complete feature rows:",
    int(complete.sum())
)

print(
    "Complete coverage (%):",
    round(100 * complete.mean(), 4)
)

print(
    "Original mismatch valid (%):",
    round(
        100 * np.isfinite(
            X[:, candidate_indices[ORIGINAL_33B]]
        ).mean(),
        4
    )
)

print(
    "Tick-floor mismatch valid (%):",
    round(
        100 * np.isfinite(
            X[:, candidate_indices[FLOORED_33B]]
        ).mean(),
        4
    )
)

# Create y arrays without copying raw source frames.
targets = {}

for task in ["signed", "absret"]:

    prefix = (
        "future_return"
        if task == "signed"
        else "future_absret"
    )

    for h in HORIZONS_33B:

        col = f"{prefix}_{h}_33a"

        targets[(task, h)] = pd.to_numeric(
            x[col],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

# Keep a compact candidate artifact for later.
calendar_exp33b_features = x[
    KEYS_33B + ["date"]
].copy()

calendar_exp33b_features[
    FLOORED_33B
] = feature_arrays[FLOORED_33B]

# Retain candidate and target arrays, but release
# redundant dictionary arrays now that X exists.
del feature_arrays, target_index
gc.collect()

# ------------------------------------------------------------
# 10. Model configurations
# ------------------------------------------------------------

PARAMS_33B = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    colsample_bytree=1.0,
    subsample=1.0,
    reg_lambda=2.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

MODELS_33B = {
    "signed": {
        "A_full37": base_indices,
        "B_original_mismatch": (
            base_indices
            + [candidate_indices[ORIGINAL_33B]]
        ),
        "C_tick_floor_mismatch": (
            base_indices
            + [candidate_indices[FLOORED_33B]]
        )
    },

    "absret": {
        "A_full37": base_indices,
        "D_ofi_price_alignment": (
            base_indices
            + [candidate_indices[ALIGNMENT_33B]]
        )
    }
}

session_dates = x["date"].to_numpy()
timestamps = x["ts_sec"].to_numpy(
    dtype=np.int64
)

unique_dates = sorted(
    pd.unique(session_dates)
)

if len(unique_dates) <= MIN_TRAIN_DAYS_33B:
    raise ValueError(
        "Not enough sessions for walk-forward"
    )

# ------------------------------------------------------------
# 11. Chronological expanding walk-forward
#
# No random row splitting.
# Train on earlier dates only.
#
# Same test rows and training rows for each
# competing model at a given task/horizon.
# ------------------------------------------------------------

records = []
importance_records = []

for i in range(
    MIN_TRAIN_DAYS_33B,
    len(unique_dates)
):

    test_date = unique_dates[i]

    train_date_mask = (
        np.isin(
            session_dates,
            unique_dates[:i]
        )
    )

    train_base = (
        train_date_mask
        & complete
        & ((timestamps % TRAIN_STRIDE_33B) == 0)
    )

    test_base = (
        (session_dates == test_date)
        & complete
    )

    for task, models in MODELS_33B.items():

        for h in HORIZONS_33B:

            y = targets[(task, h)]

            train_rows = np.flatnonzero(
                train_base & np.isfinite(y)
            )

            test_rows = np.flatnonzero(
                test_base & np.isfinite(y)
            )

            if (
                len(train_rows) < 1000
                or len(test_rows) < 200
            ):
                continue

            y_train = y[train_rows]
            y_test = y[test_rows].astype(
                np.float64
            )

            for model_name, selected_columns in (
                models.items()
            ):

                model = lgb.LGBMRegressor(
                    **PARAMS_33B
                )

                X_train = X[
                    np.ix_(
                        train_rows,
                        selected_columns
                    )
                ]

                X_test = X[
                    np.ix_(
                        test_rows,
                        selected_columns
                    )
                ]

                model.fit(
                    X_train,
                    y_train
                )

                predictions = model.predict(
                    X_test
                )

                errors = (
                    y_test - predictions
                )

                records.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test_rows),
                    "sse": float(
                        np.sum(errors**2)
                    ),
                    "sae": float(
                        np.sum(np.abs(errors))
                    ),
                    "mse": float(
                        np.mean(errors**2)
                    ),
                    "mae": float(
                        np.mean(np.abs(errors))
                    )
                })

                if model_name != "A_full37":

                    gains = (
                        model.booster_
                        .feature_importance(
                            importance_type="gain"
                        )
                    )

                    total_gain = float(
                        gains.sum()
                    )

                    candidate_gain = float(
                        gains[-1]
                    )

                    importance_records.append({
                        "date": test_date,
                        "task": task,
                        "horizon": h,
                        "model": model_name,
                        "candidate_gain_share_pct": (
                            100 * candidate_gain
                            / total_gain
                            if total_gain > 0
                            else 0.0
                        )
                    })

                del (
                    model,
                    X_train,
                    X_test,
                    predictions,
                    errors
                )

    print("Completed:", test_date)

calendar_exp33b_daily = pd.DataFrame(
    records
)

calendar_exp33b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp33b_daily.empty:
    raise ValueError(
        "No evaluation results produced"
    )

# ------------------------------------------------------------
# 12. Paired comparisons
# ------------------------------------------------------------

COMPARISONS_33B = {
    "signed": {
        "original_vs_full37": (
            "B_original_mismatch",
            "A_full37"
        ),
        "tickfloor_vs_full37": (
            "C_tick_floor_mismatch",
            "A_full37"
        ),
        "tickfloor_vs_original": (
            "C_tick_floor_mismatch",
            "B_original_mismatch"
        )
    },

    "absret": {
        "alignment_vs_full37": (
            "D_ofi_price_alignment",
            "A_full37"
        )
    }
}

summaries = []
daily_gain_records = []

for (task, h), block in (
    calendar_exp33b_daily.groupby(
        ["task", "horizon"]
    )
):

    pooled = block.groupby("model")[
        ["sse", "sae"]
    ].sum()

    mse_wide = block.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    mae_wide = block.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    for comparison, (candidate, baseline) in (
        COMPARISONS_33B[task].items()
    ):

        daily_mse = 100 * (
            1
            - mse_wide[candidate]
            / mse_wide[baseline]
        )

        daily_mae = 100 * (
            1
            - mae_wide[candidate]
            / mae_wide[baseline]
        )

        for date in daily_mse.index:

            daily_gain_records.append({
                "date": date,
                "task": task,
                "horizon": h,
                "comparison": comparison,
                "mse_gain_pct": float(
                    daily_mse.loc[date]
                ),
                "mae_gain_pct": float(
                    daily_mae.loc[date]
                )
            })

        summaries.append({
            "task": task,
            "horizon": h,
            "comparison": comparison,
            "days": len(daily_mse),

            "pooled_mse_gain_pct": 100 * (
                1
                - pooled.loc[
                    candidate, "sse"
                ]
                / pooled.loc[
                    baseline, "sse"
                ]
            ),

            "pooled_mae_gain_pct": 100 * (
                1
                - pooled.loc[
                    candidate, "sae"
                ]
                / pooled.loc[
                    baseline, "sae"
                ]
            ),

            "winning_days": int(
                (daily_mse > 0).sum()
            ),

            "median_daily_gain_pct": float(
                daily_mse.median()
            ),

            "worst_day_pct": float(
                daily_mse.min()
            )
        })

calendar_exp33b_summary = pd.DataFrame(
    summaries
)

calendar_exp33b_daily_gains = pd.DataFrame(
    daily_gain_records
)

# ------------------------------------------------------------
# 13. Print final outputs
# ------------------------------------------------------------

print("\n===== 33B FULL37 ABLATION =====")

print(
    calendar_exp33b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 33B DAILY MSE GAINS =====")

for comparison in [
    "original_vs_full37",
    "tickfloor_vs_full37",
    "tickfloor_vs_original",
    "alignment_vs_full37"
]:

    subset = (
        calendar_exp33b_daily_gains.loc[
            calendar_exp33b_daily_gains[
                "comparison"
            ].eq(comparison)
        ]
    )

    print("\n", comparison)

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 33B DAILY MAE GAINS =====")

for comparison in [
    "original_vs_full37",
    "tickfloor_vs_full37",
    "tickfloor_vs_original",
    "alignment_vs_full37"
]:

    subset = (
        calendar_exp33b_daily_gains.loc[
            calendar_exp33b_daily_gains[
                "comparison"
            ].eq(comparison)
        ]
    )

    print("\n", comparison)

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mae_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 33B FEATURE IMPORTANCE =====")

print(
    calendar_exp33b_importance
    .groupby(
        ["task", "horizon", "model"]
    )["candidate_gain_share_pct"]
    .agg(["median", "mean"])
    .round(6)
    .to_string()
)

print("\n===== 33B COMPLETE =====")
print("Raw df_1s unchanged.")
print("No new features promoted.")
print("Core feature count remains 37.")

gc.collect()

===== 33B FEATURE REGISTRY =====
Core engineered features: 37
Additional controls: 9
Total baseline inputs: 46

===== 33B SAMPLE =====
Rows: 459122
Sessions: 16
core_31 match: 100.0000%
calendar_exp24a_features match: 100.0000%
calendar_exp28a_features match: 100.0000%
calendar_exp29a_features match: 100.0000%
calendar_exp31a_features match: 100.0000%
raw df_1s match: 100.0000%

===== 33B SCALE DIAGNOSTICS =====
Historical scale coverage (%): 99.5818
Zero OFI scale (% of all rows): 1.1768
Zero return scale (% of all rows): 12.3629
Median tick floor (bps): 0.260515
Rows where tick floor exceeds return scale (%): 41.6841

===== 33B MATCHED SAMPLE =====
Total rows: 459122
Complete feature rows: 402979
Complete coverage (%): 87.7717
Original mismatch valid (%): 99.5818
Tick-floor mismatch valid (%): 99.5818
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed

8